In [1]:
# ============================================================
# CELL 1: NOTEBOOK 05 BOOTSTRAP
# 05_GHSL_Bengaluru_Historical_Epoch_Pipeline.ipynb
#
# Purpose:
#   - Mount Google Drive
#   - Restore the SSH key required by remote_exec.sh
#   - Define the reusable remote command helper
#   - Define the intermediate historical epochs
#
# remote_exec.sh is the only source of truth for:
#   - current AWS public IP
#   - SSH username
#   - SSH options
#   - SSH key path
#
# This cell does NOT:
#   - contain an AWS IP
#   - perform an SSH test
#   - start or modify HDFS
#   - install software
#   - process GHSL data
# ============================================================

from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess


# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)

print("Google Drive: MOUNTED")


# ------------------------------------------------------------
# 2. Required local and Drive paths
# ------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

REMOTE_EXEC = (
    DRIVE_ROOT
    / "remote_exec.sh"
)

DRIVE_KEY = (
    DRIVE_ROOT
    / "hadoop-hdfs.pem"
)

LOCAL_KEY = Path(
    "/content/hadoop-hdfs.pem"
)


# ------------------------------------------------------------
# 3. Validate remote_exec.sh
# ------------------------------------------------------------

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(
        f"Remote execution helper not found: "
        f"{REMOTE_EXEC}"
    )

os.chmod(
    REMOTE_EXEC,
    0o700,
)

print(
    "Remote helper:",
    REMOTE_EXEC,
)


# ------------------------------------------------------------
# 4. Restore the SSH key to Colab's temporary filesystem
#
# /content is lost whenever the Colab runtime restarts.
# Therefore, this small restoration step is required in each
# fresh runtime.
# ------------------------------------------------------------

if LOCAL_KEY.is_file():
    os.chmod(
        LOCAL_KEY,
        0o600,
    )

    print(
        "Local SSH key already exists:",
        LOCAL_KEY,
    )

else:
    if not DRIVE_KEY.is_file():
        raise FileNotFoundError(
            f"SSH key not found in Google Drive: "
            f"{DRIVE_KEY}"
        )

    shutil.copy2(
        DRIVE_KEY,
        LOCAL_KEY,
    )

    os.chmod(
        LOCAL_KEY,
        0o600,
    )

    print(
        "SSH key restored from:",
        DRIVE_KEY,
    )

    print(
        "SSH key restored to:",
        LOCAL_KEY,
    )


key_permissions = oct(
    LOCAL_KEY.stat().st_mode
    & 0o777
)

if key_permissions != "0o600":
    raise RuntimeError(
        "SSH key permissions must be 0o600, "
        f"found {key_permissions}."
    )

print(
    "SSH key permissions:",
    key_permissions,
)


# ------------------------------------------------------------
# 5. Reusable remote execution helper
#
# This helper makes no separate connectivity test.
# It runs only the command requested by a later cell.
# ------------------------------------------------------------

def run_remote(
    command,
    timeout=300,
    show_output=True,
):
    """
    Execute a remote shell command using remote_exec.sh.

    Parameters
    ----------
    command : str
        Command to execute on AWS.

    timeout : int
        Maximum notebook-side wait time in seconds.

    show_output : bool
        Print captured stdout and stderr.

    Returns
    -------
    subprocess.CompletedProcess
        The completed remote command result.
    """

    result = subprocess.run(
        [
            "bash",
            str(REMOTE_EXEC),
            command,
        ],
        text=True,
        capture_output=True,
        timeout=timeout,
    )

    if show_output and result.stdout:
        print(result.stdout)

    if show_output and result.stderr:
        print("REMOTE STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "Remote command failed with return code "
            f"{result.returncode}."
        )

    return result


# ------------------------------------------------------------
# 6. Historical pipeline configuration
# ------------------------------------------------------------

CITY = "bengaluru"

PILOT_EXTENT_MOLLWEIDE = {
    "min_x": 7624177.350136818,
    "min_y": 1571479.656526347,
    "max_x": 7683017.20414696,
    "max_y": 1632690.7573197454,
}

CANONICAL_GRID_EPOCH = 1975

HISTORICAL_EPOCHS = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]

EXISTING_ENDPOINT_EPOCHS = [
    1975,
    2020,
]

INTERMEDIATE_EPOCHS = [
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
]

# First validate the parameterized pipeline on one unseen year.
VALIDATION_EPOCHS = [
    1980,
]

# These run only after the 1980 validation passes.
REMAINING_BATCH_EPOCHS = [
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
]


# ------------------------------------------------------------
# 7. Persistent HDFS paths
# ------------------------------------------------------------

HDFS_ROOT = (
    "hdfs://localhost:9000"
)

HDFS_CATALOG = (
    f"{HDFS_ROOT}"
    "/data/ghsl/metadata/india/geotiff_catalog"
)

HDFS_PILOT_ROOT = (
    f"{HDFS_ROOT}"
    "/data/ghsl/curated/pilots/bengaluru"
)

HDFS_ANALYTICS_ROOT = (
    f"{HDFS_ROOT}"
    "/data/ghsl/analytics/pilots/bengaluru"
)


# ------------------------------------------------------------
# 8. Expected canonical grid structure
# ------------------------------------------------------------

EXPECTED_GRID_CELLS = 3658
EXPECTED_COMPLETE_CELLS = 3480
EXPECTED_PARTIAL_CELLS = 178

INPUT_RESOLUTION_METERS = 100
OUTPUT_RESOLUTION_METERS = 1000


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print()
print("=" * 80)
print("NOTEBOOK 05 BOOTSTRAP COMPLETE")
print("=" * 80)

print(
    "City:",
    CITY,
)

print(
    "Historical epochs:",
    HISTORICAL_EPOCHS,
)

print(
    "Existing endpoint epochs:",
    EXISTING_ENDPOINT_EPOCHS,
)

print(
    "Intermediate epochs:",
    INTERMEDIATE_EPOCHS,
)

print(
    "Initial validation epoch:",
    VALIDATION_EPOCHS,
)

print(
    "Remaining batch epochs:",
    REMAINING_BATCH_EPOCHS,
)

print(
    "Canonical grid cells:",
    EXPECTED_GRID_CELLS,
)

print(
    "Target aggregation:",
    f"{INPUT_RESOLUTION_METERS} m -> "
    f"{OUTPUT_RESOLUTION_METERS} m",
)

print(
    "Metadata catalog:",
    HDFS_CATALOG,
)

print()
print("No remote processing was executed.")
print("CELL 1: SUCCESS")

Mounted at /content/drive
Google Drive: MOUNTED
Remote helper: /content/drive/MyDrive/remote_exec.sh
SSH key restored from: /content/drive/MyDrive/hadoop-hdfs.pem
SSH key restored to: /content/hadoop-hdfs.pem
SSH key permissions: 0o600

NOTEBOOK 05 BOOTSTRAP COMPLETE
City: bengaluru
Historical epochs: [1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
Existing endpoint epochs: [1975, 2020]
Intermediate epochs: [1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015]
Initial validation epoch: [1980]
Remaining batch epochs: [1985, 1990, 1995, 2000, 2005, 2010, 2015]
Canonical grid cells: 3658
Target aggregation: 100 m -> 1000 m
Metadata catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog

No remote processing was executed.
CELL 1: SUCCESS


In [2]:
# ============================================================
# CELL 2: HDFS READINESS AND EXISTING OUTPUT VALIDATION
#
# Purpose:
#   - Start the existing HDFS services only if needed
#   - Wait for NameNode port 9000
#   - Wait for safe mode to turn off
#   - Verify the metadata catalog
#   - Verify the validated 1975 and 2020 endpoint products
#
# This cell does NOT:
#   - format HDFS
#   - download or upload raw data
#   - rebuild the metadata catalog
#   - process any GeoTIFF
#   - modify completed endpoint outputs
# ============================================================

hdfs_readiness_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"


# ------------------------------------------------------------
# 1. Validate the existing Hadoop installation
# ------------------------------------------------------------

if [ ! -x "${HADOOP_HOME}/bin/hdfs" ]; then
    echo "ERROR: hdfs executable not found."
    exit 20
fi

if [ ! -x "${HADOOP_HOME}/sbin/start-dfs.sh" ]; then
    echo "ERROR: start-dfs.sh not found."
    exit 21
fi

echo "HADOOP_HOME=${HADOOP_HOME}"


# ------------------------------------------------------------
# 2. Inspect HDFS daemons
# ------------------------------------------------------------

echo
echo "============================================================"
echo "HDFS PROCESSES BEFORE READINESS CHECK"
echo "============================================================"

jps || true

NAMENODE_PID="$(
    jps 2>/dev/null \
    | awk '$2 == "NameNode" {print $1}'
)"

DATANODE_PID="$(
    jps 2>/dev/null \
    | awk '$2 == "DataNode" {print $1}'
)"

SECONDARY_PID="$(
    jps 2>/dev/null \
    | awk '$2 == "SecondaryNameNode" {print $1}'
)"


# ------------------------------------------------------------
# 3. Start existing HDFS services only when required
# ------------------------------------------------------------

if [ -z "${NAMENODE_PID}" ] \
    || [ -z "${DATANODE_PID}" ] \
    || [ -z "${SECONDARY_PID}" ]
then
    echo
    echo "One or more HDFS daemons are absent."
    echo "Starting the existing HDFS installation..."

    "${HADOOP_HOME}/sbin/start-dfs.sh"
else
    echo
    echo "All required HDFS daemons are already running."
fi


# ------------------------------------------------------------
# 4. Wait for NameNode RPC port 9000
# ------------------------------------------------------------

echo
echo "Waiting for NameNode RPC port 127.0.0.1:9000..."

NAMENODE_READY=0

for attempt in $(seq 1 45); do
    if timeout 2 bash -c \
        '</dev/tcp/127.0.0.1/9000' \
        2>/dev/null
    then
        NAMENODE_READY=1
        echo "NameNode RPC port: READY"
        break
    fi

    sleep 2
done

if [ "${NAMENODE_READY}" -ne 1 ]; then
    echo "ERROR: NameNode RPC port did not become ready."
    jps || true
    exit 22
fi


# ------------------------------------------------------------
# 5. Verify all required daemons after startup
# ------------------------------------------------------------

echo
echo "============================================================"
echo "HDFS PROCESSES AFTER READINESS CHECK"
echo "============================================================"

jps || true

for daemon in NameNode DataNode SecondaryNameNode; do
    if ! jps \
        | awk '{print $2}' \
        | grep -qx "${daemon}"
    then
        echo "ERROR: ${daemon} is not running."
        exit 23
    fi
done


# ------------------------------------------------------------
# 6. Wait for HDFS safe mode to turn off
# ------------------------------------------------------------

echo
echo "============================================================"
echo "HDFS SAFE MODE"
echo "============================================================"

SAFE_MODE_OFF=0

for attempt in $(seq 1 45); do
    SAFE_MODE_STATUS="$(
        hdfs dfsadmin -safemode get 2>&1
    )"

    echo "${SAFE_MODE_STATUS}"

    if echo "${SAFE_MODE_STATUS}" \
        | grep -qi "Safe mode is OFF"
    then
        SAFE_MODE_OFF=1
        break
    fi

    sleep 2
done

if [ "${SAFE_MODE_OFF}" -ne 1 ]; then
    echo "Safe mode did not leave automatically."
    echo "Requesting safe-mode leave..."

    hdfs dfsadmin -safemode leave

    SAFE_MODE_STATUS="$(
        hdfs dfsadmin -safemode get 2>&1
    )"

    echo "${SAFE_MODE_STATUS}"

    if ! echo "${SAFE_MODE_STATUS}" \
        | grep -qi "Safe mode is OFF"
    then
        echo "ERROR: HDFS safe mode remains enabled."
        exit 24
    fi
fi


# ------------------------------------------------------------
# 7. Validate raw GHSL inventory
#
# This Hadoop build does not support:
#   hdfs dfs -find ... -type f
#
# Therefore, count GeoTIFF paths by extension.
# ------------------------------------------------------------

echo
echo "============================================================"
echo "RAW GHSL INVENTORY"
echo "============================================================"

RAW_FILE_COUNT="$(
    hdfs dfs -find /data/ghsl/raw/india \
    | grep -Ei '\.(tif|tiff)$' \
    | wc -l \
    | tr -d ' '
)"

RAW_LOGICAL_BYTES="$(
    hdfs dfs -du -s /data/ghsl/raw/india \
    | awk '{print $1}'
)"

echo "RAW_FILE_COUNT=${RAW_FILE_COUNT}"
echo "RAW_LOGICAL_BYTES=${RAW_LOGICAL_BYTES}"

if [ "${RAW_FILE_COUNT}" -ne 325 ]; then
    echo "ERROR: Expected 325 raw GHSL GeoTIFF files."
    exit 25
fi

if [ "${RAW_LOGICAL_BYTES}" -ne 7038596744 ]; then
    echo "ERROR: Unexpected raw GHSL logical byte count."
    exit 26
fi


# ------------------------------------------------------------
# 8. Verify required HDFS paths
# ------------------------------------------------------------

CATALOG_PATH="/data/ghsl/metadata/india/geotiff_catalog"

POPULATION_1975_PATH="/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1975/grid_1km"
POPULATION_2020_PATH="/data/ghsl/curated/pilots/bengaluru/product=population/epoch=2020/grid_1km"

SMOD_1975_PATH="/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1975/grid_1km"
SMOD_2020_PATH="/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=2020/grid_1km"

BUILT_1975_PATH="/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=1975/grid_1km"
BUILT_2020_PATH="/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=2020/grid_1km"

ENRICHED_1975_PATH="/data/ghsl/curated/pilots/bengaluru/enriched_grid_year/epoch=1975"
ENRICHED_2020_PATH="/data/ghsl/curated/pilots/bengaluru/enriched_grid_year/epoch=2020"

REQUIRED_PATHS="
${CATALOG_PATH}
${POPULATION_1975_PATH}
${POPULATION_2020_PATH}
${SMOD_1975_PATH}
${SMOD_2020_PATH}
${BUILT_1975_PATH}
${BUILT_2020_PATH}
${ENRICHED_1975_PATH}
${ENRICHED_2020_PATH}
"

echo
echo "============================================================"
echo "REQUIRED HDFS OUTPUTS"
echo "============================================================"

for required_path in ${REQUIRED_PATHS}; do
    if hdfs dfs -test -e "${required_path}"; then
        echo "PASS: ${required_path}"
    else
        echo "ERROR: Required HDFS path is missing:"
        echo "${required_path}"
        exit 27
    fi
done


# ------------------------------------------------------------
# 9. Confirm every Parquet dataset contains data files
# ------------------------------------------------------------

echo
echo "============================================================"
echo "PARQUET DATA-FILE VALIDATION"
echo "============================================================"

for parquet_path in ${REQUIRED_PATHS}; do
    DATA_FILE_COUNT="$(
        hdfs dfs -ls -R "${parquet_path}" \
        2>/dev/null \
        | awk '
            $1 ~ /^-/ &&
            $8 !~ /\/_SUCCESS$/ &&
            $8 !~ /\.crc$/ {
                count++
            }
            END {
                print count + 0
            }
        '
    )"

    echo "${parquet_path}: ${DATA_FILE_COUNT} data file(s)"

    if [ "${DATA_FILE_COUNT}" -lt 1 ]; then
        echo "ERROR: No data files found in ${parquet_path}"
        exit 28
    fi
done


# ------------------------------------------------------------
# 10. System resources
# ------------------------------------------------------------

echo
echo "============================================================"
echo "SYSTEM RESOURCES"
echo "============================================================"

free -h

echo
df -h / /tmp

echo
echo "ACTIVE SWAP:"
swapon --show || true


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

echo
echo "============================================================"
echo "CELL 2 RESULT"
echo "============================================================"

echo "NameNode RPC: PASS"
echo "NameNode: RUNNING"
echo "DataNode: RUNNING"
echo "SecondaryNameNode: RUNNING"
echo "Safe mode: OFF"
echo "Raw GHSL GeoTIFF files: ${RAW_FILE_COUNT}"
echo "Raw GHSL logical bytes: ${RAW_LOGICAL_BYTES}"
echo "Metadata catalog: PRESENT"
echo "Population endpoints: PRESENT"
echo "SMOD endpoints: PRESENT"
echo "Built-surface endpoints: PRESENT"
echo "Enriched endpoints: PRESENT"
echo "CELL 2: SUCCESS"
'''


result = run_remote(
    hdfs_readiness_command,
    timeout=360,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 2 NOTEBOOK RESULT")
print("=" * 80)
print(
    "Remote return code:",
    result.returncode,
)
print("CELL 2: SUCCESS")

HADOOP_HOME=/usr/local/hadoop

HDFS PROCESSES BEFORE READINESS CHECK
1248 Jps

One or more HDFS daemons are absent.
Starting the existing HDFS installation...
Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [ip-172-31-30-94]

Waiting for NameNode RPC port 127.0.0.1:9000...
NameNode RPC port: READY

HDFS PROCESSES AFTER READINESS CHECK
1618 DataNode
1843 SecondaryNameNode
1478 NameNode
1998 Jps

HDFS SAFE MODE
Safe mode is ON
Safe mode is ON
Safe mode is ON
Safe mode is ON
Safe mode is ON
Safe mode is ON
Safe mode is ON
Safe mode is OFF

RAW GHSL INVENTORY
RAW_FILE_COUNT=325
RAW_LOGICAL_BYTES=7038596744

REQUIRED HDFS OUTPUTS
PASS: /data/ghsl/metadata/india/geotiff_catalog
PASS: /data/ghsl/curated/pilots/bengaluru/product=population/epoch=1975/grid_1km
PASS: /data/ghsl/curated/pilots/bengaluru/product=population/epoch=2020/grid_1km
PASS: /data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1975/grid_1km
PASS: /data/ghsl/curated/pilots/bengaluru/produc

In [3]:
# ============================================================
# NEW CELL: READ-ONLY HISTORICAL PIPELINE RECOVERY DIAGNOSTIC
#
# Purpose:
#   - Inspect the current AWS/HDFS state before resuming
#   - Identify exactly which epoch/product outputs already exist
#   - Inspect the current batch driver, manifest, and log
#
# Safety:
#   - Does not launch Spark
#   - Does not modify HDFS
#   - Does not patch the batch driver
#   - Does not delete or overwrite anything
# ============================================================

diagnostic_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
LOG="/home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log"
MANIFEST="/home/ubuntu/ghsl_notebook05_batch/batch_manifest.tsv"

EPOCHS="1975 1980 1985 1990 1995 2000 2005 2010 2015 2020"
PRODUCTS="population smod built_surface"

echo "============================================================"
echo "1. HDFS PROCESS STATUS"
echo "============================================================"

jps || true

echo
echo "Safe-mode status:"
hdfs dfsadmin -safemode get 2>&1 || true

echo
echo "============================================================"
echo "2. ACTIVE BATCH OR SPARK PROCESSES"
echo "============================================================"

ps -ef \
    | grep -E \
      'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    || true

echo
echo "============================================================"
echo "3. BENGALURU HISTORICAL OUTPUT INVENTORY"
echo "============================================================"

printf "%-8s %-18s %-10s %-10s\n" \
    "EPOCH" \
    "STAGE" \
    "EXISTS" \
    "FILES"

printf "%-8s %-18s %-10s %-10s\n" \
    "-----" \
    "-----" \
    "------" \
    "-----"

for epoch in ${EPOCHS}; do

    for product in ${PRODUCTS}; do

        path="/data/ghsl/curated/pilots/bengaluru/product=${product}/epoch=${epoch}/grid_1km"

        if hdfs dfs -test -e "${path}"; then

            file_count="$(
                hdfs dfs -ls -R "${path}" 2>/dev/null \
                | awk '
                    $1 ~ /^-/ &&
                    $8 !~ /\/_SUCCESS$/ &&
                    $8 !~ /\.crc$/ {
                        count++
                    }
                    END {
                        print count + 0
                    }
                '
            )"

            printf "%-8s %-18s %-10s %-10s\n" \
                "${epoch}" \
                "${product}" \
                "YES" \
                "${file_count}"

        else

            printf "%-8s %-18s %-10s %-10s\n" \
                "${epoch}" \
                "${product}" \
                "NO" \
                "0"

        fi

    done

    enriched_path="/data/ghsl/curated/pilots/bengaluru/enriched_grid_year/epoch=${epoch}"

    if hdfs dfs -test -e "${enriched_path}"; then

        file_count="$(
            hdfs dfs -ls -R "${enriched_path}" 2>/dev/null \
            | awk '
                $1 ~ /^-/ &&
                $8 !~ /\/_SUCCESS$/ &&
                $8 !~ /\.crc$/ {
                    count++
                }
                END {
                    print count + 0
                }
            '
        )"

        printf "%-8s %-18s %-10s %-10s\n" \
            "${epoch}" \
            "enriched" \
            "YES" \
            "${file_count}"

    else

        printf "%-8s %-18s %-10s %-10s\n" \
            "${epoch}" \
            "enriched" \
            "NO" \
            "0"

    fi

done

echo
echo "============================================================"
echo "4. REMOTE BATCH DRIVER STATUS"
echo "============================================================"

if [ -f "${DRIVER}" ]; then

    echo "Driver: PRESENT"
    ls -lh "${DRIVER}"

    echo
    echo "Driver SHA-256:"
    sha256sum "${DRIVER}"

    echo
    echo "Python syntax:"
    python3 -m py_compile "${DRIVER}"
    echo "PASS"

    echo
    echo "Required patch markers:"

    for marker in \
        'import textwrap' \
        '"Built-surface output path"' \
        'def create_enrichment_program' \
        'from sedona.spark import SedonaContext' \
        'SedonaContext.builder()' \
        'spark = SedonaContext.create(config)'
    do
        if grep -Fq "${marker}" "${DRIVER}"; then
            echo "PASS: ${marker}"
        else
            echo "MISSING: ${marker}"
        fi
    done

    echo
    echo "Obsolete built-surface marker:"

    if grep -Fq \
        '"Built grid-reference/output paths"' \
        "${DRIVER}"
    then
        echo "PRESENT: old incorrect marker still exists"
    else
        echo "ABSENT: PASS"
    fi

    echo
    echo "Enrichment factory boundaries:"

    grep -n -E \
        '^def create_enrichment_program|^def create_validator' \
        "${DRIVER}" \
        || true

else

    echo "Driver: MISSING"

fi

echo
echo "============================================================"
echo "5. CURRENT BATCH MANIFEST"
echo "============================================================"

if [ -f "${MANIFEST}" ]; then
    cat "${MANIFEST}"
else
    echo "Manifest: MISSING"
fi

echo
echo "============================================================"
echo "6. LAST 200 LINES OF BATCH LOG"
echo "============================================================"

if [ -f "${LOG}" ]; then
    tail -n 200 "${LOG}"
else
    echo "Batch log: MISSING"
fi

echo
echo "============================================================"
echo "7. ERROR SUMMARY FROM BATCH LOG"
echo "============================================================"

if [ -f "${LOG}" ]; then

    grep -n -E \
        'Traceback|ERROR|Exception|AnalysisException|Py4J|FAILED|ExitCode|BATCH_EXIT_CODE' \
        "${LOG}" \
        | tail -n 80 \
        || true

else

    echo "Batch log: MISSING"

fi

echo
echo "============================================================"
echo "8. SYSTEM RESOURCES"
echo "============================================================"

free -h

echo
df -h / /tmp

echo
echo "============================================================"
echo "DIAGNOSTIC COMPLETE"
echo "============================================================"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "Batch driver modified: NO"
'''

result = run_remote(
    diagnostic_command,
    timeout=300,
    show_output=True,
)

print()
print("=" * 80)
print("NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("READ-ONLY RECOVERY DIAGNOSTIC: SUCCESS")

1. HDFS PROCESS STATUS
1618 DataNode
1843 SecondaryNameNode
3461 Jps
1478 NameNode

Safe-mode status:
Safe mode is OFF

2. ACTIVE BATCH OR SPARK PROCESSES

3. BENGALURU HISTORICAL OUTPUT INVENTORY
EPOCH    STAGE              EXISTS     FILES     
-----    -----              ------     -----     
1975     population         YES        4         
1975     smod               YES        2         
1975     built_surface      YES        4         
1975     enriched           YES        4         
1980     population         YES        4         
1980     smod               YES        2         
1980     built_surface      YES        4         
1980     enriched           NO         0         
1985     population         NO         0         
1985     smod               NO         0         
1985     built_surface      NO         0         
1985     enriched           NO         0         
1990     population         NO         0         
1990     smod               NO         0         
199

In [4]:
# ============================================================
# CELL 7G: ROBUST FUNCTION-SCOPED SEDONA PATCH AND BATCH RESUME
#
# Current validated state:
#   population 1980    VALID
#   SMOD 1980          VALID
#   built surface 1980 VALID
#   enriched 1980      MISSING
#
# Root cause:
#   create_enrichment_program() generates a plain SparkSession,
#   so Sedona SQL functions such as ST_AsBinary are unresolved.
#
# This cell:
#   1. Patches only create_enrichment_program()
#   2. Preserves create_validator() unchanged
#   3. Generates and statically verifies enrich_1980.py
#   4. Removes only failed generated enrichment scripts
#   5. Resumes the complete 1980-2015 batch
#
# Do not rerun Cell 7, 7A, 7B, 7C, 7D, 7E, or 7F.
# ============================================================

import base64
import textwrap

REMOTE_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_PATCHER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "patch_enrichment_factory_robust.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)

# ============================================================
# 1. BUILD A ROBUST REMOTE PATCHER
#
# The patcher first isolates the exact source range between:
#
#   def create_enrichment_program(...)
#   def create_validator(...)
#
# It modifies only that range.
# ============================================================

patch_program = textwrap.dedent(
    r'''
    from pathlib import Path
    import shutil
    import time

    DRIVER = Path(
        "/home/ubuntu/ghsl_notebook05_batch/"
        "run_intermediate_epochs.py"
    )

    BACKUP = Path(
        "/home/ubuntu/ghsl_notebook05_batch/"
        f"run_intermediate_epochs.py."
        f"before_sedona_fix_{int(time.time())}"
    )

    if not DRIVER.is_file():
        raise FileNotFoundError(
            f"Batch driver not found: {DRIVER}"
        )

    original_program = DRIVER.read_text(
        encoding="utf-8"
    )

    shutil.copy2(
        DRIVER,
        BACKUP,
    )

    # --------------------------------------------------------
    # Locate the exact enrichment-factory boundaries.
    # --------------------------------------------------------

    start_marker = "def create_enrichment_program("
    end_marker = "def create_validator("

    start_count = original_program.count(start_marker)
    end_count = original_program.count(end_marker)

    if start_count != 1:
        raise RuntimeError(
            "Expected exactly one enrichment factory, "
            f"found {start_count}."
        )

    if end_count != 1:
        raise RuntimeError(
            "Expected exactly one validator factory, "
            f"found {end_count}."
        )

    start_index = original_program.index(start_marker)
    end_index = original_program.index(
        end_marker,
        start_index,
    )

    if end_index <= start_index:
        raise RuntimeError(
            "Invalid enrichment-factory source range."
        )

    prefix = original_program[:start_index]
    enrichment_section = original_program[
        start_index:end_index
    ]
    validator_and_suffix = original_program[end_index:]

    validator_before = validator_and_suffix

    print("Enrichment factory start index:", start_index)
    print("Validator factory start index:", end_index)

    # --------------------------------------------------------
    # PATCH 1: generated enrichment import
    # --------------------------------------------------------

    old_import = (
        "from pyspark.sql import SparkSession"
    )

    new_import = (
        "from sedona.spark import SedonaContext"
    )

    old_import_count = enrichment_section.count(
        old_import
    )

    new_import_count = enrichment_section.count(
        new_import
    )

    if (
        old_import_count == 1
        and new_import_count == 0
    ):
        enrichment_section = (
            enrichment_section.replace(
                old_import,
                new_import,
                1,
            )
        )

        print(
            "Generated enrichment Sedona import: ADDED"
        )

    elif (
        old_import_count == 0
        and new_import_count == 1
    ):
        print(
            "Generated enrichment Sedona import: "
            "ALREADY PRESENT"
        )

    else:
        raise RuntimeError(
            "Unexpected enrichment import state: "
            f"SparkSession={old_import_count}, "
            f"SedonaContext={new_import_count}."
        )

    # --------------------------------------------------------
    # PATCH 2: generated enrichment builder
    # --------------------------------------------------------

    old_builder = "SparkSession.builder"
    new_builder = "SedonaContext.builder()"

    old_builder_count = enrichment_section.count(
        old_builder
    )

    new_builder_count = enrichment_section.count(
        new_builder
    )

    if (
        old_builder_count == 1
        and new_builder_count == 0
    ):
        enrichment_section = (
            enrichment_section.replace(
                old_builder,
                new_builder,
                1,
            )
        )

        print(
            "Generated enrichment Sedona builder: ADDED"
        )

    elif (
        old_builder_count == 0
        and new_builder_count == 1
    ):
        print(
            "Generated enrichment Sedona builder: "
            "ALREADY PRESENT"
        )

    else:
        raise RuntimeError(
            "Unexpected enrichment builder state: "
            f"SparkSession.builder={old_builder_count}, "
            f"SedonaContext.builder={new_builder_count}."
        )

    # --------------------------------------------------------
    # PATCH 3: register Sedona SQL functions
    #
    # Existing generated structure:
    #
    #   spark = (
    #       SedonaContext.builder()
    #       ...
    #       .getOrCreate()
    #   )
    #   spark.sparkContext.setLogLevel("WARN")
    #
    # Insert:
    #
    #   spark = SedonaContext.create(spark)
    # --------------------------------------------------------

    registration_line = (
        "spark = SedonaContext.create(spark)"
    )

    log_level_line = (
        'spark.sparkContext.setLogLevel("WARN")'
    )

    registration_count = enrichment_section.count(
        registration_line
    )

    log_level_count = enrichment_section.count(
        log_level_line
    )

    if registration_count == 0:
        if log_level_count != 1:
            raise RuntimeError(
                "Expected exactly one enrichment "
                "Spark log-level marker, found "
                f"{log_level_count}."
            )

        enrichment_section = (
            enrichment_section.replace(
                log_level_line,
                (
                    registration_line
                    + "\n"
                    + log_level_line
                ),
                1,
            )
        )

        print(
            "Generated enrichment Sedona registration: "
            "ADDED"
        )

    elif registration_count == 1:
        print(
            "Generated enrichment Sedona registration: "
            "ALREADY PRESENT"
        )

    else:
        raise RuntimeError(
            "Unexpected number of Sedona registration "
            f"statements: {registration_count}."
        )

    # --------------------------------------------------------
    # Validate only the patched enrichment factory.
    # --------------------------------------------------------

    required_enrichment_fragments = [
        "from sedona.spark import SedonaContext",
        "SedonaContext.builder()",
        "spark = SedonaContext.create(spark)",
        "ST_AsBinary(",
        "ST_PolygonFromEnvelope(",
        "geometry_wkb",
    ]

    missing_enrichment_fragments = [
        fragment
        for fragment in required_enrichment_fragments
        if fragment not in enrichment_section
    ]

    if missing_enrichment_fragments:
        raise RuntimeError(
            "Patched enrichment factory is missing:\n"
            + "\n".join(
                missing_enrichment_fragments
            )
        )

    forbidden_enrichment_fragments = [
        "from pyspark.sql import SparkSession",
        "SparkSession.builder",
    ]

    remaining_forbidden = [
        fragment
        for fragment in forbidden_enrichment_fragments
        if fragment in enrichment_section
    ]

    if remaining_forbidden:
        raise RuntimeError(
            "Plain SparkSession remains in the "
            "enrichment factory:\n"
            + "\n".join(remaining_forbidden)
        )

    # --------------------------------------------------------
    # Reassemble the driver.
    # --------------------------------------------------------

    patched_program = (
        prefix
        + enrichment_section
        + validator_and_suffix
    )

    # The validator factory must remain byte-for-byte unchanged.
    patched_validator_and_suffix = patched_program[
        patched_program.index(end_marker):
    ]

    if (
        patched_validator_and_suffix
        != validator_before
    ):
        raise RuntimeError(
            "Validator factory or following driver source "
            "was unexpectedly modified."
        )

    # --------------------------------------------------------
    # Preserve previous successful fixes.
    # --------------------------------------------------------

    required_driver_fragments = [
        "import textwrap",
        '"Built-surface output path"',
        "def create_enrichment_program(",
        "def create_validator(",
    ]

    missing_driver_fragments = [
        fragment
        for fragment in required_driver_fragments
        if fragment not in patched_program
    ]

    if missing_driver_fragments:
        raise RuntimeError(
            "Previous batch-driver fixes are missing:\n"
            + "\n".join(missing_driver_fragments)
        )

    if (
        '"Built grid-reference/output paths"'
        in patched_program
    ):
        raise RuntimeError(
            "Obsolete built-surface path-count marker "
            "remains."
        )

    # The validator should still use its normal SparkSession.
    if (
        "from pyspark.sql import SparkSession"
        not in validator_and_suffix
    ):
        raise RuntimeError(
            "Validator SparkSession import was "
            "unexpectedly removed."
        )

    # --------------------------------------------------------
    # Compile before saving.
    # --------------------------------------------------------

    compile(
        patched_program,
        str(DRIVER),
        "exec",
    )

    DRIVER.write_text(
        patched_program,
        encoding="utf-8",
    )

    # Re-read and compile the actual saved file.
    saved_program = DRIVER.read_text(
        encoding="utf-8"
    )

    compile(
        saved_program,
        str(DRIVER),
        "exec",
    )

    print()
    print("=" * 80)
    print("ROBUST ENRICHMENT FACTORY PATCH RESULT")
    print("=" * 80)
    print("Driver:", DRIVER)
    print("Backup:", BACKUP)
    print("Enrichment Sedona import: PRESENT")
    print("Enrichment Sedona builder: PRESENT")
    print("Enrichment Sedona registration: PRESENT")
    print("Validator factory unchanged: PASS")
    print("Built-surface path fix preserved: PASS")
    print("Batch-driver syntax: PASS")
    print("Spark jobs launched: NO")
    print("HDFS modified: NO")
    print("PATCH: SUCCESS")
    '''
).strip() + "\n"

encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")

# ============================================================
# 2. TRANSFER AND APPLY THE PATCH
# ============================================================

patch_command = (
    "set -e; "
    f"echo '{encoded_patch}' "
    f"| base64 -d > '{REMOTE_PATCHER}'; "
    f"python3 -m py_compile '{REMOTE_PATCHER}'; "
    f"python3 '{REMOTE_PATCHER}'; "
    f"python3 -m py_compile '{REMOTE_DRIVER}'; "
    "echo; "
    "echo 'Patched enrichment factory header:'; "
    "START=$(grep -n '^def create_enrichment_program' "
    f"'{REMOTE_DRIVER}' | head -n 1 | cut -d: -f1); "
    "END=$((START + 75)); "
    f"sed -n \"${{START}},${{END}}p\" '{REMOTE_DRIVER}'; "
    "echo; "
    "echo 'Patched driver SHA-256:'; "
    f"sha256sum '{REMOTE_DRIVER}'"
)

patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)

# ============================================================
# 3. REGENERATE ENRICHMENT 1980 WITHOUT RUNNING SPARK
#
# Import only the function definitions and constants located
# before the driver's batch-startup execution section.
#
# Then call create_enrichment_program() for 1980 and verify
# that the generated script contains real Sedona registration.
# ============================================================

generation_validation_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
EPOCH_DIR="/home/ubuntu/ghsl_notebook05_programs/epoch_1980"
ENRICH_PROGRAM="${EPOCH_DIR}/enrich_1980.py"
VALIDATOR="${EPOCH_DIR}/validate_enriched_1980.py"
GENERATOR="/home/ubuntu/ghsl_notebook05_batch/generate_enrich_1980_test.py"

mkdir -p "${EPOCH_DIR}"

# Remove only failed generated scripts.
# No HDFS output is deleted.
rm -f "${ENRICH_PROGRAM}"
rm -f "${VALIDATOR}"

python3 - "${DRIVER}" "${GENERATOR}" <<'PY'
from pathlib import Path
import sys

driver = Path(sys.argv[1])
generator = Path(sys.argv[2])

program = driver.read_text(
    encoding="utf-8"
)

startup_marker = (
    "# ========================================================\n"
    "# BATCH STARTUP VALIDATION"
)

startup_count = program.count(
    startup_marker
)

if startup_count != 1:
    raise RuntimeError(
        "Expected exactly one batch-startup marker, "
        f"found {startup_count}."
    )

definitions = program.split(
    startup_marker,
    1,
)[0]

test_program = (
    definitions
    + "\n\n"
    + 'epoch_dir = Path('
    + '"/home/ubuntu/ghsl_notebook05_programs/epoch_1980"'
    + ')\n'
    + 'epoch_dir.mkdir(parents=True, exist_ok=True)\n'
    + 'output = create_enrichment_program(1980, epoch_dir)\n'
    + 'print("Generated enrichment program:", output)\n'
)

compile(
    test_program,
    str(generator),
    "exec",
)

generator.write_text(
    test_program,
    encoding="utf-8",
)
PY

python3 -m py_compile "${GENERATOR}"
python3 "${GENERATOR}"

if [ ! -s "${ENRICH_PROGRAM}" ]; then
    echo "ERROR: enrich_1980.py was not generated."
    exit 20
fi

python3 -m py_compile "${ENRICH_PROGRAM}"

grep -F \
    'from sedona.spark import SedonaContext' \
    "${ENRICH_PROGRAM}" \
    >/dev/null \
    || {
        echo "ERROR: Generated SedonaContext import missing."
        exit 21
    }

grep -F \
    'SedonaContext.builder()' \
    "${ENRICH_PROGRAM}" \
    >/dev/null \
    || {
        echo "ERROR: Generated Sedona builder missing."
        exit 22
    }

grep -F \
    'spark = SedonaContext.create(spark)' \
    "${ENRICH_PROGRAM}" \
    >/dev/null \
    || {
        echo "ERROR: Generated Sedona registration missing."
        exit 23
    }

grep -F \
    'ST_AsBinary(' \
    "${ENRICH_PROGRAM}" \
    >/dev/null \
    || {
        echo "ERROR: ST_AsBinary expression missing."
        exit 24
    }

grep -F \
    'ST_PolygonFromEnvelope(' \
    "${ENRICH_PROGRAM}" \
    >/dev/null \
    || {
        echo "ERROR: ST_PolygonFromEnvelope expression missing."
        exit 25
    }

if grep -Fq \
    'from pyspark.sql import SparkSession' \
    "${ENRICH_PROGRAM}"
then
    echo "ERROR: Plain SparkSession import remains."
    exit 26
fi

if grep -Fq \
    'SparkSession.builder' \
    "${ENRICH_PROGRAM}"
then
    echo "ERROR: Plain SparkSession builder remains."
    exit 27
fi

echo
echo "============================================================"
echo "GENERATED ENRICHMENT 1980 VERIFICATION"
echo "============================================================"

sed -n '1,65p' "${ENRICH_PROGRAM}"

echo
echo "Generated program SHA-256:"
sha256sum "${ENRICH_PROGRAM}"

echo
echo "Generated Python syntax: PASS"
echo "Sedona import: PASS"
echo "Sedona builder: PASS"
echo "Sedona SQL registration: PASS"
echo "Geometry expressions: PASS"
echo "Spark launched: NO"
echo "HDFS modified: NO"
'''

generation_validation_result = run_remote(
    generation_validation_command,
    timeout=180,
    show_output=True,
)

# ============================================================
# 4. BATCH RESUME PRE-FLIGHT
# ============================================================

preflight_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
ENRICH_PROGRAM="/home/ubuntu/ghsl_notebook05_programs/epoch_1980/enrich_1980.py"

echo "============================================================"
echo "CELL 7G BATCH RESUME PRE-FLIGHT"
echo "============================================================"

python3 -m py_compile "${DRIVER}"
python3 -m py_compile "${ENRICH_PROGRAM}"

# Verify markers inside the enrichment factory only.
START_LINE="$(
    grep -n '^def create_enrichment_program' \
        "${DRIVER}" \
    | head -n 1 \
    | cut -d: -f1
)"

END_LINE="$(
    grep -n '^def create_validator' \
        "${DRIVER}" \
    | head -n 1 \
    | cut -d: -f1
)"

if [ -z "${START_LINE}" ] || [ -z "${END_LINE}" ]; then
    echo "ERROR: Factory boundaries were not found."
    exit 30
fi

sed -n "${START_LINE},$((END_LINE - 1))p" \
    "${DRIVER}" \
| grep -F \
    'from sedona.spark import SedonaContext' \
    >/dev/null \
    || {
        echo "ERROR: Factory Sedona import missing."
        exit 31
    }

sed -n "${START_LINE},$((END_LINE - 1))p" \
    "${DRIVER}" \
| grep -F \
    'SedonaContext.builder()' \
    >/dev/null \
    || {
        echo "ERROR: Factory Sedona builder missing."
        exit 32
    }

sed -n "${START_LINE},$((END_LINE - 1))p" \
    "${DRIVER}" \
| grep -F \
    'spark = SedonaContext.create(spark)' \
    >/dev/null \
    || {
        echo "ERROR: Factory Sedona registration missing."
        exit 33
    }

grep -F \
    '"Built-surface output path"' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: Built-surface path fix missing."
        exit 34
    }

# HDFS readiness.
for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
    | awk '{print $2}' \
    | grep -qx "${daemon}" \
    || {
        echo "ERROR: ${daemon} is not running."
        exit 35
    }
done

SAFE_MODE_STATUS="$(
    hdfs dfsadmin -safemode get 2>&1
)"

echo "${SAFE_MODE_STATUS}"

echo "${SAFE_MODE_STATUS}" \
| grep -qi "Safe mode is OFF" \
|| {
    echo "ERROR: HDFS safe mode is enabled."
    exit 36
}

# Ensure no competing job is running.
RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
      'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${RUNNING_COUNT}" -ne 0 ]; then
    echo "ERROR: Another batch or Spark process is active."

    ps -ef \
    | grep -E \
      'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep

    exit 37
fi

# Verify the three validated 1980 product inputs.
echo
echo "Validated 1980 inputs:"

for path in \
"/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1980/grid_1km" \
"/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1980/grid_1km" \
"/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=1980/grid_1km"
do
    if hdfs dfs -test -e "${path}"; then
        echo "PASS: ${path}"
    else
        echo "ERROR: Missing ${path}"
        exit 38
    fi
done

echo
echo "Memory:"
free -h

echo
echo "Disk:"
df -h / /tmp

echo
echo "Batch resume pre-flight: PASS"
'''

preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)

# ============================================================
# 5. RESUME COMPLETE INTERMEDIATE-EPOCH BATCH
#
# Expected initial sequence:
#   population 1980    validate and skip
#   SMOD 1980          validate and skip
#   built surface 1980 validate and skip
#   enriched 1980      regenerate, run, validate
#
# After completing 1980:
#   1985, 1990, 1995, 2000, 2005, 2010, 2015
# ============================================================

resume_command = r'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
LOG="/home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log"

python3 "${DRIVER}" \
    2>&1 \
    | tee "${LOG}"

status=${PIPESTATUS[0]}

echo "BATCH_EXIT_CODE=${status}"

exit "${status}"
'''

print()
print("=" * 80)
print("RESUMING INTERMEDIATE-EPOCH BATCH")
print("=" * 80)
print("1980 population: validate and skip")
print("1980 SMOD: validate and skip")
print("1980 built surface: validate and skip")
print("1980 enriched: regenerate and run")
print("Remaining epochs: 1985 through 2015")
print("Execution mode: sequential")
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)
print()

batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)

# ============================================================
# FINAL NOTEBOOK RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7G NOTEBOOK RESULT")
print("=" * 80)
print("Patch return code:", patch_result.returncode)
print(
    "Generated enrichment verification return code:",
    generation_validation_result.returncode,
)
print("Pre-flight return code:", preflight_result.returncode)
print("Batch return code:", batch_result.returncode)
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)
print("CELL 7G: SUCCESS")

Streaming output truncated to the last 5000 lines.
26/11/07 11:09:51 INFO Executor: Fetching spark://ip-172-31-30-94.ec2.internal:33317/jars/io.github.x-stream_mxparser-1.2.2.jar with timestamp 1794049790373
26/11/07 11:09:51 INFO Utils: Fetching spark://ip-172-31-30-94.ec2.internal:33317/jars/io.github.x-stream_mxparser-1.2.2.jar to /tmp/spark-de467cde-97ce-4261-9c2b-407c70f975e3/userFiles-30374881-f375-4f6d-aff4-39f663c17b1a/fetchFileTemp7462561181004662233.tmp
26/11/07 11:09:51 INFO Utils: /tmp/spark-de467cde-97ce-4261-9c2b-407c70f975e3/userFiles-30374881-f375-4f6d-aff4-39f663c17b1a/fetchFileTemp7462561181004662233.tmp has been previously copied to /tmp/spark-de467cde-97ce-4261-9c2b-407c70f975e3/userFiles-30374881-f375-4f6d-aff4-39f663c17b1a/io.github.x-stream_mxparser-1.2.2.jar
26/11/07 11:09:51 INFO Executor: Adding file:/tmp/spark-de467cde-97ce-4261-9c2b-407c70f975e3/userFiles-30374881-f375-4f6d-aff4-39f663c17b1a/io.github.x-stream_mxparser-1.2.2.jar to class loader default
26/11

RuntimeError: Remote command failed with return code 1.

In [5]:
# ============================================================
# CELL 7H: INCREASE POPULATION MEMORY AND RESUME BATCH
#
# Do not rerun Cell 7G.
#
# This cell:
#   1. Backs up the batch driver
#   2. Changes only population Spark memory: 3g -> 5g
#   3. Adds 8 GiB swap if swap is currently absent
#   4. Verifies the patch
#   5. Resumes the checkpointed historical batch
#
# Expected behavior:
#   1980 -> validate and skip
#   1985 -> validate and skip
#   1990 -> validate and skip
#   1995 population -> retry with 5g
#   Then continue through 2015
# ============================================================

import base64
import textwrap

REMOTE_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_PATCHER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "patch_population_memory.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)

# ============================================================
# 1. PATCH ONLY THE POPULATION SPARK COMMAND
# ============================================================

patch_program = textwrap.dedent(
    r'''
    from pathlib import Path
    import shutil
    import time

    driver = Path(
        "/home/ubuntu/ghsl_notebook05_batch/"
        "run_intermediate_epochs.py"
    )

    if not driver.is_file():
        raise FileNotFoundError(
            f"Batch driver not found: {driver}"
        )

    backup = driver.with_name(
        f"{driver.name}.before_population_5g_{int(time.time())}"
    )

    shutil.copy2(driver, backup)

    program = driver.read_text(
        encoding="utf-8"
    )

    # Locate the population configuration using nearby
    # population-specific text instead of globally replacing
    # every --driver-memory 3g occurrence.
    lines = program.splitlines()

    population_memory_lines = []

    for index, line in enumerate(lines):
        if '"--driver-memory", "3g"' not in line:
            continue

        context_start = max(0, index - 25)
        context_end = min(len(lines), index + 26)

        context = "\n".join(
            lines[context_start:context_end]
        ).lower()

        if "population" in context:
            population_memory_lines.append(index)

    if len(population_memory_lines) == 1:
        index = population_memory_lines[0]

        lines[index] = lines[index].replace(
            '"--driver-memory", "3g"',
            '"--driver-memory", "5g"',
            1,
        )

        print(
            "Population Spark memory changed: 3g -> 5g"
        )

    elif len(population_memory_lines) == 0:
        existing_5g_lines = []

        for index, line in enumerate(lines):
            if '"--driver-memory", "5g"' not in line:
                continue

            context_start = max(0, index - 25)
            context_end = min(len(lines), index + 26)

            context = "\n".join(
                lines[context_start:context_end]
            ).lower()

            if "population" in context:
                existing_5g_lines.append(index)

        if len(existing_5g_lines) == 1:
            print(
                "Population Spark memory already set to 5g"
            )
        else:
            raise RuntimeError(
                "Could not uniquely identify the population "
                "driver-memory configuration."
            )

    else:
        raise RuntimeError(
            "Found multiple population-specific 3g memory "
            f"configurations: {population_memory_lines}"
        )

    patched_program = "\n".join(lines) + "\n"

    # Confirm the enrichment Sedona correction remains.
    required_fragments = [
        "from sedona.spark import SedonaContext",
        "SedonaContext.builder()",
        "spark = SedonaContext.create(spark)",
        '"Built-surface output path"',
    ]

    missing = [
        fragment
        for fragment in required_fragments
        if fragment not in patched_program
    ]

    if missing:
        raise RuntimeError(
            "Prior pipeline fixes are missing:\n"
            + "\n".join(missing)
        )

    compile(
        patched_program,
        str(driver),
        "exec",
    )

    driver.write_text(
        patched_program,
        encoding="utf-8",
    )

    compile(
        driver.read_text(encoding="utf-8"),
        str(driver),
        "exec",
    )

    print()
    print("Driver:", driver)
    print("Backup:", backup)
    print("Population memory: 5g")
    print("Previous Sedona fix preserved: PASS")
    print("Python syntax: PASS")
    print("PATCH: SUCCESS")
    '''
).strip() + "\n"

encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")

# ============================================================
# 2. APPLY PATCH, CONFIGURE SWAP, AND PREFLIGHT
# ============================================================

preflight_command = f"""
set -e

DRIVER='{REMOTE_DRIVER}'
PATCHER='{REMOTE_PATCHER}'

echo '{encoded_patch}' \
| base64 -d \
> "${{PATCHER}}"

python3 -m py_compile "${{PATCHER}}"
python3 "${{PATCHER}}"
python3 -m py_compile "${{DRIVER}}"

echo
echo "============================================================"
echo "CONFIGURE SWAP"
echo "============================================================"

CURRENT_SWAP_KB="$(
    awk '
        NR > 1 {{
            total += $3
        }}
        END {{
            print total + 0
        }}
    ' /proc/swaps
)"

if [ "${{CURRENT_SWAP_KB}}" -eq 0 ]; then

    echo "No swap detected. Creating 8 GiB swap file."

    if [ ! -f /swapfile ]; then
        sudo fallocate -l 8G /swapfile
    fi

    sudo chmod 600 /swapfile

    if ! sudo file /swapfile \
        | grep -qi 'swap file'
    then
        sudo mkswap /swapfile
    fi

    sudo swapon /swapfile

    if ! grep -q '^/swapfile ' /etc/fstab; then
        echo '/swapfile none swap sw 0 0' \
        | sudo tee -a /etc/fstab \
        >/dev/null
    fi

    echo "8 GiB swap activated."

else

    echo "Existing swap detected. No new swap created."

fi

echo
echo "Memory after swap configuration:"
free -h

echo
echo "============================================================"
echo "VERIFY POPULATION MEMORY CONFIGURATION"
echo "============================================================"

grep -n -B 25 -A 25 \
    '"--driver-memory", "5g"' \
    "${{DRIVER}}" \
    | head -n 80

POPULATION_5G_COUNT="$(
    grep -n -B 25 -A 25 \
        '"--driver-memory", "5g"' \
        "${{DRIVER}}" \
    | grep -i 'population' \
    | wc -l \
    | tr -d ' '
)"

if [ "${{POPULATION_5G_COUNT}}" -eq 0 ]; then
    echo "ERROR: Population 5g configuration not verified."
    exit 40
fi

echo
echo "============================================================"
echo "HDFS AND PROCESS PREFLIGHT"
echo "============================================================"

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

for daemon in NameNode DataNode SecondaryNameNode; do

    jps \
    | awk '{{print $2}}' \
    | grep -qx "${{daemon}}" \
    || {{
        echo "ERROR: ${{daemon}} is not running."
        exit 41
    }}

done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {{
    echo "ERROR: HDFS safe mode is enabled."
    exit 42
}}

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
      'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${{RUNNING_COUNT}}" -ne 0 ]; then

    echo "ERROR: Another batch or Spark process is active."

    ps -ef \
    | grep -E \
      'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep

    exit 43

fi

echo "HDFS: READY"
echo "Competing Spark process: NONE"
echo "Preflight: PASS"
"""

preflight_result = run_remote(
    preflight_command,
    timeout=300,
    show_output=True,
)

# ============================================================
# 3. RESUME THE SAME CHECKPOINTED BATCH
# ============================================================

resume_command = r'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
LOG="/home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log"

python3 "${DRIVER}" \
    2>&1 \
    | tee "${LOG}"

status=${PIPESTATUS[0]}

echo "BATCH_EXIT_CODE=${status}"

exit "${status}"
'''

print()
print("=" * 80)
print("RESUMING HISTORICAL-EPOCH BATCH")
print("=" * 80)
print("Population Spark memory: 5g")
print("Execution mode: sequential")
print("Completed outputs will be validated and skipped.")
print("Expected restart point: population 1995")
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)
print()

batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7H RESULT")
print("=" * 80)
print(
    "Preflight return code:",
    preflight_result.returncode,
)
print(
    "Batch return code:",
    batch_result.returncode,
)
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)

REMOTE STDERR:
Traceback (most recent call last):
  File "/home/ubuntu/ghsl_notebook05_batch/patch_population_memory.py", line 81, in <module>
    raise RuntimeError(
    ...<2 lines>...
    )
RuntimeError: Could not uniquely identify the population driver-memory configuration.



RuntimeError: Remote command failed with return code 1.

In [6]:
# ============================================================
# CELL 7H-A: INSPECT ACTUAL SPARK MEMORY CONFIGURATION
#
# Read-only:
#   - Does not modify the driver
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

inspection_command = r'''
set -e

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"

echo "============================================================"
echo "1. ALL DRIVER-MEMORY REFERENCES"
echo "============================================================"

grep -n -E \
    'driver-memory|driver_memory|MEMORY|memory' \
    "${DRIVER}" \
    || true

echo
echo "============================================================"
echo "2. POPULATION COMMAND REFERENCES"
echo "============================================================"

grep -n -E \
    'population.*command|population_command|POPULATION|population' \
    "${DRIVER}" \
    | head -n 120 \
    || true

echo
echo "============================================================"
echo "3. COMMAND AROUND RUN POPULATION"
echo "============================================================"

MATCH_LINE="$(
    grep -n \
        'RUN POPULATION' \
        "${DRIVER}" \
    | head -n 1 \
    | cut -d: -f1
)"

if [ -n "${MATCH_LINE}" ]; then

    START_LINE=$((MATCH_LINE - 80))

    if [ "${START_LINE}" -lt 1 ]; then
        START_LINE=1
    fi

    END_LINE=$((MATCH_LINE + 80))

    sed -n \
        "${START_LINE},${END_LINE}p" \
        "${DRIVER}"

else

    echo "Literal RUN POPULATION marker not found."

fi

echo
echo "============================================================"
echo "4. COMMAND AROUND DRIVER-MEMORY 3G"
echo "============================================================"

MATCH_LINE="$(
    grep -n -E \
        'driver-memory.*3g|3g.*driver-memory' \
        "${DRIVER}" \
    | head -n 1 \
    | cut -d: -f1
)"

if [ -n "${MATCH_LINE}" ]; then

    START_LINE=$((MATCH_LINE - 50))

    if [ "${START_LINE}" -lt 1 ]; then
        START_LINE=1
    fi

    END_LINE=$((MATCH_LINE + 50))

    sed -n \
        "${START_LINE},${END_LINE}p" \
        "${DRIVER}"

else

    echo "No same-line driver-memory/3g expression found."

fi

echo
echo "============================================================"
echo "5. CURRENT SWAP STATUS"
echo "============================================================"

free -h

echo
cat /proc/swaps

echo
echo "============================================================"
echo "6. SAFETY CONFIRMATION"
echo "============================================================"

echo "Driver modified: NO"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "INSPECTION: SUCCESS"
'''

inspection_result = run_remote(
    inspection_command,
    timeout=180,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7H-A RESULT")
print("=" * 80)
print(
    "Inspection return code:",
    inspection_result.returncode,
)

1. ALL DRIVER-MEMORY REFERENCES
163:    driver_memory,
171:        "--driver-memory",
172:        driver_memory,
1360:    driver_memory,
1421:            driver_memory,
1560:        driver_memory="3g",
1572:        driver_memory="3g",
1584:        driver_memory="3g",
1596:        driver_memory="2g",

2. POPULATION COMMAND REFERENCES
55:POPULATION_TEMPLATE = (
57:    / "population_epoch_template.py"
71:    "population": (
317:# GENERATE POPULATION PROGRAM
320:def generate_population_program(
325:        "population",
326:        POPULATION_TEMPLATE,
357:        '"population_2020_catalog"',
358:        f'"population_{epoch}_catalog"',
365:        "FROM population_2020_catalog",
366:        f"FROM population_{epoch}_catalog",
373:        "population_2020_keys_df",
374:        f"population_{epoch}_keys_df",
380:        "CELL 10: BENGALURU 2020 POPULATION":
383:                f"{epoch} POPULATION"
404:        "2020 POPULATION SUMMARY":
405:            f"{epoch} POPULATION SUMMARY",
407:   

In [7]:
# ============================================================
# CELL 7H-B: PATCH POPULATION MEMORY, ADD SWAP, RESUME BATCH
#
# Confirmed driver structure:
#   line ~1556: stage="population"
#   line ~1560: driver_memory="3g"
#
# This cell:
#   1. Changes ONLY population memory from 3g to 5g
#   2. Preserves SMOD at 3g
#   3. Preserves built surface at 3g
#   4. Preserves enrichment at 2g
#   5. Creates 8 GiB swap
#   6. Resumes the checkpointed batch
#
# Do not rerun Cell 7G or Cell 7H.
# ============================================================

import base64
import textwrap

REMOTE_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)

REMOTE_PATCHER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "patch_population_memory_v2.py"
)

# ============================================================
# 1. CREATE A STAGE-AWARE PATCHER
#
# It finds stage="population" and changes the first
# driver_memory setting following that stage.
# ============================================================

patch_program = textwrap.dedent(
    r'''
    from pathlib import Path
    import shutil
    import time

    driver = Path(
        "/home/ubuntu/ghsl_notebook05_batch/"
        "run_intermediate_epochs.py"
    )

    if not driver.is_file():
        raise FileNotFoundError(
            f"Driver not found: {driver}"
        )

    backup = driver.with_name(
        f"{driver.name}."
        f"before_population_5g_v2_{int(time.time())}"
    )

    shutil.copy2(driver, backup)

    original = driver.read_text(
        encoding="utf-8"
    )

    lines = original.splitlines()

    # --------------------------------------------------------
    # Find all explicit stage definitions.
    # --------------------------------------------------------

    stage_positions = {}

    for index, line in enumerate(lines):
        stripped = line.strip()

        for stage in [
            "population",
            "smod",
            "built_surface",
            "enriched",
        ]:
            marker = f'stage="{stage}"'

            if marker in stripped:
                stage_positions.setdefault(
                    stage,
                    [],
                ).append(index)

    print("Stage positions:")
    for stage, positions in stage_positions.items():
        print(
            f"  {stage}: "
            + ", ".join(
                str(position + 1)
                for position in positions
            )
        )

    population_positions = stage_positions.get(
        "population",
        [],
    )

    if len(population_positions) != 1:
        raise RuntimeError(
            "Expected one population stage call, found "
            f"{len(population_positions)}."
        )

    population_stage_index = population_positions[0]

    # --------------------------------------------------------
    # Find the population driver_memory argument.
    #
    # Search only from population stage until the next
    # ensure_stage call or a maximum of 25 lines.
    # --------------------------------------------------------

    population_memory_index = None

    search_end = min(
        len(lines),
        population_stage_index + 25,
    )

    for index in range(
        population_stage_index + 1,
        search_end,
    ):
        stripped = lines[index].strip()

        if (
            stripped.startswith("ensure_stage(")
            and index > population_stage_index + 1
        ):
            break

        if stripped.startswith("driver_memory="):
            population_memory_index = index
            break

    if population_memory_index is None:
        raise RuntimeError(
            "Could not find driver_memory after "
            'stage="population".'
        )

    old_population_line = lines[
        population_memory_index
    ]

    stripped_population_line = (
        old_population_line.strip()
    )

    print()
    print(
        "Population memory before:",
        stripped_population_line,
    )

    if stripped_population_line == 'driver_memory="3g",':
        indentation = old_population_line[
            :len(old_population_line)
            - len(old_population_line.lstrip())
        ]

        lines[population_memory_index] = (
            indentation
            + 'driver_memory="5g",'
        )

        print(
            "Population memory changed: 3g -> 5g"
        )

    elif stripped_population_line == 'driver_memory="5g",':
        print(
            "Population memory already set to 5g"
        )

    else:
        raise RuntimeError(
            "Unexpected population memory setting: "
            f"{stripped_population_line}"
        )

    patched = "\n".join(lines) + "\n"

    # --------------------------------------------------------
    # Validate every stage-specific memory setting.
    # --------------------------------------------------------

    patched_lines = patched.splitlines()

    expected_memory = {
        "population": "5g",
        "smod": "3g",
        "built_surface": "3g",
        "enriched": "2g",
    }

    detected_memory = {}

    for stage, positions in stage_positions.items():
        if stage not in expected_memory:
            continue

        stage_index = positions[0]

        for index in range(
            stage_index + 1,
            min(len(patched_lines), stage_index + 25),
        ):
            stripped = patched_lines[index].strip()

            if (
                stripped.startswith("ensure_stage(")
                and index > stage_index + 1
            ):
                break

            if stripped.startswith("driver_memory="):
                detected_memory[stage] = (
                    stripped
                    .split("=", 1)[1]
                    .rstrip(",")
                    .strip('"')
                )
                break

    print()
    print("Detected stage memory settings:")

    for stage in [
        "population",
        "smod",
        "built_surface",
        "enriched",
    ]:
        actual = detected_memory.get(stage)
        expected = expected_memory[stage]

        print(
            f"  {stage}: {actual} "
            f"(expected {expected})"
        )

        if actual != expected:
            raise RuntimeError(
                f"{stage} memory is {actual}; "
                f"expected {expected}."
            )

    # --------------------------------------------------------
    # Preserve the previous Sedona enrichment correction.
    # --------------------------------------------------------

    required_fragments = [
        "from sedona.spark import SedonaContext",
        "SedonaContext.builder()",
        "spark = SedonaContext.create(spark)",
        '"Built-surface output path"',
    ]

    missing = [
        fragment
        for fragment in required_fragments
        if fragment not in patched
    ]

    if missing:
        raise RuntimeError(
            "Previous pipeline corrections are missing:\n"
            + "\n".join(missing)
        )

    # Compile before writing.
    compile(
        patched,
        str(driver),
        "exec",
    )

    driver.write_text(
        patched,
        encoding="utf-8",
    )

    # Compile the actual saved file.
    saved = driver.read_text(
        encoding="utf-8"
    )

    compile(
        saved,
        str(driver),
        "exec",
    )

    print()
    print("=" * 80)
    print("POPULATION MEMORY PATCH RESULT")
    print("=" * 80)
    print("Driver:", driver)
    print("Backup:", backup)
    print("Population memory: 5g")
    print("SMOD memory: 3g")
    print("Built-surface memory: 3g")
    print("Enriched memory: 2g")
    print("Sedona enrichment correction: PRESERVED")
    print("Python syntax: PASS")
    print("PATCH: SUCCESS")
    '''
).strip() + "\n"

encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")

# ============================================================
# 2. APPLY THE TARGETED PATCH
# ============================================================

patch_command = f"""
set -e

PATCHER='{REMOTE_PATCHER}'
DRIVER='{REMOTE_DRIVER}'

echo '{encoded_patch}' \
| base64 -d \
> "${{PATCHER}}"

python3 -m py_compile "${{PATCHER}}"
python3 "${{PATCHER}}"
python3 -m py_compile "${{DRIVER}}"

echo
echo "============================================================"
echo "PATCHED STAGE CALLS"
echo "============================================================"

sed -n '1548,1602p' "${{DRIVER}}"

echo
echo "Driver SHA-256:"
sha256sum "${{DRIVER}}"
"""

patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)

# ============================================================
# 3. CREATE AND ACTIVATE 8 GiB SWAP
#
# This protects the machine if Spark temporarily exhausts
# physical RAM. It does not alter HDFS data.
# ============================================================

swap_command = r'''
set -e

echo "============================================================"
echo "SWAP CONFIGURATION"
echo "============================================================"

CURRENT_SWAP_KB="$(
    awk '
        NR > 1 {
            total += $3
        }
        END {
            print total + 0
        }
    ' /proc/swaps
)"

if [ "${CURRENT_SWAP_KB}" -gt 0 ]; then

    echo "Swap is already active."
    cat /proc/swaps

else

    echo "No active swap detected."
    echo "Creating an 8 GiB swap file."

    if [ -f /swapfile ]; then
        echo "Existing inactive /swapfile found."

        sudo swapoff /swapfile 2>/dev/null || true
        sudo rm -f /swapfile
    fi

    # dd is more universally reliable for swap than fallocate.
    sudo dd \
        if=/dev/zero \
        of=/swapfile \
        bs=1M \
        count=8192 \
        status=progress

    sudo chmod 600 /swapfile
    sudo mkswap /swapfile
    sudo swapon /swapfile

    if ! grep -q '^/swapfile ' /etc/fstab; then
        echo '/swapfile none swap sw 0 0' \
        | sudo tee -a /etc/fstab \
        >/dev/null
    fi

fi

echo
echo "Memory and swap after configuration:"
free -h

echo
echo "Active swap:"
cat /proc/swaps

ACTIVE_SWAP_KB="$(
    awk '
        NR > 1 {
            total += $3
        }
        END {
            print total + 0
        }
    ' /proc/swaps
)"

if [ "${ACTIVE_SWAP_KB}" -lt 7000000 ]; then
    echo "ERROR: Expected approximately 8 GiB active swap."
    exit 50
fi

echo
echo "SWAP CONFIGURATION: PASS"
'''

swap_result = run_remote(
    swap_command,
    timeout=600,
    show_output=True,
)

# ============================================================
# 4. FINAL PREFLIGHT
# ============================================================

preflight_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"

echo "============================================================"
echo "FINAL RESUME PREFLIGHT"
echo "============================================================"

python3 -m py_compile "${DRIVER}"

echo
echo "Population stage configuration:"

sed -n '1550,1565p' "${DRIVER}"

sed -n '1550,1565p' "${DRIVER}" \
| grep -F 'stage="population"' \
>/dev/null \
|| {
    echo "ERROR: Population stage marker missing."
    exit 51
}

sed -n '1550,1565p' "${DRIVER}" \
| grep -F 'driver_memory="5g"' \
>/dev/null \
|| {
    echo "ERROR: Population 5g setting missing."
    exit 52
}

echo
echo "HDFS status:"

for daemon in NameNode DataNode SecondaryNameNode; do

    jps \
    | awk '{print $2}' \
    | grep -qx "${daemon}" \
    || {
        echo "ERROR: ${daemon} is not running."
        exit 53
    }

    echo "PASS: ${daemon}"

done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {
    echo "ERROR: HDFS safe mode is enabled."
    exit 54
}

echo "PASS: HDFS safe mode is OFF"

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
      'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${RUNNING_COUNT}" -ne 0 ]; then

    echo "ERROR: A batch or Spark process is active."

    ps -ef \
    | grep -E \
      'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep

    exit 55

fi

echo "PASS: No competing Spark process"

echo
echo "Available resources:"
free -h
df -h / /tmp

echo
echo "FINAL PREFLIGHT: PASS"
'''

preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)

# ============================================================
# 5. RESUME THE CHECKPOINTED BATCH
#
# Existing valid stages will be validated and skipped.
# The first missing stage should be population 1995.
# ============================================================

resume_command = r'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
LOG="/home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log"

python3 "${DRIVER}" \
    2>&1 \
    | tee "${LOG}"

status=${PIPESTATUS[0]}

echo "BATCH_EXIT_CODE=${status}"

exit "${status}"
'''

print()
print("=" * 80)
print("RESUMING CHECKPOINTED HISTORICAL BATCH")
print("=" * 80)
print("Population driver memory: 5g")
print("System swap: 8 GiB")
print("Expected first missing stage: population 1995")
print("Existing valid stages: validate and skip")
print("Remaining target: 1995 through 2015")
print()

batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7H-B RESULT")
print("=" * 80)
print("Patch return code:", patch_result.returncode)
print("Swap return code:", swap_result.returncode)
print(
    "Preflight return code:",
    preflight_result.returncode,
)
print("Batch return code:", batch_result.returncode)
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)

Streaming output truncated to the last 5000 lines.
26/11/07 12:07:39 INFO Utils: Fetching spark://ip-172-31-30-94.ec2.internal:43593/jars/xml-apis_xml-apis-ext-1.3.04.jar to /tmp/spark-7b5bb8bd-a943-4b23-b73a-7f7d076ec182/userFiles-4ed6966b-25c1-4557-ae4b-ce50e803c5c5/fetchFileTemp13543196195396948685.tmp
26/11/07 12:07:39 INFO Utils: /tmp/spark-7b5bb8bd-a943-4b23-b73a-7f7d076ec182/userFiles-4ed6966b-25c1-4557-ae4b-ce50e803c5c5/fetchFileTemp13543196195396948685.tmp has been previously copied to /tmp/spark-7b5bb8bd-a943-4b23-b73a-7f7d076ec182/userFiles-4ed6966b-25c1-4557-ae4b-ce50e803c5c5/xml-apis_xml-apis-ext-1.3.04.jar
26/11/07 12:07:39 INFO Executor: Adding file:/tmp/spark-7b5bb8bd-a943-4b23-b73a-7f7d076ec182/userFiles-4ed6966b-25c1-4557-ae4b-ce50e803c5c5/xml-apis_xml-apis-ext-1.3.04.jar to class loader default
26/11/07 12:07:39 INFO Executor: Fetching spark://ip-172-31-30-94.ec2.internal:43593/jars/org.apache.sedona_sedona-common-1.9.1.jar with timestamp 1794053258018
26/11/07 12:07

Final result

The historical batch completed all eight intermediate epochs:

Plain Text
1980 COMPLETE
1985 COMPLETE
1990 COMPLETE
1995 COMPLETE
2000 COMPLETE
2005 COMPLETE
2010 COMPLETE
2015 COMPLETE

The batch summary reports:

Plain Text
Completed epochs: 8
Expected outputs: 32
Execution mode: Sequential
Total runtime: 0.86 hours, about 52 minutes
Batch exit code: 0

Therefore, the memory correction worked and the batch finished normally.

What we now have for Bengaluru

Including the two endpoint epochs:

Plain Text
1975 COMPLETE
1980 COMPLETE
1985 COMPLETE
1990 COMPLETE
1995 COMPLETE
2000 COMPLETE
2005 COMPLETE
2010 COMPLETE
2015 COMPLETE
2020 COMPLETE

For every epoch, we should now have all four outputs:

Plain Text
Population
SMOD
Built surface
Enriched grid-year dataset

That gives the intended inventory:

Plain Text
10 epochs × 3 individual GHSL products = 30 outputs
10 epochs × 1 enriched dataset = 10 outputs
------------------------------------------------------
Total = 40 outputs

The batch itself completed the 32 intermediate-epoch outputs, while the eight endpoint outputs for 1975 and 2020 were already present.

Evidence from the final epochs

For 2010:

Built surface succeeded with 3,658 grid cells
3,480 complete cells and 178 partial boundary cells
Total built surface: 252,980,792 m²
Enriched population: 10,350,317.8079
Conservation, grid alignment, and HDFS read-back passed

For 2015:

Population succeeded with 3,658 grid cells
Population total: 12,337,715.5687
SMOD had complete population coverage and no unmatched cells
Built surface total: 305,757,963 m²
Residential built surface: 280,259,187 m²
Non-residential built surface: 25,498,776 m²
Enriched output succeeded and passed HDFS read-back

The shown product validations also report:

Plain Text
Rows: 3,658
Duplicate keys: 0
Invalid rows: 0

The final log marks 2005, 2010, and 2015 complete, and the batch summary confirms completion of all intermediate epochs.

Next step

We have completed the historical data-generation stage for Bengaluru. Do not rerun the batch.

The next cell should be a final read-only inventory and validation across all 40 outputs. After that, we should build:

One combined longitudinal Bengaluru dataset containing all ten epochs
Epoch-level comparison summaries
Population-growth comparisons
Built-surface expansion comparisons
SMOD settlement-transition comparisons
Benchmark metrics for Spark/Sedona processing

So yes bro, this is the major milestone we wanted: all epochs for all three GHSL datasets have now been processed successfully for Bengaluru.

the core processing used the required project stack, especially Apache Spark, Apache Sedona, and HDFS.

What was used
HDFS stored the original GHSL GeoTIFFs and the resulting curated Parquet datasets.
Apache Spark handled distributed reading, transformations, aggregation, joins, validation, and Parquet writing.
Apache Sedona handled the geospatial and raster-specific operations.
Python/PySpark served as the orchestration layer for submitting Spark jobs and expressing transformations.

The Spark jobs explicitly loaded:

Plain Text
org.apache.sedona:sedona-spark-3.5_2.12:1.9.1
org.datasyslab:geotools-wrapper:1.9.1-33.5

The enrichment failure itself proved that Sedona functions were expected: ST_AsBinary initially failed because Sedona SQL had not been registered. After switching the generated enrichment program to SedonaContext, the intermediate epochs completed successfully.

Where Sedona was used

For the individual GHSL products, Sedona processed raster content through functions such as:

Plain Text
RS_FromGeoTiff
RS_Tile
RS_BandAsArray

For enriched spatial outputs, Sedona supplied spatial SQL functions such as:

Plain Text
ST_PolygonFromEnvelope
ST_AsBinary

So this was not a conventional Python-only raster workflow. The architecture was:

Plain Text
GHSL GeoTIFFs in HDFS
↓
Spark binaryFile reader
↓
Apache Sedona raster decoding and tiling
↓
Spark SQL aggregation from 100 m to 1 km
↓
Sedona geometry creation
↓
Spark joins and validation
↓
Parquet outputs in HDFS
What was not used as the primary processing engine

In [13]:
# ============================================================
# CELL 8: FINAL HDFS INVENTORY AND STRUCTURAL VALIDATION
#
# REPLACE the existing Cell 8 completely.
#
# Checks:
#   - 40 expected HDFS outputs
#   - Every Parquet output is readable by Spark
#   - Every output contains 3,658 Bengaluru grid cells
#   - grid_cell_id is present, non-null, and unique
#   - grid_1km_col + grid_1km_row are present and unique
#
# Safety:
#   - Does not rerun the historical pipeline
#   - Does not write to HDFS
#   - Does not modify any dataset
# ============================================================

import base64
import hashlib
import textwrap

REMOTE_VALIDATOR = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "final_validate_all_40.py"
)

REMOTE_LOG = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "final_validate_all_40.log"
)

# ============================================================
# REMOTE PYSPARK VALIDATOR
# ============================================================

validator_code = textwrap.dedent(
    """
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    import sys
    import time

    EPOCHS = [
        1975,
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
        2020,
    ]

    STAGES = [
        "population",
        "smod",
        "built_surface",
        "enriched",
    ]

    EXPECTED_ROWS = 3658
    EXPECTED_OUTPUTS = 40

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    spark = (
        SparkSession.builder
        .appName("GHSL_Bengaluru_Final_Inventory")
        .config("spark.ui.enabled", "false")
        .config("spark.sql.shuffle.partitions", "2")
        .config("spark.default.parallelism", "2")
        .config("spark.sql.adaptive.enabled", "false")
        .config("spark.sql.codegen.wholeStage", "false")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")

    jvm = spark.sparkContext._jvm

    hadoop_conf = (
        spark.sparkContext
        ._jsc
        .hadoopConfiguration()
    )

    hdfs = (
        jvm.org.apache.hadoop.fs.FileSystem.get(
            hadoop_conf
        )
    )

    HadoopPath = jvm.org.apache.hadoop.fs.Path


    def output_path(epoch, stage):
        if stage == "enriched":
            return (
                f"{HDFS_BASE}/enriched_grid_year/"
                f"epoch={epoch}"
            )

        return (
            f"{HDFS_BASE}/product={stage}/"
            f"epoch={epoch}/grid_1km"
        )


    def hdfs_exists(path):
        return bool(
            hdfs.exists(
                HadoopPath(path)
            )
        )


    def validate_output(epoch, stage):
        path = output_path(epoch, stage)
        started = time.time()

        result = {
            "epoch": epoch,
            "stage": stage,
            "path": path,
            "status": "FAIL",
            "rows": None,
            "null_keys": None,
            "duplicate_ids": None,
            "duplicate_xy": None,
            "message": "",
            "seconds": None,
        }

        try:
            if not hdfs_exists(path):
                result["message"] = "HDFS path missing"
                return result

            df = spark.read.parquet(path)

            required_keys = {
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
            }

            missing_keys = sorted(
                required_keys - set(df.columns)
            )

            if missing_keys:
                result["message"] = (
                    "Missing key columns: "
                    + ",".join(missing_keys)
                )
                return result

            row_count = df.count()

            null_key_count = df.filter(
                F.col("grid_cell_id").isNull()
                | F.col("grid_1km_col").isNull()
                | F.col("grid_1km_row").isNull()
            ).count()

            distinct_id_count = (
                df.select("grid_cell_id")
                .distinct()
                .count()
            )

            distinct_xy_count = (
                df.select(
                    "grid_1km_col",
                    "grid_1km_row",
                )
                .distinct()
                .count()
            )

            duplicate_id_count = (
                row_count - distinct_id_count
            )

            duplicate_xy_count = (
                row_count - distinct_xy_count
            )

            result["rows"] = row_count
            result["null_keys"] = null_key_count
            result["duplicate_ids"] = (
                duplicate_id_count
            )
            result["duplicate_xy"] = (
                duplicate_xy_count
            )

            failures = []

            if row_count != EXPECTED_ROWS:
                failures.append(
                    f"rows={row_count}, "
                    f"expected={EXPECTED_ROWS}"
                )

            if null_key_count != 0:
                failures.append(
                    f"null_keys={null_key_count}"
                )

            if duplicate_id_count != 0:
                failures.append(
                    f"duplicate_ids={duplicate_id_count}"
                )

            if duplicate_xy_count != 0:
                failures.append(
                    f"duplicate_xy={duplicate_xy_count}"
                )

            if failures:
                result["message"] = "; ".join(
                    failures
                )
                return result

            result["status"] = "PASS"
            result["message"] = "Valid"

            return result

        except Exception as error:
            result["message"] = (
                f"{type(error).__name__}: {error}"
            )

            return result

        finally:
            result["seconds"] = round(
                time.time() - started,
                2,
            )


    results = []

    print("=" * 104)
    print("FINAL HDFS INVENTORY AND STRUCTURAL VALIDATION")
    print("=" * 104)

    for epoch in EPOCHS:
        for stage in STAGES:
            result = validate_output(
                epoch,
                stage,
            )

            results.append(result)

            print(
                f"{epoch}  "
                f"{stage:<15}  "
                f"{result['status']:<4}  "
                f"rows={str(result['rows']):<5}  "
                f"null={str(result['null_keys']):<4}  "
                f"dup_id={str(result['duplicate_ids']):<4}  "
                f"dup_xy={str(result['duplicate_xy']):<4}  "
                f"time={result['seconds']}s"
            )

            if result["status"] == "FAIL":
                print(
                    f"    Reason: {result['message']}"
                )
                print(
                    f"    Path: {result['path']}"
                )

    print()
    print("=" * 104)
    print("VALIDATION BY EPOCH")
    print("=" * 104)

    for epoch in EPOCHS:
        epoch_results = [
            result
            for result in results
            if result["epoch"] == epoch
        ]

        passed = sum(
            result["status"] == "PASS"
            for result in epoch_results
        )

        failed = len(epoch_results) - passed

        print(
            f"{epoch}: "
            f"passed={passed}, "
            f"failed={failed}"
        )

    total_checked = len(results)

    total_passed = sum(
        result["status"] == "PASS"
        for result in results
    )

    total_failed = (
        total_checked - total_passed
    )

    print()
    print("=" * 104)
    print("FINAL 40-OUTPUT VALIDATION SUMMARY")
    print("=" * 104)

    print(
        f"Expected outputs : {EXPECTED_OUTPUTS}"
    )

    print(
        f"Outputs checked  : {total_checked}"
    )

    print(
        f"Passed           : {total_passed}"
    )

    print(
        f"Failed           : {total_failed}"
    )

    if total_checked != EXPECTED_OUTPUTS:
        print()
        print(
            "FINAL VALIDATION: FAILED "
            "because output count is incorrect"
        )

        spark.stop()
        sys.exit(2)

    if total_failed != 0:
        print()
        print("FAILED OUTPUTS:")

        for result in results:
            if result["status"] == "FAIL":
                print(
                    f"{result['epoch']} "
                    f"{result['stage']}: "
                    f"{result['message']}"
                )

        print()
        print("FINAL VALIDATION: FAILED")

        spark.stop()
        sys.exit(3)

    print()
    print("All 40 HDFS paths exist: PASS")
    print("All 40 Parquet outputs are readable: PASS")
    print("All outputs contain 3,658 rows: PASS")
    print("All grid keys are non-null: PASS")
    print("All grid_cell_id values are unique: PASS")
    print("All grid coordinate pairs are unique: PASS")
    print("FINAL VALIDATION: SUCCESS")

    spark.stop()
    """
).strip() + "\n"

# ============================================================
# COMPILE THE EXACT SOURCE IN COLAB
# ============================================================

compile(
    validator_code,
    "final_validate_all_40.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    validator_code.encode("utf-8")
).hexdigest()

print("LOCAL PYTHON COMPILE CHECK: PASS")
print("Expected validator SHA-256:", local_sha256)

# ============================================================
# TRANSFER THE EXACT SOURCE USING BASE64
# ============================================================

encoded_validator = base64.b64encode(
    validator_code.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

VALIDATOR="{REMOTE_VALIDATOR}"
LOG="{REMOTE_LOG}"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_validator}" \
| base64 -d \
> "${{VALIDATOR}}"

ACTUAL_SHA256="$(
    sha256sum "${{VALIDATOR}}" \
    | awk '{{print $1}}'
)"

echo "Remote validator SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Transferred validator checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM CHECK: PASS"

python3 -m py_compile "${{VALIDATOR}}"

echo "REMOTE PYTHON COMPILE CHECK: PASS"

echo
echo "HDFS readiness:"

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
    | awk '{{print $2}}' \
    | grep -qx "${{daemon}}" \
    || {{
        echo "ERROR: ${{daemon}} is not running."
        exit 21
    }}

    echo "PASS: ${{daemon}}"
done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {{
    echo "ERROR: HDFS safe mode is enabled."
    exit 22
}}

echo "PASS: HDFS safe mode is OFF"

echo
echo "RUNNING HDFS-BACKED SPARK VALIDATION"

spark-submit \
    --master 'local[1]' \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=2 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    "${{VALIDATOR}}" \
    2>&1 \
    | tee "${{LOG}}"

status=${{PIPESTATUS[0]}}

echo
echo "FINAL_VALIDATION_EXIT_CODE=${{status}}"
echo "Validation log: ${{LOG}}"

exit "${{status}}"
"""

validation_result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 8 RESULT")
print("=" * 80)

print(
    "Validation return code:",
    validation_result.returncode,
)

print(
    "Expected result:",
    "40 passed, 0 failed",
)

print(
    "Remote log:",
    REMOTE_LOG,
)

LOCAL PYTHON COMPILE CHECK: PASS
Expected validator SHA-256: a32434364c3345dcae24feabb58217a6b242e427d7b3e24462179794347ebfa0
Remote validator SHA-256: a32434364c3345dcae24feabb58217a6b242e427d7b3e24462179794347ebfa0
REMOTE CHECKSUM CHECK: PASS
REMOTE PYTHON COMPILE CHECK: PASS

HDFS readiness:
PASS: NameNode
PASS: DataNode
PASS: SecondaryNameNode
PASS: HDFS safe mode is OFF

RUNNING HDFS-BACKED SPARK VALIDATION
26/11/07 13:01:10 INFO SparkContext: Running Spark version 3.5.9
26/11/07 13:01:10 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/07 13:01:10 INFO SparkContext: Java version 11.0.32.1
26/11/07 13:01:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 13:01:10 INFO ResourceUtils: ==============================================================
26/11/07 13:01:10 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/07 13:01:10 INFO ResourceUtils: ================

The final HDFS-backed validation passed exactly as required:

Plain Text
Expected outputs : 40
Outputs checked : 40
Passed : 40
Failed : 0
FINAL_VALIDATION_EXIT_CODE=0

For every epoch from 1975 through 2020, all four outputs passed:

Population
SMOD
Built surface
Enriched grid-year dataset

Each output contains:

Plain Text
3,658 Bengaluru grid cells
0 null grid keys
0 duplicate grid_cell_id values
0 duplicate grid coordinate pairs

The checksum and compilation checks also proved that the exact validator compiled in Colab was transferred and executed on EC2 without corruption.

Final validated inventory
Plain Text
10 epochs × 4 output groups = 40 validated outputs

1975 4/4 PASS
1980 4/4 PASS
1985 4/4 PASS
1990 4/4 PASS
1995 4/4 PASS
2000 4/4 PASS
2005 4/4 PASS
2010 4/4 PASS
2015 4/4 PASS
2020 4/4 PASS
What this proves

The Bengaluru historical data-generation stage is now formally validated:

All expected HDFS paths exist.
Apache Spark can read every Parquet output.
Every dataset uses the complete 3,658-cell Bengaluru grid.
Grid-cell identities are non-null and unique.
Coordinate pairs are unique.
No epoch or product is missing from the inventory.

Build one combined longitudinal Bengaluru dataset

10 epochs × 3,658 cells
= 36,580 expected grid-year rows

The source should be the ten validated enriched datasets:

Plain Text
.../enriched_grid_year/epoch=1975
.../enriched_grid_year/epoch=1980
...
.../enriched_grid_year/epoch=2020

The combined HDFS output can use a stable path such as:

Plain Text
/data/ghsl/curated/pilots/bengaluru/
longitudinal_grid_year/all_epochs

What we will do next

We should proceed in the following order, one cell at a time.

Step 1: Build the longitudinal dataset

Read the ten enriched epoch datasets and stack them into one table:

Plain Text
3,658 grid cells × 10 epochs = 36,580 grid-year rows

Conceptually:

Plain Text
grid_cell_id epoch population SMOD built_surface geometry
BLR_... 1975 ...
BLR_... 1980 ...
BLR_... 1985 ...
...
BLR_... 2020 ...

Proposed HDFS output:

Plain Text
/data/ghsl/curated/pilots/bengaluru/
longitudinal_grid_year/all_epochs

This will become the primary analytical dataset for all subsequent comparisons.

Step 2: Create epoch-level summaries

Using the longitudinal dataset, calculate one summary row per epoch:

Total population
Population change from previous epoch
Population growth percentage
Total built surface
Residential built surface
Non-residential built surface
Built-surface change and growth percentage
Population density
Population per built hectare
Counts and shares of SMOD classes

Expected output:

Plain Text
10 epoch-summary rows
Step 3: Population-growth comparisons

At both city and grid-cell levels, calculate:

Plain Text
Absolute change
Percentage change
Five-year growth
1975–2020 total growth
Top-growing grid cells
Grid cells with population decline

This tells us where Bengaluru’s population growth occurred, rather than only showing the city-wide total.

Step 4: Built-surface expansion comparisons

Calculate:

Plain Text
Built-surface growth per epoch
Residential versus non-residential growth
1975–2020 built expansion
Newly urbanized grid cells
Population growth versus built-surface growth

This lets us assess whether Bengaluru’s growth was primarily:

Densification within existing built areas
Outward spatial expansion
A combination of both
Step 5: SMOD settlement transitions

Track each grid cell’s settlement-class changes across epochs.

Examples:

Plain Text
Rural → Suburban/peri-urban
Suburban/peri-urban → Urban centre
Rural → Urban centre
No class change

We will produce transition counts and matrices for consecutive epochs and for the full 1975–2020 period.

Step 6: Benchmark metrics

Record technical performance for the project report:

Execution time by product and epoch
Total batch runtime
Spark driver-memory configuration
Number of cells and output rows
HDFS input/output structure
Sequential resumability behavior
Validation counts
Sedona raster-processing role
Memory failure encountered at 1995 and the corrected configuration

In [15]:
# ============================================================
# CELL 9A: READ-ONLY ENRICHED SCHEMA COMPARISON
#
# Compares:
#   1975 endpoint output
#   1980 historical output
#   2020 endpoint output
#
# Safety:
#   - Reads schemas only
#   - Does not count or process raster data
#   - Does not write to HDFS
#   - Does not modify Cell 9 output
# ============================================================

import base64
import hashlib
import textwrap

REMOTE_PROGRAM = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "compare_enriched_schemas.py"
)

REMOTE_LOG = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "compare_enriched_schemas.log"
)

program_code = textwrap.dedent(
    """
    from pyspark.sql import SparkSession

    EPOCHS = [
        1975,
        1980,
        2020,
    ]

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
        "/enriched_grid_year"
    )

    spark = (
        SparkSession.builder
        .appName("GHSL_Enriched_Schema_Comparison")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")

    schemas = {}

    for epoch in EPOCHS:
        path = f"{HDFS_BASE}/epoch={epoch}"

        df = spark.read.parquet(path)

        schemas[epoch] = {
            field.name: field.dataType.simpleString()
            for field in df.schema.fields
        }

        print()
        print("=" * 100)
        print(f"ENRICHED SCHEMA: {epoch}")
        print("=" * 100)
        print(f"HDFS path: {path}")
        print(f"Column count: {len(df.columns)}")

        for position, field in enumerate(
            df.schema.fields,
            start=1,
        ):
            print(
                f"{position:02d}. "
                f"{field.name:<48} "
                f"{field.dataType.simpleString():<24} "
                f"nullable={field.nullable}"
            )

    all_column_names = sorted(
        set().union(
            *[
                set(schema.keys())
                for schema in schemas.values()
            ]
        )
    )

    shared_column_names = sorted(
        set.intersection(
            *[
                set(schema.keys())
                for schema in schemas.values()
            ]
        )
    )

    print()
    print("=" * 100)
    print("COLUMN-PRESENCE COMPARISON")
    print("=" * 100)

    print(
        "Columns present in all compared epochs:",
        len(shared_column_names),
    )

    for epoch in EPOCHS:
        other_epochs = [
            other
            for other in EPOCHS
            if other != epoch
        ]

        other_columns = set().union(
            *[
                set(schemas[other].keys())
                for other in other_epochs
            ]
        )

        unique_columns = sorted(
            set(schemas[epoch].keys())
            - other_columns
        )

        print()
        print(
            f"Columns unique to {epoch}: "
            f"{len(unique_columns)}"
        )

        for column in unique_columns:
            print(f"  {column}")

    print()
    print("=" * 100)
    print("COLUMN AVAILABILITY MATRIX")
    print("=" * 100)

    for column in all_column_names:
        status_values = []

        for epoch in EPOCHS:
            if column in schemas[epoch]:
                status_values.append(
                    f"{epoch}=YES"
                )
            else:
                status_values.append(
                    f"{epoch}=NO"
                )

        print(
            f"{column:<48} "
            + "  ".join(status_values)
        )

    print()
    print("=" * 100)
    print("SHARED-COLUMN TYPE DIFFERENCES")
    print("=" * 100)

    type_difference_count = 0

    for column in shared_column_names:
        epoch_types = {
            epoch: schemas[epoch][column]
            for epoch in EPOCHS
        }

        if len(set(epoch_types.values())) > 1:
            type_difference_count += 1

            print(
                f"{column}: "
                + ", ".join(
                    f"{epoch}={epoch_types[epoch]}"
                    for epoch in EPOCHS
                )
            )

    print(
        "Shared columns with type differences:",
        type_difference_count,
    )

    print()
    print("=" * 100)
    print("CORE LONGITUDINAL CANDIDATES")
    print("=" * 100)

    core_candidates = [
        "city",
        "epoch",
        "grid_cell_id",
        "grid_1km_col",
        "grid_1km_row",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "population_total",
        "smod_class",
        "built_surface_total_m2",
        "built_surface_residential_m2",
        "built_surface_nres_m2",
        "geometry_wkb",
        "geometry_crs",
    ]

    for column in core_candidates:
        availability = []

        for epoch in EPOCHS:
            data_type = schemas[epoch].get(
                column,
                "MISSING",
            )

            availability.append(
                f"{epoch}={data_type}"
            )

        print(
            f"{column:<42} "
            + "  ".join(availability)
        )

    print()
    print("SCHEMA COMPARISON: SUCCESS")

    spark.stop()
    """
).strip() + "\n"

compile(
    program_code,
    "compare_enriched_schemas.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    program_code.encode("utf-8")
).hexdigest()

print("LOCAL PYTHON COMPILE CHECK: PASS")
print("Expected program SHA-256:", local_sha256)

encoded_program = base64.b64encode(
    program_code.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoopHOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"
LOG="{REMOTE_LOG}"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_program}" \
| base64 -d \
> "${{PROGRAM}}"

ACTUAL_SHA256="$(
    sha256sum "${{PROGRAM}}" \
    | awk '{{print $1}}'
)"

echo "Remote program SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM CHECK: PASS"

python3 -m py_compile "${{PROGRAM}}"

echo "REMOTE PYTHON COMPILE CHECK: PASS"

spark-submit \
    --master 'local[1]' \
    --driver-memory 2g \
    --conf spark.ui.enabled=false \
    "${{PROGRAM}}" \
    2>&1 \
    | tee "${{LOG}}"

status=${{PIPESTATUS[0]}}

echo
echo "SCHEMA_COMPARISON_EXIT_CODE=${{status}}"
echo "Schema log: ${{LOG}}"

exit "${{status}}"
"""

schema_result = run_remote(
    remote_command,
    timeout=900,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 9A RESULT")
print("=" * 80)
print(
    "Return code:",
    schema_result.returncode,
)
print(
    "Remote log:",
    REMOTE_LOG,
)

LOCAL PYTHON COMPILE CHECK: PASS
Expected program SHA-256: e0afe9e567d734d64d3f43e25f9c06b2af5efbf7a7824ec37dd45761e77ff25e
Remote program SHA-256: e0afe9e567d734d64d3f43e25f9c06b2af5efbf7a7824ec37dd45761e77ff25e
REMOTE CHECKSUM CHECK: PASS
REMOTE PYTHON COMPILE CHECK: PASS
26/11/07 13:09:41 INFO SparkContext: Running Spark version 3.5.9
26/11/07 13:09:41 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/07 13:09:41 INFO SparkContext: Java version 11.0.32.1
26/11/07 13:09:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 13:09:41 INFO ResourceUtils: ==============================================================
26/11/07 13:09:41 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/07 13:09:41 INFO ResourceUtils: ==============================================================
26/11/07 13:09:41 INFO SparkContext: Submitted application: GHSL_Enriched_Schema_Comparison
26

In [16]:
# ============================================================
# CELL 9: BUILD CANONICAL BENGALURU LONGITUDINAL DATASET
#
# REPLACE the failed Cell 9 completely.
#
# Input:
#   10 validated enriched epoch datasets in HDFS
#
# Output:
#   /data/ghsl/curated/pilots/bengaluru/
#   longitudinal_grid_year/all_epochs
#
# Expected:
#   10 epochs
#   3,658 cells per epoch
#   36,580 grid-year rows
#
# The output uses a canonical 16-column analytical schema
# shared consistently by all ten epochs.
# ============================================================

import base64
import hashlib
import textwrap

REMOTE_PROGRAM = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_longitudinal_grid_year.py"
)

REMOTE_LOG = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_longitudinal_grid_year.log"
)

program_code = textwrap.dedent(
    """
    from functools import reduce
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    import time

    EPOCHS = [
        1975, 1980, 1985, 1990, 1995,
        2000, 2005, 2010, 2015, 2020,
    ]

    EXPECTED_CELLS = 3658
    EXPECTED_ROWS = EXPECTED_CELLS * len(EPOCHS)

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    INPUT_BASE = (
        f"{HDFS_BASE}/enriched_grid_year"
    )

    OUTPUT_PATH = (
        f"{HDFS_BASE}/longitudinal_grid_year/"
        "all_epochs"
    )

    CANONICAL_COLUMNS = [
        "city",
        "epoch",
        "grid_cell_id",
        "grid_1km_col",
        "grid_1km_row",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "population_total",
        "smod_class",
        "built_surface_total_m2",
        "built_surface_residential_m2",
        "built_surface_nres_m2",
        "geometry_wkb",
        "geometry_crs",
    ]

    spark = (
        SparkSession.builder
        .appName(
            "GHSL_Bengaluru_Canonical_Longitudinal"
        )
        .config("spark.ui.enabled", "false")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.default.parallelism", "2")
        .config("spark.sql.adaptive.enabled", "false")
        .config("spark.sql.codegen.wholeStage", "false")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")

    jvm = spark.sparkContext._jvm
    hadoop_conf = (
        spark.sparkContext
        ._jsc
        .hadoopConfiguration()
    )

    hdfs = (
        jvm.org.apache.hadoop.fs.FileSystem.get(
            hadoop_conf
        )
    )

    HadoopPath = jvm.org.apache.hadoop.fs.Path


    def require(condition, message):
        if not condition:
            raise RuntimeError(message)


    def exists(path):
        return bool(
            hdfs.exists(
                HadoopPath(path)
            )
        )


    def input_path(epoch):
        return (
            f"{INPUT_BASE}/epoch={epoch}"
        )


    def select_canonical(df, epoch):
        missing = sorted(
            set(CANONICAL_COLUMNS)
            - set(df.columns)
        )

        require(
            not missing,
            (
                f"Epoch {epoch}: missing canonical "
                f"columns: {missing}"
            ),
        )

        return df.select(
            F.col("city").cast("string").alias("city"),
            F.lit(epoch).cast("int").alias("epoch"),
            F.col("grid_cell_id")
            .cast("string")
            .alias("grid_cell_id"),
            F.col("grid_1km_col")
            .cast("long")
            .alias("grid_1km_col"),
            F.col("grid_1km_row")
            .cast("long")
            .alias("grid_1km_row"),
            F.col("grid_min_x")
            .cast("double")
            .alias("grid_min_x"),
            F.col("grid_max_x")
            .cast("double")
            .alias("grid_max_x"),
            F.col("grid_min_y")
            .cast("double")
            .alias("grid_min_y"),
            F.col("grid_max_y")
            .cast("double")
            .alias("grid_max_y"),
            F.col("population_total")
            .cast("double")
            .alias("population_total"),
            F.col("smod_class")
            .cast("int")
            .alias("smod_class"),
            F.col("built_surface_total_m2")
            .cast("double")
            .alias("built_surface_total_m2"),
            F.col("built_surface_residential_m2")
            .cast("double")
            .alias(
                "built_surface_residential_m2"
            ),
            F.col("built_surface_nres_m2")
            .cast("double")
            .alias("built_surface_nres_m2"),
            F.col("geometry_wkb")
            .cast("binary")
            .alias("geometry_wkb"),
            F.col("geometry_crs")
            .cast("string")
            .alias("geometry_crs"),
        )


    def validate(df, label):
        row_count = df.count()

        epoch_values = sorted(
            int(row["epoch"])
            for row in (
                df.select("epoch")
                .distinct()
                .collect()
            )
        )

        distinct_cells = (
            df.select("grid_cell_id")
            .distinct()
            .count()
        )

        distinct_grid_years = (
            df.select(
                "grid_cell_id",
                "epoch",
            )
            .distinct()
            .count()
        )

        duplicate_grid_years = (
            row_count - distinct_grid_years
        )

        distinct_coordinate_years = (
            df.select(
                "grid_1km_col",
                "grid_1km_row",
                "epoch",
            )
            .distinct()
            .count()
        )

        duplicate_coordinate_years = (
            row_count
            - distinct_coordinate_years
        )

        null_keys = df.filter(
            F.col("grid_cell_id").isNull()
            | F.col("epoch").isNull()
            | F.col("grid_1km_col").isNull()
            | F.col("grid_1km_row").isNull()
        ).count()

        invalid_values = df.filter(
            F.col("population_total").isNull()
            | F.isnan("population_total")
            | (F.col("population_total") < 0)
            | F.col("smod_class").isNull()
            | F.col(
                "built_surface_total_m2"
            ).isNull()
            | F.isnan("built_surface_total_m2")
            | (
                F.col("built_surface_total_m2")
                < 0
            )
            | F.col(
                "built_surface_residential_m2"
            ).isNull()
            | (
                F.col(
                    "built_surface_residential_m2"
                )
                < 0
            )
            | F.col(
                "built_surface_nres_m2"
            ).isNull()
            | (
                F.col("built_surface_nres_m2")
                < 0
            )
            | F.col("geometry_wkb").isNull()
            | F.col("geometry_crs").isNull()
        ).count()

        per_epoch = {
            int(row["epoch"]): int(row["count"])
            for row in (
                df.groupBy("epoch")
                .count()
                .collect()
            )
        }

        require(
            row_count == EXPECTED_ROWS,
            (
                f"{label}: rows={row_count}, "
                f"expected={EXPECTED_ROWS}"
            ),
        )

        require(
            epoch_values == EPOCHS,
            (
                f"{label}: epochs={epoch_values}, "
                f"expected={EPOCHS}"
            ),
        )

        require(
            distinct_cells == EXPECTED_CELLS,
            (
                f"{label}: distinct cells="
                f"{distinct_cells}, "
                f"expected={EXPECTED_CELLS}"
            ),
        )

        require(
            duplicate_grid_years == 0,
            (
                f"{label}: duplicate grid-years="
                f"{duplicate_grid_years}"
            ),
        )

        require(
            duplicate_coordinate_years == 0,
            (
                f"{label}: duplicate coordinate-years="
                f"{duplicate_coordinate_years}"
            ),
        )

        require(
            null_keys == 0,
            f"{label}: null keys={null_keys}",
        )

        require(
            invalid_values == 0,
            (
                f"{label}: invalid core values="
                f"{invalid_values}"
            ),
        )

        for epoch in EPOCHS:
            require(
                per_epoch.get(epoch, 0)
                == EXPECTED_CELLS,
                (
                    f"{label}: epoch {epoch} has "
                    f"{per_epoch.get(epoch, 0)} rows"
                ),
            )

        require(
            df.columns == CANONICAL_COLUMNS,
            (
                f"{label}: output columns differ "
                "from canonical schema"
            ),
        )

        return {
            "rows": row_count,
            "epochs": epoch_values,
            "distinct_cells": distinct_cells,
            "duplicate_grid_years": (
                duplicate_grid_years
            ),
            "duplicate_coordinate_years": (
                duplicate_coordinate_years
            ),
            "null_keys": null_keys,
            "invalid_values": invalid_values,
            "per_epoch": per_epoch,
        }


    started = time.time()
    epoch_frames = []

    print("=" * 100)
    print("READING CANONICAL EPOCH DATA")
    print("=" * 100)

    for epoch in EPOCHS:
        path = input_path(epoch)

        require(
            exists(path),
            f"Missing HDFS input: {path}",
        )

        source_df = spark.read.parquet(path)

        canonical_df = select_canonical(
            source_df,
            epoch,
        )

        row_count = canonical_df.count()

        duplicate_ids = (
            row_count
            - canonical_df.select(
                "grid_cell_id"
            ).distinct().count()
        )

        require(
            row_count == EXPECTED_CELLS,
            (
                f"Epoch {epoch}: rows={row_count}, "
                f"expected={EXPECTED_CELLS}"
            ),
        )

        require(
            duplicate_ids == 0,
            (
                f"Epoch {epoch}: duplicate IDs="
                f"{duplicate_ids}"
            ),
        )

        epoch_frames.append(canonical_df)

        print(
            f"{epoch}: rows={row_count}, "
            f"columns={len(canonical_df.columns)}, "
            f"duplicate_ids={duplicate_ids}, "
            "status=PASS"
        )


    print()
    print("=" * 100)
    print("UNIONING TEN EPOCHS")
    print("=" * 100)

    longitudinal_df = reduce(
        lambda left, right: (
            left.unionByName(right)
        ),
        epoch_frames,
    )

    pre_write = validate(
        longitudinal_df,
        "PRE-WRITE",
    )

    print("Pre-write validation: PASS")
    print("Rows:", pre_write["rows"])
    print(
        "Distinct cells:",
        pre_write["distinct_cells"],
    )
    print(
        "Duplicate grid-year keys:",
        pre_write["duplicate_grid_years"],
    )


    print()
    print("=" * 100)
    print("WRITING CANONICAL DATASET TO HDFS")
    print("=" * 100)

    (
        longitudinal_df
        .repartition(10, "epoch")
        .write
        .mode("overwrite")
        .partitionBy("epoch")
        .parquet(OUTPUT_PATH)
    )

    print("HDFS write: COMPLETE")
    print("Output:", OUTPUT_PATH)


    print()
    print("=" * 100)
    print("HDFS READ-BACK VALIDATION")
    print("=" * 100)

    require(
        exists(OUTPUT_PATH),
        "Output path missing after write",
    )

    saved_df = (
        spark.read
        .option("basePath", OUTPUT_PATH)
        .parquet(OUTPUT_PATH)
        .select(*CANONICAL_COLUMNS)
    )

    read_back = validate(
        saved_df,
        "READ-BACK",
    )

    elapsed = round(
        time.time() - started,
        2,
    )

    print("Read-back validation: PASS")
    print("Rows:", read_back["rows"])
    print("Epochs:", read_back["epochs"])
    print(
        "Distinct cells:",
        read_back["distinct_cells"],
    )
    print(
        "Duplicate grid-year keys:",
        read_back["duplicate_grid_years"],
    )
    print(
        "Duplicate coordinate-year keys:",
        read_back[
            "duplicate_coordinate_years"
        ],
    )
    print(
        "Null keys:",
        read_back["null_keys"],
    )
    print(
        "Invalid core values:",
        read_back["invalid_values"],
    )

    print()
    print("=" * 100)
    print("LONGITUDINAL DATASET SUMMARY")
    print("=" * 100)
    print("Canonical columns:", len(CANONICAL_COLUMNS))
    print("Epochs:", len(read_back["epochs"]))
    print("Cells per epoch:", EXPECTED_CELLS)
    print("Grid-year rows:", read_back["rows"])
    print("Elapsed seconds:", elapsed)
    print("Output:", OUTPUT_PATH)
    print()
    print("LONGITUDINAL DATASET: SUCCESS")

    spark.stop()
    """
).strip() + "\n"


# Compile exact source before transfer.
compile(
    program_code,
    "build_longitudinal_grid_year.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    program_code.encode("utf-8")
).hexdigest()

print("LOCAL PYTHON COMPILE CHECK: PASS")
print("Expected program SHA-256:", local_sha256)

encoded_program = base64.b64encode(
    program_code.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"
LOG="{REMOTE_LOG}"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_program}" \
| base64 -d \
> "${{PROGRAM}}"

ACTUAL_SHA256="$(
    sha256sum "${{PROGRAM}}" \
    | awk '{{print $1}}'
)"

echo "Remote program SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Program checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM CHECK: PASS"

python3 -m py_compile "${{PROGRAM}}"

echo "REMOTE PYTHON COMPILE CHECK: PASS"

echo
echo "HDFS readiness:"

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
    | awk '{{print $2}}' \
    | grep -qx "${{daemon}}" \
    || {{
        echo "ERROR: ${{daemon}} is not running."
        exit 21
    }}

    echo "PASS: ${{daemon}}"
done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {{
    echo "ERROR: HDFS safe mode is enabled."
    exit 22
}}

echo "PASS: HDFS safe mode is OFF"

echo
echo "RUNNING CANONICAL LONGITUDINAL JOB"

spark-submit \
    --master 'local[1]' \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    "${{PROGRAM}}" \
    2>&1 \
    | tee "${{LOG}}"

status=${{PIPESTATUS[0]}}

echo
echo "LONGITUDINAL_EXIT_CODE=${{status}}"
echo "Job log: ${{LOG}}"

exit "${{status}}"
"""

longitudinal_result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 9 RESULT")
print("=" * 80)
print(
    "Return code:",
    longitudinal_result.returncode,
)
print(
    "Expected rows:",
    36580,
)
print(
    "Canonical columns:",
    16,
)
print(
    "Output:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/longitudinal_grid_year/"
    "all_epochs",
)
print(
    "Remote log:",
    REMOTE_LOG,
)

LOCAL PYTHON COMPILE CHECK: PASS
Expected program SHA-256: 14d3840fd8dbde05b5b3e172ed2fe7fc7c0f6476df0e8458b1c6c7965abf7e3f
Remote program SHA-256: 14d3840fd8dbde05b5b3e172ed2fe7fc7c0f6476df0e8458b1c6c7965abf7e3f
REMOTE CHECKSUM CHECK: PASS
REMOTE PYTHON COMPILE CHECK: PASS

HDFS readiness:
PASS: NameNode
PASS: DataNode
PASS: SecondaryNameNode
PASS: HDFS safe mode is OFF

RUNNING CANONICAL LONGITUDINAL JOB
26/11/07 13:10:55 INFO SparkContext: Running Spark version 3.5.9
26/11/07 13:10:55 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/07 13:10:55 INFO SparkContext: Java version 11.0.32.1
26/11/07 13:10:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 13:10:56 INFO ResourceUtils: ==============================================================
26/11/07 13:10:56 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/07 13:10:56 INFO ResourceUtils: ======================

We now have a single canonical Bengaluru longitudinal dataset in HDFS:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/
longitudinal_grid_year/all_epochs
Validated structure
Plain Text
Epochs: 10
Grid cells per epoch: 3,658
Total grid-year rows: 36,580
Canonical columns: 16
Distinct grid cells: 3,658
Duplicate grid-year keys: 0
Duplicate coordinate-year keys: 0
Null keys: 0
Invalid core values: 0
Runtime: 28.52 seconds

The dataset was successfully written to HDFS and then read back and independently validated.

What one row represents

Each row represents:

Plain Text
One Bengaluru 1 km grid cell
×
One historical epoch

The canonical fields include:

Plain Text
Grid identity and coordinates
Epoch
Population total
SMOD class
Total built surface
Residential built surface
Non-residential built surface
Geometry WKB
Geometry CRS
Current completed milestones
Plain Text
Cell 8
Final inventory and structural validation
40 of 40 outputs passed

Cell 9
Canonical longitudinal dataset
36,580 of 36,580 rows validated

We will generate an epoch-level comparison summary, with one row for each epoch:

Total population
Population change from the preceding epoch
Population growth percentage
Total built surface in m² and km²
Residential and non-residential built surface
Built-surface change and growth percentage
Population per built hectare
Populated and built grid-cell counts
Counts and shares by SMOD class

In [17]:
# ============================================================
# CELL 10: BUILD BENGALURU EPOCH-LEVEL COMPARISON SUMMARY
#
# Input:
#   Canonical longitudinal HDFS dataset from Cell 9
#
# Output:
#   /data/ghsl/curated/pilots/bengaluru/
#   historical_summaries/epoch_summary
#
# Expected:
#   10 rows, one per epoch
#
# Metrics:
#   - Total population
#   - Five-year population change and growth
#   - Total, residential, and non-residential built surface
#   - Five-year built-surface change and growth
#   - Population per built hectare
#   - Populated and built grid-cell counts
#   - Urban and urban-centre cell counts
#   - SMOD class counts as JSON
#
# SMOD interpretation:
#   smod_class >= 21 → urban-class cell
#   smod_class == 30 → urban-centre cell
#
# Architecture:
#   Storage: HDFS
#   Processing: Apache Spark
#   Format: Parquet
#
# This cell does not rerun raster or epoch processing.
# ============================================================

import base64
import hashlib
import textwrap

REMOTE_PROGRAM = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_epoch_summary.py"
)

REMOTE_LOG = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_epoch_summary.log"
)

program_code = textwrap.dedent(
    """
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    import time


    # ========================================================
    # CONFIGURATION
    # ========================================================

    EPOCHS = [
        1975,
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
        2020,
    ]

    EXPECTED_INPUT_ROWS = 36580
    EXPECTED_CELLS_PER_EPOCH = 3658
    EXPECTED_SUMMARY_ROWS = 10

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    INPUT_PATH = (
        f"{HDFS_BASE}/longitudinal_grid_year/"
        "all_epochs"
    )

    OUTPUT_PATH = (
        f"{HDFS_BASE}/historical_summaries/"
        "epoch_summary"
    )


    # ========================================================
    # SPARK SESSION
    # ========================================================

    spark = (
        SparkSession.builder
        .appName(
            "GHSL_Bengaluru_Epoch_Comparison_Summary"
        )
        .config("spark.ui.enabled", "false")
        .config(
            "spark.sql.shuffle.partitions",
            "4",
        )
        .config(
            "spark.default.parallelism",
            "2",
        )
        .config(
            "spark.sql.adaptive.enabled",
            "false",
        )
        .config(
            "spark.sql.codegen.wholeStage",
            "false",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")


    # ========================================================
    # HDFS ACCESS
    # ========================================================

    jvm = spark.sparkContext._jvm

    hadoop_conf = (
        spark.sparkContext
        ._jsc
        .hadoopConfiguration()
    )

    hdfs = (
        jvm.org.apache.hadoop.fs.FileSystem.get(
            hadoop_conf
        )
    )

    HadoopPath = jvm.org.apache.hadoop.fs.Path


    def hdfs_exists(path):
        return bool(
            hdfs.exists(
                HadoopPath(path)
            )
        )


    def require(condition, message):
        if not condition:
            raise RuntimeError(message)


    # ========================================================
    # READ AND VALIDATE LONGITUDINAL INPUT
    # ========================================================

    started = time.time()

    require(
        hdfs_exists(INPUT_PATH),
        f"Input HDFS path is missing: {INPUT_PATH}",
    )

    longitudinal_df = spark.read.parquet(
        INPUT_PATH
    )

    input_rows = longitudinal_df.count()

    input_epochs = sorted(
        int(row["epoch"])
        for row in (
            longitudinal_df
            .select("epoch")
            .distinct()
            .collect()
        )
    )

    distinct_grid_years = (
        longitudinal_df
        .select(
            "grid_cell_id",
            "epoch",
        )
        .distinct()
        .count()
    )

    require(
        input_rows == EXPECTED_INPUT_ROWS,
        (
            f"Input rows={input_rows}; expected "
            f"{EXPECTED_INPUT_ROWS}"
        ),
    )

    require(
        input_epochs == EPOCHS,
        (
            f"Input epochs={input_epochs}; "
            f"expected {EPOCHS}"
        ),
    )

    require(
        distinct_grid_years
        == EXPECTED_INPUT_ROWS,
        (
            "Input contains duplicate grid-year keys: "
            f"distinct={distinct_grid_years}, "
            f"rows={input_rows}"
        ),
    )

    print("=" * 104)
    print("LONGITUDINAL INPUT VALIDATION")
    print("=" * 104)
    print("Input rows:", input_rows)
    print("Input epochs:", input_epochs)
    print(
        "Distinct grid-year keys:",
        distinct_grid_years,
    )
    print("Input validation: PASS")


    # ========================================================
    # BUILD SMOD CLASS COUNTS
    #
    # Example JSON:
    # {"10":1200,"11":250,"21":900,"30":100}
    # ========================================================

    smod_counts_df = (
        longitudinal_df
        .groupBy(
            "epoch",
            "smod_class",
        )
        .agg(
            F.count("*").alias(
                "smod_cell_count"
            )
        )
    )

    smod_json_df = (
        smod_counts_df
        .groupBy("epoch")
        .agg(
            F.to_json(
                F.map_from_entries(
                    F.sort_array(
                        F.collect_list(
                            F.struct(
                                F.col(
                                    "smod_class"
                                )
                                .cast("string")
                                .alias("key"),
                                F.col(
                                    "smod_cell_count"
                                )
                                .cast("long")
                                .alias("value"),
                            )
                        )
                    )
                )
            ).alias(
                "smod_class_counts_json"
            ),
            F.countDistinct(
                "smod_class"
            ).alias(
                "distinct_smod_classes"
            ),
        )
    )


    # ========================================================
    # BUILD BASE EPOCH SUMMARY
    # ========================================================

    base_summary_df = (
        longitudinal_df
        .groupBy(
            "city",
            "epoch",
        )
        .agg(
            F.count("*").alias(
                "grid_cell_count"
            ),
            F.sum(
                "population_total"
            ).alias(
                "population_total"
            ),
            F.sum(
                "built_surface_total_m2"
            ).alias(
                "built_surface_total_m2"
            ),
            F.sum(
                "built_surface_residential_m2"
            ).alias(
                "built_surface_residential_m2"
            ),
            F.sum(
                "built_surface_nres_m2"
            ).alias(
                "built_surface_nres_m2"
            ),
            F.sum(
                F.when(
                    F.col(
                        "population_total"
                    ) > F.lit(0),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).cast("long").alias(
                "populated_grid_cells"
            ),
            F.sum(
                F.when(
                    F.col(
                        "built_surface_total_m2"
                    ) > F.lit(0),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).cast("long").alias(
                "built_grid_cells"
            ),
            F.sum(
                F.when(
                    F.col("smod_class")
                    >= F.lit(21),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).cast("long").alias(
                "urban_class_grid_cells"
            ),
            F.sum(
                F.when(
                    F.col("smod_class")
                    == F.lit(30),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).cast("long").alias(
                "urban_centre_grid_cells"
            ),
            F.avg(
                "population_total"
            ).alias(
                "mean_population_per_grid_cell"
            ),
            F.max(
                "population_total"
            ).alias(
                "maximum_population_grid_cell"
            ),
            F.avg(
                "built_surface_total_m2"
            ).alias(
                "mean_built_surface_m2_per_grid_cell"
            ),
            F.max(
                "built_surface_total_m2"
            ).alias(
                "maximum_built_surface_m2_grid_cell"
            ),
        )
        .join(
            smod_json_df,
            on="epoch",
            how="left",
        )
    )


    # ========================================================
    # DERIVED EPOCH METRICS
    # ========================================================

    summary_df = (
        base_summary_df
        .withColumn(
            "built_surface_total_km2",
            F.col(
                "built_surface_total_m2"
            )
            / F.lit(1_000_000.0),
        )
        .withColumn(
            "built_surface_residential_km2",
            F.col(
                "built_surface_residential_m2"
            )
            / F.lit(1_000_000.0),
        )
        .withColumn(
            "built_surface_nres_km2",
            F.col(
                "built_surface_nres_m2"
            )
            / F.lit(1_000_000.0),
        )
        .withColumn(
            "population_per_built_hectare",
            F.when(
                F.col(
                    "built_surface_total_m2"
                ) > F.lit(0),
                F.col("population_total")
                / (
                    F.col(
                        "built_surface_total_m2"
                    )
                    / F.lit(10_000.0)
                ),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "residential_share_of_built_percent",
            F.when(
                F.col(
                    "built_surface_total_m2"
                ) > F.lit(0),
                (
                    F.col(
                        "built_surface_residential_m2"
                    )
                    / F.col(
                        "built_surface_total_m2"
                    )
                )
                * F.lit(100.0),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "urban_class_cell_share_percent",
            (
                F.col(
                    "urban_class_grid_cells"
                )
                / F.col(
                    "grid_cell_count"
                )
            )
            * F.lit(100.0),
        )
        .withColumn(
            "urban_centre_cell_share_percent",
            (
                F.col(
                    "urban_centre_grid_cells"
                )
                / F.col(
                    "grid_cell_count"
                )
            )
            * F.lit(100.0),
        )
    )


    # ========================================================
    # FIVE-YEAR CHANGE USING SPARK WINDOW FUNCTIONS
    # ========================================================

    epoch_window = Window.orderBy(
        "epoch"
    )

    summary_df = (
        summary_df
        .withColumn(
            "previous_epoch",
            F.lag("epoch").over(
                epoch_window
            ),
        )
        .withColumn(
            "previous_population_total",
            F.lag(
                "population_total"
            ).over(
                epoch_window
            ),
        )
        .withColumn(
            "previous_built_surface_total_m2",
            F.lag(
                "built_surface_total_m2"
            ).over(
                epoch_window
            ),
        )
        .withColumn(
            "population_change_from_previous",
            F.when(
                F.col(
                    "previous_population_total"
                ).isNotNull(),
                F.col("population_total")
                - F.col(
                    "previous_population_total"
                ),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "population_growth_percent",
            F.when(
                F.col(
                    "previous_population_total"
                ) > F.lit(0),
                (
                    F.col(
                        "population_change_from_previous"
                    )
                    / F.col(
                        "previous_population_total"
                    )
                )
                * F.lit(100.0),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "built_surface_change_m2",
            F.when(
                F.col(
                    "previous_built_surface_total_m2"
                ).isNotNull(),
                F.col(
                    "built_surface_total_m2"
                )
                - F.col(
                    "previous_built_surface_total_m2"
                ),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "built_surface_growth_percent",
            F.when(
                F.col(
                    "previous_built_surface_total_m2"
                ) > F.lit(0),
                (
                    F.col(
                        "built_surface_change_m2"
                    )
                    / F.col(
                        "previous_built_surface_total_m2"
                    )
                )
                * F.lit(100.0),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "built_surface_change_km2",
            F.col(
                "built_surface_change_m2"
            )
            / F.lit(1_000_000.0),
        )
    )


    # ========================================================
    # FINAL COLUMN ORDER
    # ========================================================

    final_columns = [
        "city",
        "epoch",
        "previous_epoch",
        "grid_cell_count",
        "population_total",
        "previous_population_total",
        "population_change_from_previous",
        "population_growth_percent",
        "populated_grid_cells",
        "mean_population_per_grid_cell",
        "maximum_population_grid_cell",
        "built_surface_total_m2",
        "built_surface_total_km2",
        "previous_built_surface_total_m2",
        "built_surface_change_m2",
        "built_surface_change_km2",
        "built_surface_growth_percent",
        "built_surface_residential_m2",
        "built_surface_residential_km2",
        "built_surface_nres_m2",
        "built_surface_nres_km2",
        "residential_share_of_built_percent",
        "population_per_built_hectare",
        "built_grid_cells",
        "mean_built_surface_m2_per_grid_cell",
        "maximum_built_surface_m2_grid_cell",
        "urban_class_grid_cells",
        "urban_class_cell_share_percent",
        "urban_centre_grid_cells",
        "urban_centre_cell_share_percent",
        "distinct_smod_classes",
        "smod_class_counts_json",
    ]

    summary_df = (
        summary_df
        .select(*final_columns)
        .orderBy("epoch")
    )


    # ========================================================
    # PRE-WRITE VALIDATION
    # ========================================================

    summary_rows = summary_df.count()

    summary_epochs = [
        int(row["epoch"])
        for row in (
            summary_df
            .select("epoch")
            .orderBy("epoch")
            .collect()
        )
    ]

    duplicate_epochs = (
        summary_rows
        - summary_df
        .select("epoch")
        .distinct()
        .count()
    )

    invalid_summary_rows = summary_df.filter(
        F.col("population_total").isNull()
        | F.isnan("population_total")
        | (F.col("population_total") < 0)
        | F.col(
            "built_surface_total_m2"
        ).isNull()
        | F.isnan(
            "built_surface_total_m2"
        )
        | (
            F.col(
                "built_surface_total_m2"
            ) < 0
        )
        | (
            F.col("grid_cell_count")
            != F.lit(
                EXPECTED_CELLS_PER_EPOCH
            )
        )
        | F.col(
            "smod_class_counts_json"
        ).isNull()
    ).count()

    require(
        summary_rows
        == EXPECTED_SUMMARY_ROWS,
        (
            f"Summary rows={summary_rows}; expected "
            f"{EXPECTED_SUMMARY_ROWS}"
        ),
    )

    require(
        summary_epochs == EPOCHS,
        (
            f"Summary epochs={summary_epochs}; "
            f"expected {EPOCHS}"
        ),
    )

    require(
        duplicate_epochs == 0,
        (
            f"Duplicate summary epochs="
            f"{duplicate_epochs}"
        ),
    )

    require(
        invalid_summary_rows == 0,
        (
            f"Invalid summary rows="
            f"{invalid_summary_rows}"
        ),
    )

    first_epoch_change_values = (
        summary_df
        .filter(
            F.col("epoch") == F.lit(1975)
        )
        .select(
            "previous_epoch",
            "population_change_from_previous",
            "population_growth_percent",
            "built_surface_change_m2",
            "built_surface_growth_percent",
        )
        .first()
    )

    require(
        all(
            value is None
            for value in first_epoch_change_values
        ),
        (
            "1975 previous-epoch metrics should "
            "all be null"
        ),
    )

    later_null_changes = summary_df.filter(
        (F.col("epoch") > F.lit(1975))
        & (
            F.col("previous_epoch").isNull()
            | F.col(
                "population_change_from_previous"
            ).isNull()
            | F.col(
                "population_growth_percent"
            ).isNull()
            | F.col(
                "built_surface_change_m2"
            ).isNull()
            | F.col(
                "built_surface_growth_percent"
            ).isNull()
        )
    ).count()

    require(
        later_null_changes == 0,
        (
            "One or more post-1975 epochs have "
            "missing change metrics"
        ),
    )

    print()
    print("=" * 104)
    print("EPOCH SUMMARY PRE-WRITE VALIDATION")
    print("=" * 104)
    print("Rows:", summary_rows)
    print("Epochs:", summary_epochs)
    print("Duplicate epochs:", duplicate_epochs)
    print("Invalid rows:", invalid_summary_rows)
    print("Pre-write validation: PASS")


    # ========================================================
    # DISPLAY SUMMARY
    # ========================================================

    print()
    print("=" * 104)
    print("BENGALURU EPOCH COMPARISON")
    print("=" * 104)

    summary_df.select(
        "epoch",
        F.round(
            "population_total",
            2,
        ).alias(
            "population_total"
        ),
        F.round(
            "population_change_from_previous",
            2,
        ).alias(
            "population_change"
        ),
        F.round(
            "population_growth_percent",
            2,
        ).alias(
            "population_growth_pct"
        ),
        F.round(
            "built_surface_total_km2",
            3,
        ).alias(
            "built_km2"
        ),
        F.round(
            "built_surface_change_km2",
            3,
        ).alias(
            "built_change_km2"
        ),
        F.round(
            "built_surface_growth_percent",
            2,
        ).alias(
            "built_growth_pct"
        ),
        F.round(
            "population_per_built_hectare",
            2,
        ).alias(
            "people_per_built_hectare"
        ),
        "populated_grid_cells",
        "built_grid_cells",
        "urban_class_grid_cells",
        "urban_centre_grid_cells",
    ).show(
        20,
        truncate=False,
    )


    # ========================================================
    # WRITE TO HDFS
    # ========================================================

    print()
    print("=" * 104)
    print("WRITING EPOCH SUMMARY TO HDFS")
    print("=" * 104)

    (
        summary_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(OUTPUT_PATH)
    )

    require(
        hdfs_exists(OUTPUT_PATH),
        (
            "Summary output path is missing "
            "after write"
        ),
    )

    print("HDFS write: COMPLETE")
    print("Output:", OUTPUT_PATH)


    # ========================================================
    # HDFS READ-BACK VALIDATION
    # ========================================================

    saved_df = (
        spark.read
        .parquet(OUTPUT_PATH)
        .orderBy("epoch")
    )

    saved_rows = saved_df.count()

    saved_epochs = [
        int(row["epoch"])
        for row in (
            saved_df
            .select("epoch")
            .orderBy("epoch")
            .collect()
        )
    ]

    saved_duplicate_epochs = (
        saved_rows
        - saved_df
        .select("epoch")
        .distinct()
        .count()
    )

    require(
        saved_rows == EXPECTED_SUMMARY_ROWS,
        (
            f"Read-back rows={saved_rows}; "
            f"expected={EXPECTED_SUMMARY_ROWS}"
        ),
    )

    require(
        saved_epochs == EPOCHS,
        (
            f"Read-back epochs={saved_epochs}; "
            f"expected={EPOCHS}"
        ),
    )

    require(
        saved_duplicate_epochs == 0,
        (
            f"Read-back duplicate epochs="
            f"{saved_duplicate_epochs}"
        ),
    )

    elapsed_seconds = round(
        time.time() - started,
        2,
    )


    # ========================================================
    # FINAL REPORT
    # ========================================================

    first_summary = saved_df.first()

    last_summary = (
        saved_df
        .orderBy(
            F.col("epoch").desc()
        )
        .first()
    )

    total_population_change = (
        float(
            last_summary[
                "population_total"
            ]
        )
        - float(
            first_summary[
                "population_total"
            ]
        )
    )

    total_built_change_km2 = (
        float(
            last_summary[
                "built_surface_total_km2"
            ]
        )
        - float(
            first_summary[
                "built_surface_total_km2"
            ]
        )
    )

    print()
    print("=" * 104)
    print("EPOCH SUMMARY RESULT")
    print("=" * 104)
    print("Summary rows:", saved_rows)
    print("Epochs:", saved_epochs)
    print(
        "Duplicate epoch rows:",
        saved_duplicate_epochs,
    )
    print(
        "1975 population:",
        first_summary["population_total"],
    )
    print(
        "2020 population:",
        last_summary["population_total"],
    )
    print(
        "1975-2020 population change:",
        total_population_change,
    )
    print(
        "1975 built surface km2:",
        first_summary[
            "built_surface_total_km2"
        ],
    )
    print(
        "2020 built surface km2:",
        last_summary[
            "built_surface_total_km2"
        ],
    )
    print(
        "1975-2020 built change km2:",
        total_built_change_km2,
    )
    print(
        "Elapsed seconds:",
        elapsed_seconds,
    )
    print("Output:", OUTPUT_PATH)
    print()
    print("EPOCH SUMMARY: SUCCESS")

    spark.stop()
    """
).strip() + "\n"


# ============================================================
# COMPILE EXACT SOURCE BEFORE TRANSFER
# ============================================================

compile(
    program_code,
    "build_epoch_summary.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    program_code.encode("utf-8")
).hexdigest()

print("LOCAL PYTHON COMPILE CHECK: PASS")
print("Expected program SHA-256:", local_sha256)


# ============================================================
# TRANSFER EXACT SOURCE AND RUN REMOTELY
# ============================================================

encoded_program = base64.b64encode(
    program_code.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"
LOG="{REMOTE_LOG}"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_program}" \
| base64 -d \
> "${{PROGRAM}}"

ACTUAL_SHA256="$(
    sha256sum "${{PROGRAM}}" \
    | awk '{{print $1}}'
)"

echo "Remote program SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Program checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM CHECK: PASS"

python3 -m py_compile "${{PROGRAM}}"

echo "REMOTE PYTHON COMPILE CHECK: PASS"

echo
echo "HDFS readiness:"

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
    | awk '{{print $2}}' \
    | grep -qx "${{daemon}}" \
    || {{
        echo "ERROR: ${{daemon}} is not running."
        exit 21
    }}

    echo "PASS: ${{daemon}}"
done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {{
    echo "ERROR: HDFS safe mode is enabled."
    exit 22
}}

echo "PASS: HDFS safe mode is OFF"

echo
echo "RUNNING EPOCH-SUMMARY JOB"

spark-submit \
    --master 'local[1]' \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    "${{PROGRAM}}" \
    2>&1 \
    | tee "${{LOG}}"

status=${{PIPESTATUS[0]}}

echo
echo "EPOCH_SUMMARY_EXIT_CODE=${{status}}"
echo "Job log: ${{LOG}}"

exit "${{status}}"
"""

summary_result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 10 RESULT")
print("=" * 80)

print(
    "Return code:",
    summary_result.returncode,
)

print(
    "Expected summary rows:",
    10,
)

print(
    "Output:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/historical_summaries/"
    "epoch_summary",
)

print(
    "Remote log:",
    REMOTE_LOG,
)

LOCAL PYTHON COMPILE CHECK: PASS
Expected program SHA-256: 3abf77c24d977770a5811d3eb2a5c11f685f9220091988c97fec84fb8f01376b
Remote program SHA-256: 3abf77c24d977770a5811d3eb2a5c11f685f9220091988c97fec84fb8f01376b
REMOTE CHECKSUM CHECK: PASS
REMOTE PYTHON COMPILE CHECK: PASS

HDFS readiness:
PASS: NameNode
PASS: DataNode
PASS: SecondaryNameNode
PASS: HDFS safe mode is OFF

RUNNING EPOCH-SUMMARY JOB
26/11/07 13:15:33 INFO SparkContext: Running Spark version 3.5.9
26/11/07 13:15:33 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/07 13:15:33 INFO SparkContext: Java version 11.0.32.1
26/11/07 13:15:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 13:15:33 INFO ResourceUtils: ==============================================================
26/11/07 13:15:33 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/07 13:15:33 INFO ResourceUtils: ===============================

Main historical findings
Metric	1975	2020	Change
Population	3.06 million	14.61 million	+11.55 million
Built surface	67.09 km²	342.43 km²	+275.34 km²
Urban-class grid cells	398	2,011	+1,613
Urban-centre grid cells	202	950	+748
Populated grid cells	3,249	3,601	+352
Built grid cells	3,249	3,601	+352
Population trend

Population increased approximately 4.77 times:

Plain Text
14.61 million / 3.06 million ≈ 4.77

The largest absolute five-year increase occurred from 2015 to 2020:

Plain Text
Population increase: approximately 2.27 million
Five-year growth: 18.41%
Built-surface trend

Built surface increased approximately 5.10 times:

Plain Text
342.43 km² / 67.09 km² ≈ 5.10


The largest absolute five-year expansion occurred from 2010 to 2015:

Plain Text
Built-surface increase: approximately 52.78 km²
Five-year growth: 20.86%

The highest percentage expansion occurred from 1990 to 1995:

Plain Text
Built-surface increase: 30.70 km²
Five-year growth: 32.06%
Density interpretation

Population per built hectare changed as follows:

Plain Text
1975: 456.18 people per built hectare
1990: 561.82 people per built hectare
2015: 403.51 people per built hectare
2020: 426.63 people per built hectare

This suggests:

Densification dominated through approximately 1990.
Rapid outward built expansion accelerated during 1990–2015.
Population growth again slightly outpaced built expansion during 2015–2020.

Next step: grid-level population-growth comparisons

Cell 11 should calculate changes for every grid cell between consecutive epochs:

Plain Text
1975→1980
1980→1985
...
2015→2020

Expected size:

Plain Text
3,658 grid cells × 9 transitions
= 32,922 transition rows

For each transition, we should calculate:

Previous and current population
Absolute population change
Percentage change where the previous population is positive
Annualized growth rate
Growth category
Previous and current SMOD classes
Previous and current built surface
Grid geometry

In [18]:
# ============================================================
# CELL 11: GRID-LEVEL POPULATION-GROWTH TRANSITIONS
#
# Input:
#   /data/ghsl/curated/pilots/bengaluru/
#   longitudinal_grid_year/all_epochs
#
# Output:
#   /data/ghsl/curated/pilots/bengaluru/
#   historical_comparisons/population_growth_transitions
#
# Expected:
#   3,658 grid cells x 9 consecutive transitions
#   = 32,922 transition rows
#
# Each row represents:
#   One 1 km Bengaluru grid cell
#   across one consecutive five-year transition
#
# Metrics:
#   - Population before and after
#   - Absolute population change
#   - Percentage population change
#   - Annualized population growth rate
#   - Population-growth category
#   - Built surface before and after
#   - Built-surface change
#   - SMOD class before and after
#   - SMOD transition label
#   - Geometry and grid coordinates
#
# Architecture:
#   Storage: HDFS
#   Processing: Apache Spark window functions
#   Format: Parquet partitioned by to_epoch
#
# This cell does not rerun raster processing.
# ============================================================

import base64
import hashlib
import textwrap

REMOTE_PROGRAM = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_population_growth_transitions.py"
)

REMOTE_LOG = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_population_growth_transitions.log"
)

program_code = textwrap.dedent(
    """
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    import time


    # ========================================================
    # CONFIGURATION
    # ========================================================

    EPOCHS = [
        1975,
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
        2020,
    ]

    EXPECTED_GRID_CELLS = 3658
    EXPECTED_TRANSITIONS_PER_CELL = 9

    EXPECTED_TRANSITION_ROWS = (
        EXPECTED_GRID_CELLS
        * EXPECTED_TRANSITIONS_PER_CELL
    )

    EXPECTED_TO_EPOCHS = EPOCHS[1:]

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    INPUT_PATH = (
        f"{HDFS_BASE}/longitudinal_grid_year/"
        "all_epochs"
    )

    OUTPUT_PATH = (
        f"{HDFS_BASE}/historical_comparisons/"
        "population_growth_transitions"
    )


    # ========================================================
    # SPARK SESSION
    # ========================================================

    spark = (
        SparkSession.builder
        .appName(
            "GHSL_Bengaluru_Population_Growth_Transitions"
        )
        .config("spark.ui.enabled", "false")
        .config(
            "spark.sql.shuffle.partitions",
            "4",
        )
        .config(
            "spark.default.parallelism",
            "2",
        )
        .config(
            "spark.sql.adaptive.enabled",
            "false",
        )
        .config(
            "spark.sql.codegen.wholeStage",
            "false",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")


    # ========================================================
    # HDFS ACCESS
    # ========================================================

    jvm = spark.sparkContext._jvm

    hadoop_conf = (
        spark.sparkContext
        ._jsc
        .hadoopConfiguration()
    )

    hdfs = (
        jvm.org.apache.hadoop.fs.FileSystem.get(
            hadoop_conf
        )
    )

    HadoopPath = jvm.org.apache.hadoop.fs.Path


    def hdfs_exists(path):
        return bool(
            hdfs.exists(
                HadoopPath(path)
            )
        )


    def require(condition, message):
        if not condition:
            raise RuntimeError(message)


    # ========================================================
    # READ AND VALIDATE LONGITUDINAL INPUT
    # ========================================================

    started = time.time()

    require(
        hdfs_exists(INPUT_PATH),
        f"Input HDFS path is missing: {INPUT_PATH}",
    )

    longitudinal_df = spark.read.parquet(
        INPUT_PATH
    )

    required_columns = {
        "city",
        "epoch",
        "grid_cell_id",
        "grid_1km_col",
        "grid_1km_row",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "population_total",
        "smod_class",
        "built_surface_total_m2",
        "built_surface_residential_m2",
        "built_surface_nres_m2",
        "geometry_wkb",
        "geometry_crs",
    }

    missing_columns = sorted(
        required_columns
        - set(longitudinal_df.columns)
    )

    require(
        not missing_columns,
        (
            "Longitudinal input is missing columns: "
            + ",".join(missing_columns)
        ),
    )

    input_rows = longitudinal_df.count()

    input_epochs = sorted(
        int(row["epoch"])
        for row in (
            longitudinal_df
            .select("epoch")
            .distinct()
            .collect()
        )
    )

    distinct_grid_cells = (
        longitudinal_df
        .select("grid_cell_id")
        .distinct()
        .count()
    )

    distinct_grid_years = (
        longitudinal_df
        .select(
            "grid_cell_id",
            "epoch",
        )
        .distinct()
        .count()
    )

    require(
        input_rows
        == EXPECTED_GRID_CELLS * len(EPOCHS),
        (
            f"Input rows={input_rows}; expected "
            f"{EXPECTED_GRID_CELLS * len(EPOCHS)}"
        ),
    )

    require(
        input_epochs == EPOCHS,
        (
            f"Input epochs={input_epochs}; "
            f"expected={EPOCHS}"
        ),
    )

    require(
        distinct_grid_cells
        == EXPECTED_GRID_CELLS,
        (
            f"Distinct grid cells={distinct_grid_cells}; "
            f"expected={EXPECTED_GRID_CELLS}"
        ),
    )

    require(
        distinct_grid_years == input_rows,
        (
            "Duplicate grid-year rows detected: "
            f"rows={input_rows}, "
            f"distinct={distinct_grid_years}"
        ),
    )

    print("=" * 108)
    print("LONGITUDINAL INPUT VALIDATION")
    print("=" * 108)
    print("Input rows:", input_rows)
    print("Input epochs:", input_epochs)
    print("Distinct grid cells:", distinct_grid_cells)
    print("Distinct grid-year keys:", distinct_grid_years)
    print("Input validation: PASS")


    # ========================================================
    # CREATE PREVIOUS-EPOCH VALUES WITH SPARK WINDOWS
    # ========================================================

    grid_window = (
        Window
        .partitionBy("grid_cell_id")
        .orderBy("epoch")
    )

    lagged_df = (
        longitudinal_df
        .withColumn(
            "from_epoch",
            F.lag("epoch").over(grid_window),
        )
        .withColumn(
            "from_population_total",
            F.lag(
                "population_total"
            ).over(grid_window),
        )
        .withColumn(
            "from_smod_class",
            F.lag(
                "smod_class"
            ).over(grid_window),
        )
        .withColumn(
            "from_built_surface_total_m2",
            F.lag(
                "built_surface_total_m2"
            ).over(grid_window),
        )
        .withColumn(
            "from_built_surface_residential_m2",
            F.lag(
                "built_surface_residential_m2"
            ).over(grid_window),
        )
        .withColumn(
            "from_built_surface_nres_m2",
            F.lag(
                "built_surface_nres_m2"
            ).over(grid_window),
        )
        .withColumnRenamed(
            "epoch",
            "to_epoch",
        )
        .withColumnRenamed(
            "population_total",
            "to_population_total",
        )
        .withColumnRenamed(
            "smod_class",
            "to_smod_class",
        )
        .withColumnRenamed(
            "built_surface_total_m2",
            "to_built_surface_total_m2",
        )
        .withColumnRenamed(
            "built_surface_residential_m2",
            "to_built_surface_residential_m2",
        )
        .withColumnRenamed(
            "built_surface_nres_m2",
            "to_built_surface_nres_m2",
        )
        .filter(
            F.col("from_epoch").isNotNull()
        )
    )


    # ========================================================
    # CALCULATE POPULATION-GROWTH METRICS
    # ========================================================

    transition_df = (
        lagged_df
        .withColumn(
            "transition_years",
            (
                F.col("to_epoch")
                - F.col("from_epoch")
            ).cast("int"),
        )
        .withColumn(
            "transition_label",
            F.concat(
                F.col("from_epoch").cast("string"),
                F.lit("_to_"),
                F.col("to_epoch").cast("string"),
            ),
        )
        .withColumn(
            "population_change",
            F.col("to_population_total")
            - F.col("from_population_total"),
        )
        .withColumn(
            "population_change_percent",
            F.when(
                F.col(
                    "from_population_total"
                ) > F.lit(0),
                (
                    F.col("population_change")
                    / F.col(
                        "from_population_total"
                    )
                )
                * F.lit(100.0),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "population_annualized_growth_percent",
            F.when(
                (
                    F.col(
                        "from_population_total"
                    ) > F.lit(0)
                )
                & (
                    F.col(
                        "to_population_total"
                    ) >= F.lit(0)
                )
                & (
                    F.col(
                        "transition_years"
                    ) > F.lit(0)
                ),
                (
                    F.pow(
                        (
                            F.col(
                                "to_population_total"
                            )
                            / F.col(
                                "from_population_total"
                            )
                        ),
                        (
                            F.lit(1.0)
                            / F.col(
                                "transition_years"
                            )
                        ),
                    )
                    - F.lit(1.0)
                )
                * F.lit(100.0),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "population_growth_category",
            F.when(
                (
                    F.col(
                        "from_population_total"
                    ) == F.lit(0)
                )
                & (
                    F.col(
                        "to_population_total"
                    ) > F.lit(0)
                ),
                F.lit("newly_populated"),
            )
            .when(
                (
                    F.col(
                        "from_population_total"
                    ) == F.lit(0)
                )
                & (
                    F.col(
                        "to_population_total"
                    ) == F.lit(0)
                ),
                F.lit("remained_unpopulated"),
            )
            .when(
                F.col("population_change")
                < F.lit(-0.000001),
                F.lit("population_decline"),
            )
            .when(
                F.abs(
                    F.col("population_change")
                ) <= F.lit(0.000001),
                F.lit("stable_population"),
            )
            .when(
                F.col(
                    "population_change_percent"
                ) < F.lit(10.0),
                F.lit("low_growth"),
            )
            .when(
                F.col(
                    "population_change_percent"
                ) < F.lit(25.0),
                F.lit("moderate_growth"),
            )
            .when(
                F.col(
                    "population_change_percent"
                ) < F.lit(50.0),
                F.lit("high_growth"),
            )
            .otherwise(
                F.lit("very_high_growth")
            ),
        )
    )


    # ========================================================
    # CALCULATE BUILT-SURFACE AND SMOD TRANSITIONS
    # ========================================================

    transition_df = (
        transition_df
        .withColumn(
            "built_surface_change_m2",
            F.col(
                "to_built_surface_total_m2"
            )
            - F.col(
                "from_built_surface_total_m2"
            ),
        )
        .withColumn(
            "built_surface_change_km2",
            F.col(
                "built_surface_change_m2"
            )
            / F.lit(1_000_000.0),
        )
        .withColumn(
            "built_surface_change_percent",
            F.when(
                F.col(
                    "from_built_surface_total_m2"
                ) > F.lit(0),
                (
                    F.col(
                        "built_surface_change_m2"
                    )
                    / F.col(
                        "from_built_surface_total_m2"
                    )
                )
                * F.lit(100.0),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "residential_built_change_m2",
            F.col(
                "to_built_surface_residential_m2"
            )
            - F.col(
                "from_built_surface_residential_m2"
            ),
        )
        .withColumn(
            "nonresidential_built_change_m2",
            F.col(
                "to_built_surface_nres_m2"
            )
            - F.col(
                "from_built_surface_nres_m2"
            ),
        )
        .withColumn(
            "smod_transition",
            F.concat(
                F.col(
                    "from_smod_class"
                ).cast("string"),
                F.lit("_to_"),
                F.col(
                    "to_smod_class"
                ).cast("string"),
            ),
        )
        .withColumn(
            "smod_class_changed",
            (
                F.col("from_smod_class")
                != F.col("to_smod_class")
            ),
        )
        .withColumn(
            "smod_class_direction",
            F.when(
                F.col("to_smod_class")
                > F.col("from_smod_class"),
                F.lit("higher_class"),
            )
            .when(
                F.col("to_smod_class")
                < F.col("from_smod_class"),
                F.lit("lower_class"),
            )
            .otherwise(
                F.lit("unchanged")
            ),
        )
    )


    # ========================================================
    # FINAL COLUMN ORDER
    # ========================================================

    FINAL_COLUMNS = [
        "city",
        "grid_cell_id",
        "grid_1km_col",
        "grid_1km_row",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "from_epoch",
        "to_epoch",
        "transition_years",
        "transition_label",
        "from_population_total",
        "to_population_total",
        "population_change",
        "population_change_percent",
        "population_annualized_growth_percent",
        "population_growth_category",
        "from_built_surface_total_m2",
        "to_built_surface_total_m2",
        "built_surface_change_m2",
        "built_surface_change_km2",
        "built_surface_change_percent",
        "from_built_surface_residential_m2",
        "to_built_surface_residential_m2",
        "residential_built_change_m2",
        "from_built_surface_nres_m2",
        "to_built_surface_nres_m2",
        "nonresidential_built_change_m2",
        "from_smod_class",
        "to_smod_class",
        "smod_transition",
        "smod_class_changed",
        "smod_class_direction",
        "geometry_wkb",
        "geometry_crs",
    ]

    transition_df = (
        transition_df
        .select(*FINAL_COLUMNS)
    )


    # ========================================================
    # VALIDATION FUNCTION
    # ========================================================

    def validate_transitions(df, label):
        row_count = df.count()

        to_epochs = sorted(
            int(row["to_epoch"])
            for row in (
                df.select("to_epoch")
                .distinct()
                .collect()
            )
        )

        distinct_transition_keys = (
            df.select(
                "grid_cell_id",
                "from_epoch",
                "to_epoch",
            )
            .distinct()
            .count()
        )

        duplicate_transition_keys = (
            row_count
            - distinct_transition_keys
        )

        distinct_coordinate_transitions = (
            df.select(
                "grid_1km_col",
                "grid_1km_row",
                "from_epoch",
                "to_epoch",
            )
            .distinct()
            .count()
        )

        duplicate_coordinate_transitions = (
            row_count
            - distinct_coordinate_transitions
        )

        null_keys = df.filter(
            F.col("grid_cell_id").isNull()
            | F.col("grid_1km_col").isNull()
            | F.col("grid_1km_row").isNull()
            | F.col("from_epoch").isNull()
            | F.col("to_epoch").isNull()
        ).count()

        invalid_transition_years = df.filter(
            F.col("transition_years")
            != F.lit(5)
        ).count()

        nonconsecutive_transitions = df.filter(
            F.col("to_epoch")
            != (
                F.col("from_epoch")
                + F.lit(5)
            )
        ).count()

        invalid_population_rows = df.filter(
            F.col(
                "from_population_total"
            ).isNull()
            | F.isnan(
                "from_population_total"
            )
            | (
                F.col(
                    "from_population_total"
                ) < F.lit(0)
            )
            | F.col(
                "to_population_total"
            ).isNull()
            | F.isnan(
                "to_population_total"
            )
            | (
                F.col(
                    "to_population_total"
                ) < F.lit(0)
            )
            | F.col(
                "population_change"
            ).isNull()
            | F.isnan(
                "population_change"
            )
            | F.col(
                "population_growth_category"
            ).isNull()
        ).count()

        invalid_context_rows = df.filter(
            F.col(
                "from_built_surface_total_m2"
            ).isNull()
            | F.col(
                "to_built_surface_total_m2"
            ).isNull()
            | F.col(
                "from_smod_class"
            ).isNull()
            | F.col(
                "to_smod_class"
            ).isNull()
            | F.col("geometry_wkb").isNull()
            | F.col("geometry_crs").isNull()
        ).count()

        per_transition_rows = {
            (
                int(row["from_epoch"]),
                int(row["to_epoch"]),
            ): int(row["count"])
            for row in (
                df.groupBy(
                    "from_epoch",
                    "to_epoch",
                )
                .count()
                .collect()
            )
        }

        category_rows = (
            df.groupBy(
                "population_growth_category"
            )
            .count()
            .collect()
        )

        category_counts = {
            row["population_growth_category"]:
            int(row["count"])
            for row in category_rows
        }

        require(
            row_count
            == EXPECTED_TRANSITION_ROWS,
            (
                f"{label}: rows={row_count}; "
                f"expected={EXPECTED_TRANSITION_ROWS}"
            ),
        )

        require(
            to_epochs == EXPECTED_TO_EPOCHS,
            (
                f"{label}: to_epochs={to_epochs}; "
                f"expected={EXPECTED_TO_EPOCHS}"
            ),
        )

        require(
            duplicate_transition_keys == 0,
            (
                f"{label}: duplicate transition keys="
                f"{duplicate_transition_keys}"
            ),
        )

        require(
            duplicate_coordinate_transitions == 0,
            (
                f"{label}: duplicate coordinate "
                f"transitions="
                f"{duplicate_coordinate_transitions}"
            ),
        )

        require(
            null_keys == 0,
            f"{label}: null keys={null_keys}",
        )

        require(
            invalid_transition_years == 0,
            (
                f"{label}: invalid transition years="
                f"{invalid_transition_years}"
            ),
        )

        require(
            nonconsecutive_transitions == 0,
            (
                f"{label}: nonconsecutive transitions="
                f"{nonconsecutive_transitions}"
            ),
        )

        require(
            invalid_population_rows == 0,
            (
                f"{label}: invalid population rows="
                f"{invalid_population_rows}"
            ),
        )

        require(
            invalid_context_rows == 0,
            (
                f"{label}: invalid context rows="
                f"{invalid_context_rows}"
            ),
        )

        for index in range(
            len(EPOCHS) - 1
        ):
            transition_key = (
                EPOCHS[index],
                EPOCHS[index + 1],
            )

            transition_count = (
                per_transition_rows.get(
                    transition_key,
                    0,
                )
            )

            require(
                transition_count
                == EXPECTED_GRID_CELLS,
                (
                    f"{label}: transition "
                    f"{transition_key} has "
                    f"{transition_count} rows; "
                    f"expected={EXPECTED_GRID_CELLS}"
                ),
            )

        require(
            df.columns == FINAL_COLUMNS,
            (
                f"{label}: columns differ from "
                "expected transition schema"
            ),
        )

        return {
            "rows": row_count,
            "to_epochs": to_epochs,
            "duplicate_transition_keys": (
                duplicate_transition_keys
            ),
            "duplicate_coordinate_transitions": (
                duplicate_coordinate_transitions
            ),
            "null_keys": null_keys,
            "invalid_transition_years": (
                invalid_transition_years
            ),
            "nonconsecutive_transitions": (
                nonconsecutive_transitions
            ),
            "invalid_population_rows": (
                invalid_population_rows
            ),
            "invalid_context_rows": (
                invalid_context_rows
            ),
            "per_transition_rows": (
                per_transition_rows
            ),
            "category_counts": category_counts,
        }


    # ========================================================
    # PRE-WRITE VALIDATION
    # ========================================================

    print()
    print("=" * 108)
    print("POPULATION TRANSITION PRE-WRITE VALIDATION")
    print("=" * 108)

    pre_write = validate_transitions(
        transition_df,
        "PRE-WRITE",
    )

    print("Rows:", pre_write["rows"])
    print(
        "Destination epochs:",
        pre_write["to_epochs"],
    )
    print(
        "Duplicate transition keys:",
        pre_write[
            "duplicate_transition_keys"
        ],
    )
    print(
        "Invalid population rows:",
        pre_write[
            "invalid_population_rows"
        ],
    )
    print("Pre-write validation: PASS")


    # ========================================================
    # DISPLAY TRANSITION SUMMARIES
    # ========================================================

    print()
    print("=" * 108)
    print("POPULATION CHANGE BY TRANSITION")
    print("=" * 108)

    transition_summary_df = (
        transition_df
        .groupBy(
            "from_epoch",
            "to_epoch",
        )
        .agg(
            F.count("*").alias(
                "grid_cells"
            ),
            F.sum(
                "population_change"
            ).alias(
                "total_population_change"
            ),
            F.avg(
                "population_change"
            ).alias(
                "mean_population_change"
            ),
            F.expr(
                "percentile_approx("
                "population_change, 0.5)"
            ).alias(
                "median_population_change"
            ),
            F.max(
                "population_change"
            ).alias(
                "maximum_population_increase"
            ),
            F.min(
                "population_change"
            ).alias(
                "maximum_population_decline"
            ),
            F.sum(
                F.when(
                    F.col(
                        "population_change"
                    ) > F.lit(0),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).alias(
                "growing_cells"
            ),
            F.sum(
                F.when(
                    F.col(
                        "population_change"
                    ) < F.lit(0),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).alias(
                "declining_cells"
            ),
            F.sum(
                F.when(
                    F.col(
                        "population_growth_category"
                    )
                    == F.lit(
                        "newly_populated"
                    ),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).alias(
                "newly_populated_cells"
            ),
        )
        .orderBy(
            "from_epoch",
            "to_epoch",
        )
    )

    transition_summary_df.select(
        "from_epoch",
        "to_epoch",
        "grid_cells",
        F.round(
            "total_population_change",
            2,
        ).alias(
            "total_population_change"
        ),
        F.round(
            "mean_population_change",
            2,
        ).alias(
            "mean_change_per_cell"
        ),
        F.round(
            "median_population_change",
            2,
        ).alias(
            "median_change_per_cell"
        ),
        F.round(
            "maximum_population_increase",
            2,
        ).alias(
            "maximum_increase"
        ),
        F.round(
            "maximum_population_decline",
            2,
        ).alias(
            "maximum_decline"
        ),
        "growing_cells",
        "declining_cells",
        "newly_populated_cells",
    ).show(
        20,
        truncate=False,
    )

    print()
    print("=" * 108)
    print("POPULATION GROWTH CATEGORIES")
    print("=" * 108)

    (
        transition_df
        .groupBy(
            "from_epoch",
            "to_epoch",
            "population_growth_category",
        )
        .count()
        .orderBy(
            "from_epoch",
            "to_epoch",
            "population_growth_category",
        )
        .show(
            200,
            truncate=False,
        )
    )


    # ========================================================
    # WRITE TO HDFS
    # ========================================================

    print()
    print("=" * 108)
    print("WRITING POPULATION TRANSITIONS TO HDFS")
    print("=" * 108)

    (
        transition_df
        .repartition(
            9,
            "to_epoch",
        )
        .write
        .mode("overwrite")
        .partitionBy("to_epoch")
        .parquet(OUTPUT_PATH)
    )

    require(
        hdfs_exists(OUTPUT_PATH),
        (
            "Transition output path is missing "
            "after write"
        ),
    )

    print("HDFS write: COMPLETE")
    print("Output:", OUTPUT_PATH)


    # ========================================================
    # HDFS READ-BACK VALIDATION
    # ========================================================

    print()
    print("=" * 108)
    print("HDFS READ-BACK VALIDATION")
    print("=" * 108)

    saved_df = (
        spark.read
        .option(
            "basePath",
            OUTPUT_PATH,
        )
        .parquet(OUTPUT_PATH)
        .select(*FINAL_COLUMNS)
    )

    read_back = validate_transitions(
        saved_df,
        "READ-BACK",
    )

    elapsed_seconds = round(
        time.time() - started,
        2,
    )

    print("Read-back validation: PASS")
    print("Rows:", read_back["rows"])
    print(
        "Destination epochs:",
        read_back["to_epochs"],
    )
    print(
        "Duplicate transition keys:",
        read_back[
            "duplicate_transition_keys"
        ],
    )
    print(
        "Duplicate coordinate transitions:",
        read_back[
            "duplicate_coordinate_transitions"
        ],
    )
    print(
        "Null keys:",
        read_back["null_keys"],
    )
    print(
        "Invalid transition years:",
        read_back[
            "invalid_transition_years"
        ],
    )
    print(
        "Nonconsecutive transitions:",
        read_back[
            "nonconsecutive_transitions"
        ],
    )
    print(
        "Invalid population rows:",
        read_back[
            "invalid_population_rows"
        ],
    )
    print(
        "Invalid context rows:",
        read_back[
            "invalid_context_rows"
        ],
    )


    # ========================================================
    # FINAL REPORT
    # ========================================================

    print()
    print("=" * 108)
    print("POPULATION-GROWTH TRANSITION RESULT")
    print("=" * 108)

    print(
        "Grid cells:",
        EXPECTED_GRID_CELLS,
    )

    print(
        "Transitions per grid cell:",
        EXPECTED_TRANSITIONS_PER_CELL,
    )

    print(
        "Expected transition rows:",
        EXPECTED_TRANSITION_ROWS,
    )

    print(
        "Validated transition rows:",
        read_back["rows"],
    )

    print(
        "Transition periods:",
        len(
            read_back["per_transition_rows"]
        ),
    )

    print(
        "Growth categories:",
        sorted(
            read_back[
                "category_counts"
            ].keys()
        ),
    )

    print(
        "Elapsed seconds:",
        elapsed_seconds,
    )

    print(
        "Output:",
        OUTPUT_PATH,
    )

    print()
    print(
        "POPULATION-GROWTH TRANSITIONS: SUCCESS"
    )

    spark.stop()
    """
).strip() + "\n"


# ============================================================
# COMPILE EXACT SOURCE BEFORE TRANSFER
# ============================================================

compile(
    program_code,
    "build_population_growth_transitions.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    program_code.encode("utf-8")
).hexdigest()

print("LOCAL PYTHON COMPILE CHECK: PASS")
print("Expected program SHA-256:", local_sha256)


# ============================================================
# TRANSFER EXACT SOURCE AND RUN REMOTELY
# ============================================================

encoded_program = base64.b64encode(
    program_code.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"
LOG="{REMOTE_LOG}"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_program}" \
| base64 -d \
> "${{PROGRAM}}"

ACTUAL_SHA256="$(
    sha256sum "${{PROGRAM}}" \
    | awk '{{print $1}}'
)"

echo "Remote program SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Program checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM CHECK: PASS"

python3 -m py_compile "${{PROGRAM}}"

echo "REMOTE PYTHON COMPILE CHECK: PASS"

echo
echo "HDFS readiness:"

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
    | awk '{{print $2}}' \
    | grep -qx "${{daemon}}" \
    || {{
        echo "ERROR: ${{daemon}} is not running."
        exit 21
    }}

    echo "PASS: ${{daemon}}"
done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {{
    echo "ERROR: HDFS safe mode is enabled."
    exit 22
}}

echo "PASS: HDFS safe mode is OFF"

echo
echo "RUNNING POPULATION-GROWTH TRANSITION JOB"

spark-submit \
    --master 'local[1]' \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    "${{PROGRAM}}" \
    2>&1 \
    | tee "${{LOG}}"

status=${{PIPESTATUS[0]}}

echo
echo "POPULATION_TRANSITIONS_EXIT_CODE=${{status}}"
echo "Job log: ${{LOG}}"

exit "${{status}}"
"""

transition_result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 11 RESULT")
print("=" * 80)

print(
    "Return code:",
    transition_result.returncode,
)

print(
    "Expected transition rows:",
    32922,
)

print(
    "Output:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/historical_comparisons/"
    "population_growth_transitions",
)

print(
    "Remote log:",
    REMOTE_LOG,
)

LOCAL PYTHON COMPILE CHECK: PASS
Expected program SHA-256: 050d0fd28fe85f156f72db6e0eb9a1022a3a3ecae4937e17f9c3c2f9b51ebdaf
Remote program SHA-256: 050d0fd28fe85f156f72db6e0eb9a1022a3a3ecae4937e17f9c3c2f9b51ebdaf
REMOTE CHECKSUM CHECK: PASS
REMOTE PYTHON COMPILE CHECK: PASS

HDFS readiness:
PASS: NameNode
PASS: DataNode
PASS: SecondaryNameNode
PASS: HDFS safe mode is OFF

RUNNING POPULATION-GROWTH TRANSITION JOB
26/11/07 13:21:33 INFO SparkContext: Running Spark version 3.5.9
26/11/07 13:21:33 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/07 13:21:33 INFO SparkContext: Java version 11.0.32.1
26/11/07 13:21:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 13:21:33 INFO ResourceUtils: ==============================================================
26/11/07 13:21:33 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/07 13:21:33 INFO ResourceUtils: ================

The grid-level population-growth transition dataset is now stored in HDFS:

Plain Text
/data/ghsl/curated/pilots/bengaluru/
historical_comparisons/population_growth_transitions

Main findings

Population growth accelerated substantially across the study period:

Transition	Total increase	Mean per cell	Growing cells	Declining cells
1975→1980	854,057	233	2,251	1,169
1980→1985	708,415	194	2,363	1,094
1985→1990	758,065	207	2,481	997
1990→1995	876,095	240	1,900	1,625
1995→2000	977,774	267	2,143	1,396
2000→2005	1,409,572	385	2,777	817
2005→2010	1,706,020	466	2,816	784
2010→2015	1,987,398	543	2,962	639
2015→2020	2,271,170	621	2,948	653
Key interpretation
The strongest total population increase occurred during 2015–2020, at approximately 2.27 million.
The median grid-cell increase rose from approximately 1 person during 1975–1980 to 35 people during 2015–2020.
The number of newly populated cells declined from 171 in 1975–1980 to zero in 2015–2020.
This suggests that later growth increasingly occurred within grid cells that were already populated, rather than through first-time occupation of previously empty cells.
The unusual 1990–1995 transition had 1,625 declining cells despite positive city-wide growth. We should treat this as a real analytical signal for now, but later map it and check whether GHSL redistribution or classification effects contribute to the pattern.

The next planned stage is the built-surface expansion comparison. It should use the same 32,922 grid-transition structure and classify cells into patterns such as:

Plain Text
newly_built
built_expansion
stable_built_surface
built_surface_decline
remained_unbuilt

It should also compare population and built growth to classify development patterns such as:

Plain Text
population-led densification
built-led expansion
combined growth
population decline with expansion
stable or low-change

In [19]:
# ============================================================
# CELL 12: GRID-LEVEL BUILT-SURFACE EXPANSION TRANSITIONS
#
# Input:
#   /data/ghsl/curated/pilots/bengaluru/
#   historical_comparisons/population_growth_transitions
#
# Output:
#   /data/ghsl/curated/pilots/bengaluru/
#   historical_comparisons/built_surface_transitions
#
# Expected:
#   3,658 grid cells x 9 consecutive transitions
#   = 32,922 rows
#
# Adds:
#   - Total built-surface change
#   - Residential and non-residential change
#   - Built-surface growth category
#   - Population versus built growth classification
#   - Change in population per built hectare
#   - SMOD context and geometry
#
# Built-surface categories:
#   newly_built
#   remained_unbuilt
#   built_surface_decline
#   stable_built_surface
#   low_built_growth
#   moderate_built_growth
#   high_built_growth
#   very_high_built_growth
#
# Development-pattern categories:
#   newly_built_and_populated
#   built_expansion_without_population_growth
#   population_growth_without_built_expansion
#   population_led_densification
#   built_led_expansion
#   balanced_population_and_built_growth
#   population_decline_with_built_expansion
#   population_growth_with_built_decline
#   joint_decline
#   stable_or_low_change
#
# Architecture:
#   Storage: HDFS
#   Processing: Apache Spark
#   Format: Parquet partitioned by to_epoch
#
# This cell does not rerun raster processing.
# ============================================================

import base64
import hashlib
import textwrap

REMOTE_PROGRAM = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_built_surface_transitions.py"
)

REMOTE_LOG = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_built_surface_transitions.log"
)

program_code = textwrap.dedent(
    """
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    import time


    # ========================================================
    # CONFIGURATION
    # ========================================================

    EPOCHS = [
        1975,
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
        2020,
    ]

    EXPECTED_GRID_CELLS = 3658
    EXPECTED_TRANSITIONS = 9

    EXPECTED_ROWS = (
        EXPECTED_GRID_CELLS
        * EXPECTED_TRANSITIONS
    )

    EXPECTED_TO_EPOCHS = EPOCHS[1:]

    # Numerical tolerance for treating tiny floating-point
    # differences as effectively zero.
    BUILT_STABLE_TOLERANCE_M2 = 1.0
    POPULATION_STABLE_TOLERANCE = 0.000001

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    INPUT_PATH = (
        f"{HDFS_BASE}/historical_comparisons/"
        "population_growth_transitions"
    )

    OUTPUT_PATH = (
        f"{HDFS_BASE}/historical_comparisons/"
        "built_surface_transitions"
    )


    # ========================================================
    # SPARK SESSION
    # ========================================================

    spark = (
        SparkSession.builder
        .appName(
            "GHSL_Bengaluru_Built_Surface_Transitions"
        )
        .config("spark.ui.enabled", "false")
        .config(
            "spark.sql.shuffle.partitions",
            "4",
        )
        .config(
            "spark.default.parallelism",
            "2",
        )
        .config(
            "spark.sql.adaptive.enabled",
            "false",
        )
        .config(
            "spark.sql.codegen.wholeStage",
            "false",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")


    # ========================================================
    # HDFS ACCESS
    # ========================================================

    jvm = spark.sparkContext._jvm

    hadoop_conf = (
        spark.sparkContext
        ._jsc
        .hadoopConfiguration()
    )

    hdfs = (
        jvm.org.apache.hadoop.fs.FileSystem.get(
            hadoop_conf
        )
    )

    HadoopPath = jvm.org.apache.hadoop.fs.Path


    def hdfs_exists(path):
        return bool(
            hdfs.exists(
                HadoopPath(path)
            )
        )


    def require(condition, message):
        if not condition:
            raise RuntimeError(message)


    # ========================================================
    # READ AND VALIDATE CELL 11 INPUT
    # ========================================================

    started = time.time()

    require(
        hdfs_exists(INPUT_PATH),
        f"Input HDFS path is missing: {INPUT_PATH}",
    )

    source_df = (
        spark.read
        .option("basePath", INPUT_PATH)
        .parquet(INPUT_PATH)
    )

    required_input_columns = {
        "city",
        "grid_cell_id",
        "grid_1km_col",
        "grid_1km_row",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "from_epoch",
        "to_epoch",
        "transition_years",
        "transition_label",
        "from_population_total",
        "to_population_total",
        "population_change",
        "population_change_percent",
        "population_annualized_growth_percent",
        "population_growth_category",
        "from_built_surface_total_m2",
        "to_built_surface_total_m2",
        "built_surface_change_m2",
        "built_surface_change_km2",
        "built_surface_change_percent",
        "from_built_surface_residential_m2",
        "to_built_surface_residential_m2",
        "residential_built_change_m2",
        "from_built_surface_nres_m2",
        "to_built_surface_nres_m2",
        "nonresidential_built_change_m2",
        "from_smod_class",
        "to_smod_class",
        "smod_transition",
        "smod_class_changed",
        "smod_class_direction",
        "geometry_wkb",
        "geometry_crs",
    }

    missing_input_columns = sorted(
        required_input_columns
        - set(source_df.columns)
    )

    require(
        not missing_input_columns,
        (
            "Cell 11 input is missing columns: "
            + ",".join(missing_input_columns)
        ),
    )

    input_rows = source_df.count()

    input_to_epochs = sorted(
        int(row["to_epoch"])
        for row in (
            source_df
            .select("to_epoch")
            .distinct()
            .collect()
        )
    )

    distinct_input_keys = (
        source_df
        .select(
            "grid_cell_id",
            "from_epoch",
            "to_epoch",
        )
        .distinct()
        .count()
    )

    require(
        input_rows == EXPECTED_ROWS,
        (
            f"Input rows={input_rows}; "
            f"expected={EXPECTED_ROWS}"
        ),
    )

    require(
        input_to_epochs == EXPECTED_TO_EPOCHS,
        (
            f"Input destination epochs="
            f"{input_to_epochs}; expected="
            f"{EXPECTED_TO_EPOCHS}"
        ),
    )

    require(
        distinct_input_keys == input_rows,
        (
            "Duplicate Cell 11 transition keys: "
            f"rows={input_rows}, "
            f"distinct={distinct_input_keys}"
        ),
    )

    print("=" * 110)
    print("CELL 11 INPUT VALIDATION")
    print("=" * 110)
    print("Input rows:", input_rows)
    print("Destination epochs:", input_to_epochs)
    print(
        "Distinct transition keys:",
        distinct_input_keys,
    )
    print("Input validation: PASS")


    # ========================================================
    # BUILT-SURFACE GROWTH CATEGORY
    # ========================================================

    built_df = (
        source_df
        .withColumn(
            "from_built_surface_total_km2",
            F.col(
                "from_built_surface_total_m2"
            )
            / F.lit(1_000_000.0),
        )
        .withColumn(
            "to_built_surface_total_km2",
            F.col(
                "to_built_surface_total_m2"
            )
            / F.lit(1_000_000.0),
        )
        .withColumn(
            "from_population_per_built_hectare",
            F.when(
                F.col(
                    "from_built_surface_total_m2"
                ) > F.lit(0),
                F.col(
                    "from_population_total"
                )
                / (
                    F.col(
                        "from_built_surface_total_m2"
                    )
                    / F.lit(10_000.0)
                ),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "to_population_per_built_hectare",
            F.when(
                F.col(
                    "to_built_surface_total_m2"
                ) > F.lit(0),
                F.col(
                    "to_population_total"
                )
                / (
                    F.col(
                        "to_built_surface_total_m2"
                    )
                    / F.lit(10_000.0)
                ),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "population_per_built_hectare_change",
            F.when(
                F.col(
                    "from_population_per_built_hectare"
                ).isNotNull()
                & F.col(
                    "to_population_per_built_hectare"
                ).isNotNull(),
                F.col(
                    "to_population_per_built_hectare"
                )
                - F.col(
                    "from_population_per_built_hectare"
                ),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "population_per_built_hectare_change_percent",
            F.when(
                F.col(
                    "from_population_per_built_hectare"
                ) > F.lit(0),
                (
                    F.col(
                        "population_per_built_hectare_change"
                    )
                    / F.col(
                        "from_population_per_built_hectare"
                    )
                )
                * F.lit(100.0),
            ).otherwise(
                F.lit(None).cast("double")
            ),
        )
        .withColumn(
            "built_surface_growth_category",
            F.when(
                (
                    F.col(
                        "from_built_surface_total_m2"
                    ) <= F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                )
                & (
                    F.col(
                        "to_built_surface_total_m2"
                    ) > F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                ),
                F.lit("newly_built"),
            )
            .when(
                (
                    F.col(
                        "from_built_surface_total_m2"
                    ) <= F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                )
                & (
                    F.col(
                        "to_built_surface_total_m2"
                    ) <= F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                ),
                F.lit("remained_unbuilt"),
            )
            .when(
                F.col(
                    "built_surface_change_m2"
                ) < F.lit(
                    -BUILT_STABLE_TOLERANCE_M2
                ),
                F.lit("built_surface_decline"),
            )
            .when(
                F.abs(
                    F.col(
                        "built_surface_change_m2"
                    )
                ) <= F.lit(
                    BUILT_STABLE_TOLERANCE_M2
                ),
                F.lit("stable_built_surface"),
            )
            .when(
                F.col(
                    "built_surface_change_percent"
                ) < F.lit(10.0),
                F.lit("low_built_growth"),
            )
            .when(
                F.col(
                    "built_surface_change_percent"
                ) < F.lit(25.0),
                F.lit("moderate_built_growth"),
            )
            .when(
                F.col(
                    "built_surface_change_percent"
                ) < F.lit(50.0),
                F.lit("high_built_growth"),
            )
            .otherwise(
                F.lit("very_high_built_growth")
            ),
        )
    )


    # ========================================================
    # COMBINED POPULATION-BUILT DEVELOPMENT PATTERN
    #
    # For cells with positive starting population and built
    # surface, percentage growth rates are compared.
    #
    # A +/-5 percentage-point band is used for "balanced"
    # growth to avoid treating small differences as meaningful.
    # ========================================================

    POP_BUILT_BALANCE_BAND = 5.0

    built_df = (
        built_df
        .withColumn(
            "development_pattern",
            F.when(
                (
                    F.col(
                        "from_built_surface_total_m2"
                    ) <= F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                )
                & (
                    F.col(
                        "to_built_surface_total_m2"
                    ) > F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                )
                & (
                    F.col(
                        "from_population_total"
                    ) <= F.lit(
                        POPULATION_STABLE_TOLERANCE
                    )
                )
                & (
                    F.col(
                        "to_population_total"
                    ) > F.lit(
                        POPULATION_STABLE_TOLERANCE
                    )
                ),
                F.lit("newly_built_and_populated"),
            )
            .when(
                (
                    F.col(
                        "built_surface_change_m2"
                    ) > F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                )
                & (
                    F.col("population_change")
                    <= F.lit(
                        POPULATION_STABLE_TOLERANCE
                    )
                ),
                F.lit(
                    "built_expansion_without_"
                    "population_growth"
                ),
            )
            .when(
                (
                    F.col("population_change")
                    > F.lit(
                        POPULATION_STABLE_TOLERANCE
                    )
                )
                & (
                    F.col(
                        "built_surface_change_m2"
                    ) <= F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                )
                & (
                    F.col(
                        "built_surface_change_m2"
                    ) >= F.lit(
                        -BUILT_STABLE_TOLERANCE_M2
                    )
                ),
                F.lit(
                    "population_growth_without_"
                    "built_expansion"
                ),
            )
            .when(
                (
                    F.col("population_change")
                    > F.lit(
                        POPULATION_STABLE_TOLERANCE
                    )
                )
                & (
                    F.col(
                        "built_surface_change_m2"
                    ) < F.lit(
                        -BUILT_STABLE_TOLERANCE_M2
                    )
                ),
                F.lit(
                    "population_growth_with_"
                    "built_decline"
                ),
            )
            .when(
                (
                    F.col("population_change")
                    < F.lit(
                        -POPULATION_STABLE_TOLERANCE
                    )
                )
                & (
                    F.col(
                        "built_surface_change_m2"
                    ) > F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                ),
                F.lit(
                    "population_decline_with_"
                    "built_expansion"
                ),
            )
            .when(
                (
                    F.col("population_change")
                    < F.lit(
                        -POPULATION_STABLE_TOLERANCE
                    )
                )
                & (
                    F.col(
                        "built_surface_change_m2"
                    ) < F.lit(
                        -BUILT_STABLE_TOLERANCE_M2
                    )
                ),
                F.lit("joint_decline"),
            )
            .when(
                (
                    F.col("population_change")
                    > F.lit(
                        POPULATION_STABLE_TOLERANCE
                    )
                )
                & (
                    F.col(
                        "built_surface_change_m2"
                    ) > F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                )
                & F.col(
                    "population_change_percent"
                ).isNotNull()
                & F.col(
                    "built_surface_change_percent"
                ).isNotNull()
                & (
                    F.col(
                        "population_change_percent"
                    )
                    > (
                        F.col(
                            "built_surface_change_percent"
                        )
                        + F.lit(
                            POP_BUILT_BALANCE_BAND
                        )
                    )
                ),
                F.lit(
                    "population_led_densification"
                ),
            )
            .when(
                (
                    F.col("population_change")
                    > F.lit(
                        POPULATION_STABLE_TOLERANCE
                    )
                )
                & (
                    F.col(
                        "built_surface_change_m2"
                    ) > F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                )
                & F.col(
                    "population_change_percent"
                ).isNotNull()
                & F.col(
                    "built_surface_change_percent"
                ).isNotNull()
                & (
                    F.col(
                        "built_surface_change_percent"
                    )
                    > (
                        F.col(
                            "population_change_percent"
                        )
                        + F.lit(
                            POP_BUILT_BALANCE_BAND
                        )
                    )
                ),
                F.lit("built_led_expansion"),
            )
            .when(
                (
                    F.col("population_change")
                    > F.lit(
                        POPULATION_STABLE_TOLERANCE
                    )
                )
                & (
                    F.col(
                        "built_surface_change_m2"
                    ) > F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    )
                ),
                F.lit(
                    "balanced_population_and_"
                    "built_growth"
                ),
            )
            .otherwise(
                F.lit("stable_or_low_change")
            ),
        )
        .withColumn(
            "densification_indicator",
            F.when(
                F.col(
                    "population_per_built_hectare_change"
                ) > F.lit(0),
                F.lit("increasing_density"),
            )
            .when(
                F.col(
                    "population_per_built_hectare_change"
                ) < F.lit(0),
                F.lit("decreasing_density"),
            )
            .when(
                F.col(
                    "population_per_built_hectare_change"
                ).isNull(),
                F.lit("not_comparable"),
            )
            .otherwise(
                F.lit("stable_density")
            ),
        )
    )


    # ========================================================
    # FINAL OUTPUT SCHEMA
    # ========================================================

    FINAL_COLUMNS = [
        "city",
        "grid_cell_id",
        "grid_1km_col",
        "grid_1km_row",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "from_epoch",
        "to_epoch",
        "transition_years",
        "transition_label",
        "from_built_surface_total_m2",
        "to_built_surface_total_m2",
        "from_built_surface_total_km2",
        "to_built_surface_total_km2",
        "built_surface_change_m2",
        "built_surface_change_km2",
        "built_surface_change_percent",
        "built_surface_growth_category",
        "from_built_surface_residential_m2",
        "to_built_surface_residential_m2",
        "residential_built_change_m2",
        "from_built_surface_nres_m2",
        "to_built_surface_nres_m2",
        "nonresidential_built_change_m2",
        "from_population_total",
        "to_population_total",
        "population_change",
        "population_change_percent",
        "population_growth_category",
        "from_population_per_built_hectare",
        "to_population_per_built_hectare",
        "population_per_built_hectare_change",
        "population_per_built_hectare_change_percent",
        "densification_indicator",
        "development_pattern",
        "from_smod_class",
        "to_smod_class",
        "smod_transition",
        "smod_class_changed",
        "smod_class_direction",
        "geometry_wkb",
        "geometry_crs",
    ]

    built_df = built_df.select(
        *FINAL_COLUMNS
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    def validate_output(df, label):
        row_count = df.count()

        to_epochs = sorted(
            int(row["to_epoch"])
            for row in (
                df.select("to_epoch")
                .distinct()
                .collect()
            )
        )

        distinct_keys = (
            df.select(
                "grid_cell_id",
                "from_epoch",
                "to_epoch",
            )
            .distinct()
            .count()
        )

        duplicate_keys = (
            row_count - distinct_keys
        )

        distinct_coordinate_keys = (
            df.select(
                "grid_1km_col",
                "grid_1km_row",
                "from_epoch",
                "to_epoch",
            )
            .distinct()
            .count()
        )

        duplicate_coordinate_keys = (
            row_count
            - distinct_coordinate_keys
        )

        null_keys = df.filter(
            F.col("grid_cell_id").isNull()
            | F.col("grid_1km_col").isNull()
            | F.col("grid_1km_row").isNull()
            | F.col("from_epoch").isNull()
            | F.col("to_epoch").isNull()
        ).count()

        invalid_periods = df.filter(
            (
                F.col("transition_years")
                != F.lit(5)
            )
            | (
                F.col("to_epoch")
                != (
                    F.col("from_epoch")
                    + F.lit(5)
                )
            )
        ).count()

        invalid_built_values = df.filter(
            F.col(
                "from_built_surface_total_m2"
            ).isNull()
            | F.isnan(
                "from_built_surface_total_m2"
            )
            | (
                F.col(
                    "from_built_surface_total_m2"
                ) < F.lit(0)
            )
            | F.col(
                "to_built_surface_total_m2"
            ).isNull()
            | F.isnan(
                "to_built_surface_total_m2"
            )
            | (
                F.col(
                    "to_built_surface_total_m2"
                ) < F.lit(0)
            )
            | F.col(
                "built_surface_change_m2"
            ).isNull()
            | F.isnan(
                "built_surface_change_m2"
            )
            | F.col(
                "built_surface_growth_category"
            ).isNull()
            | F.col(
                "development_pattern"
            ).isNull()
            | F.col(
                "densification_indicator"
            ).isNull()
        ).count()

        invalid_context_values = df.filter(
            F.col(
                "from_population_total"
            ).isNull()
            | F.col(
                "to_population_total"
            ).isNull()
            | F.col(
                "from_smod_class"
            ).isNull()
            | F.col(
                "to_smod_class"
            ).isNull()
            | F.col("geometry_wkb").isNull()
            | F.col("geometry_crs").isNull()
        ).count()

        per_transition = {
            (
                int(row["from_epoch"]),
                int(row["to_epoch"]),
            ): int(row["count"])
            for row in (
                df.groupBy(
                    "from_epoch",
                    "to_epoch",
                )
                .count()
                .collect()
            )
        }

        built_category_counts = {
            row[
                "built_surface_growth_category"
            ]: int(row["count"])
            for row in (
                df.groupBy(
                    "built_surface_growth_category"
                )
                .count()
                .collect()
            )
        }

        development_pattern_counts = {
            row["development_pattern"]:
            int(row["count"])
            for row in (
                df.groupBy(
                    "development_pattern"
                )
                .count()
                .collect()
            )
        }

        require(
            row_count == EXPECTED_ROWS,
            (
                f"{label}: rows={row_count}; "
                f"expected={EXPECTED_ROWS}"
            ),
        )

        require(
            to_epochs == EXPECTED_TO_EPOCHS,
            (
                f"{label}: to_epochs={to_epochs}; "
                f"expected={EXPECTED_TO_EPOCHS}"
            ),
        )

        require(
            duplicate_keys == 0,
            (
                f"{label}: duplicate keys="
                f"{duplicate_keys}"
            ),
        )

        require(
            duplicate_coordinate_keys == 0,
            (
                f"{label}: duplicate coordinate keys="
                f"{duplicate_coordinate_keys}"
            ),
        )

        require(
            null_keys == 0,
            f"{label}: null keys={null_keys}",
        )

        require(
            invalid_periods == 0,
            (
                f"{label}: invalid periods="
                f"{invalid_periods}"
            ),
        )

        require(
            invalid_built_values == 0,
            (
                f"{label}: invalid built rows="
                f"{invalid_built_values}"
            ),
        )

        require(
            invalid_context_values == 0,
            (
                f"{label}: invalid context rows="
                f"{invalid_context_values}"
            ),
        )

        for index in range(
            len(EPOCHS) - 1
        ):
            key = (
                EPOCHS[index],
                EPOCHS[index + 1],
            )

            actual = per_transition.get(
                key,
                0,
            )

            require(
                actual == EXPECTED_GRID_CELLS,
                (
                    f"{label}: transition {key} "
                    f"has {actual} rows; expected "
                    f"{EXPECTED_GRID_CELLS}"
                ),
            )

        require(
            df.columns == FINAL_COLUMNS,
            (
                f"{label}: output schema differs "
                "from expected schema"
            ),
        )

        return {
            "rows": row_count,
            "to_epochs": to_epochs,
            "duplicate_keys": duplicate_keys,
            "duplicate_coordinate_keys": (
                duplicate_coordinate_keys
            ),
            "null_keys": null_keys,
            "invalid_periods": invalid_periods,
            "invalid_built_values": (
                invalid_built_values
            ),
            "invalid_context_values": (
                invalid_context_values
            ),
            "per_transition": per_transition,
            "built_category_counts": (
                built_category_counts
            ),
            "development_pattern_counts": (
                development_pattern_counts
            ),
        }


    # ========================================================
    # PRE-WRITE VALIDATION
    # ========================================================

    print()
    print("=" * 110)
    print("BUILT-SURFACE PRE-WRITE VALIDATION")
    print("=" * 110)

    pre_write = validate_output(
        built_df,
        "PRE-WRITE",
    )

    print("Rows:", pre_write["rows"])
    print(
        "Destination epochs:",
        pre_write["to_epochs"],
    )
    print(
        "Duplicate transition keys:",
        pre_write["duplicate_keys"],
    )
    print(
        "Invalid built rows:",
        pre_write[
            "invalid_built_values"
        ],
    )
    print("Pre-write validation: PASS")


    # ========================================================
    # TRANSITION-LEVEL SUMMARY
    # ========================================================

    print()
    print("=" * 110)
    print("BUILT-SURFACE CHANGE BY TRANSITION")
    print("=" * 110)

    transition_summary_df = (
        built_df
        .groupBy(
            "from_epoch",
            "to_epoch",
        )
        .agg(
            F.count("*").alias(
                "grid_cells"
            ),
            F.sum(
                "built_surface_change_m2"
            ).alias(
                "total_built_change_m2"
            ),
            (
                F.sum(
                    "built_surface_change_m2"
                )
                / F.lit(1_000_000.0)
            ).alias(
                "total_built_change_km2"
            ),
            F.avg(
                "built_surface_change_m2"
            ).alias(
                "mean_built_change_m2"
            ),
            F.expr(
                "percentile_approx("
                "built_surface_change_m2, 0.5)"
            ).alias(
                "median_built_change_m2"
            ),
            F.max(
                "built_surface_change_m2"
            ).alias(
                "maximum_built_increase_m2"
            ),
            F.min(
                "built_surface_change_m2"
            ).alias(
                "maximum_built_decline_m2"
            ),
            F.sum(
                F.when(
                    F.col(
                        "built_surface_change_m2"
                    ) > F.lit(
                        BUILT_STABLE_TOLERANCE_M2
                    ),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).alias(
                "expanding_cells"
            ),
            F.sum(
                F.when(
                    F.col(
                        "built_surface_change_m2"
                    ) < F.lit(
                        -BUILT_STABLE_TOLERANCE_M2
                    ),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).alias(
                "declining_built_cells"
            ),
            F.sum(
                F.when(
                    F.col(
                        "built_surface_growth_category"
                    ) == F.lit("newly_built"),
                    F.lit(1),
                ).otherwise(
                    F.lit(0)
                )
            ).alias(
                "newly_built_cells"
            ),
            F.sum(
                "residential_built_change_m2"
            ).alias(
                "residential_change_m2"
            ),
            F.sum(
                "nonresidential_built_change_m2"
            ).alias(
                "nonresidential_change_m2"
            ),
        )
        .orderBy(
            "from_epoch",
            "to_epoch",
        )
    )

    transition_summary_df.select(
        "from_epoch",
        "to_epoch",
        "grid_cells",
        F.round(
            "total_built_change_km2",
            3,
        ).alias(
            "built_change_km2"
        ),
        F.round(
            "mean_built_change_m2",
            2,
        ).alias(
            "mean_change_m2"
        ),
        F.round(
            "median_built_change_m2",
            2,
        ).alias(
            "median_change_m2"
        ),
        F.round(
            "maximum_built_increase_m2",
            2,
        ).alias(
            "maximum_increase_m2"
        ),
        F.round(
            "maximum_built_decline_m2",
            2,
        ).alias(
            "maximum_decline_m2"
        ),
        "expanding_cells",
        "declining_built_cells",
        "newly_built_cells",
        F.round(
            (
                F.col(
                    "residential_change_m2"
                )
                / F.lit(1_000_000.0)
            ),
            3,
        ).alias(
            "residential_change_km2"
        ),
        F.round(
            (
                F.col(
                    "nonresidential_change_m2"
                )
                / F.lit(1_000_000.0)
            ),
            3,
        ).alias(
            "nonresidential_change_km2"
        ),
    ).show(
        20,
        truncate=False,
    )


    # ========================================================
    # CATEGORY SUMMARIES
    # ========================================================

    print()
    print("=" * 110)
    print("BUILT-SURFACE GROWTH CATEGORIES")
    print("=" * 110)

    (
        built_df
        .groupBy(
            "from_epoch",
            "to_epoch",
            "built_surface_growth_category",
        )
        .count()
        .orderBy(
            "from_epoch",
            "to_epoch",
            "built_surface_growth_category",
        )
        .show(
            200,
            truncate=False,
        )
    )

    print()
    print("=" * 110)
    print("POPULATION-BUILT DEVELOPMENT PATTERNS")
    print("=" * 110)

    (
        built_df
        .groupBy(
            "from_epoch",
            "to_epoch",
            "development_pattern",
        )
        .count()
        .orderBy(
            "from_epoch",
            "to_epoch",
            "development_pattern",
        )
        .show(
            200,
            truncate=False,
        )
    )


    # ========================================================
    # WRITE TO HDFS
    # ========================================================

    print()
    print("=" * 110)
    print("WRITING BUILT-SURFACE TRANSITIONS TO HDFS")
    print("=" * 110)

    (
        built_df
        .repartition(
            9,
            "to_epoch",
        )
        .write
        .mode("overwrite")
        .partitionBy("to_epoch")
        .parquet(OUTPUT_PATH)
    )

    require(
        hdfs_exists(OUTPUT_PATH),
        (
            "Built-transition output path "
            "is missing after write"
        ),
    )

    print("HDFS write: COMPLETE")
    print("Output:", OUTPUT_PATH)


    # ========================================================
    # READ-BACK VALIDATION
    # ========================================================

    print()
    print("=" * 110)
    print("HDFS READ-BACK VALIDATION")
    print("=" * 110)

    saved_df = (
        spark.read
        .option("basePath", OUTPUT_PATH)
        .parquet(OUTPUT_PATH)
        .select(*FINAL_COLUMNS)
    )

    read_back = validate_output(
        saved_df,
        "READ-BACK",
    )

    elapsed_seconds = round(
        time.time() - started,
        2,
    )

    print("Read-back validation: PASS")
    print("Rows:", read_back["rows"])
    print(
        "Destination epochs:",
        read_back["to_epochs"],
    )
    print(
        "Duplicate transition keys:",
        read_back["duplicate_keys"],
    )
    print(
        "Duplicate coordinate keys:",
        read_back[
            "duplicate_coordinate_keys"
        ],
    )
    print(
        "Null keys:",
        read_back["null_keys"],
    )
    print(
        "Invalid periods:",
        read_back["invalid_periods"],
    )
    print(
        "Invalid built rows:",
        read_back[
            "invalid_built_values"
        ],
    )
    print(
        "Invalid context rows:",
        read_back[
            "invalid_context_values"
        ],
    )


    # ========================================================
    # FINAL REPORT
    # ========================================================

    print()
    print("=" * 110)
    print("BUILT-SURFACE TRANSITION RESULT")
    print("=" * 110)

    print(
        "Grid cells:",
        EXPECTED_GRID_CELLS,
    )

    print(
        "Transitions per grid cell:",
        EXPECTED_TRANSITIONS,
    )

    print(
        "Expected rows:",
        EXPECTED_ROWS,
    )

    print(
        "Validated rows:",
        read_back["rows"],
    )

    print(
        "Transition periods:",
        len(
            read_back["per_transition"]
        ),
    )

    print(
        "Built-growth categories:",
        sorted(
            read_back[
                "built_category_counts"
            ].keys()
        ),
    )

    print(
        "Development patterns:",
        sorted(
            read_back[
                "development_pattern_counts"
            ].keys()
        ),
    )

    print(
        "Elapsed seconds:",
        elapsed_seconds,
    )

    print(
        "Output:",
        OUTPUT_PATH,
    )

    print()
    print(
        "BUILT-SURFACE TRANSITIONS: SUCCESS"
    )

    spark.stop()
    """
).strip() + "\n"


# ============================================================
# COMPILE EXACT SOURCE BEFORE TRANSFER
# ============================================================

compile(
    program_code,
    "build_built_surface_transitions.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    program_code.encode("utf-8")
).hexdigest()

print("LOCAL PYTHON COMPILE CHECK: PASS")
print("Expected program SHA-256:", local_sha256)


# ============================================================
# TRANSFER EXACT SOURCE AND RUN REMOTELY
# ============================================================

encoded_program = base64.b64encode(
    program_code.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"
LOG="{REMOTE_LOG}"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_program}" \
| base64 -d \
> "${{PROGRAM}}"

ACTUAL_SHA256="$(
    sha256sum "${{PROGRAM}}" \
    | awk '{{print $1}}'
)"

echo "Remote program SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Program checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM CHECK: PASS"

python3 -m py_compile "${{PROGRAM}}"

echo "REMOTE PYTHON COMPILE CHECK: PASS"

echo
echo "HDFS readiness:"

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
    | awk '{{print $2}}' \
    | grep -qx "${{daemon}}" \
    || {{
        echo "ERROR: ${{daemon}} is not running."
        exit 21
    }}

    echo "PASS: ${{daemon}}"
done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {{
    echo "ERROR: HDFS safe mode is enabled."
    exit 22
}}

echo "PASS: HDFS safe mode is OFF"

echo
echo "RUNNING BUILT-SURFACE TRANSITION JOB"

spark-submit \
    --master 'local[1]' \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    "${{PROGRAM}}" \
    2>&1 \
    | tee "${{LOG}}"

status=${{PIPESTATUS[0]}}

echo
echo "BUILT_TRANSITIONS_EXIT_CODE=${{status}}"
echo "Job log: ${{LOG}}"

exit "${{status}}"
"""

built_transition_result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 12 RESULT")
print("=" * 80)

print(
    "Return code:",
    built_transition_result.returncode,
)

print(
    "Expected transition rows:",
    32922,
)

print(
    "Output:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/historical_comparisons/"
    "built_surface_transitions",
)

print(
    "Remote log:",
    REMOTE_LOG,
)

LOCAL PYTHON COMPILE CHECK: PASS
Expected program SHA-256: 2157418f492b27c38cd444de17be9359210fae6b8a5653c5314226bea6b113e5
Remote program SHA-256: 2157418f492b27c38cd444de17be9359210fae6b8a5653c5314226bea6b113e5
REMOTE CHECKSUM CHECK: PASS
REMOTE PYTHON COMPILE CHECK: PASS

HDFS readiness:
PASS: NameNode
PASS: DataNode
PASS: SecondaryNameNode
PASS: HDFS safe mode is OFF

RUNNING BUILT-SURFACE TRANSITION JOB
26/11/07 13:27:32 INFO SparkContext: Running Spark version 3.5.9
26/11/07 13:27:32 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/07 13:27:32 INFO SparkContext: Java version 11.0.32.1
26/11/07 13:27:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 13:27:32 INFO ResourceUtils: ==============================================================
26/11/07 13:27:32 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/07 13:27:32 INFO ResourceUtils: ====================

he built-surface transition dataset is now stored in HDFS:

Plain Text
/data/ghsl/curated/pilots/bengaluru/
historical_comparisons/built_surface_transitions

Main built-surface findings
Transition	Built increase	Expanding cells	Newly built cells
1975→1980	7.29 km²	3,356	175
1980→1985	9.87 km²	3,398	34
1985→1990	11.53 km²	3,430	21
1990→1995	30.70 km²	3,196	49
1995→2000	40.12 km²	3,209	14
2000→2005	36.76 km²	3,578	58
2005→2010	49.62 km²	3,587	6
2010→2015	52.78 km²	3,587	1
2015→2020	36.67 km²	3,505	0

What this indicates
The largest absolute built expansion occurred during 2010–2015, adding approximately 52.78 km².
No grid cell experienced a net built-surface decline above the 1 m² tolerance in any transition.
Newly built cells decreased from 175 during 1975–1980 to zero during 2015–2020.
Later growth therefore occurred mainly by adding built surface within already-built grid cells, not by first-time construction in previously unbuilt cells.
Across all periods, residential expansion dominated non-residential expansion.

Built-led expansion:                       1,361 cells
Population-led densification:                836 cells
Balanced population and built growth:        669 cells
Built expansion without population growth:   639 cells
Population growth without built expansion:    82 cells
Stable or low change:                         71 cells

In [20]:
# ============================================================
# CELL 13: SMOD SETTLEMENT-CLASS TRANSITIONS
#
# Inputs:
#   Longitudinal Bengaluru dataset from Cell 9
#
# Outputs:
#   1. Grid-level consecutive transitions:
#      historical_comparisons/smod_transitions
#
#   2. Consecutive transition matrix:
#      historical_summaries/smod_transition_matrix
#
#   3. Full-period 1975-to-2020 matrix:
#      historical_summaries/smod_1975_2020_matrix
#
# Expected:
#   Consecutive grid transitions:
#       3,658 cells x 9 periods = 32,922 rows
#
#   Direct full-period transitions:
#       3,658 grid cells
#
# SMOD category logic:
#   class >= 21  -> urban-class
#   class == 30  -> urban-centre
#
# Architecture:
#   Storage: HDFS
#   Processing: Apache Spark window functions and joins
#   Format: Parquet
#
# This cell does not rerun raster processing.
# ============================================================

import base64
import hashlib
import textwrap

REMOTE_PROGRAM = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_smod_transitions.py"
)

REMOTE_LOG = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_smod_transitions.log"
)

program_code = textwrap.dedent(
    """
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    import time


    # ========================================================
    # CONFIGURATION
    # ========================================================

    EPOCHS = [
        1975,
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
        2020,
    ]

    EXPECTED_GRID_CELLS = 3658
    EXPECTED_CONSECUTIVE_ROWS = 32922
    EXPECTED_FULL_PERIOD_ROWS = 3658
    EXPECTED_TO_EPOCHS = EPOCHS[1:]

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    INPUT_PATH = (
        f"{HDFS_BASE}/longitudinal_grid_year/"
        "all_epochs"
    )

    GRID_OUTPUT_PATH = (
        f"{HDFS_BASE}/historical_comparisons/"
        "smod_transitions"
    )

    MATRIX_OUTPUT_PATH = (
        f"{HDFS_BASE}/historical_summaries/"
        "smod_transition_matrix"
    )

    FULL_MATRIX_OUTPUT_PATH = (
        f"{HDFS_BASE}/historical_summaries/"
        "smod_1975_2020_matrix"
    )


    # ========================================================
    # SPARK SESSION
    # ========================================================

    spark = (
        SparkSession.builder
        .appName(
            "GHSL_Bengaluru_SMOD_Transitions"
        )
        .config("spark.ui.enabled", "false")
        .config(
            "spark.sql.shuffle.partitions",
            "4",
        )
        .config(
            "spark.default.parallelism",
            "2",
        )
        .config(
            "spark.sql.adaptive.enabled",
            "false",
        )
        .config(
            "spark.sql.codegen.wholeStage",
            "false",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")


    # ========================================================
    # HDFS ACCESS
    # ========================================================

    jvm = spark.sparkContext._jvm

    hadoop_conf = (
        spark.sparkContext
        ._jsc
        .hadoopConfiguration()
    )

    hdfs = (
        jvm.org.apache.hadoop.fs.FileSystem.get(
            hadoop_conf
        )
    )

    HadoopPath = jvm.org.apache.hadoop.fs.Path


    def hdfs_exists(path):
        return bool(
            hdfs.exists(
                HadoopPath(path)
            )
        )


    def require(condition, message):
        if not condition:
            raise RuntimeError(message)


    def add_smod_group(column_name):
        return (
            F.when(
                F.col(column_name) == F.lit(30),
                F.lit("urban_centre"),
            )
            .when(
                F.col(column_name) >= F.lit(21),
                F.lit("urban_class"),
            )
            .otherwise(
                F.lit("non_urban_class")
            )
        )


    def add_transition_category(df):
        return df.withColumn(
            "smod_transition_category",
            F.when(
                F.col("from_smod_class")
                == F.col("to_smod_class"),
                F.lit("unchanged"),
            )
            .when(
                (
                    F.col("from_smod_class")
                    < F.lit(21)
                )
                & (
                    F.col("to_smod_class")
                    >= F.lit(21)
                ),
                F.lit("newly_urban_class"),
            )
            .when(
                (
                    F.col("from_smod_class")
                    != F.lit(30)
                )
                & (
                    F.col("to_smod_class")
                    == F.lit(30)
                ),
                F.lit("newly_urban_centre"),
            )
            .when(
                (
                    F.col("from_smod_class")
                    == F.lit(30)
                )
                & (
                    F.col("to_smod_class")
                    == F.lit(30)
                ),
                F.lit("remained_urban_centre"),
            )
            .when(
                (
                    F.col("from_smod_class")
                    == F.lit(30)
                )
                & (
                    F.col("to_smod_class")
                    != F.lit(30)
                ),
                F.lit("left_urban_centre"),
            )
            .when(
                F.col("to_smod_class")
                > F.col("from_smod_class"),
                F.lit("upward_class_transition"),
            )
            .when(
                F.col("to_smod_class")
                < F.col("from_smod_class"),
                F.lit("downward_class_transition"),
            )
            .otherwise(
                F.lit("other_transition")
            ),
        )


    # ========================================================
    # READ AND VALIDATE LONGITUDINAL INPUT
    # ========================================================

    started = time.time()

    require(
        hdfs_exists(INPUT_PATH),
        f"Input HDFS path is missing: {INPUT_PATH}",
    )

    source_df = spark.read.parquet(
        INPUT_PATH
    )

    required_columns = {
        "city",
        "epoch",
        "grid_cell_id",
        "grid_1km_col",
        "grid_1km_row",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "population_total",
        "smod_class",
        "built_surface_total_m2",
        "geometry_wkb",
        "geometry_crs",
    }

    missing_columns = sorted(
        required_columns - set(source_df.columns)
    )

    require(
        not missing_columns,
        (
            "Input is missing columns: "
            + ",".join(missing_columns)
        ),
    )

    input_rows = source_df.count()

    input_epochs = sorted(
        int(row["epoch"])
        for row in (
            source_df
            .select("epoch")
            .distinct()
            .collect()
        )
    )

    distinct_grid_years = (
        source_df
        .select(
            "grid_cell_id",
            "epoch",
        )
        .distinct()
        .count()
    )

    null_smod_rows = source_df.filter(
        F.col("smod_class").isNull()
    ).count()

    require(
        input_rows == 36580,
        (
            f"Input rows={input_rows}; "
            "expected=36580"
        ),
    )

    require(
        input_epochs == EPOCHS,
        (
            f"Input epochs={input_epochs}; "
            f"expected={EPOCHS}"
        ),
    )

    require(
        distinct_grid_years == input_rows,
        (
            "Duplicate grid-year rows found: "
            f"rows={input_rows}, "
            f"distinct={distinct_grid_years}"
        ),
    )

    require(
        null_smod_rows == 0,
        (
            f"Input null SMOD rows="
            f"{null_smod_rows}"
        ),
    )

    print("=" * 108)
    print("LONGITUDINAL INPUT VALIDATION")
    print("=" * 108)
    print("Input rows:", input_rows)
    print("Input epochs:", input_epochs)
    print(
        "Distinct grid-year keys:",
        distinct_grid_years,
    )
    print("Null SMOD rows:", null_smod_rows)
    print("Input validation: PASS")


    # ========================================================
    # BUILD CONSECUTIVE GRID-LEVEL TRANSITIONS
    # ========================================================

    grid_window = (
        Window
        .partitionBy("grid_cell_id")
        .orderBy("epoch")
    )

    consecutive_df = (
        source_df
        .withColumn(
            "from_epoch",
            F.lag("epoch").over(grid_window),
        )
        .withColumn(
            "from_smod_class",
            F.lag(
                "smod_class"
            ).over(grid_window),
        )
        .withColumn(
            "from_population_total",
            F.lag(
                "population_total"
            ).over(grid_window),
        )
        .withColumn(
            "from_built_surface_total_m2",
            F.lag(
                "built_surface_total_m2"
            ).over(grid_window),
        )
        .withColumnRenamed(
            "epoch",
            "to_epoch",
        )
        .withColumnRenamed(
            "smod_class",
            "to_smod_class",
        )
        .withColumnRenamed(
            "population_total",
            "to_population_total",
        )
        .withColumnRenamed(
            "built_surface_total_m2",
            "to_built_surface_total_m2",
        )
        .filter(
            F.col("from_epoch").isNotNull()
        )
        .withColumn(
            "transition_years",
            (
                F.col("to_epoch")
                - F.col("from_epoch")
            ).cast("int"),
        )
        .withColumn(
            "transition_label",
            F.concat(
                F.col("from_epoch").cast("string"),
                F.lit("_to_"),
                F.col("to_epoch").cast("string"),
            ),
        )
        .withColumn(
            "from_smod_group",
            add_smod_group(
                "from_smod_class"
            ),
        )
        .withColumn(
            "to_smod_group",
            add_smod_group(
                "to_smod_class"
            ),
        )
        .withColumn(
            "smod_transition",
            F.concat(
                F.col(
                    "from_smod_class"
                ).cast("string"),
                F.lit("_to_"),
                F.col(
                    "to_smod_class"
                ).cast("string"),
            ),
        )
        .withColumn(
            "smod_class_change",
            F.col("to_smod_class")
            - F.col("from_smod_class"),
        )
        .withColumn(
            "smod_class_changed",
            F.col("from_smod_class")
            != F.col("to_smod_class"),
        )
        .withColumn(
            "population_change",
            F.col("to_population_total")
            - F.col("from_population_total"),
        )
        .withColumn(
            "built_surface_change_m2",
            F.col(
                "to_built_surface_total_m2"
            )
            - F.col(
                "from_built_surface_total_m2"
            ),
        )
    )

    consecutive_df = add_transition_category(
        consecutive_df
    )

    GRID_COLUMNS = [
        "city",
        "grid_cell_id",
        "grid_1km_col",
        "grid_1km_row",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "from_epoch",
        "to_epoch",
        "transition_years",
        "transition_label",
        "from_smod_class",
        "to_smod_class",
        "from_smod_group",
        "to_smod_group",
        "smod_transition",
        "smod_class_change",
        "smod_class_changed",
        "smod_transition_category",
        "from_population_total",
        "to_population_total",
        "population_change",
        "from_built_surface_total_m2",
        "to_built_surface_total_m2",
        "built_surface_change_m2",
        "geometry_wkb",
        "geometry_crs",
    ]

    consecutive_df = consecutive_df.select(
        *GRID_COLUMNS
    )


    # ========================================================
    # VALIDATE CONSECUTIVE TRANSITIONS
    # ========================================================

    def validate_consecutive(df, label):
        rows = df.count()

        to_epochs = sorted(
            int(row["to_epoch"])
            for row in (
                df.select("to_epoch")
                .distinct()
                .collect()
            )
        )

        distinct_keys = (
            df.select(
                "grid_cell_id",
                "from_epoch",
                "to_epoch",
            )
            .distinct()
            .count()
        )

        duplicate_keys = rows - distinct_keys

        null_keys = df.filter(
            F.col("grid_cell_id").isNull()
            | F.col("from_epoch").isNull()
            | F.col("to_epoch").isNull()
            | F.col("from_smod_class").isNull()
            | F.col("to_smod_class").isNull()
            | F.col(
                "smod_transition_category"
            ).isNull()
        ).count()

        invalid_periods = df.filter(
            (F.col("transition_years") != 5)
            | (
                F.col("to_epoch")
                != F.col("from_epoch") + F.lit(5)
            )
        ).count()

        invalid_context = df.filter(
            F.col("from_population_total").isNull()
            | F.col("to_population_total").isNull()
            | F.col(
                "from_built_surface_total_m2"
            ).isNull()
            | F.col(
                "to_built_surface_total_m2"
            ).isNull()
            | F.col("geometry_wkb").isNull()
            | F.col("geometry_crs").isNull()
        ).count()

        per_transition = {
            (
                int(row["from_epoch"]),
                int(row["to_epoch"]),
            ): int(row["count"])
            for row in (
                df.groupBy(
                    "from_epoch",
                    "to_epoch",
                )
                .count()
                .collect()
            )
        }

        require(
            rows == EXPECTED_CONSECUTIVE_ROWS,
            (
                f"{label}: rows={rows}; expected="
                f"{EXPECTED_CONSECUTIVE_ROWS}"
            ),
        )

        require(
            to_epochs == EXPECTED_TO_EPOCHS,
            (
                f"{label}: to epochs={to_epochs}; "
                f"expected={EXPECTED_TO_EPOCHS}"
            ),
        )

        require(
            duplicate_keys == 0,
            (
                f"{label}: duplicate keys="
                f"{duplicate_keys}"
            ),
        )

        require(
            null_keys == 0,
            f"{label}: null keys={null_keys}",
        )

        require(
            invalid_periods == 0,
            (
                f"{label}: invalid periods="
                f"{invalid_periods}"
            ),
        )

        require(
            invalid_context == 0,
            (
                f"{label}: invalid context="
                f"{invalid_context}"
            ),
        )

        for index in range(len(EPOCHS) - 1):
            key = (
                EPOCHS[index],
                EPOCHS[index + 1],
            )

            actual = per_transition.get(key, 0)

            require(
                actual == EXPECTED_GRID_CELLS,
                (
                    f"{label}: transition {key} "
                    f"has {actual} rows"
                ),
            )

        require(
            df.columns == GRID_COLUMNS,
            (
                f"{label}: grid output schema "
                "does not match"
            ),
        )

        return {
            "rows": rows,
            "to_epochs": to_epochs,
            "duplicate_keys": duplicate_keys,
            "null_keys": null_keys,
            "invalid_periods": invalid_periods,
            "invalid_context": invalid_context,
            "per_transition": per_transition,
        }


    print()
    print("=" * 108)
    print("CONSECUTIVE SMOD PRE-WRITE VALIDATION")
    print("=" * 108)

    grid_validation = validate_consecutive(
        consecutive_df,
        "CONSECUTIVE PRE-WRITE",
    )

    print("Rows:", grid_validation["rows"])
    print(
        "Destination epochs:",
        grid_validation["to_epochs"],
    )
    print(
        "Duplicate keys:",
        grid_validation["duplicate_keys"],
    )
    print("Validation: PASS")


    # ========================================================
    # CONSECUTIVE TRANSITION MATRIX
    # ========================================================

    transition_matrix_df = (
        consecutive_df
        .groupBy(
            "from_epoch",
            "to_epoch",
            "from_smod_class",
            "to_smod_class",
            "from_smod_group",
            "to_smod_group",
            "smod_transition_category",
        )
        .agg(
            F.count("*").cast("long").alias(
                "grid_cell_count"
            ),
            F.sum(
                "population_change"
            ).alias(
                "population_change_total"
            ),
            F.sum(
                "built_surface_change_m2"
            ).alias(
                "built_surface_change_m2_total"
            ),
            F.avg(
                "population_change"
            ).alias(
                "mean_population_change"
            ),
            F.avg(
                "built_surface_change_m2"
            ).alias(
                "mean_built_surface_change_m2"
            ),
        )
        .withColumn(
            "transition_grid_share_percent",
            (
                F.col("grid_cell_count")
                / F.lit(
                    EXPECTED_GRID_CELLS
                )
            )
            * F.lit(100.0),
        )
        .orderBy(
            "from_epoch",
            "to_epoch",
            "from_smod_class",
            "to_smod_class",
        )
    )


    # ========================================================
    # DIRECT FULL-PERIOD 1975-2020 TRANSITIONS
    # ========================================================

    start_df = (
        source_df
        .filter(F.col("epoch") == 1975)
        .select(
            "city",
            "grid_cell_id",
            "grid_1km_col",
            "grid_1km_row",
            F.col("smod_class").alias(
                "from_smod_class"
            ),
            F.col("population_total").alias(
                "from_population_total"
            ),
            F.col(
                "built_surface_total_m2"
            ).alias(
                "from_built_surface_total_m2"
            ),
        )
    )

    end_df = (
        source_df
        .filter(F.col("epoch") == 2020)
        .select(
            "grid_cell_id",
            F.col("smod_class").alias(
                "to_smod_class"
            ),
            F.col("population_total").alias(
                "to_population_total"
            ),
            F.col(
                "built_surface_total_m2"
            ).alias(
                "to_built_surface_total_m2"
            ),
        )
    )

    full_period_df = (
        start_df
        .join(
            end_df,
            on="grid_cell_id",
            how="inner",
        )
        .withColumn(
            "from_epoch",
            F.lit(1975),
        )
        .withColumn(
            "to_epoch",
            F.lit(2020),
        )
        .withColumn(
            "from_smod_group",
            add_smod_group(
                "from_smod_class"
            ),
        )
        .withColumn(
            "to_smod_group",
            add_smod_group(
                "to_smod_class"
            ),
        )
        .withColumn(
            "smod_transition",
            F.concat(
                F.col(
                    "from_smod_class"
                ).cast("string"),
                F.lit("_to_"),
                F.col(
                    "to_smod_class"
                ).cast("string"),
            ),
        )
        .withColumn(
            "smod_class_change",
            F.col("to_smod_class")
            - F.col("from_smod_class"),
        )
        .withColumn(
            "smod_class_changed",
            F.col("from_smod_class")
            != F.col("to_smod_class"),
        )
        .withColumn(
            "population_change",
            F.col("to_population_total")
            - F.col("from_population_total"),
        )
        .withColumn(
            "built_surface_change_m2",
            F.col(
                "to_built_surface_total_m2"
            )
            - F.col(
                "from_built_surface_total_m2"
            ),
        )
    )

    full_period_df = add_transition_category(
        full_period_df
    )

    full_period_rows = full_period_df.count()

    full_period_distinct = (
        full_period_df
        .select("grid_cell_id")
        .distinct()
        .count()
    )

    full_period_nulls = full_period_df.filter(
        F.col("from_smod_class").isNull()
        | F.col("to_smod_class").isNull()
        | F.col(
            "smod_transition_category"
        ).isNull()
    ).count()

    require(
        full_period_rows
        == EXPECTED_FULL_PERIOD_ROWS,
        (
            f"Full-period rows={full_period_rows}; "
            f"expected={EXPECTED_FULL_PERIOD_ROWS}"
        ),
    )

    require(
        full_period_distinct
        == EXPECTED_FULL_PERIOD_ROWS,
        (
            "Full-period duplicate grid cells: "
            f"distinct={full_period_distinct}"
        ),
    )

    require(
        full_period_nulls == 0,
        (
            f"Full-period null transitions="
            f"{full_period_nulls}"
        ),
    )

    full_matrix_df = (
        full_period_df
        .groupBy(
            "from_epoch",
            "to_epoch",
            "from_smod_class",
            "to_smod_class",
            "from_smod_group",
            "to_smod_group",
            "smod_transition_category",
        )
        .agg(
            F.count("*").cast("long").alias(
                "grid_cell_count"
            ),
            F.sum(
                "population_change"
            ).alias(
                "population_change_total"
            ),
            F.sum(
                "built_surface_change_m2"
            ).alias(
                "built_surface_change_m2_total"
            ),
        )
        .withColumn(
            "grid_share_percent",
            (
                F.col("grid_cell_count")
                / F.lit(
                    EXPECTED_GRID_CELLS
                )
            )
            * F.lit(100.0),
        )
        .orderBy(
            "from_smod_class",
            "to_smod_class",
        )
    )


    # ========================================================
    # DISPLAY SUMMARIES
    # ========================================================

    print()
    print("=" * 108)
    print("SMOD TRANSITION CATEGORIES BY PERIOD")
    print("=" * 108)

    (
        consecutive_df
        .groupBy(
            "from_epoch",
            "to_epoch",
            "smod_transition_category",
        )
        .count()
        .orderBy(
            "from_epoch",
            "to_epoch",
            "smod_transition_category",
        )
        .show(
            200,
            truncate=False,
        )
    )

    print()
    print("=" * 108)
    print("SMOD CLASS TRANSITION MATRIX BY PERIOD")
    print("=" * 108)

    transition_matrix_df.select(
        "from_epoch",
        "to_epoch",
        "from_smod_class",
        "to_smod_class",
        "smod_transition_category",
        "grid_cell_count",
        F.round(
            "transition_grid_share_percent",
            3,
        ).alias(
            "grid_share_percent"
        ),
        F.round(
            "population_change_total",
            2,
        ).alias(
            "population_change_total"
        ),
        F.round(
            (
                F.col(
                    "built_surface_change_m2_total"
                )
                / F.lit(1_000_000.0)
            ),
            3,
        ).alias(
            "built_change_km2"
        ),
    ).show(
        300,
        truncate=False,
    )

    print()
    print("=" * 108)
    print("FULL-PERIOD 1975-2020 SMOD TRANSITIONS")
    print("=" * 108)

    full_matrix_df.select(
        "from_smod_class",
        "to_smod_class",
        "smod_transition_category",
        "grid_cell_count",
        F.round(
            "grid_share_percent",
            3,
        ).alias(
            "grid_share_percent"
        ),
        F.round(
            "population_change_total",
            2,
        ).alias(
            "population_change_total"
        ),
        F.round(
            (
                F.col(
                    "built_surface_change_m2_total"
                )
                / F.lit(1_000_000.0)
            ),
            3,
        ).alias(
            "built_change_km2"
        ),
    ).show(
        200,
        truncate=False,
    )


    # ========================================================
    # WRITE OUTPUTS TO HDFS
    # ========================================================

    print()
    print("=" * 108)
    print("WRITING SMOD OUTPUTS TO HDFS")
    print("=" * 108)

    (
        consecutive_df
        .repartition(9, "to_epoch")
        .write
        .mode("overwrite")
        .partitionBy("to_epoch")
        .parquet(GRID_OUTPUT_PATH)
    )

    (
        transition_matrix_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(MATRIX_OUTPUT_PATH)
    )

    (
        full_matrix_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(FULL_MATRIX_OUTPUT_PATH)
    )

    require(
        hdfs_exists(GRID_OUTPUT_PATH),
        "Grid SMOD output path missing",
    )

    require(
        hdfs_exists(MATRIX_OUTPUT_PATH),
        "SMOD matrix output path missing",
    )

    require(
        hdfs_exists(FULL_MATRIX_OUTPUT_PATH),
        "Full-period matrix path missing",
    )

    print("Grid transitions: COMPLETE")
    print("Period matrices: COMPLETE")
    print("1975-2020 matrix: COMPLETE")


    # ========================================================
    # READ-BACK VALIDATION
    # ========================================================

    print()
    print("=" * 108)
    print("HDFS READ-BACK VALIDATION")
    print("=" * 108)

    saved_grid_df = (
        spark.read
        .option(
            "basePath",
            GRID_OUTPUT_PATH,
        )
        .parquet(GRID_OUTPUT_PATH)
        .select(*GRID_COLUMNS)
    )

    saved_grid_validation = (
        validate_consecutive(
            saved_grid_df,
            "GRID READ-BACK",
        )
    )

    saved_matrix_df = spark.read.parquet(
        MATRIX_OUTPUT_PATH
    )

    saved_full_matrix_df = spark.read.parquet(
        FULL_MATRIX_OUTPUT_PATH
    )

    matrix_total_cells = (
        saved_matrix_df
        .groupBy(
            "from_epoch",
            "to_epoch",
        )
        .agg(
            F.sum(
                "grid_cell_count"
            ).alias("cells")
        )
    )

    invalid_matrix_periods = (
        matrix_total_cells
        .filter(
            F.col("cells")
            != F.lit(
                EXPECTED_GRID_CELLS
            )
        )
        .count()
    )

    full_matrix_cells = (
        saved_full_matrix_df
        .agg(
            F.sum(
                "grid_cell_count"
            ).alias("cells")
        )
        .first()["cells"]
    )

    require(
        invalid_matrix_periods == 0,
        (
            "One or more period matrices do not "
            "sum to 3,658 cells"
        ),
    )

    require(
        int(full_matrix_cells)
        == EXPECTED_GRID_CELLS,
        (
            "Full-period matrix sums to "
            f"{full_matrix_cells}; expected "
            f"{EXPECTED_GRID_CELLS}"
        ),
    )

    elapsed_seconds = round(
        time.time() - started,
        2,
    )

    print("Grid transition rows:",
          saved_grid_validation["rows"])

    print(
        "Grid duplicate keys:",
        saved_grid_validation[
            "duplicate_keys"
        ],
    )

    print(
        "Invalid matrix periods:",
        invalid_matrix_periods,
    )

    print(
        "Full-period matrix cells:",
        full_matrix_cells,
    )

    print("Read-back validation: PASS")


    # ========================================================
    # FINAL REPORT
    # ========================================================

    category_counts = {
        row["smod_transition_category"]:
        int(row["count"])
        for row in (
            full_period_df
            .groupBy(
                "smod_transition_category"
            )
            .count()
            .collect()
        )
    }

    print()
    print("=" * 108)
    print("SMOD TRANSITION RESULT")
    print("=" * 108)

    print(
        "Grid cells:",
        EXPECTED_GRID_CELLS,
    )

    print(
        "Consecutive transition periods:",
        9,
    )

    print(
        "Consecutive transition rows:",
        saved_grid_validation["rows"],
    )

    print(
        "Full-period comparison rows:",
        full_period_rows,
    )

    print(
        "Full-period categories:",
        category_counts,
    )

    print(
        "Elapsed seconds:",
        elapsed_seconds,
    )

    print(
        "Grid output:",
        GRID_OUTPUT_PATH,
    )

    print(
        "Period matrix output:",
        MATRIX_OUTPUT_PATH,
    )

    print(
        "Full-period matrix output:",
        FULL_MATRIX_OUTPUT_PATH,
    )

    print()
    print("SMOD TRANSITIONS: SUCCESS")

    spark.stop()
    """
).strip() + "\n"


# ============================================================
# COMPILE EXACT SOURCE BEFORE TRANSFER
# ============================================================

compile(
    program_code,
    "build_smod_transitions.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    program_code.encode("utf-8")
).hexdigest()

print("LOCAL PYTHON COMPILE CHECK: PASS")
print("Expected program SHA-256:", local_sha256)


# ============================================================
# TRANSFER EXACT SOURCE AND RUN REMOTELY
# ============================================================

encoded_program = base64.b64encode(
    program_code.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"
LOG="{REMOTE_LOG}"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_program}" \
| base64 -d \
> "${{PROGRAM}}"

ACTUAL_SHA256="$(
    sha256sum "${{PROGRAM}}" \
    | awk '{{print $1}}'
)"

echo "Remote program SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Program checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM CHECK: PASS"

python3 -m py_compile "${{PROGRAM}}"

echo "REMOTE PYTHON COMPILE CHECK: PASS"

echo
echo "HDFS readiness:"

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
    | awk '{{print $2}}' \
    | grep -qx "${{daemon}}" \
    || {{
        echo "ERROR: ${{daemon}} is not running."
        exit 21
    }}

    echo "PASS: ${{daemon}}"
done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {{
    echo "ERROR: HDFS safe mode is enabled."
    exit 22
}}

echo "PASS: HDFS safe mode is OFF"

echo
echo "RUNNING SMOD TRANSITION JOB"

spark-submit \
    --master 'local[1]' \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    "${{PROGRAM}}" \
    2>&1 \
    | tee "${{LOG}}"

status=${{PIPESTATUS[0]}}

echo
echo "SMOD_TRANSITIONS_EXIT_CODE=${{status}}"
echo "Job log: ${{LOG}}"

exit "${{status}}"
"""

smod_result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 13 RESULT")
print("=" * 80)

print(
    "Return code:",
    smod_result.returncode,
)

print(
    "Expected consecutive rows:",
    32922,
)

print(
    "Grid output:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/historical_comparisons/"
    "smod_transitions",
)

print(
    "Period matrix:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/historical_summaries/"
    "smod_transition_matrix",
)

print(
    "Full-period matrix:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/historical_summaries/"
    "smod_1975_2020_matrix",
)

print(
    "Remote log:",
    REMOTE_LOG,
)

LOCAL PYTHON COMPILE CHECK: PASS
Expected program SHA-256: 5f17b5b9fe594f52de5c1db35fa176d16db412ba464eb8c98083c19ca96d1e90
Remote program SHA-256: 5f17b5b9fe594f52de5c1db35fa176d16db412ba464eb8c98083c19ca96d1e90
REMOTE CHECKSUM CHECK: PASS
REMOTE PYTHON COMPILE CHECK: PASS

HDFS readiness:
PASS: NameNode
PASS: DataNode
PASS: SecondaryNameNode
PASS: HDFS safe mode is OFF

RUNNING SMOD TRANSITION JOB
26/11/07 13:32:16 INFO SparkContext: Running Spark version 3.5.9
26/11/07 13:32:16 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/07 13:32:16 INFO SparkContext: Java version 11.0.32.1
26/11/07 13:32:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 13:32:16 INFO ResourceUtils: ==============================================================
26/11/07 13:32:16 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/07 13:32:16 INFO ResourceUtils: =============================

HDFS outputs
Plain Text
Grid-level consecutive transitions:
/data/ghsl/curated/pilots/bengaluru/
historical_comparisons/smod_transitions

Five-year transition matrices:
/data/ghsl/curated/pilots/bengaluru/
historical_summaries/smod_transition_matrix

Direct 1975–2020 matrix:
/data/ghsl/curated/pilots/bengaluru/
historical_summaries/smod_1975_2020_matrix

Full-period settlement findings

Across the 3,658 Bengaluru grid cells from 1975 to 2020:

Transition category	Grid cells	Share
Newly urban-class	1,630	44.56%
Unchanged	1,179	32.23%
Upward class transition	537	14.68%
Downward class transition	165	4.51%
Newly urban-centre	147	4.02%

Thus, approximately 63.3% of cells experienced either a newly urban classification, a newly urban-centre classification, or another upward class transition.

The direct 1975–2020 matrix also shows:

416 cells moved from class 11 directly to class 30.
172 cells moved from class 12 to class 30.
120 cells moved from class 21 to class 30.
202 cells remained class 30 throughout.
The 202 persistent urban-centre cells accounted for approximately 5.30 million of the total population increase and 28.01 km² of built-surface expansion.
Consecutive-period findings

The number of newly urban-class cells was highest during:

Plain Text
2015–2020: 301 cells
2000–2005: 264 cells
2005–2010: 222 cells
2010–2015: 218 cells

The largest formation of new urban-centre cells occurred during:

Plain Text
2000–2005: 122 cells
2010–2015: 114 cells
2005–2010: 98 cells

Only the final 2015–2020 period explicitly recorded cells leaving the urban-centre class:

Plain Text
8 cells:
2 changed from class 30 to class 21
6 changed from class 30 to class 23

These should be interpreted carefully because settlement classes can shift due to both real spatial change and GHSL classification/model effects.

/data/ghsl/curated/pilots/bengaluru/benchmarks/pipeline_manifest
/data/ghsl/curated/pilots/bengaluru/benchmarks/dataset_inventory
/data/ghsl/curated/pilots/bengaluru/benchmarks/job_runtime_summary

In [21]:
# ============================================================
# CELL 14: SPARK / SEDONA BENCHMARK AND PIPELINE MANIFEST
#
# Purpose:
#   Create reproducible technical benchmark artifacts for the
#   Bengaluru GHSL pipeline.
#
# Captures:
#   - Spark, Hadoop, Java, Python, and Sedona versions
#   - EC2 CPU, memory, swap, and filesystem information
#   - HDFS capacity and output sizes
#   - Dataset row counts and file counts
#   - Cell 9-13 runtimes from completed job logs
#   - Historical-batch configuration and recovery details
#   - Architecture and processing-engine responsibilities
#
# HDFS outputs:
#   benchmarks/pipeline_manifest
#   benchmarks/dataset_inventory
#   benchmarks/job_runtime_summary
#
# Safety:
#   - Does not rerun raster processing
#   - Does not modify analytical datasets
#   - Reads existing HDFS datasets and logs
#   - Writes only benchmark/manifest outputs
# ============================================================

import base64
import hashlib
import textwrap

REMOTE_PROGRAM = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_pipeline_benchmark.py"
)

REMOTE_LOG = (
    "/home/ubuntu/ghsl_notebook05_batch/"
    "build_pipeline_benchmark.log"
)

program_code = textwrap.dedent(
    """
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        BooleanType,
        DoubleType,
        LongType,
        StringType,
        StructField,
        StructType,
    )

    from importlib import metadata
    from pathlib import Path

    import json
    import os
    import platform
    import re
    import subprocess
    import time


    # ========================================================
    # CONFIGURATION
    # ========================================================

    EPOCHS = [
        1975,
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
        2020,
    ]

    EXPECTED_GRID_CELLS = 3658
    EXPECTED_LONGITUDINAL_ROWS = 36580
    EXPECTED_TRANSITION_ROWS = 32922

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    BENCHMARK_BASE = (
        f"{HDFS_BASE}/benchmarks"
    )

    MANIFEST_OUTPUT = (
        f"{BENCHMARK_BASE}/pipeline_manifest"
    )

    INVENTORY_OUTPUT = (
        f"{BENCHMARK_BASE}/dataset_inventory"
    )

    RUNTIME_OUTPUT = (
        f"{BENCHMARK_BASE}/job_runtime_summary"
    )

    LONGITUDINAL_PATH = (
        f"{HDFS_BASE}/longitudinal_grid_year/"
        "all_epochs"
    )

    EPOCH_SUMMARY_PATH = (
        f"{HDFS_BASE}/historical_summaries/"
        "epoch_summary"
    )

    POPULATION_TRANSITION_PATH = (
        f"{HDFS_BASE}/historical_comparisons/"
        "population_growth_transitions"
    )

    BUILT_TRANSITION_PATH = (
        f"{HDFS_BASE}/historical_comparisons/"
        "built_surface_transitions"
    )

    SMOD_TRANSITION_PATH = (
        f"{HDFS_BASE}/historical_comparisons/"
        "smod_transitions"
    )

    SMOD_MATRIX_PATH = (
        f"{HDFS_BASE}/historical_summaries/"
        "smod_transition_matrix"
    )

    SMOD_FULL_MATRIX_PATH = (
        f"{HDFS_BASE}/historical_summaries/"
        "smod_1975_2020_matrix"
    )

    LOCAL_LOGS = {
        "historical_epoch_batch": (
            "/home/ubuntu/ghsl_notebook05_batch/"
            "intermediate_epochs_batch.log"
        ),
        "final_40_output_validation": (
            "/home/ubuntu/ghsl_notebook05_batch/"
            "final_validate_all_40.log"
        ),
        "longitudinal_dataset": (
            "/home/ubuntu/ghsl_notebook05_batch/"
            "build_longitudinal_grid_year.log"
        ),
        "epoch_summary": (
            "/home/ubuntu/ghsl_notebook05_batch/"
            "build_epoch_summary.log"
        ),
        "population_transitions": (
            "/home/ubuntu/ghsl_notebook05_batch/"
            "build_population_growth_transitions.log"
        ),
        "built_surface_transitions": (
            "/home/ubuntu/ghsl_notebook05_batch/"
            "build_built_surface_transitions.log"
        ),
        "smod_transitions": (
            "/home/ubuntu/ghsl_notebook05_batch/"
            "build_smod_transitions.log"
        ),
    }


    # ========================================================
    # SPARK SESSION
    # ========================================================

    job_started = time.time()

    spark = (
        SparkSession.builder
        .appName(
            "GHSL_Bengaluru_Pipeline_Benchmark"
        )
        .config("spark.ui.enabled", "false")
        .config(
            "spark.sql.shuffle.partitions",
            "4",
        )
        .config(
            "spark.default.parallelism",
            "2",
        )
        .config(
            "spark.sql.adaptive.enabled",
            "false",
        )
        .config(
            "spark.sql.codegen.wholeStage",
            "false",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")


    # ========================================================
    # HDFS ACCESS
    # ========================================================

    jvm = spark.sparkContext._jvm

    hadoop_conf = (
        spark.sparkContext
        ._jsc
        .hadoopConfiguration()
    )

    hdfs = (
        jvm.org.apache.hadoop.fs.FileSystem.get(
            hadoop_conf
        )
    )

    HadoopPath = jvm.org.apache.hadoop.fs.Path


    def hdfs_exists(path):
        return bool(
            hdfs.exists(
                HadoopPath(path)
            )
        )


    def require(condition, message):
        if not condition:
            raise RuntimeError(message)


    def run_command(command):
        result = subprocess.run(
            command,
            shell=True,
            text=True,
            capture_output=True,
            check=False,
        )

        if result.returncode != 0:
            return None

        return result.stdout.strip()


    def package_version(package_names):
        for package_name in package_names:
            try:
                return metadata.version(
                    package_name
                )
            except metadata.PackageNotFoundError:
                continue

        return "not_detected"


    def content_summary(path):
        require(
            hdfs_exists(path),
            f"HDFS path missing: {path}",
        )

        summary = hdfs.getContentSummary(
            HadoopPath(path)
        )

        return {
            "length_bytes": int(
                summary.getLength()
            ),
            "file_count": int(
                summary.getFileCount()
            ),
            "directory_count": int(
                summary.getDirectoryCount()
            ),
            "space_consumed_bytes": int(
                summary.getSpaceConsumed()
            ),
        }


    def safe_count(path):
        require(
            hdfs_exists(path),
            f"HDFS dataset missing: {path}",
        )

        return int(
            spark.read.parquet(path).count()
        )


    def read_log(log_path):
        path = Path(log_path)

        if not path.is_file():
            return ""

        return path.read_text(
            encoding="utf-8",
            errors="replace",
        )


    def last_float_match(
        text,
        patterns,
    ):
        for pattern in patterns:
            matches = re.findall(
                pattern,
                text,
                flags=re.IGNORECASE,
            )

            if matches:
                return float(matches[-1])

        return None


    # ========================================================
    # VERSION AND SYSTEM INFORMATION
    # ========================================================

    spark_version = spark.version

    hadoop_version = str(
        jvm.org.apache.hadoop.util.VersionInfo
        .getVersion()
    )

    java_version = str(
        jvm.java.lang.System.getProperty(
            "java.version"
        )
    )

    python_version = platform.python_version()

    sedona_version = package_version(
        [
            "apache-sedona",
            "sedona",
        ]
    )

    pyspark_version = package_version(
        [
            "pyspark",
        ]
    )

    cpu_count = os.cpu_count()

    cpu_model = run_command(
        "lscpu | awk -F: "
        "'/Model name/ {gsub(/^ +/, \\\"\\\", $2); "
        "print $2; exit}'"
    )

    kernel_version = platform.release()
    operating_system = platform.platform()

    memory_total_kib = run_command(
        "awk '/MemTotal/ {print $2}' "
        "/proc/meminfo"
    )

    memory_available_kib = run_command(
        "awk '/MemAvailable/ {print $2}' "
        "/proc/meminfo"
    )

    swap_total_kib = run_command(
        "awk '/SwapTotal/ {print $2}' "
        "/proc/meminfo"
    )

    root_disk_total_bytes = run_command(
        "df -B1 / | awk 'NR==2 {print $2}'"
    )

    root_disk_available_bytes = run_command(
        "df -B1 / | awk 'NR==2 {print $4}'"
    )

    hdfs_capacity_bytes = int(
        hdfs.getStatus().getCapacity()
    )

    hdfs_used_bytes = int(
        hdfs.getStatus().getUsed()
    )

    hdfs_remaining_bytes = int(
        hdfs.getStatus().getRemaining()
    )


    # ========================================================
    # DATASET INVENTORY
    # ========================================================

    inventory_specs = [
        (
            "longitudinal_grid_year",
            LONGITUDINAL_PATH,
            EXPECTED_LONGITUDINAL_ROWS,
            "grid-year",
        ),
        (
            "epoch_summary",
            EPOCH_SUMMARY_PATH,
            10,
            "epoch",
        ),
        (
            "population_growth_transitions",
            POPULATION_TRANSITION_PATH,
            EXPECTED_TRANSITION_ROWS,
            "grid-transition",
        ),
        (
            "built_surface_transitions",
            BUILT_TRANSITION_PATH,
            EXPECTED_TRANSITION_ROWS,
            "grid-transition",
        ),
        (
            "smod_transitions",
            SMOD_TRANSITION_PATH,
            EXPECTED_TRANSITION_ROWS,
            "grid-transition",
        ),
        (
            "smod_transition_matrix",
            SMOD_MATRIX_PATH,
            None,
            "class-transition-summary",
        ),
        (
            "smod_1975_2020_matrix",
            SMOD_FULL_MATRIX_PATH,
            None,
            "class-transition-summary",
        ),
    ]

    inventory_rows = []

    print("=" * 108)
    print("HDFS DATASET INVENTORY")
    print("=" * 108)

    for (
        dataset_name,
        dataset_path,
        expected_rows,
        grain,
    ) in inventory_specs:
        row_count = safe_count(dataset_path)

        summary = content_summary(
            dataset_path
        )

        if expected_rows is None:
            row_validation_passed = True
        else:
            row_validation_passed = (
                row_count == expected_rows
            )

        require(
            row_validation_passed,
            (
                f"{dataset_name}: rows={row_count}; "
                f"expected={expected_rows}"
            ),
        )

        inventory_rows.append(
            {
                "dataset_name": dataset_name,
                "hdfs_path": dataset_path,
                "grain": grain,
                "row_count": row_count,
                "expected_row_count": (
                    expected_rows
                ),
                "row_validation_passed": (
                    row_validation_passed
                ),
                "logical_length_bytes": (
                    summary["length_bytes"]
                ),
                "space_consumed_bytes": (
                    summary[
                        "space_consumed_bytes"
                    ]
                ),
                "file_count": (
                    summary["file_count"]
                ),
                "directory_count": (
                    summary[
                        "directory_count"
                    ]
                ),
            }
        )

        print(
            f"{dataset_name:<34} "
            f"rows={row_count:<7} "
            f"files={summary['file_count']:<4} "
            f"bytes={summary['length_bytes']:<12} "
            "status=PASS"
        )


    # ========================================================
    # VERIFY THE 40 EPOCH-PRODUCT OUTPUTS
    # ========================================================

    epoch_output_count = 0
    epoch_output_rows = 0
    epoch_output_bytes = 0
    epoch_output_files = 0

    for epoch in EPOCHS:
        product_paths = [
            (
                f"{HDFS_BASE}/product=population/"
                f"epoch={epoch}/grid_1km"
            ),
            (
                f"{HDFS_BASE}/product=smod/"
                f"epoch={epoch}/grid_1km"
            ),
            (
                f"{HDFS_BASE}/product=built_surface/"
                f"epoch={epoch}/grid_1km"
            ),
            (
                f"{HDFS_BASE}/enriched_grid_year/"
                f"epoch={epoch}"
            ),
        ]

        for path in product_paths:
            require(
                hdfs_exists(path),
                f"Missing historical output: {path}",
            )

            rows = safe_count(path)

            require(
                rows == EXPECTED_GRID_CELLS,
                (
                    f"{path}: rows={rows}; "
                    f"expected={EXPECTED_GRID_CELLS}"
                ),
            )

            summary = content_summary(path)

            epoch_output_count += 1
            epoch_output_rows += rows
            epoch_output_bytes += (
                summary["length_bytes"]
            )
            epoch_output_files += (
                summary["file_count"]
            )

    require(
        epoch_output_count == 40,
        (
            f"Historical output count="
            f"{epoch_output_count}; expected=40"
        ),
    )

    require(
        epoch_output_rows
        == 40 * EXPECTED_GRID_CELLS,
        (
            f"Historical output row total="
            f"{epoch_output_rows}"
        ),
    )

    print()
    print("Historical outputs:", epoch_output_count)
    print(
        "Historical output row total:",
        epoch_output_rows,
    )
    print(
        "Historical output logical bytes:",
        epoch_output_bytes,
    )
    print(
        "Historical output files:",
        epoch_output_files,
    )


    # ========================================================
    # PARSE COMPLETED JOB RUNTIMES
    # ========================================================

    runtime_rows = []

    runtime_patterns = {
        "historical_epoch_batch": [
            (
                r"total runtime[^0-9]*"
                r"([0-9]+(?:\\.[0-9]+)?)"
                r"\\s*hours"
            ),
            (
                r"([0-9]+(?:\\.[0-9]+)?)"
                r"\\s*hours"
            ),
        ],
        "longitudinal_dataset": [
            (
                r"elapsed seconds[^0-9]*"
                r"([0-9]+(?:\\.[0-9]+)?)"
            ),
        ],
        "epoch_summary": [
            (
                r"elapsed seconds[^0-9]*"
                r"([0-9]+(?:\\.[0-9]+)?)"
            ),
        ],
        "population_transitions": [
            (
                r"elapsed seconds[^0-9]*"
                r"([0-9]+(?:\\.[0-9]+)?)"
            ),
        ],
        "built_surface_transitions": [
            (
                r"elapsed seconds[^0-9]*"
                r"([0-9]+(?:\\.[0-9]+)?)"
            ),
        ],
        "smod_transitions": [
            (
                r"elapsed seconds[^0-9]*"
                r"([0-9]+(?:\\.[0-9]+)?)"
            ),
        ],
    }

    known_success_markers = {
        "historical_epoch_batch": [
            "BATCH_EXIT_CODE=0",
            "BATCH PIPELINE: SUCCESS",
        ],
        "final_40_output_validation": [
            "FINAL VALIDATION: SUCCESS",
        ],
        "longitudinal_dataset": [
            "LONGITUDINAL DATASET: SUCCESS",
        ],
        "epoch_summary": [
            "EPOCH SUMMARY: SUCCESS",
        ],
        "population_transitions": [
            (
                "POPULATION-GROWTH "
                "TRANSITIONS: SUCCESS"
            ),
        ],
        "built_surface_transitions": [
            (
                "BUILT-SURFACE "
                "TRANSITIONS: SUCCESS"
            ),
        ],
        "smod_transitions": [
            "SMOD TRANSITIONS: SUCCESS",
        ],
    }

    for job_name, log_path in LOCAL_LOGS.items():
        text = read_log(log_path)

        log_exists = bool(text)

        success_markers = (
            known_success_markers.get(
                job_name,
                [],
            )
        )

        success = (
            log_exists
            and any(
                marker in text
                for marker in success_markers
            )
        )

        runtime_seconds = None
        runtime_source = "not_available"

        if job_name == "historical_epoch_batch":
            runtime_hours = last_float_match(
                text,
                runtime_patterns.get(
                    job_name,
                    [],
                ),
            )

            if runtime_hours is not None:
                runtime_seconds = (
                    runtime_hours * 3600.0
                )
                runtime_source = (
                    "parsed_hours_from_log"
                )

        elif job_name in runtime_patterns:
            runtime_seconds = last_float_match(
                text,
                runtime_patterns[job_name],
            )

            if runtime_seconds is not None:
                runtime_source = (
                    "parsed_elapsed_seconds_from_log"
                )

        runtime_rows.append(
            {
                "job_name": job_name,
                "log_path": log_path,
                "log_exists": log_exists,
                "success_marker_found": success,
                "runtime_seconds": (
                    runtime_seconds
                ),
                "runtime_source": runtime_source,
            }
        )


    # ========================================================
    # PIPELINE MANIFEST
    # ========================================================

    manifest_entries = {
        "project": (
            "GHSL Bengaluru historical "
            "data-engineering pipeline"
        ),
        "city": "Bengaluru",
        "study_epochs": json.dumps(EPOCHS),
        "epoch_count": str(len(EPOCHS)),
        "grid_resolution_m": "1000",
        "grid_cells_per_epoch": str(
            EXPECTED_GRID_CELLS
        ),
        "historical_output_count": "40",
        "longitudinal_row_count": str(
            EXPECTED_LONGITUDINAL_ROWS
        ),
        "transition_row_count_each": str(
            EXPECTED_TRANSITION_ROWS
        ),
        "spark_version": spark_version,
        "pyspark_version": pyspark_version,
        "sedona_version": sedona_version,
        "hadoop_version": hadoop_version,
        "java_version": java_version,
        "python_version": python_version,
        "operating_system": operating_system,
        "kernel_version": kernel_version,
        "cpu_count": str(cpu_count),
        "cpu_model": str(cpu_model),
        "memory_total_kib": str(
            memory_total_kib
        ),
        "memory_available_kib_at_manifest": str(
            memory_available_kib
        ),
        "swap_total_kib": str(
            swap_total_kib
        ),
        "root_disk_total_bytes": str(
            root_disk_total_bytes
        ),
        "root_disk_available_bytes": str(
            root_disk_available_bytes
        ),
        "hdfs_capacity_bytes": str(
            hdfs_capacity_bytes
        ),
        "hdfs_used_bytes": str(
            hdfs_used_bytes
        ),
        "hdfs_remaining_bytes": str(
            hdfs_remaining_bytes
        ),
        "historical_output_logical_bytes": str(
            epoch_output_bytes
        ),
        "historical_output_file_count": str(
            epoch_output_files
        ),
        "pipeline_storage": (
            "HDFS with Parquet curated outputs"
        ),
        "distributed_processing_engine": (
            "Apache Spark 3.5"
        ),
        "geospatial_engine": (
            "Apache Sedona raster and spatial SQL"
        ),
        "sedona_raster_functions": (
            "RS_FromGeoTiff, RS_Tile, "
            "RS_BandAsArray"
        ),
        "sedona_spatial_functions": (
            "ST_PolygonFromEnvelope, ST_AsBinary"
        ),
        "orchestration_language": (
            "Python with PySpark"
        ),
        "historical_execution_mode": (
            "Sequential epochs with validation "
            "and checkpoint skipping"
        ),
        "historical_population_driver_memory": (
            "5g"
        ),
        "historical_smod_driver_memory": "3g",
        "historical_built_driver_memory": "3g",
        "historical_enriched_driver_memory": "2g",
        "analytical_driver_memory": "3g",
        "swap_configuration": (
            "8 GiB swap enabled during recovery"
        ),
        "memory_incident": (
            "Initial 1995 population run failed "
            "with Java heap space at 3g"
        ),
        "memory_resolution": (
            "Population driver heap increased "
            "to 5g and 8 GiB swap enabled"
        ),
        "input_raster_storage": (
            "GHSL GeoTIFFs in HDFS"
        ),
        "source_processing_pattern": (
            "100 m GHSL pixels aggregated to "
            "canonical 1 km Bengaluru grid"
        ),
        "validation_policy": (
            "HDFS path, Parquet readability, "
            "row count, key uniqueness, nulls, "
            "and read-back validation"
        ),
        "benchmark_created_utc": (
            time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime(),
            )
        ),
    }

    manifest_rows = [
        (
            key,
            value,
        )
        for key, value in sorted(
            manifest_entries.items()
        )
    ]

    manifest_schema = StructType(
        [
            StructField(
                "metric_name",
                StringType(),
                False,
            ),
            StructField(
                "metric_value",
                StringType(),
                True,
            ),
        ]
    )

    manifest_df = spark.createDataFrame(
        manifest_rows,
        schema=manifest_schema,
    )


    # ========================================================
    # CREATE INVENTORY AND RUNTIME DATAFRAMES
    # ========================================================

    inventory_schema = StructType(
        [
            StructField(
                "dataset_name",
                StringType(),
                False,
            ),
            StructField(
                "hdfs_path",
                StringType(),
                False,
            ),
            StructField(
                "grain",
                StringType(),
                False,
            ),
            StructField(
                "row_count",
                LongType(),
                False,
            ),
            StructField(
                "expected_row_count",
                LongType(),
                True,
            ),
            StructField(
                "row_validation_passed",
                BooleanType(),
                False,
            ),
            StructField(
                "logical_length_bytes",
                LongType(),
                False,
            ),
            StructField(
                "space_consumed_bytes",
                LongType(),
                False,
            ),
            StructField(
                "file_count",
                LongType(),
                False,
            ),
            StructField(
                "directory_count",
                LongType(),
                False,
            ),
        ]
    )

    inventory_df = spark.createDataFrame(
        inventory_rows,
        schema=inventory_schema,
    )

    runtime_schema = StructType(
        [
            StructField(
                "job_name",
                StringType(),
                False,
            ),
            StructField(
                "log_path",
                StringType(),
                False,
            ),
            StructField(
                "log_exists",
                BooleanType(),
                False,
            ),
            StructField(
                "success_marker_found",
                BooleanType(),
                False,
            ),
            StructField(
                "runtime_seconds",
                DoubleType(),
                True,
            ),
            StructField(
                "runtime_source",
                StringType(),
                False,
            ),
        ]
    )

    runtime_df = spark.createDataFrame(
        runtime_rows,
        schema=runtime_schema,
    )


    # ========================================================
    # DISPLAY BENCHMARK SUMMARY
    # ========================================================

    print()
    print("=" * 108)
    print("SOFTWARE AND INFRASTRUCTURE")
    print("=" * 108)

    print("Spark version:", spark_version)
    print("PySpark version:", pyspark_version)
    print("Sedona version:", sedona_version)
    print("Hadoop version:", hadoop_version)
    print("Java version:", java_version)
    print("Python version:", python_version)
    print("CPU count:", cpu_count)
    print("CPU model:", cpu_model)
    print("Memory total KiB:", memory_total_kib)
    print("Swap total KiB:", swap_total_kib)
    print("HDFS capacity bytes:", hdfs_capacity_bytes)
    print("HDFS used bytes:", hdfs_used_bytes)

    print()
    print("=" * 108)
    print("JOB RUNTIME SUMMARY")
    print("=" * 108)

    runtime_df.orderBy(
        "job_name"
    ).show(
        20,
        truncate=False,
    )

    print()
    print("=" * 108)
    print("ANALYTICAL DATASET INVENTORY")
    print("=" * 108)

    inventory_df.select(
        "dataset_name",
        "grain",
        "row_count",
        "expected_row_count",
        "row_validation_passed",
        "logical_length_bytes",
        "file_count",
    ).orderBy(
        "dataset_name"
    ).show(
        20,
        truncate=False,
    )


    # ========================================================
    # WRITE BENCHMARK OUTPUTS
    # ========================================================

    print()
    print("=" * 108)
    print("WRITING BENCHMARK ARTIFACTS TO HDFS")
    print("=" * 108)

    (
        manifest_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(MANIFEST_OUTPUT)
    )

    (
        inventory_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(INVENTORY_OUTPUT)
    )

    (
        runtime_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(RUNTIME_OUTPUT)
    )

    require(
        hdfs_exists(MANIFEST_OUTPUT),
        "Manifest output path missing",
    )

    require(
        hdfs_exists(INVENTORY_OUTPUT),
        "Inventory output path missing",
    )

    require(
        hdfs_exists(RUNTIME_OUTPUT),
        "Runtime output path missing",
    )


    # ========================================================
    # READ-BACK VALIDATION
    # ========================================================

    saved_manifest_df = spark.read.parquet(
        MANIFEST_OUTPUT
    )

    saved_inventory_df = spark.read.parquet(
        INVENTORY_OUTPUT
    )

    saved_runtime_df = spark.read.parquet(
        RUNTIME_OUTPUT
    )

    manifest_count = saved_manifest_df.count()
    inventory_count = saved_inventory_df.count()
    runtime_count = saved_runtime_df.count()

    duplicate_manifest_metrics = (
        manifest_count
        - saved_manifest_df
        .select("metric_name")
        .distinct()
        .count()
    )

    duplicate_inventory_names = (
        inventory_count
        - saved_inventory_df
        .select("dataset_name")
        .distinct()
        .count()
    )

    duplicate_runtime_jobs = (
        runtime_count
        - saved_runtime_df
        .select("job_name")
        .distinct()
        .count()
    )

    require(
        manifest_count
        == len(manifest_entries),
        (
            f"Manifest rows={manifest_count}; "
            f"expected={len(manifest_entries)}"
        ),
    )

    require(
        inventory_count
        == len(inventory_specs),
        (
            f"Inventory rows={inventory_count}; "
            f"expected={len(inventory_specs)}"
        ),
    )

    require(
        runtime_count
        == len(LOCAL_LOGS),
        (
            f"Runtime rows={runtime_count}; "
            f"expected={len(LOCAL_LOGS)}"
        ),
    )

    require(
        duplicate_manifest_metrics == 0,
        (
            "Duplicate manifest metrics="
            f"{duplicate_manifest_metrics}"
        ),
    )

    require(
        duplicate_inventory_names == 0,
        (
            "Duplicate inventory names="
            f"{duplicate_inventory_names}"
        ),
    )

    require(
        duplicate_runtime_jobs == 0,
        (
            "Duplicate runtime jobs="
            f"{duplicate_runtime_jobs}"
        ),
    )

    benchmark_elapsed_seconds = round(
        time.time() - job_started,
        2,
    )


    # ========================================================
    # FINAL REPORT
    # ========================================================

    print()
    print("=" * 108)
    print("PIPELINE BENCHMARK RESULT")
    print("=" * 108)

    print(
        "Historical outputs validated:",
        epoch_output_count,
    )

    print(
        "Historical output rows checked:",
        epoch_output_rows,
    )

    print(
        "Analytical datasets inventoried:",
        inventory_count,
    )

    print(
        "Runtime logs inventoried:",
        runtime_count,
    )

    print(
        "Manifest metrics:",
        manifest_count,
    )

    print(
        "Duplicate manifest metrics:",
        duplicate_manifest_metrics,
    )

    print(
        "Benchmark elapsed seconds:",
        benchmark_elapsed_seconds,
    )

    print(
        "Manifest output:",
        MANIFEST_OUTPUT,
    )

    print(
        "Inventory output:",
        INVENTORY_OUTPUT,
    )

    print(
        "Runtime output:",
        RUNTIME_OUTPUT,
    )

    print()
    print(
        "PIPELINE BENCHMARK: SUCCESS"
    )

    spark.stop()
    """
).strip() + "\n"


# ============================================================
# COMPILE EXACT SOURCE BEFORE TRANSFER
# ============================================================

compile(
    program_code,
    "build_pipeline_benchmark.py",
    "exec",
)

local_sha256 = hashlib.sha256(
    program_code.encode("utf-8")
).hexdigest()

print("LOCAL PYTHON COMPILE CHECK: PASS")
print("Expected program SHA-256:", local_sha256)


# ============================================================
# TRANSFER EXACT SOURCE AND RUN REMOTELY
# ============================================================

encoded_program = base64.b64encode(
    program_code.encode("utf-8")
).decode("ascii")

remote_command = f"""
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"
LOG="{REMOTE_LOG}"
EXPECTED_SHA256="{local_sha256}"

echo "{encoded_program}" \
| base64 -d \
> "${{PROGRAM}}"

ACTUAL_SHA256="$(
    sha256sum "${{PROGRAM}}" \
    | awk '{{print $1}}'
)"

echo "Remote program SHA-256: ${{ACTUAL_SHA256}}"

if [ "${{ACTUAL_SHA256}}" != "${{EXPECTED_SHA256}}" ]; then
    echo "ERROR: Program checksum mismatch."
    exit 20
fi

echo "REMOTE CHECKSUM CHECK: PASS"

python3 -m py_compile "${{PROGRAM}}"

echo "REMOTE PYTHON COMPILE CHECK: PASS"

echo
echo "HDFS readiness:"

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
    | awk '{{print $2}}' \
    | grep -qx "${{daemon}}" \
    || {{
        echo "ERROR: ${{daemon}} is not running."
        exit 21
    }}

    echo "PASS: ${{daemon}}"
done

hdfs dfsadmin -safemode get \
| grep -qi "Safe mode is OFF" \
|| {{
    echo "ERROR: HDFS safe mode is enabled."
    exit 22
}}

echo "PASS: HDFS safe mode is OFF"

echo
echo "RUNNING PIPELINE BENCHMARK JOB"

spark-submit \
    --master 'local[1]' \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    "${{PROGRAM}}" \
    2>&1 \
    | tee "${{LOG}}"

status=${{PIPESTATUS[0]}}

echo
echo "PIPELINE_BENCHMARK_EXIT_CODE=${{status}}"
echo "Job log: ${{LOG}}"

exit "${{status}}"
"""

benchmark_result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 14 RESULT")
print("=" * 80)

print(
    "Return code:",
    benchmark_result.returncode,
)

print(
    "Manifest output:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/benchmarks/"
    "pipeline_manifest",
)

print(
    "Inventory output:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/benchmarks/"
    "dataset_inventory",
)

print(
    "Runtime output:",
    "hdfs://localhost:9000/data/ghsl/curated/"
    "pilots/bengaluru/benchmarks/"
    "job_runtime_summary",
)

print(
    "Remote log:",
    REMOTE_LOG,
)

LOCAL PYTHON COMPILE CHECK: PASS
Expected program SHA-256: ee18d4abb5af91115ca3fb00d71d5715e2e7430fc3917ed320c1edee0cd407d7
Remote program SHA-256: ee18d4abb5af91115ca3fb00d71d5715e2e7430fc3917ed320c1edee0cd407d7
REMOTE CHECKSUM CHECK: PASS
REMOTE PYTHON COMPILE CHECK: PASS

HDFS readiness:
PASS: NameNode
PASS: DataNode
PASS: SecondaryNameNode
PASS: HDFS safe mode is OFF

RUNNING PIPELINE BENCHMARK JOB
26/11/07 13:38:24 INFO SparkContext: Running Spark version 3.5.9
26/11/07 13:38:24 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/07 13:38:24 INFO SparkContext: Java version 11.0.32.1
26/11/07 13:38:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/07 13:38:24 INFO ResourceUtils: ==============================================================
26/11/07 13:38:24 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/07 13:38:24 INFO ResourceUtils: ==========================

Benchmark artifacts
Plain Text
/data/ghsl/curated/pilots/bengaluru/benchmarks/
├── pipeline_manifest
├── dataset_inventory
└── job_runtime_summary
Infrastructure captured
Plain Text
Spark: 3.5.9
PySpark: 3.5.9
Apache Sedona: 1.9.1
Hadoop: 3.3.4
Java: 11.0.32.1
Python: 3.12.16

CPU: 2 vCPUs
Processor: Intel Xeon Platinum 8488C
Physical RAM: approximately 7.6 GiB
Swap: approximately 8 GiB
HDFS capacity: approximately 76.45 GiB
HDFS used: approximately 6.63 GiB
Validated project inventory
Historical datasets
Plain Text
Historical outputs: 40
Rows checked across outputs: 146,320
Logical HDFS size: 9,816,624 bytes
Files: 180
Analytical outputs
Dataset	Rows	Logical size	Files
Longitudinal grid-year	36,580	2.05 MB	11
Epoch summary	10	12.74 KB	2
Population transitions	32,922	3.90 MB	10
Built-surface transitions	32,922	5.23 MB	10
SMOD transitions	32,922	2.31 MB	10
SMOD period matrix	230	13.08 KB	2
SMOD 1975–2020 matrix	30	4.64 KB	2

All datasets with predetermined row expectations matched those expectations.

Recorded analytical runtimes
Plain Text
Longitudinal dataset: 28.52 seconds
Epoch summary: 18.04 seconds
Population transitions: 26.61 seconds
Built-surface transitions: 24.31 seconds
SMOD transitions: 26.54 seconds
Benchmark manifest: 23.26 seconds

Bengaluru results currently show
Population
Plain Text
1975: 3.06 million
2020: 14.61 million
Change: +11.55 million
Built surface
Plain Text
1975: 67.09 km²
2020: 342.43 km²
Change: +275.34 km²
Settlement transitions, 1975–2020

Of the 3,658 grid cells:

Plain Text
1,630 newly urban-class
1,179 unchanged
537 other upward-class transitions
165 downward-class transitions
147 newly urban-centre

DO NOT RUN BELOW CELLS

In [ ]:
# ============================================================
# CELL 3: VALIDATE PROVEN NOTEBOOK 04 REMOTE PROGRAMS
#
# Purpose:
#   - Confirm the validated endpoint programs remain on AWS
#   - Compile each program
#   - Inspect active epoch/path markers
#   - Create immutable backup copies for Notebook 05
#
# This cell does NOT:
#   - process a GeoTIFF
#   - launch Spark
#   - modify HDFS
#   - overwrite the validated programs
# ============================================================

template_validation_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

TEMPLATE_DIR="/home/ubuntu/ghsl_notebook05_templates"

mkdir -p "${TEMPLATE_DIR}"


# ------------------------------------------------------------
# Validated Notebook 04 programs
# ------------------------------------------------------------

POPULATION_TEMPLATE_SOURCE="/home/ubuntu/ghsl_notebook04_cell10_population_2020.py"
SMOD_TEMPLATE_SOURCE="/home/ubuntu/ghsl_notebook04_cell11_smod_2020.py"
BUILT_TEMPLATE_SOURCE="/home/ubuntu/ghsl_notebook04_cell16_built_surface_2020.py"
ENRICH_TEMPLATE_SOURCE="/home/ubuntu/ghsl_notebook04_cell17_enrich_endpoints.py"


# ------------------------------------------------------------
# Notebook 05 immutable template copies
# ------------------------------------------------------------

POPULATION_TEMPLATE="${TEMPLATE_DIR}/population_epoch_template.py"
SMOD_TEMPLATE="${TEMPLATE_DIR}/smod_epoch_template.py"
BUILT_TEMPLATE="${TEMPLATE_DIR}/built_surface_epoch_template.py"
ENRICH_TEMPLATE="${TEMPLATE_DIR}/enrich_epoch_template.py"


echo "============================================================"
echo "VALIDATING NOTEBOOK 04 SOURCE PROGRAMS"
echo "============================================================"


# ------------------------------------------------------------
# Check source files
# ------------------------------------------------------------

for source_file in \
    "${POPULATION_TEMPLATE_SOURCE}" \
    "${SMOD_TEMPLATE_SOURCE}" \
    "${BUILT_TEMPLATE_SOURCE}" \
    "${ENRICH_TEMPLATE_SOURCE}"
do
    if [ ! -f "${source_file}" ]; then
        echo "ERROR: Validated source program is missing:"
        echo "${source_file}"
        exit 20
    fi

    if [ ! -s "${source_file}" ]; then
        echo "ERROR: Validated source program is empty:"
        echo "${source_file}"
        exit 21
    fi

    echo
    echo "SOURCE: ${source_file}"
    ls -lh "${source_file}"

    python3 -m py_compile "${source_file}"

    echo "Python syntax: PASS"
done


# ------------------------------------------------------------
# Validate characteristic markers in each program
# ------------------------------------------------------------

echo
echo "============================================================"
echo "POPULATION TEMPLATE MARKERS"
echo "============================================================"

grep -F 'EPOCH = 2020' \
    "${POPULATION_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Population epoch marker missing."
        exit 22
    }

grep -F 'PRODUCT = "population"' \
    "${POPULATION_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Population product marker missing."
        exit 23
    }

grep -F 'RS_BandAsArray' \
    "${POPULATION_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Population Sedona extraction missing."
        exit 24
    }

grep -F 'grid_1km' \
    "${POPULATION_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Population output marker missing."
        exit 25
    }

echo "Population template markers: PASS"


echo
echo "============================================================"
echo "SMOD TEMPLATE MARKERS"
echo "============================================================"

grep -F 'EPOCH = 2020' \
    "${SMOD_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: SMOD epoch marker missing."
        exit 26
    }

grep -F 'PRODUCT = "smod"' \
    "${SMOD_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: SMOD product marker missing."
        exit 27
    }

grep -F 'RS_BandAsArray' \
    "${SMOD_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: SMOD Sedona extraction missing."
        exit 28
    }

echo "SMOD template markers: PASS"


echo
echo "============================================================"
echo "BUILT-SURFACE TEMPLATE MARKERS"
echo "============================================================"

grep -F 'EPOCH = 2020' \
    "${BUILT_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Built-surface epoch marker missing."
        exit 29
    }

grep -F 'PRODUCT = "built_surface"' \
    "${BUILT_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Built-surface product marker missing."
        exit 30
    }

grep -F 'ARRAY(1, 2)' \
    "${BUILT_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Two-band tiling marker missing."
        exit 31
    }

grep -F 'built_surface_nres_m2' \
    "${BUILT_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: NRES output marker missing."
        exit 32
    }

echo "Built-surface template markers: PASS"


echo
echo "============================================================"
echo "ENRICHMENT TEMPLATE MARKERS"
echo "============================================================"

grep -F 'EPOCHS = [1975, 2020]' \
    "${ENRICH_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Enrichment epoch-list marker missing."
        exit 33
    }

grep -F 'population_per_built_hectare' \
    "${ENRICH_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Enrichment density marker missing."
        exit 34
    }

grep -F 'enriched_grid_year' \
    "${ENRICH_TEMPLATE_SOURCE}" \
    >/dev/null \
    || {
        echo "ERROR: Enriched output marker missing."
        exit 35
    }

echo "Enrichment template markers: PASS"


# ------------------------------------------------------------
# Copy to immutable Notebook 05 template directory
# ------------------------------------------------------------

cp \
    "${POPULATION_TEMPLATE_SOURCE}" \
    "${POPULATION_TEMPLATE}"

cp \
    "${SMOD_TEMPLATE_SOURCE}" \
    "${SMOD_TEMPLATE}"

cp \
    "${BUILT_TEMPLATE_SOURCE}" \
    "${BUILT_TEMPLATE}"

cp \
    "${ENRICH_TEMPLATE_SOURCE}" \
    "${ENRICH_TEMPLATE}"


chmod 444 \
    "${POPULATION_TEMPLATE}" \
    "${SMOD_TEMPLATE}" \
    "${BUILT_TEMPLATE}" \
    "${ENRICH_TEMPLATE}"


# ------------------------------------------------------------
# Verify copied templates
# ------------------------------------------------------------

echo
echo "============================================================"
echo "NOTEBOOK 05 TEMPLATE COPIES"
echo "============================================================"

for template_file in \
    "${POPULATION_TEMPLATE}" \
    "${SMOD_TEMPLATE}" \
    "${BUILT_TEMPLATE}" \
    "${ENRICH_TEMPLATE}"
do
    python3 -m py_compile "${template_file}"

    echo
    ls -lh "${template_file}"
    sha256sum "${template_file}"
done


# ------------------------------------------------------------
# Confirm source and copies are byte-identical
# ------------------------------------------------------------

cmp -s \
    "${POPULATION_TEMPLATE_SOURCE}" \
    "${POPULATION_TEMPLATE}" \
    || {
        echo "ERROR: Population template copy differs."
        exit 36
    }

cmp -s \
    "${SMOD_TEMPLATE_SOURCE}" \
    "${SMOD_TEMPLATE}" \
    || {
        echo "ERROR: SMOD template copy differs."
        exit 37
    }

cmp -s \
    "${BUILT_TEMPLATE_SOURCE}" \
    "${BUILT_TEMPLATE}" \
    || {
        echo "ERROR: Built template copy differs."
        exit 38
    }

cmp -s \
    "${ENRICH_TEMPLATE_SOURCE}" \
    "${ENRICH_TEMPLATE}" \
    || {
        echo "ERROR: Enrichment template copy differs."
        exit 39
    }


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

echo
echo "============================================================"
echo "CELL 3 RESULT"
echo "============================================================"

echo "Population validated template: PASS"
echo "SMOD validated template: PASS"
echo "Built-surface validated template: PASS"
echo "Enrichment validated template: PASS"
echo "Template syntax checks: PASS"
echo "Template SHA-256 checksums: RECORDED"
echo "Template directory: ${TEMPLATE_DIR}"
echo "Spark jobs launched: NO"
echo "HDFS modified: NO"
echo "CELL 3: SUCCESS"
'''


result = run_remote(
    template_validation_command,
    timeout=180,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 3 NOTEBOOK RESULT")
print("=" * 80)
print(
    "Remote return code:",
    result.returncode,
)
print("CELL 3: SUCCESS")

VALIDATING NOTEBOOK 04 SOURCE PROGRAMS

SOURCE: /home/ubuntu/ghsl_notebook04_cell10_population_2020.py
-rw-rw-r-- 1 ubuntu ubuntu 23K Nov  4 05:20 /home/ubuntu/ghsl_notebook04_cell10_population_2020.py
Python syntax: PASS

SOURCE: /home/ubuntu/ghsl_notebook04_cell11_smod_2020.py
-rw-rw-r-- 1 ubuntu ubuntu 26K Nov  4 05:31 /home/ubuntu/ghsl_notebook04_cell11_smod_2020.py
Python syntax: PASS

SOURCE: /home/ubuntu/ghsl_notebook04_cell16_built_surface_2020.py
-rw-rw-r-- 1 ubuntu ubuntu 31K Nov  4 06:29 /home/ubuntu/ghsl_notebook04_cell16_built_surface_2020.py
Python syntax: PASS

SOURCE: /home/ubuntu/ghsl_notebook04_cell17_enrich_endpoints.py
-rw-rw-r-- 1 ubuntu ubuntu 33K Nov  4 06:38 /home/ubuntu/ghsl_notebook04_cell17_enrich_endpoints.py
Python syntax: PASS

POPULATION TEMPLATE MARKERS
Population template markers: PASS

SMOD TEMPLATE MARKERS
SMOD template markers: PASS

BUILT-SURFACE TEMPLATE MARKERS
Built-surface template markers: PASS

ENRICHMENT TEMPLATE MARKERS
Enrichment template m

In [ ]:
# ============================================================
# CELL 4: GENERATE POPULATION 1980 PROGRAM
#
# Source:
#   Protected, validated population 2020 template
#
# Result:
#   Separate population_1980.py program
#
# Safety:
#   - no blanket "2020" replacement
#   - protected template remains unchanged
#   - completed 2020 program remains unchanged
#   - canonical 1975 grid reference remains unchanged
#   - no Spark job is launched
#   - no HDFS data is modified
# ============================================================

import base64
import textwrap


generator_program = textwrap.dedent(
    r"""
    from pathlib import Path
    import hashlib


    # ========================================================
    # CONFIGURATION
    # ========================================================

    TARGET_EPOCH = 1980

    TEMPLATE_PATH = Path(
        "/home/ubuntu/ghsl_notebook05_templates/"
        "population_epoch_template.py"
    )

    OUTPUT_DIR = Path(
        "/home/ubuntu/ghsl_notebook05_programs/"
        f"epoch_{TARGET_EPOCH}"
    )

    OUTPUT_PATH = (
        OUTPUT_DIR
        / f"population_{TARGET_EPOCH}.py"
    )

    MANIFEST_PATH = (
        OUTPUT_DIR
        / "population_manifest.txt"
    )

    EXPECTED_TEMPLATE_SHA256 = (
        "f764cd402f7dafe00734c3f7afda7e947"
        "f597a46eb3e2d5618c4831f028b93aa"
    )


    # ========================================================
    # HELPERS
    # ========================================================

    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def replace_exact_count(
        text,
        old_text,
        new_text,
        expected_count,
        description,
    ):
        actual_count = text.count(old_text)

        if actual_count != expected_count:
            raise RuntimeError(
                f"{description}: expected "
                f"{expected_count} occurrence(s), "
                f"found {actual_count}.\n"
                f"Marker: {old_text}"
            )

        return text.replace(
            old_text,
            new_text,
            expected_count,
        )


    def require_fragments(
        text,
        fragments,
        description,
    ):
        missing = [
            fragment
            for fragment in fragments
            if fragment not in text
        ]

        if missing:
            raise RuntimeError(
                f"{description} is missing:\n"
                + "\n".join(missing)
            )


    def forbid_fragments(
        text,
        fragments,
        description,
    ):
        present = [
            fragment
            for fragment in fragments
            if fragment in text
        ]

        if present:
            raise RuntimeError(
                f"{description} contains forbidden content:\n"
                + "\n".join(present)
            )


    # ========================================================
    # VERIFY PROTECTED TEMPLATE
    # ========================================================

    if not TEMPLATE_PATH.is_file():
        raise FileNotFoundError(
            f"Population template not found: "
            f"{TEMPLATE_PATH}"
        )

    template_checksum = sha256_file(
        TEMPLATE_PATH
    )

    if template_checksum != EXPECTED_TEMPLATE_SHA256:
        raise RuntimeError(
            "Protected population template checksum mismatch.\n"
            f"Expected: {EXPECTED_TEMPLATE_SHA256}\n"
            f"Actual:   {template_checksum}"
        )

    template_text = TEMPLATE_PATH.read_text(
        encoding="utf-8"
    )

    print(
        "Protected template:",
        TEMPLATE_PATH,
    )

    print(
        "Template SHA-256:",
        template_checksum,
    )

    print("Template checksum validation: PASS")


    # ========================================================
    # VALIDATE CONFIRMED TEMPLATE STRUCTURE
    # ========================================================

    required_template_fragments = [
        '    "product=population/"',
        '    "epoch=2020/"',
        '    "epoch=1975/"',
        "EPOCH = 2020",
        'PRODUCT = "population"',
        '"GHSL-Bengaluru-2020-"',
        '"population_2020_catalog"',
        "FROM population_2020_catalog",
        "population_2020_keys_df",
        "RS_FromGeoTiff",
        "RS_Tile",
        "RS_BandAsArray",
        'format("binaryFile")',
        ".parquet(OUTPUT_PATH)",
        "EXPECTED_SOURCE_TILES = 1024",
        "EXPECTED_SELECTED_TILES = 12",
        "EXPECTED_GRID_CELLS = 3658",
        "INVALID_SENTINEL = -200.0",
    ]

    require_fragments(
        template_text,
        required_template_fragments,
        "Protected population template",
    )

    if template_text.count(
        '    "epoch=2020/"'
    ) != 1:
        raise RuntimeError(
            "Expected exactly one active 2020 output-path "
            "epoch marker."
        )

    if template_text.count(
        '    "epoch=1975/"'
    ) != 1:
        raise RuntimeError(
            "Expected exactly one canonical 1975 "
            "reference-path marker."
        )

    print("Exact template structure: PASS")


    # ========================================================
    # CREATE A SEPARATE 1980 PROGRAM IN MEMORY
    # ========================================================

    program = template_text


    # --------------------------------------------------------
    # 1. Active output-path epoch
    #
    # Only the one eight-space-indented 2020 path line is
    # changed. The separate 1975 reference line is untouched.
    # --------------------------------------------------------

    program = replace_exact_count(
        program,
        '    "epoch=2020/"',
        f'    "epoch={TARGET_EPOCH}/"',
        1,
        "Population output path",
    )


    # --------------------------------------------------------
    # 2. Active epoch constant
    # --------------------------------------------------------

    program = replace_exact_count(
        program,
        "EPOCH = 2020",
        f"EPOCH = {TARGET_EPOCH}",
        1,
        "Active epoch constant",
    )


    # --------------------------------------------------------
    # 3. Spark application name
    # --------------------------------------------------------

    program = replace_exact_count(
        program,
        '"GHSL-Bengaluru-2020-"',
        f'"GHSL-Bengaluru-{TARGET_EPOCH}-"',
        1,
        "Spark application name",
    )


    # --------------------------------------------------------
    # 4. Temporary Spark catalog view name
    # --------------------------------------------------------

    program = replace_exact_count(
        program,
        '"population_2020_catalog"',
        f'"population_{TARGET_EPOCH}_catalog"',
        1,
        "Temporary catalog view name",
    )

    program = replace_exact_count(
        program,
        "FROM population_2020_catalog",
        f"FROM population_{TARGET_EPOCH}_catalog",
        1,
        "Temporary catalog SQL reference",
    )


    # --------------------------------------------------------
    # 5. Target grid-key DataFrame variable
    #
    # The canonical population_1975_df variable and path are
    # intentionally preserved.
    # --------------------------------------------------------

    program = replace_exact_count(
        program,
        "population_2020_keys_df",
        f"population_{TARGET_EPOCH}_keys_df",
        3,
        "Target grid-key DataFrame variable",
    )


    # --------------------------------------------------------
    # 6. Target-specific messages and headings
    #
    # These affect only diagnostics, not processing logic.
    # --------------------------------------------------------

    message_replacements = {
        "CELL 10: BENGALURU 2020 POPULATION":
            (
                "NOTEBOOK 05: BENGALURU "
                f"{TARGET_EPOCH} POPULATION"
            ),

        "DISCOVER THE 2020 POPULATION SHARD":
            (
                "DISCOVER THE "
                f"{TARGET_EPOCH} POPULATION SHARD"
            ),

        "2020 Bengaluru source shards:":
            f"{TARGET_EPOCH} Bengaluru source shards:",

        "Expected 15 population catalog records for 2020":
            (
                "Expected 15 population catalog "
                f"records for {TARGET_EPOCH}"
            ),

        "Expected exactly one 2020 population shard":
            (
                "Expected exactly one "
                f"{TARGET_EPOCH} population shard"
            ),

        "Unexpected 2020 population shard dimensions.":
            (
                "Unexpected "
                f"{TARGET_EPOCH} population shard "
                "dimensions."
            ),

        "2020 source metadata validation failed.":
            (
                f"{TARGET_EPOCH} source metadata "
                "validation failed."
            ),

        "2020 population resolution is not 100 m.":
            (
                f"{TARGET_EPOCH} population resolution "
                "is not 100 m."
            ),

        "Valid 2020 pilot pixels:":
            f"Valid {TARGET_EPOCH} pilot pixels:",

        "No valid 2020 pilot pixels were found.":
            (
                "No valid "
                f"{TARGET_EPOCH} pilot pixels were found."
            ),

        "Unexpected 2020 grid-cell count.":
            (
                "Unexpected "
                f"{TARGET_EPOCH} grid-cell count."
            ),

        "Duplicate 2020 grid cells were produced.":
            (
                "Duplicate "
                f"{TARGET_EPOCH} grid cells were produced."
            ),

        "One or more 2020 grid cells failed validation.":
            (
                "One or more "
                f"{TARGET_EPOCH} grid cells failed validation."
            ),

        "2020 population conservation failed.":
            (
                f"{TARGET_EPOCH} population "
                "conservation failed."
            ),

        "1975-2020 GRID KEY COMPARISON":
            f"1975-{TARGET_EPOCH} GRID KEY COMPARISON",

        "1975 cells missing from 2020:":
            (
                "1975 cells missing from "
                f"{TARGET_EPOCH}:"
            ),

        "2020 cells absent from 1975:":
            (
                f"{TARGET_EPOCH} cells absent from 1975:"
            ),

        "The 2020 grid keys do not match 1975.":
            (
                "The "
                f"{TARGET_EPOCH} grid keys do not "
                "match 1975."
            ),

        "2020 POPULATION SUMMARY":
            f"{TARGET_EPOCH} POPULATION SUMMARY",

        "HDFS 2020 population write: COMPLETE":
            (
                "HDFS "
                f"{TARGET_EPOCH} population write: COMPLETE"
            ),

        "2020 population read-back count mismatch.":
            (
                f"{TARGET_EPOCH} population read-back "
                "count mismatch."
            ),

        "2020 population read-back total mismatch.":
            (
                f"{TARGET_EPOCH} population read-back "
                "total mismatch."
            ),

        "CELL 10 REMOTE RESULT":
            f"POPULATION {TARGET_EPOCH} REMOTE RESULT",

        "2020 source discovery: PASS":
            f"{TARGET_EPOCH} source discovery: PASS",

        "1975-2020 grid-key agreement: PASS":
            (
                f"1975-{TARGET_EPOCH} "
                "grid-key agreement: PASS"
            ),

        "CELL 10: SUCCESS":
            f"POPULATION {TARGET_EPOCH}: SUCCESS",
    }

    for old_message, new_message in (
        message_replacements.items()
    ):
        if old_message in program:
            program = program.replace(
                old_message,
                new_message,
            )


    # ========================================================
    # STATIC VALIDATION
    # ========================================================

    required_generated_fragments = [
        f'    "epoch={TARGET_EPOCH}/"',
        '    "epoch=1975/"',
        f"EPOCH = {TARGET_EPOCH}",
        'PRODUCT = "population"',
        f'"GHSL-Bengaluru-{TARGET_EPOCH}-"',
        f'"population_{TARGET_EPOCH}_catalog"',
        f"FROM population_{TARGET_EPOCH}_catalog",
        f"population_{TARGET_EPOCH}_keys_df",
        "RS_FromGeoTiff",
        "RS_Tile",
        "RS_BandAsArray",
        'format("binaryFile")',
        ".parquet(OUTPUT_PATH)",
        "EXPECTED_GRID_CELLS = 3658",
        "EXPECTED_SELECTED_TILES = 12",
        "INVALID_SENTINEL = -200.0",
        (
            f"1975-{TARGET_EPOCH} "
            "grid-key agreement: PASS"
        ),
    ]

    require_fragments(
        program,
        required_generated_fragments,
        "Generated population 1980 program",
    )


    forbidden_active_fragments = [
        '    "epoch=2020/"',
        "EPOCH = 2020",
        '"GHSL-Bengaluru-2020-"',
        '"population_2020_catalog"',
        "FROM population_2020_catalog",
        "population_2020_keys_df",
    ]

    forbid_fragments(
        program,
        forbidden_active_fragments,
        "Generated population 1980 program",
    )


    # --------------------------------------------------------
    # Canonical 1975 reference must remain exactly once
    # --------------------------------------------------------

    if program.count(
        '    "epoch=1975/"'
    ) != 1:
        raise RuntimeError(
            "Canonical 1975 reference was modified "
            "or duplicated."
        )


    # --------------------------------------------------------
    # Spark/Sedona-first architecture
    # --------------------------------------------------------

    required_architecture = [
        "spark.read",
        'format("binaryFile")',
        "RS_FromGeoTiff",
        "RS_Tile",
        "RS_BandAsArray",
        "valid_population_pixels",
        "GROUP BY",
        ".parquet(OUTPUT_PATH)",
    ]

    require_fragments(
        program,
        required_architecture,
        "Generated Spark/Sedona architecture",
    )


    forbidden_architecture = [
        "import numpy",
        "import rasterio",
        "@udf",
        "RS_AsGeoTiff(",
        "RS_Clip(",
    ]

    forbid_fragments(
        program,
        forbidden_architecture,
        "Generated Spark/Sedona architecture",
    )


    print("Targeted configuration replacement: PASS")
    print("Canonical 1975 reference preserved: PASS")
    print("Spark/Sedona architecture validation: PASS")


    # ========================================================
    # COMPILE AND SAVE
    # ========================================================

    compile(
        program,
        str(OUTPUT_PATH),
        "exec",
    )

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    OUTPUT_PATH.write_text(
        program,
        encoding="utf-8",
    )

    output_checksum = sha256_file(
        OUTPUT_PATH
    )

    print(
        "Generated program:",
        OUTPUT_PATH,
    )

    print(
        "Generated size:",
        OUTPUT_PATH.stat().st_size,
        "bytes",
    )

    print(
        "Generated SHA-256:",
        output_checksum,
    )

    print("Generated Python syntax: PASS")


    # ========================================================
    # WRITE MANIFEST
    # ========================================================

    manifest_lines = [
        "target_product=population",
        f"target_epoch={TARGET_EPOCH}",
        f"template_path={TEMPLATE_PATH}",
        f"template_sha256={template_checksum}",
        f"program_path={OUTPUT_PATH}",
        f"program_sha256={output_checksum}",
        "canonical_reference_epoch=1975",
        "replacement_strategy=targeted_exact_markers",
        "blanket_replacement=no",
        "spark_jobs_launched=no",
        "hdfs_modified=no",
        "status=generated_not_executed",
    ]

    MANIFEST_PATH.write_text(
        "\n".join(manifest_lines) + "\n",
        encoding="utf-8",
    )


    # ========================================================
    # FINAL RESULT
    # ========================================================

    print()
    print("=" * 80)
    print("CELL 4 REMOTE RESULT")
    print("=" * 80)

    print("Target product: population")
    print("Target epoch:", TARGET_EPOCH)
    print("Template preserved:", TEMPLATE_PATH)
    print("Generated program:", OUTPUT_PATH)
    print("Manifest:", MANIFEST_PATH)
    print("Targeted replacements only: PASS")
    print("Blanket 2020 replacement used: NO")
    print("Canonical 1975 reference preserved: PASS")
    print("Python syntax check: PASS")
    print("Spark/Sedona architecture check: PASS")
    print("Spark jobs launched: NO")
    print("HDFS modified: NO")
    print("CELL 4: SUCCESS")
    """
).strip() + "\n"


# ============================================================
# TRANSFER AND RUN GENERATOR
# ============================================================

encoded_generator = base64.b64encode(
    generator_program.encode("utf-8")
).decode("ascii")

remote_generator_path = (
    "/home/ubuntu/"
    "ghsl_notebook05_generate_population_1980.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_generator}' "
    f"| base64 -d > {remote_generator_path}; "

    f"python3 -m py_compile "
    f"{remote_generator_path}; "

    "echo 'Remote generator syntax: PASS'; "

    f"python3 {remote_generator_path}; "

    "echo; "
    "echo 'Generated files:'; "

    "ls -lh "
    "/home/ubuntu/"
    "ghsl_notebook05_programs/"
    "epoch_1980; "

    "echo; "
    "echo 'Population manifest:'; "

    "cat "
    "/home/ubuntu/"
    "ghsl_notebook05_programs/"
    "epoch_1980/"
    "population_manifest.txt"
)


result = run_remote(
    remote_command,
    timeout=240,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 4 NOTEBOOK RESULT")
print("=" * 80)

print(
    "Remote return code:",
    result.returncode,
)

print("CELL 4: SUCCESS")

Remote generator syntax: PASS
Protected template: /home/ubuntu/ghsl_notebook05_templates/population_epoch_template.py
Template SHA-256: f764cd402f7dafe00734c3f7afda7e947f597a46eb3e2d5618c4831f028b93aa
Template checksum validation: PASS
Exact template structure: PASS
Targeted configuration replacement: PASS
Canonical 1975 reference preserved: PASS
Spark/Sedona architecture validation: PASS
Generated program: /home/ubuntu/ghsl_notebook05_programs/epoch_1980/population_1980.py
Generated size: 23434 bytes
Generated SHA-256: 34d0cbe8c66778588c46d283dbdd4ac7ed28cef3de934db38ff831ba7912fb45
Generated Python syntax: PASS

CELL 4 REMOTE RESULT
Target product: population
Target epoch: 1980
Template preserved: /home/ubuntu/ghsl_notebook05_templates/population_epoch_template.py
Generated program: /home/ubuntu/ghsl_notebook05_programs/epoch_1980/population_1980.py
Manifest: /home/ubuntu/ghsl_notebook05_programs/epoch_1980/population_manifest.txt
Targeted replacements only: PASS
Blanket 2020 replace

In [ ]:
# ============================================================
# CELL 5: RUN AND VALIDATE POPULATION 1980
#
# Executes:
#   /home/ubuntu/ghsl_notebook05_programs/
#   epoch_1980/population_1980.py
#
# Expected output:
#   /data/ghsl/curated/pilots/bengaluru/
#   product=population/epoch=1980/grid_1km
#
# Processing stack:
#   HDFS GeoTIFF
#   -> Spark binaryFile
#   -> Sedona raster decoding
#   -> Sedona 256 x 256 tiling
#   -> Sedona band-array extraction
#   -> Spark SQL finite-value filtering
#   -> Spark SQL 100 m to 1 km aggregation
#   -> HDFS Parquet
#
# This cell runs only population 1980.
# It does not process SMOD or built surface.
# ============================================================

import base64
import textwrap


# ------------------------------------------------------------
# Remote paths
# ------------------------------------------------------------

REMOTE_PROGRAM = (
    "/home/ubuntu/"
    "ghsl_notebook05_programs/"
    "epoch_1980/"
    "population_1980.py"
)

REMOTE_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_programs/"
    "epoch_1980/"
    "population_manifest.txt"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_programs/"
    "epoch_1980/"
    "population_1980.log"
)

HDFS_OUTPUT = (
    "hdfs://localhost:9000"
    "/data/ghsl/curated/pilots/bengaluru/"
    "product=population/"
    "epoch=1980/"
    "grid_1km"
)


# ============================================================
# 1. PRE-FLIGHT VALIDATION
#
# No Spark processing yet.
# ============================================================

preflight_command = f'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"
MANIFEST="{REMOTE_MANIFEST}"

echo "============================================================"
echo "POPULATION 1980 PRE-FLIGHT"
echo "============================================================"

if [ ! -f "${{PROGRAM}}" ]; then
    echo "ERROR: Generated population program is missing:"
    echo "${{PROGRAM}}"
    exit 20
fi

if [ ! -s "${{PROGRAM}}" ]; then
    echo "ERROR: Generated population program is empty."
    exit 21
fi

if [ ! -f "${{MANIFEST}}" ]; then
    echo "ERROR: Population manifest is missing."
    exit 22
fi

python3 -m py_compile "${{PROGRAM}}"

echo "Generated Python syntax: PASS"

grep -F 'EPOCH = 1980' "${{PROGRAM}}" >/dev/null \
    || {{
        echo "ERROR: Active 1980 epoch marker missing."
        exit 23
    }}

grep -F '    "epoch=1980/"' "${{PROGRAM}}" >/dev/null \
    || {{
        echo "ERROR: 1980 output path is missing."
        exit 24
    }}

grep -F '    "epoch=1975/"' "${{PROGRAM}}" >/dev/null \
    || {{
        echo "ERROR: Canonical 1975 reference is missing."
        exit 25
    }}

if grep -Fq 'EPOCH = 2020' "${{PROGRAM}}"; then
    echo "ERROR: Active 2020 epoch remains."
    exit 26
fi

if grep -Fq '    "epoch=2020/"' "${{PROGRAM}}"; then
    echo "ERROR: Active 2020 output path remains."
    exit 27
fi

for marker in \
    'RS_FromGeoTiff' \
    'RS_Tile' \
    'RS_BandAsArray' \
    'valid_population_pixels' \
    '.parquet(OUTPUT_PATH)'
do
    grep -Fq "${{marker}}" "${{PROGRAM}}" \
        || {{
            echo "ERROR: Required architecture marker missing:"
            echo "${{marker}}"
            exit 28
        }}
done

if grep -Eq \
    'import numpy|import rasterio|@udf|RS_AsGeoTiff|RS_Clip\\(' \
    "${{PROGRAM}}"
then
    echo "ERROR: Forbidden Python/raster architecture detected."
    exit 29
fi

for daemon in NameNode DataNode SecondaryNameNode; do
    jps | awk '{{print $2}}' | grep -qx "${{daemon}}" \
        || {{
            echo "ERROR: ${{daemon}} is not running."
            exit 30
        }}
done

SAFE_MODE_STATUS="$(hdfs dfsadmin -safemode get 2>&1)"
echo "${{SAFE_MODE_STATUS}}"

echo "${{SAFE_MODE_STATUS}}" \
    | grep -qi "Safe mode is OFF" \
    || {{
        echo "ERROR: HDFS safe mode is enabled."
        exit 31
    }}

echo
echo "Program:"
ls -lh "${{PROGRAM}}"

echo
echo "Program SHA-256:"
sha256sum "${{PROGRAM}}"

echo
echo "Manifest:"
cat "${{MANIFEST}}"

echo
echo "System memory:"
free -h

echo
echo "Disk:"
df -h / /tmp

echo
echo "Existing Spark processes:"
ps -ef \
    | grep -E \
    'SparkSubmit|spark-submit|population_1980.py' \
    | grep -v grep \
    || true

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
    'SparkSubmit|spark-submit|population_1980.py' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${{RUNNING_COUNT}}" -ne 0 ]; then
    echo "ERROR: Another Spark job is already running."
    exit 32
fi

echo
echo "Population 1980 pre-flight: PASS"
'''


preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 2. EXECUTE POPULATION 1980 SPARK/SEDONA PROGRAM
#
# Output is streamed to the remote log and returned after the
# Spark job finishes.
# ============================================================

run_command = f'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

rm -f "{REMOTE_LOG}"

spark-submit \
    --master local[1] \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    --packages \
org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,org.datasyslab:geotools-wrapper:1.9.1-33.5 \
    --repositories \
https://artifacts.unidata.ucar.edu/repository/unidata-all \
    "{REMOTE_PROGRAM}" \
    2>&1 \
    | tee "{REMOTE_LOG}"

status=${{PIPESTATUS[0]}}

echo "SPARK_EXIT_CODE=${{status}}"

exit "${{status}}"
'''


print()
print("=" * 80)
print("STARTING POPULATION 1980 SPARK/SEDONA JOB")
print("=" * 80)
print("Program:", REMOTE_PROGRAM)
print("HDFS output:", HDFS_OUTPUT)
print("Remote log:", REMOTE_LOG)
print()


result = run_remote(
    run_command,
    timeout=3600,
    show_output=True,
)


# ============================================================
# 3. INDEPENDENT HDFS OUTPUT VALIDATION
#
# This validation runs in a separate Spark process and does not
# rely only on the processing program's success message.
# ============================================================

validation_program = textwrap.dedent(
    r'''
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F


    OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=1980/"
        "grid_1km"
    )

    REFERENCE_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=1975/"
        "grid_1km"
    )

    EXPECTED_ROWS = 3658
    EXPECTED_COMPLETE = 3480
    EXPECTED_PARTIAL = 178


    spark = (
        SparkSession.builder
        .appName(
            "GHSL-Population-1980-"
            "Independent-Validation"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    try:
        output_df = spark.read.parquet(
            OUTPUT_PATH
        )

        reference_df = spark.read.parquet(
            REFERENCE_PATH
        )

        output_count = output_df.count()

        if output_count != EXPECTED_ROWS:
            raise RuntimeError(
                "Expected "
                f"{EXPECTED_ROWS} output rows, "
                f"found {output_count}."
            )

        duplicate_count = (
            output_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if duplicate_count != 0:
            raise RuntimeError(
                "Population 1980 contains duplicate keys."
            )

        invalid_epoch_count = (
            output_df
            .filter(
                F.col("epoch") != F.lit(1980)
            )
            .count()
        )

        if invalid_epoch_count != 0:
            raise RuntimeError(
                "Population 1980 contains rows with "
                "the wrong epoch."
            )

        invalid_value_count = (
            output_df
            .filter(
                F.col("population_total").isNull()
                | F.isnan("population_total")
                | (
                    F.col("population_total")
                    < F.lit(0.0)
                )
                | (
                    F.col("valid_100m_pixels")
                    <= F.lit(0)
                )
                | (
                    F.col("valid_100m_pixels")
                    > F.lit(100)
                )
                | (
                    F.col(
                        "valid_coverage_fraction"
                    )
                    <= F.lit(0.0)
                )
                | (
                    F.col(
                        "valid_coverage_fraction"
                    )
                    > F.lit(1.0)
                )
            )
            .count()
        )

        if invalid_value_count != 0:
            raise RuntimeError(
                "Population 1980 contains invalid values."
            )

        complete_count = (
            output_df
            .filter(
                F.col("valid_100m_pixels")
                == F.lit(100)
            )
            .count()
        )

        partial_count = (
            output_df
            .filter(
                F.col("valid_100m_pixels")
                < F.lit(100)
            )
            .count()
        )

        if complete_count != EXPECTED_COMPLETE:
            raise RuntimeError(
                "Expected "
                f"{EXPECTED_COMPLETE} complete cells, "
                f"found {complete_count}."
            )

        if partial_count != EXPECTED_PARTIAL:
            raise RuntimeError(
                "Expected "
                f"{EXPECTED_PARTIAL} partial cells, "
                f"found {partial_count}."
            )

        output_keys = output_df.select(
            "grid_cell_id"
        )

        reference_keys = reference_df.select(
            "grid_cell_id"
        )

        missing_from_1980 = (
            reference_keys
            .join(
                output_keys,
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        extra_in_1980 = (
            output_keys
            .join(
                reference_keys,
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        if (
            missing_from_1980 != 0
            or extra_in_1980 != 0
        ):
            raise RuntimeError(
                "Population 1980 grid keys do not "
                "match the 1975 canonical grid."
            )

        summary = (
            output_df
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "population_total"
                ).alias(
                    "population_total"
                ),

                F.avg(
                    "population_total"
                ).alias(
                    "average_population_per_cell"
                ),

                F.max(
                    "population_total"
                ).alias(
                    "maximum_population_cell"
                ),

                F.sum(
                    "positive_population_pixels"
                ).alias(
                    "positive_population_pixels"
                ),

                F.sum(
                    "zero_population_pixels"
                ).alias(
                    "zero_population_pixels"
                ),

                F.min(
                    "valid_coverage_fraction"
                ).alias(
                    "minimum_coverage"
                ),

                F.max(
                    "valid_coverage_fraction"
                ).alias(
                    "maximum_coverage"
                ),
            )
            .first()
        )

        population_total = float(
            summary["population_total"]
        )

        if population_total <= 0.0:
            raise RuntimeError(
                "Population 1980 total is not positive."
            )

        print("=" * 80)
        print("POPULATION 1980 INDEPENDENT VALIDATION")
        print("=" * 80)
        print(
            "Grid cells:",
            summary["grid_cell_count"],
        )
        print(
            "Complete cells:",
            complete_count,
        )
        print(
            "Partial cells:",
            partial_count,
        )
        print(
            "Population total:",
            population_total,
        )
        print(
            "Average population per cell:",
            summary[
                "average_population_per_cell"
            ],
        )
        print(
            "Maximum population cell:",
            summary[
                "maximum_population_cell"
            ],
        )
        print(
            "Positive population pixels:",
            summary[
                "positive_population_pixels"
            ],
        )
        print(
            "Zero population pixels:",
            summary[
                "zero_population_pixels"
            ],
        )
        print(
            "Coverage range:",
            (
                summary["minimum_coverage"],
                summary["maximum_coverage"],
            ),
        )
        print("Duplicate keys: 0")
        print("Invalid values: 0")
        print("Canonical 1975 key agreement: PASS")
        print("Independent HDFS validation: PASS")

    finally:
        spark.stop()
    '''
).strip() + "\n"


encoded_validation = base64.b64encode(
    validation_program.encode("utf-8")
).decode("ascii")

REMOTE_VALIDATION_PROGRAM = (
    "/home/ubuntu/"
    "ghsl_notebook05_programs/"
    "epoch_1980/"
    "validate_population_1980.py"
)


validation_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_validation}' "
    f"| base64 -d > "
    f"{REMOTE_VALIDATION_PROGRAM}; "

    f"python3 -m py_compile "
    f"{REMOTE_VALIDATION_PROGRAM}; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 1g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=2 "
    "--conf spark.default.parallelism=2 "
    f"{REMOTE_VALIDATION_PROGRAM}"
)


validation_result = run_remote(
    validation_command,
    timeout=600,
    show_output=True,
)


# ============================================================
# 4. UPDATE MANIFEST AFTER SUCCESS
# ============================================================

manifest_update_command = f'''
set -e

MANIFEST="{REMOTE_MANIFEST}"
LOG="{REMOTE_LOG}"

if [ ! -f "${{MANIFEST}}" ]; then
    echo "ERROR: Manifest is missing."
    exit 40
fi

if [ ! -f "${{LOG}}" ]; then
    echo "ERROR: Population log is missing."
    exit 41
fi

grep -F 'POPULATION 1980: SUCCESS' \
    "${{LOG}}" \
    >/dev/null \
    || {{
        echo "ERROR: Population success marker not found."
        exit 42
    }}

sed -i \
    's/status=generated_not_executed/status=executed_and_validated/' \
    "${{MANIFEST}}"

sed -i \
    's/spark_jobs_launched=no/spark_jobs_launched=yes/' \
    "${{MANIFEST}}"

sed -i \
    's/hdfs_modified=no/hdfs_modified=yes_population_1980_output/' \
    "${{MANIFEST}}"

if ! grep -Fq \
    'validation_status=pass' \
    "${{MANIFEST}}"
then
    echo 'validation_status=pass' \
        >> "${{MANIFEST}}"
fi

if ! grep -Fq \
    'hdfs_output={HDFS_OUTPUT}' \
    "${{MANIFEST}}"
then
    echo 'hdfs_output={HDFS_OUTPUT}' \
        >> "${{MANIFEST}}"
fi

if ! grep -Fq \
    'remote_log={REMOTE_LOG}' \
    "${{MANIFEST}}"
then
    echo 'remote_log={REMOTE_LOG}' \
        >> "${{MANIFEST}}"
fi

echo "============================================================"
echo "UPDATED POPULATION 1980 MANIFEST"
echo "============================================================"

cat "${{MANIFEST}}"
'''


manifest_result = run_remote(
    manifest_update_command,
    timeout=120,
    show_output=True,
)


# ============================================================
# FINAL NOTEBOOK RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 5 NOTEBOOK RESULT")
print("=" * 80)

print(
    "Population 1980 Spark return code:",
    result.returncode,
)

print(
    "Independent validation return code:",
    validation_result.returncode,
)

print(
    "Manifest update return code:",
    manifest_result.returncode,
)

print("HDFS output:", HDFS_OUTPUT)
print("Remote log:", REMOTE_LOG)
print("CELL 5: SUCCESS")

POPULATION 1980 PRE-FLIGHT
Generated Python syntax: PASS
Safe mode is OFF

Program:
-rw-rw-r-- 1 ubuntu ubuntu 23K Nov  4 07:20 /home/ubuntu/ghsl_notebook05_programs/epoch_1980/population_1980.py

Program SHA-256:
34d0cbe8c66778588c46d283dbdd4ac7ed28cef3de934db38ff831ba7912fb45  /home/ubuntu/ghsl_notebook05_programs/epoch_1980/population_1980.py

Manifest:
target_product=population
target_epoch=1980
template_path=/home/ubuntu/ghsl_notebook05_templates/population_epoch_template.py
template_sha256=f764cd402f7dafe00734c3f7afda7e947f597a46eb3e2d5618c4831f028b93aa
program_path=/home/ubuntu/ghsl_notebook05_programs/epoch_1980/population_1980.py
program_sha256=34d0cbe8c66778588c46d283dbdd4ac7ed28cef3de934db38ff831ba7912fb45
canonical_reference_epoch=1975
replacement_strategy=targeted_exact_markers
blanket_replacement=no
spark_jobs_launched=no
hdfs_modified=no
status=generated_not_executed

System memory:
               total        used        free      shared  buff/cache   available
Mem:    

In [ ]:
# ============================================================
# CELL 6: GENERATE, RUN, AND VALIDATE SMOD 1980
#
# Input template:
#   /home/ubuntu/ghsl_notebook05_templates/
#   smod_epoch_template.py
#
# Required prior output:
#   population/epoch=1980/grid_1km
#
# New output:
#   product=smod/epoch=1980/grid_1km
#
# Safety:
#   - protected 2020 template remains unchanged
#   - 1975 SMOD reference remains unchanged
#   - only explicit active 2020 configuration is changed
#   - no blanket replacement is used
# ============================================================

import base64
import textwrap


TARGET_EPOCH = 1980

REMOTE_TEMPLATE = (
    "/home/ubuntu/"
    "ghsl_notebook05_templates/"
    "smod_epoch_template.py"
)

REMOTE_DIR = (
    "/home/ubuntu/"
    "ghsl_notebook05_programs/"
    "epoch_1980"
)

REMOTE_PROGRAM = (
    f"{REMOTE_DIR}/smod_1980.py"
)

REMOTE_GENERATOR = (
    "/home/ubuntu/"
    "ghsl_notebook05_generate_smod_1980.py"
)

REMOTE_VALIDATOR = (
    f"{REMOTE_DIR}/validate_smod_1980.py"
)

REMOTE_LOG = (
    f"{REMOTE_DIR}/smod_1980.log"
)

REMOTE_MANIFEST = (
    f"{REMOTE_DIR}/smod_manifest.txt"
)

HDFS_POPULATION_OUTPUT = (
    "hdfs://localhost:9000"
    "/data/ghsl/curated/pilots/bengaluru/"
    "product=population/"
    "epoch=1980/"
    "grid_1km"
)

HDFS_SMOD_OUTPUT = (
    "hdfs://localhost:9000"
    "/data/ghsl/curated/pilots/bengaluru/"
    "product=smod/"
    "epoch=1980/"
    "grid_1km"
)


# ============================================================
# 1. GENERATE THE SMOD 1980 PROGRAM
# ============================================================

generator_program = textwrap.dedent(
    r"""
    from pathlib import Path
    import hashlib


    TARGET_EPOCH = 1980

    TEMPLATE_PATH = Path(
        "/home/ubuntu/ghsl_notebook05_templates/"
        "smod_epoch_template.py"
    )

    OUTPUT_DIR = Path(
        "/home/ubuntu/ghsl_notebook05_programs/"
        f"epoch_{TARGET_EPOCH}"
    )

    OUTPUT_PATH = (
        OUTPUT_DIR
        / f"smod_{TARGET_EPOCH}.py"
    )

    MANIFEST_PATH = (
        OUTPUT_DIR
        / "smod_manifest.txt"
    )

    EXPECTED_TEMPLATE_SHA256 = (
        "361fea01efcde8ffb3f5d85067749352"
        "f354e8ad1bef85f2b9c9e2bdb5dc085c"
    )


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def replace_exact_count(
        text,
        old_text,
        new_text,
        expected_count,
        description,
    ):
        actual_count = text.count(old_text)

        if actual_count != expected_count:
            raise RuntimeError(
                f"{description}: expected "
                f"{expected_count} occurrence(s), "
                f"found {actual_count}.\n"
                f"Marker: {old_text}"
            )

        return text.replace(
            old_text,
            new_text,
            expected_count,
        )


    def require_fragments(
        text,
        fragments,
        description,
    ):
        missing = [
            fragment
            for fragment in fragments
            if fragment not in text
        ]

        if missing:
            raise RuntimeError(
                f"{description} is missing:\n"
                + "\n".join(missing)
            )


    def forbid_fragments(
        text,
        fragments,
        description,
    ):
        present = [
            fragment
            for fragment in fragments
            if fragment in text
        ]

        if present:
            raise RuntimeError(
                f"{description} contains forbidden content:\n"
                + "\n".join(present)
            )


    # --------------------------------------------------------
    # Verify protected template
    # --------------------------------------------------------

    if not TEMPLATE_PATH.is_file():
        raise FileNotFoundError(
            f"SMOD template not found: {TEMPLATE_PATH}"
        )

    template_checksum = sha256_file(
        TEMPLATE_PATH
    )

    if template_checksum != EXPECTED_TEMPLATE_SHA256:
        raise RuntimeError(
            "Protected SMOD template checksum mismatch.\n"
            f"Expected: {EXPECTED_TEMPLATE_SHA256}\n"
            f"Actual:   {template_checksum}"
        )

    template_text = TEMPLATE_PATH.read_text(
        encoding="utf-8"
    )

    print("Protected template:", TEMPLATE_PATH)
    print("Template SHA-256:", template_checksum)
    print("Template checksum validation: PASS")


    # --------------------------------------------------------
    # Validate known template structure
    #
    # Two active paths contain epoch=2020:
    #   population reference
    #   SMOD output
    #
    # One canonical SMOD path contains epoch=1975.
    # --------------------------------------------------------

    required_template_fragments = [
        'PRODUCT = "smod"',
        "EPOCH = 2020",
        '    "epoch=2020/"',
        '    "epoch=1975/"',
        '"GHSL-Bengaluru-2020-SMOD-1km"',
        "RS_FromGeoTiff",
        "RS_Tile",
        "RS_BandAsArray",
        "smod_class",
        "EXPECTED_POPULATION_CELLS = 3658",
        ".parquet(SMOD_OUTPUT_PATH)",
    ]

    require_fragments(
        template_text,
        required_template_fragments,
        "Protected SMOD template",
    )

    active_2020_path_count = template_text.count(
        '    "epoch=2020/"'
    )

    if active_2020_path_count != 2:
        raise RuntimeError(
            "Expected exactly two active 2020 path markers "
            "in the SMOD template, found "
            f"{active_2020_path_count}."
        )

    if template_text.count(
        '    "epoch=1975/"'
    ) != 1:
        raise RuntimeError(
            "Expected exactly one canonical 1975 "
            "SMOD-reference path."
        )

    print("Exact template structure: PASS")


    # --------------------------------------------------------
    # Create separate 1980 program
    # --------------------------------------------------------

    program = template_text

    program = replace_exact_count(
        program,
        '    "epoch=2020/"',
        f'    "epoch={TARGET_EPOCH}/"',
        2,
        "Population-reference and SMOD-output paths",
    )

    program = replace_exact_count(
        program,
        "EPOCH = 2020",
        f"EPOCH = {TARGET_EPOCH}",
        1,
        "Active epoch constant",
    )

    program = replace_exact_count(
        program,
        '"GHSL-Bengaluru-2020-SMOD-1km"',
        f'"GHSL-Bengaluru-{TARGET_EPOCH}-SMOD-1km"',
        1,
        "Spark application name",
    )


    # --------------------------------------------------------
    # Diagnostic-label replacements
    #
    # These do not change processing logic.
    # --------------------------------------------------------

    message_replacements = {
        "CELL 11: BENGALURU 2020 SMOD 1 KM":
            f"NOTEBOOK 05: BENGALURU {TARGET_EPOCH} SMOD 1 KM",

        "2020 SMOD catalog records:":
            f"{TARGET_EPOCH} SMOD catalog records:",

        "Expected exactly one 2020 SMOD source":
            f"Expected exactly one {TARGET_EPOCH} SMOD source",

        "No 2020 SMOD tile intersects Bengaluru.":
            f"No {TARGET_EPOCH} SMOD tile intersects Bengaluru.",

        "2020 SMOD CLASS DISTRIBUTION":
            f"{TARGET_EPOCH} SMOD CLASS DISTRIBUTION",

        "2020 population-SMOD coverage":
            f"{TARGET_EPOCH} population-SMOD coverage",

        "1975-2020 SMOD GRID KEY COMPARISON":
            f"1975-{TARGET_EPOCH} SMOD GRID KEY COMPARISON",

        "1975 SMOD cells missing from 2020:":
            f"1975 SMOD cells missing from {TARGET_EPOCH}:",

        "2020 SMOD cells absent from 1975:":
            f"{TARGET_EPOCH} SMOD cells absent from 1975:",

        "HDFS 2020 SMOD Parquet write: COMPLETE":
            f"HDFS {TARGET_EPOCH} SMOD Parquet write: COMPLETE",

        "SAMPLE 2020 SMOD RECORDS":
            f"SAMPLE {TARGET_EPOCH} SMOD RECORDS",

        "CELL 11 REMOTE RESULT":
            f"SMOD {TARGET_EPOCH} REMOTE RESULT",

        "2020 SMOD source discovery from catalog: PASS":
            (
                f"{TARGET_EPOCH} SMOD source discovery "
                "from catalog: PASS"
            ),

        "HDFS 2020 SMOD GeoTIFF read: PASS":
            f"HDFS {TARGET_EPOCH} SMOD GeoTIFF read: PASS",

        "2020 population-SMOD coverage: PASS":
            f"{TARGET_EPOCH} population-SMOD coverage: PASS",

        "1975-2020 SMOD grid-key agreement: PASS":
            (
                f"1975-{TARGET_EPOCH} SMOD "
                "grid-key agreement: PASS"
            ),

        "HDFS 2020 SMOD Parquet write: PASS":
            f"HDFS {TARGET_EPOCH} SMOD Parquet write: PASS",

        "HDFS 2020 SMOD Parquet read-back: PASS":
            (
                f"HDFS {TARGET_EPOCH} SMOD "
                "Parquet read-back: PASS"
            ),

        "CELL 11: SUCCESS":
            f"SMOD {TARGET_EPOCH}: SUCCESS",
    }

    for old_text, new_text in message_replacements.items():
        if old_text in program:
            program = program.replace(
                old_text,
                new_text,
            )


    # --------------------------------------------------------
    # Static validation
    # --------------------------------------------------------

    required_generated_fragments = [
        f"EPOCH = {TARGET_EPOCH}",
        'PRODUCT = "smod"',
        f'    "epoch={TARGET_EPOCH}/"',
        '    "epoch=1975/"',
        (
            f'"GHSL-Bengaluru-'
            f'{TARGET_EPOCH}-SMOD-1km"'
        ),
        "RS_FromGeoTiff",
        "RS_Tile",
        "RS_BandAsArray",
        "smod_class",
        "EXPECTED_POPULATION_CELLS = 3658",
        ".parquet(SMOD_OUTPUT_PATH)",
        f"SMOD {TARGET_EPOCH}: SUCCESS",
    ]

    require_fragments(
        program,
        required_generated_fragments,
        "Generated SMOD 1980 program",
    )

    forbidden_active_fragments = [
        "EPOCH = 2020",
        '    "epoch=2020/"',
        '"GHSL-Bengaluru-2020-SMOD-1km"',
    ]

    forbid_fragments(
        program,
        forbidden_active_fragments,
        "Generated SMOD 1980 program",
    )

    if program.count(
        '    "epoch=1980/"'
    ) != 2:
        raise RuntimeError(
            "Generated program must contain exactly two "
            "active 1980 paths."
        )

    if program.count(
        '    "epoch=1975/"'
    ) != 1:
        raise RuntimeError(
            "Canonical 1975 SMOD reference was modified."
        )

    forbidden_architecture = [
        "import numpy",
        "import rasterio",
        "@udf",
        "RS_AsGeoTiff(",
        "RS_Clip(",
    ]

    forbid_fragments(
        program,
        forbidden_architecture,
        "Generated SMOD architecture",
    )

    print("Targeted configuration replacement: PASS")
    print("Canonical 1975 SMOD reference preserved: PASS")
    print("Spark/Sedona architecture validation: PASS")


    # --------------------------------------------------------
    # Compile and save
    # --------------------------------------------------------

    compile(
        program,
        str(OUTPUT_PATH),
        "exec",
    )

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    OUTPUT_PATH.write_text(
        program,
        encoding="utf-8",
    )

    output_checksum = sha256_file(
        OUTPUT_PATH
    )

    print("Generated program:", OUTPUT_PATH)
    print(
        "Generated size:",
        OUTPUT_PATH.stat().st_size,
        "bytes",
    )
    print("Generated SHA-256:", output_checksum)
    print("Generated Python syntax: PASS")


    # --------------------------------------------------------
    # Write manifest
    # --------------------------------------------------------

    manifest_lines = [
        "target_product=smod",
        f"target_epoch={TARGET_EPOCH}",
        f"template_path={TEMPLATE_PATH}",
        f"template_sha256={template_checksum}",
        f"program_path={OUTPUT_PATH}",
        f"program_sha256={output_checksum}",
        "population_reference_epoch=1980",
        "canonical_smod_reference_epoch=1975",
        "replacement_strategy=targeted_exact_markers",
        "blanket_replacement=no",
        "spark_jobs_launched=no",
        "hdfs_modified=no",
        "status=generated_not_executed",
    ]

    MANIFEST_PATH.write_text(
        "\n".join(manifest_lines) + "\n",
        encoding="utf-8",
    )

    print()
    print("=" * 80)
    print("SMOD 1980 GENERATOR RESULT")
    print("=" * 80)
    print("Targeted replacements only: PASS")
    print("Blanket 2020 replacement used: NO")
    print("Canonical 1975 reference preserved: PASS")
    print("Population 1980 path configured: PASS")
    print("SMOD 1980 output configured: PASS")
    print("Spark jobs launched: NO")
    print("HDFS modified: NO")
    print("SMOD 1980 GENERATION: SUCCESS")
    """
).strip() + "\n"


encoded_generator = base64.b64encode(
    generator_program.encode("utf-8")
).decode("ascii")

generate_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_generator}' "
    f"| base64 -d > {REMOTE_GENERATOR}; "

    f"python3 -m py_compile {REMOTE_GENERATOR}; "
    "echo 'Remote generator syntax: PASS'; "

    f"python3 {REMOTE_GENERATOR}; "

    "echo; "
    "echo 'SMOD manifest:'; "
    f"cat {REMOTE_MANIFEST}"
)

generate_result = run_remote(
    generate_command,
    timeout=240,
    show_output=True,
)


# ============================================================
# 2. PRE-FLIGHT CHECK
# ============================================================

preflight_command = f'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

PROGRAM="{REMOTE_PROGRAM}"

echo "============================================================"
echo "SMOD 1980 PRE-FLIGHT"
echo "============================================================"

python3 -m py_compile "${{PROGRAM}}"

grep -F 'EPOCH = 1980' "${{PROGRAM}}" >/dev/null
grep -F '    "epoch=1980/"' "${{PROGRAM}}" >/dev/null
grep -F '    "epoch=1975/"' "${{PROGRAM}}" >/dev/null
grep -F 'SMOD 1980: SUCCESS' "${{PROGRAM}}" >/dev/null

if grep -Fq 'EPOCH = 2020' "${{PROGRAM}}"; then
    echo "ERROR: Active 2020 epoch remains."
    exit 20
fi

if grep -Fq '    "epoch=2020/"' "${{PROGRAM}}"; then
    echo "ERROR: Active 2020 path remains."
    exit 21
fi

hdfs dfs -test -e \
"/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1980/grid_1km" \
    || {{
        echo "ERROR: Population 1980 output is missing."
        exit 22
    }}

SAFE_MODE_STATUS="$(hdfs dfsadmin -safemode get 2>&1)"
echo "${{SAFE_MODE_STATUS}}"

echo "${{SAFE_MODE_STATUS}}" \
    | grep -qi "Safe mode is OFF" \
    || {{
        echo "ERROR: HDFS safe mode is enabled."
        exit 23
    }}

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
    'SparkSubmit|spark-submit|smod_1980.py' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${{RUNNING_COUNT}}" -ne 0 ]; then
    echo "ERROR: Another Spark job is already running."
    exit 24
fi

echo "Program:"
ls -lh "${{PROGRAM}}"

echo
echo "Program SHA-256:"
sha256sum "${{PROGRAM}}"

echo
echo "Memory:"
free -h

echo
echo "SMOD 1980 pre-flight: PASS"
'''

preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 3. RUN SMOD 1980
# ============================================================

run_command = f'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

rm -f "{REMOTE_LOG}"

spark-submit \
    --master local[1] \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=2 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    --packages \
org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,org.datasyslab:geotools-wrapper:1.9.1-33.5 \
    --repositories \
https://artifacts.unidata.ucar.edu/repository/unidata-all \
    "{REMOTE_PROGRAM}" \
    2>&1 \
    | tee "{REMOTE_LOG}"

status=${{PIPESTATUS[0]}}

echo "SPARK_EXIT_CODE=${{status}}"

exit "${{status}}"
'''

print()
print("=" * 80)
print("STARTING SMOD 1980 SPARK/SEDONA JOB")
print("=" * 80)
print("Program:", REMOTE_PROGRAM)
print("Output:", HDFS_SMOD_OUTPUT)
print("Log:", REMOTE_LOG)

run_result = run_remote(
    run_command,
    timeout=1800,
    show_output=True,
)


# ============================================================
# 4. INDEPENDENT OUTPUT VALIDATION
# ============================================================

validation_program = textwrap.dedent(
    r'''
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F


    SMOD_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=smod/epoch=1980/grid_1km"
    )

    POPULATION_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/epoch=1980/grid_1km"
    )

    SMOD_1975_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=smod/epoch=1975/grid_1km"
    )

    EXPECTED_ROWS = 3658


    spark = (
        SparkSession.builder
        .appName(
            "GHSL-SMOD-1980-Independent-Validation"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    try:
        smod_df = spark.read.parquet(
            SMOD_PATH
        )

        population_df = spark.read.parquet(
            POPULATION_PATH
        )

        smod_1975_df = spark.read.parquet(
            SMOD_1975_PATH
        )

        row_count = smod_df.count()

        if row_count != EXPECTED_ROWS:
            raise RuntimeError(
                f"Expected {EXPECTED_ROWS} SMOD rows, "
                f"found {row_count}."
            )

        duplicate_count = (
            smod_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if duplicate_count != 0:
            raise RuntimeError(
                "SMOD 1980 contains duplicate keys."
            )

        invalid_epoch_count = (
            smod_df
            .filter(
                F.col("epoch") != F.lit(1980)
            )
            .count()
        )

        if invalid_epoch_count != 0:
            raise RuntimeError(
                "SMOD output contains the wrong epoch."
            )

        invalid_class_count = (
            smod_df
            .filter(
                F.col("smod_class").isNull()
                | (
                    F.col("smod_class")
                    < F.lit(0)
                )
            )
            .count()
        )

        if invalid_class_count != 0:
            raise RuntimeError(
                "SMOD 1980 contains invalid classes."
            )

        population_without_smod = (
            population_df
            .select("grid_cell_id")
            .join(
                smod_df.select("grid_cell_id"),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        smod_without_population = (
            smod_df
            .select("grid_cell_id")
            .join(
                population_df.select("grid_cell_id"),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        if (
            population_without_smod != 0
            or smod_without_population != 0
        ):
            raise RuntimeError(
                "Population-SMOD key coverage failed."
            )

        missing_from_1980 = (
            smod_1975_df
            .select("grid_cell_id")
            .join(
                smod_df.select("grid_cell_id"),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        extra_in_1980 = (
            smod_df
            .select("grid_cell_id")
            .join(
                smod_1975_df.select("grid_cell_id"),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        if (
            missing_from_1980 != 0
            or extra_in_1980 != 0
        ):
            raise RuntimeError(
                "SMOD 1980 keys differ from SMOD 1975."
            )

        print("=" * 80)
        print("SMOD 1980 CLASS DISTRIBUTION")
        print("=" * 80)

        (
            smod_df
            .groupBy("smod_class")
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                )
            )
            .orderBy("smod_class")
            .show(
                100,
                truncate=False,
            )
        )

        print("=" * 80)
        print("SMOD 1980 INDEPENDENT VALIDATION")
        print("=" * 80)
        print("Grid cells:", row_count)
        print("Duplicate keys: 0")
        print("Invalid classes: 0")
        print("Population cells without SMOD: 0")
        print("SMOD cells without population: 0")
        print("Canonical 1975 key agreement: PASS")
        print("Independent HDFS validation: PASS")

    finally:
        spark.stop()
    '''
).strip() + "\n"

encoded_validation = base64.b64encode(
    validation_program.encode("utf-8")
).decode("ascii")

validation_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_validation}' "
    f"| base64 -d > {REMOTE_VALIDATOR}; "

    f"python3 -m py_compile {REMOTE_VALIDATOR}; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 1g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=2 "
    f"{REMOTE_VALIDATOR}"
)

validation_result = run_remote(
    validation_command,
    timeout=600,
    show_output=True,
)


# ============================================================
# 5. UPDATE MANIFEST
# ============================================================

manifest_update_command = f'''
set -e

grep -F 'SMOD 1980: SUCCESS' \
    "{REMOTE_LOG}" \
    >/dev/null \
    || {{
        echo "ERROR: SMOD success marker is absent."
        exit 40
    }}

sed -i \
    's/status=generated_not_executed/status=executed_and_validated/' \
    "{REMOTE_MANIFEST}"

sed -i \
    's/spark_jobs_launched=no/spark_jobs_launched=yes/' \
    "{REMOTE_MANIFEST}"

sed -i \
    's/hdfs_modified=no/hdfs_modified=yes_smod_1980_output/' \
    "{REMOTE_MANIFEST}"

grep -Fq 'validation_status=pass' \
    "{REMOTE_MANIFEST}" \
    || echo 'validation_status=pass' \
       >> "{REMOTE_MANIFEST}"

grep -Fq 'hdfs_output={HDFS_SMOD_OUTPUT}' \
    "{REMOTE_MANIFEST}" \
    || echo 'hdfs_output={HDFS_SMOD_OUTPUT}' \
       >> "{REMOTE_MANIFEST}"

grep -Fq 'remote_log={REMOTE_LOG}' \
    "{REMOTE_MANIFEST}" \
    || echo 'remote_log={REMOTE_LOG}' \
       >> "{REMOTE_MANIFEST}"

echo "============================================================"
echo "UPDATED SMOD 1980 MANIFEST"
echo "============================================================"

cat "{REMOTE_MANIFEST}"
'''

manifest_result = run_remote(
    manifest_update_command,
    timeout=120,
    show_output=True,
)


# ============================================================
# FINAL NOTEBOOK RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 6 NOTEBOOK RESULT")
print("=" * 80)
print(
    "Generator return code:",
    generate_result.returncode,
)
print(
    "Pre-flight return code:",
    preflight_result.returncode,
)
print(
    "SMOD Spark return code:",
    run_result.returncode,
)
print(
    "Independent validation return code:",
    validation_result.returncode,
)
print(
    "Manifest update return code:",
    manifest_result.returncode,
)
print("HDFS output:", HDFS_SMOD_OUTPUT)
print("Remote log:", REMOTE_LOG)
print("CELL 6: SUCCESS")

Remote generator syntax: PASS
Protected template: /home/ubuntu/ghsl_notebook05_templates/smod_epoch_template.py
Template SHA-256: 361fea01efcde8ffb3f5d85067749352f354e8ad1bef85f2b9c9e2bdb5dc085c
Template checksum validation: PASS
Exact template structure: PASS
Targeted configuration replacement: PASS
Canonical 1975 SMOD reference preserved: PASS
Spark/Sedona architecture validation: PASS
Generated program: /home/ubuntu/ghsl_notebook05_programs/epoch_1980/smod_1980.py
Generated size: 26107 bytes
Generated SHA-256: 7b0886a25d4a0c805f34e000bea4ee3eb8413bd7bc88f01c2aaaab9bba84b5fe
Generated Python syntax: PASS

SMOD 1980 GENERATOR RESULT
Targeted replacements only: PASS
Blanket 2020 replacement used: NO
Canonical 1975 reference preserved: PASS
Population 1980 path configured: PASS
SMOD 1980 output configured: PASS
Spark jobs launched: NO
HDFS modified: NO
SMOD 1980 GENERATION: SUCCESS

SMOD manifest:
target_product=smod
target_epoch=1980
template_path=/home/ubuntu/ghsl_notebook05_templates

In [ ]:
# ============================================================
# CELL 7: RESUMABLE INTERMEDIATE-EPOCH BATCH PIPELINE
#
# Epochs:
#   1980, 1985, 1990, 1995,
#   2000, 2005, 2010, 2015
#
# For every epoch:
#   1. Validate or build population 1 km
#   2. Validate or build SMOD 1 km
#   3. Validate or build built surface 1 km
#   4. Validate or build enriched grid-year
#   5. Write an epoch manifest
#
# Important:
#   - Includes 1980 in the authoritative batch.
#   - Existing valid population/SMOD 1980 outputs are skipped.
#   - Runs sequentially, one Spark job at a time.
#   - Preserves 1975 and 2020 outputs.
#   - Does not run products concurrently.
#   - Safe to rerun after interruption.
#
# Primary stack:
#   HDFS + Apache Spark + Apache Sedona + Parquet
#
# Python is orchestration only.
# ============================================================

import base64
import textwrap


# ============================================================
# BATCH CONFIGURATION
# ============================================================

BATCH_EPOCHS = [
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
]

REMOTE_BATCH_ROOT = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch"
)

REMOTE_BATCH_DRIVER = (
    f"{REMOTE_BATCH_ROOT}/"
    "run_intermediate_epochs.py"
)

REMOTE_BATCH_LOG = (
    f"{REMOTE_BATCH_ROOT}/"
    "intermediate_epochs_batch.log"
)

REMOTE_BATCH_MANIFEST = (
    f"{REMOTE_BATCH_ROOT}/"
    "batch_manifest.tsv"
)


# ============================================================
# REMOTE BATCH DRIVER
# ============================================================

batch_driver = textwrap.dedent(
    r"""
    from pathlib import Path
    import hashlib
    import subprocess
    import sys
    import time


    # ========================================================
    # CONFIGURATION
    # ========================================================

    EPOCHS = [
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
    ]

    EXPECTED_ROWS = 3658
    EXPECTED_COMPLETE = 3480
    EXPECTED_PARTIAL = 178

    TEMPLATE_ROOT = Path(
        "/home/ubuntu/ghsl_notebook05_templates"
    )

    PROGRAM_ROOT = Path(
        "/home/ubuntu/ghsl_notebook05_programs"
    )

    BATCH_ROOT = Path(
        "/home/ubuntu/ghsl_notebook05_batch"
    )

    BATCH_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    MANIFEST_PATH = (
        BATCH_ROOT
        / "batch_manifest.tsv"
    )

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    POPULATION_TEMPLATE = (
        TEMPLATE_ROOT
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        TEMPLATE_ROOT
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        TEMPLATE_ROOT
        / "built_surface_epoch_template.py"
    )

    EXPECTED_TEMPLATE_SHA256 = {
        "population": (
            "f764cd402f7dafe00734c3f7afda7e947"
            "f597a46eb3e2d5618c4831f028b93aa"
        ),
        "smod": (
            "361fea01efcde8ffb3f5d85067749352"
            "f354e8ad1bef85f2b9c9e2bdb5dc085c"
        ),
        "built_surface": (
            "d21f49e979bec0d20dea8e42107382509"
            "620a750fad89f77ce9331ea23dba87a"
        ),
    }

    SEDONA_PACKAGES = (
        "org.apache.sedona:"
        "sedona-spark-3.5_2.12:1.9.1,"
        "org.datasyslab:"
        "geotools-wrapper:1.9.1-33.5"
    )

    SEDONA_REPOSITORY = (
        "https://artifacts.unidata.ucar.edu/"
        "repository/unidata-all"
    )


    # ========================================================
    # HELPERS
    # ========================================================

    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(1024 * 1024),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def run_command(
        command,
        description,
        timeout_seconds,
    ):
        print()
        print("=" * 80)
        print(description)
        print("=" * 80)
        print("Command:", " ".join(command))
        print()

        started_at = time.time()

        process = subprocess.run(
            command,
            text=True,
            stdout=sys.stdout,
            stderr=sys.stderr,
            timeout=timeout_seconds,
        )

        elapsed_seconds = (
            time.time() - started_at
        )

        print()
        print(
            "Return code:",
            process.returncode,
        )

        print(
            "Elapsed seconds:",
            round(elapsed_seconds, 2),
        )

        if process.returncode != 0:
            raise RuntimeError(
                f"{description} failed with return code "
                f"{process.returncode}."
            )

        return elapsed_seconds


    def spark_submit_command(
        program_path,
        driver_memory,
        shuffle_partitions,
        use_sedona,
    ):
        command = [
            "spark-submit",
            "--master",
            "local[1]",
            "--driver-memory",
            driver_memory,
            "--conf",
            "spark.ui.enabled=false",
            "--conf",
            (
                "spark.sql.shuffle.partitions="
                f"{shuffle_partitions}"
            ),
            "--conf",
            "spark.default.parallelism=2",
            "--conf",
            "spark.sql.adaptive.enabled=false",
            "--conf",
            "spark.sql.codegen.wholeStage=false",
            "--conf",
            "spark.driver.maxResultSize=256m",
        ]

        if use_sedona:
            command.extend(
                [
                    "--packages",
                    SEDONA_PACKAGES,
                    "--repositories",
                    SEDONA_REPOSITORY,
                ]
            )

        command.append(
            str(program_path)
        )

        return command


    def replace_exact_count(
        text,
        old_text,
        new_text,
        expected_count,
        description,
    ):
        actual_count = text.count(old_text)

        if actual_count != expected_count:
            raise RuntimeError(
                f"{description}: expected "
                f"{expected_count} occurrence(s), "
                f"found {actual_count}.\n"
                f"Marker: {old_text}"
            )

        return text.replace(
            old_text,
            new_text,
            expected_count,
        )


    def require_fragments(
        text,
        fragments,
        description,
    ):
        missing = [
            fragment
            for fragment in fragments
            if fragment not in text
        ]

        if missing:
            raise RuntimeError(
                f"{description} is missing:\n"
                + "\n".join(missing)
            )


    def verify_template(
        product,
        template_path,
    ):
        if not template_path.is_file():
            raise FileNotFoundError(
                f"{product} template missing: "
                f"{template_path}"
            )

        actual_checksum = sha256_file(
            template_path
        )

        expected_checksum = (
            EXPECTED_TEMPLATE_SHA256[product]
        )

        if actual_checksum != expected_checksum:
            raise RuntimeError(
                f"{product} template checksum mismatch.\n"
                f"Expected: {expected_checksum}\n"
                f"Actual:   {actual_checksum}"
            )

        return template_path.read_text(
            encoding="utf-8"
        )


    def hdfs_output_exists(path):
        result = subprocess.run(
            [
                "hdfs",
                "dfs",
                "-test",
                "-e",
                path,
            ],
            text=True,
        )

        return result.returncode == 0


    def append_manifest(
        epoch,
        stage,
        status,
        elapsed_seconds,
        hdfs_path,
    ):
        line = (
            f"{epoch}\t"
            f"{stage}\t"
            f"{status}\t"
            f"{elapsed_seconds:.2f}\t"
            f"{hdfs_path}\n"
        )

        with MANIFEST_PATH.open(
            "a",
            encoding="utf-8",
        ) as handle:
            handle.write(line)


    # ========================================================
    # GENERATE POPULATION PROGRAM
    # ========================================================

    def generate_population_program(
        epoch,
        epoch_dir,
    ):
        template = verify_template(
            "population",
            POPULATION_TEMPLATE,
        )

        program = template

        program = replace_exact_count(
            program,
            '    "epoch=2020/"',
            f'    "epoch={epoch}/"',
            1,
            "Population output path",
        )

        program = replace_exact_count(
            program,
            "EPOCH = 2020",
            f"EPOCH = {epoch}",
            1,
            "Population active epoch",
        )

        program = replace_exact_count(
            program,
            '"GHSL-Bengaluru-2020-"',
            f'"GHSL-Bengaluru-{epoch}-"',
            1,
            "Population application name",
        )

        program = replace_exact_count(
            program,
            '"population_2020_catalog"',
            f'"population_{epoch}_catalog"',
            1,
            "Population catalog view",
        )

        program = replace_exact_count(
            program,
            "FROM population_2020_catalog",
            f"FROM population_{epoch}_catalog",
            1,
            "Population catalog SQL reference",
        )

        program = replace_exact_count(
            program,
            "population_2020_keys_df",
            f"population_{epoch}_keys_df",
            3,
            "Population target key variable",
        )

        message_replacements = {
            "CELL 10: BENGALURU 2020 POPULATION":
                (
                    "BATCH: BENGALURU "
                    f"{epoch} POPULATION"
                ),

            "2020 Bengaluru source shards:":
                f"{epoch} Bengaluru source shards:",

            "1975-2020 GRID KEY COMPARISON":
                f"1975-{epoch} GRID KEY COMPARISON",

            "1975 cells missing from 2020:":
                f"1975 cells missing from {epoch}:",

            "2020 cells absent from 1975:":
                f"{epoch} cells absent from 1975:",

            "1975-2020 grid-key agreement: PASS":
                (
                    f"1975-{epoch} "
                    "grid-key agreement: PASS"
                ),

            "2020 POPULATION SUMMARY":
                f"{epoch} POPULATION SUMMARY",

            "HDFS 2020 population write: COMPLETE":
                (
                    f"HDFS {epoch} population "
                    "write: COMPLETE"
                ),

            "CELL 10 REMOTE RESULT":
                f"POPULATION {epoch} REMOTE RESULT",

            "2020 source discovery: PASS":
                f"{epoch} source discovery: PASS",

            "CELL 10: SUCCESS":
                f"POPULATION {epoch}: SUCCESS",
        }

        for old_text, new_text in (
            message_replacements.items()
        ):
            program = program.replace(
                old_text,
                new_text,
            )

        require_fragments(
            program,
            [
                f"EPOCH = {epoch}",
                f'    "epoch={epoch}/"',
                '    "epoch=1975/"',
                "RS_FromGeoTiff",
                "RS_Tile",
                "RS_BandAsArray",
                ".parquet(OUTPUT_PATH)",
            ],
            f"Population {epoch}",
        )

        if "EPOCH = 2020" in program:
            raise RuntimeError(
                f"Population {epoch} retains active 2020."
            )

        output_path = (
            epoch_dir
            / f"population_{epoch}.py"
        )

        compile(
            program,
            str(output_path),
            "exec",
        )

        output_path.write_text(
            program,
            encoding="utf-8",
        )

        return output_path


    # ========================================================
    # GENERATE SMOD PROGRAM
    # ========================================================

    def generate_smod_program(
        epoch,
        epoch_dir,
    ):
        template = verify_template(
            "smod",
            SMOD_TEMPLATE,
        )

        program = template

        program = replace_exact_count(
            program,
            '    "epoch=2020/"',
            f'    "epoch={epoch}/"',
            2,
            "SMOD population/output paths",
        )

        program = replace_exact_count(
            program,
            "EPOCH = 2020",
            f"EPOCH = {epoch}",
            1,
            "SMOD active epoch",
        )

        program = replace_exact_count(
            program,
            '"GHSL-Bengaluru-2020-SMOD-1km"',
            f'"GHSL-Bengaluru-{epoch}-SMOD-1km"',
            1,
            "SMOD application name",
        )

        message_replacements = {
            "CELL 11: BENGALURU 2020 SMOD 1 KM":
                (
                    "BATCH: BENGALURU "
                    f"{epoch} SMOD 1 KM"
                ),

            "2020 SMOD catalog records:":
                f"{epoch} SMOD catalog records:",

            "2020 SMOD CLASS DISTRIBUTION":
                f"{epoch} SMOD CLASS DISTRIBUTION",

            "1975-2020 SMOD GRID KEY COMPARISON":
                (
                    f"1975-{epoch} "
                    "SMOD GRID KEY COMPARISON"
                ),

            "1975 SMOD cells missing from 2020:":
                (
                    "1975 SMOD cells missing from "
                    f"{epoch}:"
                ),

            "2020 SMOD cells absent from 1975:":
                (
                    f"{epoch} SMOD cells absent "
                    "from 1975:"
                ),

            "CELL 11 REMOTE RESULT":
                f"SMOD {epoch} REMOTE RESULT",

            "1975-2020 SMOD grid-key agreement: PASS":
                (
                    f"1975-{epoch} SMOD "
                    "grid-key agreement: PASS"
                ),

            "CELL 11: SUCCESS":
                f"SMOD {epoch}: SUCCESS",
        }

        for old_text, new_text in (
            message_replacements.items()
        ):
            program = program.replace(
                old_text,
                new_text,
            )

        require_fragments(
            program,
            [
                f"EPOCH = {epoch}",
                f'    "epoch={epoch}/"',
                '    "epoch=1975/"',
                "RS_FromGeoTiff",
                "RS_Tile",
                "RS_BandAsArray",
                ".parquet(SMOD_OUTPUT_PATH)",
            ],
            f"SMOD {epoch}",
        )

        if "EPOCH = 2020" in program:
            raise RuntimeError(
                f"SMOD {epoch} retains active 2020."
            )

        output_path = (
            epoch_dir
            / f"smod_{epoch}.py"
        )

        compile(
            program,
            str(output_path),
            "exec",
        )

        output_path.write_text(
            program,
            encoding="utf-8",
        )

        return output_path


    # ========================================================
    # GENERATE BUILT-SURFACE PROGRAM
    # ========================================================

    def generate_built_program(
        epoch,
        epoch_dir,
    ):
        template = verify_template(
            "built_surface",
            BUILT_TEMPLATE,
        )

        program = template

        # The validated 2020 built template contains:
        #   1. grid-year 2020 reference path
        #   2. built-surface 2020 output path
        program = replace_exact_count(
            program,
            '    "epoch=2020/"',
            f'    "epoch={epoch}/"',
            2,
            "Built grid-reference/output paths",
        )

        program = replace_exact_count(
            program,
            "EPOCH = 2020",
            f"EPOCH = {epoch}",
            1,
            "Built active epoch",
        )

        program = replace_exact_count(
            program,
            '"GHSL-Bengaluru-2020-"',
            f'"GHSL-Bengaluru-{epoch}-"',
            1,
            "Built application name",
        )

        message_replacements = {
            "CELL 16: BENGALURU 2020 BUILT SURFACE":
                (
                    "BATCH: BENGALURU "
                    f"{epoch} BUILT SURFACE"
                ),

            "2020 BUILT-SURFACE SUMMARY":
                f"{epoch} BUILT-SURFACE SUMMARY",

            "2020 analytical-grid alignment: PASS":
                (
                    f"{epoch} analytical-grid "
                    "alignment: PASS"
                ),

            "2020 built-surface discovery: PASS":
                (
                    f"{epoch} built-surface "
                    "discovery: PASS"
                ),

            "CELL 16 REMOTE RESULT":
                (
                    f"BUILT SURFACE {epoch} "
                    "REMOTE RESULT"
                ),

            "CELL 16: SUCCESS":
                f"BUILT SURFACE {epoch}: SUCCESS",
        }

        for old_text, new_text in (
            message_replacements.items()
        ):
            program = program.replace(
                old_text,
                new_text,
            )

        require_fragments(
            program,
            [
                f"EPOCH = {epoch}",
                f'    "epoch={epoch}/"',
                'PRODUCT = "built_surface"',
                "ARRAY(1, 2)",
                "built_surface_total_m2",
                "built_surface_nres_m2",
                ".parquet(OUTPUT_PATH)",
            ],
            f"Built surface {epoch}",
        )

        if "EPOCH = 2020" in program:
            raise RuntimeError(
                f"Built surface {epoch} retains "
                "active 2020."
            )

        output_path = (
            epoch_dir
            / f"built_surface_{epoch}.py"
        )

        compile(
            program,
            str(output_path),
            "exec",
        )

        output_path.write_text(
            program,
            encoding="utf-8",
        )

        return output_path


    # ========================================================
    # CREATE GENERIC ENRICHMENT PROGRAM
    # ========================================================

    def create_enrichment_program(
        epoch,
        epoch_dir,
    ):
        output_path = (
            epoch_dir
            / f"enrich_{epoch}.py"
        )

        program = f'''
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F


    EPOCH = {epoch}
    EXPECTED_ROWS = 3658
    EXPECTED_COMPLETE = 3480
    EXPECTED_PARTIAL = 178

    BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    POPULATION_PATH = (
        f"{{BASE}}/product=population/"
        f"epoch={{EPOCH}}/grid_1km"
    )

    SMOD_PATH = (
        f"{{BASE}}/product=smod/"
        f"epoch={{EPOCH}}/grid_1km"
    )

    BUILT_PATH = (
        f"{{BASE}}/product=built_surface/"
        f"epoch={{EPOCH}}/grid_1km"
    )

    OUTPUT_PATH = (
        f"{{BASE}}/enriched_grid_year/"
        f"epoch={{EPOCH}}"
    )


    spark = (
        SparkSession.builder
        .appName(
            f"GHSL-Bengaluru-{{EPOCH}}-Enrichment"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")


    try:
        population_df = spark.read.parquet(
            POPULATION_PATH
        )

        smod_df = spark.read.parquet(
            SMOD_PATH
        )

        built_df = spark.read.parquet(
            BUILT_PATH
        )


        for name, dataframe in [
            ("population", population_df),
            ("smod", smod_df),
            ("built_surface", built_df),
        ]:
            count = dataframe.count()

            if count != EXPECTED_ROWS:
                raise RuntimeError(
                    f"{{name}} {{EPOCH}} expected "
                    f"{{EXPECTED_ROWS}} rows, found {{count}}."
                )

            duplicate_count = (
                dataframe
                .groupBy("grid_cell_id", "epoch")
                .count()
                .filter(F.col("count") != 1)
                .count()
            )

            if duplicate_count != 0:
                raise RuntimeError(
                    f"{{name}} {{EPOCH}} contains "
                    "duplicate keys."
                )


        joined_df = (
            population_df.alias("p")
            .join(
                smod_df.alias("s"),
                on=[
                    F.col("p.grid_cell_id")
                    == F.col("s.grid_cell_id"),
                    F.col("p.epoch")
                    == F.col("s.epoch"),
                ],
                how="inner",
            )
            .join(
                built_df.alias("b"),
                on=[
                    F.col("p.grid_cell_id")
                    == F.col("b.grid_cell_id"),
                    F.col("p.epoch")
                    == F.col("b.epoch"),
                ],
                how="inner",
            )
        )


        coordinate_mismatch_count = (
            joined_df
            .filter(
                (
                    F.col("p.grid_1km_col")
                    != F.col("s.grid_1km_col")
                )
                | (
                    F.col("p.grid_1km_col")
                    != F.col("b.grid_1km_col")
                )
                | (
                    F.col("p.grid_1km_row")
                    != F.col("s.grid_1km_row")
                )
                | (
                    F.col("p.grid_1km_row")
                    != F.col("b.grid_1km_row")
                )
            )
            .count()
        )

        if coordinate_mismatch_count != 0:
            raise RuntimeError(
                f"Coordinate mismatch for "
                f"{{coordinate_mismatch_count}} cells."
            )


        enriched_df = (
            joined_df
            .select(
                F.lit("bengaluru").alias("city"),
                F.col("p.grid_cell_id"),
                F.col("p.epoch"),
                F.col("p.grid_1km_col"),
                F.col("p.grid_1km_row"),
                F.col("p.grid_min_x"),
                F.col("p.grid_max_x"),
                F.col("p.grid_min_y"),
                F.col("p.grid_max_y"),
                F.col("p.valid_100m_pixels"),
                F.col(
                    "p.valid_coverage_fraction"
                ).alias(
                    "population_coverage_fraction"
                ),
                F.col("p.population_total"),
                F.col("p.population_mean_100m"),
                F.col("p.population_min_100m"),
                F.col("p.population_max_100m"),
                F.col("p.zero_population_pixels"),
                F.col("p.positive_population_pixels"),
                F.col("s.smod_class"),

                F.when(
                    F.col("s.smod_class") == 10,
                    F.lit("water"),
                )
                .when(
                    F.col("s.smod_class").between(
                        11,
                        13,
                    ),
                    F.lit("rural"),
                )
                .when(
                    F.col("s.smod_class").between(
                        21,
                        23,
                    ),
                    F.lit("urban_cluster"),
                )
                .when(
                    F.col("s.smod_class") == 30,
                    F.lit("urban_centre"),
                )
                .otherwise(
                    F.lit("other_or_unknown")
                )
                .alias("settlement_group"),

                F.col(
                    "s.smod_class"
                ).isin(
                    21,
                    22,
                    23,
                    30,
                ).alias("is_urban"),

                (
                    F.col("s.smod_class")
                    == F.lit(30)
                ).alias("is_urban_centre"),

                F.col(
                    "b.valid_100m_pixels"
                ).alias(
                    "built_valid_100m_pixels"
                ),

                F.col(
                    "b.observed_grid_area_m2"
                ).alias(
                    "built_observed_grid_area_m2"
                ),

                F.col("b.built_coverage_fraction"),
                F.col("b.built_surface_total_m2"),
                F.col("b.built_surface_nres_m2"),
                F.col(
                    "b.built_surface_residential_m2"
                ),
                F.col("b.built_surface_fraction"),
                F.col("b.built_surface_percent"),
                F.col(
                    "b.built_surface_nres_fraction"
                ),
                F.col(
                    "b.built_surface_residential_fraction"
                ),
                F.col(
                    "b.nres_share_of_built_surface"
                ),
                F.col("b.built_100m_pixels"),
                F.col("b.nres_100m_pixels"),
                F.col("b.is_complete_built_cell"),
                F.col("b.built_quality_flag"),
                F.col("p.source_path").alias(
                    "population_source_path"
                ),
                F.col("s.source_path").alias(
                    "smod_source_path"
                ),
                F.col("b.source_path").alias(
                    "built_surface_source_path"
                ),
            )
            .withColumn(
                "population_quality_flag",
                F.when(
                    F.col("valid_100m_pixels") == 100,
                    F.lit("complete"),
                ).otherwise(
                    F.lit("partial_pilot_boundary")
                ),
            )
            .withColumn(
                "is_complete_population_cell",
                F.col("valid_100m_pixels") == 100,
            )
            .withColumn(
                "is_complete_all_products",
                (
                    F.col("valid_100m_pixels") == 100
                )
                & F.col("is_complete_built_cell"),
            )
            .withColumn(
                "population_per_built_hectare",
                F.when(
                    F.col("built_surface_total_m2")
                    > 0.0,
                    F.col("population_total")
                    / (
                        F.col("built_surface_total_m2")
                        / 10000.0
                    ),
                ),
            )
            .withColumn(
                "population_per_residential_built_hectare",
                F.when(
                    F.col(
                        "built_surface_residential_m2"
                    ) > 0.0,
                    F.col("population_total")
                    / (
                        F.col(
                            "built_surface_residential_m2"
                        )
                        / 10000.0
                    ),
                ),
            )
            .withColumn(
                "geometry_wkb",
                F.expr(
                    "ST_AsBinary("
                    "ST_PolygonFromEnvelope("
                    "grid_min_x, grid_min_y, "
                    "grid_max_x, grid_max_y"
                    "))"
                ),
            )
            .withColumn(
                "geometry_crs",
                F.lit("ESRI:54009"),
            )
            .withColumn(
                "grid_resolution_m",
                F.lit(1000.0),
            )
            .withColumn(
                "processed_at_utc",
                F.current_timestamp(),
            )
        )


        output_count = enriched_df.count()

        if output_count != EXPECTED_ROWS:
            raise RuntimeError(
                f"Expected {{EXPECTED_ROWS}} enriched "
                f"rows, found {{output_count}}."
            )


        complete_count = (
            enriched_df
            .filter(
                F.col("is_complete_all_products")
            )
            .count()
        )

        partial_count = (
            output_count - complete_count
        )

        if complete_count != EXPECTED_COMPLETE:
            raise RuntimeError(
                f"Expected {{EXPECTED_COMPLETE}} complete "
                f"cells, found {{complete_count}}."
            )

        if partial_count != EXPECTED_PARTIAL:
            raise RuntimeError(
                f"Expected {{EXPECTED_PARTIAL}} partial "
                f"cells, found {{partial_count}}."
            )


        (
            enriched_df
            .repartition(4)
            .write
            .mode("overwrite")
            .parquet(OUTPUT_PATH)
        )


        readback_df = spark.read.parquet(
            OUTPUT_PATH
        )

        if readback_df.count() != EXPECTED_ROWS:
            raise RuntimeError(
                "Enriched read-back count mismatch."
            )


        totals = readback_df.agg(
            F.sum("population_total").alias(
                "population_total"
            ),
            F.sum("built_surface_total_m2").alias(
                "built_surface_total_m2"
            ),
            F.sum("built_surface_nres_m2").alias(
                "built_surface_nres_m2"
            ),
        ).first()


        print("=" * 80)
        print(f"ENRICHED {{EPOCH}} RESULT")
        print("=" * 80)
        print("Grid cells:", EXPECTED_ROWS)
        print("Complete cells:", complete_count)
        print("Partial cells:", partial_count)
        print(
            "Population total:",
            totals["population_total"],
        )
        print(
            "Built-surface total m2:",
            totals["built_surface_total_m2"],
        )
        print(
            "NRES built-surface total m2:",
            totals["built_surface_nres_m2"],
        )
        print("HDFS write/read-back: PASS")
        print(f"ENRICHED {{EPOCH}}: SUCCESS")

    finally:
        spark.stop()
    '''

        program = textwrap.dedent(
            program
        ).strip() + "\n"

        compile(
            program,
            str(output_path),
            "exec",
        )

        output_path.write_text(
            program,
            encoding="utf-8",
        )

        return output_path


    # ========================================================
    # VALIDATION PROGRAM FACTORY
    # ========================================================

    def create_validator(
        epoch,
        stage,
        hdfs_path,
        epoch_dir,
    ):
        output_path = (
            epoch_dir
            / f"validate_{stage}_{epoch}.py"
        )

        stage_specific_validation = ""

        if stage == "population":
            stage_specific_validation = '''
        invalid_count = (
            dataframe
            .filter(
                F.col("population_total").isNull()
                | F.isnan("population_total")
                | (F.col("population_total") < 0.0)
            )
            .count()
        )
    '''

        elif stage == "smod":
            stage_specific_validation = '''
        invalid_count = (
            dataframe
            .filter(
                F.col("smod_class").isNull()
                | (F.col("smod_class") < 0)
            )
            .count()
        )
    '''

        elif stage == "built_surface":
            stage_specific_validation = '''
        invalid_count = (
            dataframe
            .filter(
                F.col("built_surface_total_m2").isNull()
                | F.isnan("built_surface_total_m2")
                | (F.col("built_surface_total_m2") < 0.0)
                | (
                    F.col("built_surface_nres_m2")
                    > F.col("built_surface_total_m2")
                )
            )
            .count()
        )
    '''

        elif stage == "enriched":
            stage_specific_validation = '''
        invalid_count = (
            dataframe
            .filter(
                F.col("population_total").isNull()
                | F.col("smod_class").isNull()
                | F.col("built_surface_total_m2").isNull()
                | F.col("geometry_wkb").isNull()
            )
            .count()
        )
    '''

        else:
            raise RuntimeError(
                f"Unsupported validation stage: {stage}"
            )

        program = f'''
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F


    EPOCH = {epoch}
    STAGE = "{stage}"
    HDFS_PATH = "{hdfs_path}"
    EXPECTED_ROWS = 3658


    spark = (
        SparkSession.builder
        .appName(
            f"GHSL-Validate-{{STAGE}}-{{EPOCH}}"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")


    try:
        dataframe = spark.read.parquet(
            HDFS_PATH
        )

        row_count = dataframe.count()

        if row_count != EXPECTED_ROWS:
            raise RuntimeError(
                f"Expected {{EXPECTED_ROWS}} rows, "
                f"found {{row_count}}."
            )


        duplicate_count = (
            dataframe
            .groupBy("grid_cell_id", "epoch")
            .count()
            .filter(F.col("count") != 1)
            .count()
        )

        if duplicate_count != 0:
            raise RuntimeError(
                f"{{STAGE}} {{EPOCH}} contains "
                "duplicate keys."
            )


        invalid_epoch_count = (
            dataframe
            .filter(F.col("epoch") != F.lit(EPOCH))
            .count()
        )

        if invalid_epoch_count != 0:
            raise RuntimeError(
                f"{{STAGE}} output contains "
                "an incorrect epoch."
            )

    {stage_specific_validation}

        if invalid_count != 0:
            raise RuntimeError(
                f"{{STAGE}} {{EPOCH}} has "
                f"{{invalid_count}} invalid rows."
            )


        print("=" * 80)
        print(
            f"VALIDATION: {{STAGE}} {{EPOCH}}"
        )
        print("=" * 80)
        print("Rows:", row_count)
        print("Duplicate keys: 0")
        print("Invalid rows: 0")
        print("VALIDATION: PASS")

    finally:
        spark.stop()
    '''

        program = textwrap.dedent(
            program
        ).strip() + "\n"

        compile(
            program,
            str(output_path),
            "exec",
        )

        output_path.write_text(
            program,
            encoding="utf-8",
        )

        return output_path


    # ========================================================
    # STAGE EXECUTION
    # ========================================================

    def ensure_stage(
        epoch,
        stage,
        hdfs_path,
        program_factory,
        epoch_dir,
        driver_memory,
        shuffle_partitions,
        use_sedona,
    ):
        validator = create_validator(
            epoch,
            stage,
            hdfs_path,
            epoch_dir,
        )

        if hdfs_output_exists(hdfs_path):
            try:
                elapsed = run_command(
                    spark_submit_command(
                        validator,
                        "1g",
                        2,
                        False,
                    ),
                    (
                        f"VALIDATE EXISTING "
                        f"{stage.upper()} {epoch}"
                    ),
                    900,
                )

                append_manifest(
                    epoch,
                    stage,
                    "existing_valid_skipped",
                    elapsed,
                    hdfs_path,
                )

                print(
                    f"{stage} {epoch}: "
                    "VALID, SKIPPING RECOMPUTATION"
                )

                return

            except Exception as error:
                print(
                    f"Existing {stage} {epoch} "
                    f"is invalid: {error}"
                )

                print(
                    "Rebuilding invalid output..."
                )


        program = program_factory(
            epoch,
            epoch_dir,
        )

        elapsed = run_command(
            spark_submit_command(
                program,
                driver_memory,
                shuffle_partitions,
                use_sedona,
            ),
            f"RUN {stage.upper()} {epoch}",
            7200,
        )

        validation_elapsed = run_command(
            spark_submit_command(
                validator,
                "1g",
                2,
                False,
            ),
            (
                f"VALIDATE NEW "
                f"{stage.upper()} {epoch}"
            ),
            900,
        )

        append_manifest(
            epoch,
            stage,
            "built_and_validated",
            elapsed + validation_elapsed,
            hdfs_path,
        )

        print(
            f"{stage} {epoch}: "
            "BUILT AND VALIDATED"
        )


    # ========================================================
    # BATCH STARTUP VALIDATION
    # ========================================================

    for product, template_path in [
        ("population", POPULATION_TEMPLATE),
        ("smod", SMOD_TEMPLATE),
        ("built_surface", BUILT_TEMPLATE),
    ]:
        verify_template(
            product,
            template_path,
        )

    print("Protected template checksums: PASS")


    if MANIFEST_PATH.exists():
        timestamp = int(time.time())

        backup_path = (
            BATCH_ROOT
            / f"batch_manifest_{timestamp}.tsv"
        )

        MANIFEST_PATH.replace(
            backup_path
        )

        print(
            "Previous manifest backed up to:",
            backup_path,
        )


    MANIFEST_PATH.write_text(
        (
            "epoch\tstage\tstatus\t"
            "elapsed_seconds\thdfs_path\n"
        ),
        encoding="utf-8",
    )


    # ========================================================
    # PROCESS ALL INTERMEDIATE EPOCHS SEQUENTIALLY
    # ========================================================

    batch_started_at = time.time()

    for epoch_index, epoch in enumerate(
        EPOCHS,
        start=1,
    ):
        epoch_started_at = time.time()

        print()
        print("#" * 80)
        print(
            f"EPOCH {epoch} "
            f"({epoch_index}/{len(EPOCHS)})"
        )
        print("#" * 80)


        epoch_dir = (
            PROGRAM_ROOT
            / f"epoch_{epoch}"
        )

        epoch_dir.mkdir(
            parents=True,
            exist_ok=True,
        )


        population_hdfs = (
            f"{HDFS_BASE}/product=population/"
            f"epoch={epoch}/grid_1km"
        )

        smod_hdfs = (
            f"{HDFS_BASE}/product=smod/"
            f"epoch={epoch}/grid_1km"
        )

        built_hdfs = (
            f"{HDFS_BASE}/product=built_surface/"
            f"epoch={epoch}/grid_1km"
        )

        enriched_hdfs = (
            f"{HDFS_BASE}/enriched_grid_year/"
            f"epoch={epoch}"
        )


        ensure_stage(
            epoch=epoch,
            stage="population",
            hdfs_path=population_hdfs,
            program_factory=generate_population_program,
            epoch_dir=epoch_dir,
            driver_memory="3g",
            shuffle_partitions=4,
            use_sedona=True,
        )


        ensure_stage(
            epoch=epoch,
            stage="smod",
            hdfs_path=smod_hdfs,
            program_factory=generate_smod_program,
            epoch_dir=epoch_dir,
            driver_memory="3g",
            shuffle_partitions=2,
            use_sedona=True,
        )


        ensure_stage(
            epoch=epoch,
            stage="built_surface",
            hdfs_path=built_hdfs,
            program_factory=generate_built_program,
            epoch_dir=epoch_dir,
            driver_memory="3g",
            shuffle_partitions=4,
            use_sedona=True,
        )


        ensure_stage(
            epoch=epoch,
            stage="enriched",
            hdfs_path=enriched_hdfs,
            program_factory=create_enrichment_program,
            epoch_dir=epoch_dir,
            driver_memory="2g",
            shuffle_partitions=4,
            use_sedona=True,
        )


        epoch_elapsed = (
            time.time() - epoch_started_at
        )

        print()
        print(
            f"EPOCH {epoch}: COMPLETE"
        )

        print(
            "Epoch elapsed minutes:",
            round(epoch_elapsed / 60.0, 2),
        )


    # ========================================================
    # FINAL BATCH SUMMARY
    # ========================================================

    batch_elapsed = (
        time.time() - batch_started_at
    )

    print()
    print("=" * 80)
    print("INTERMEDIATE-EPOCH BATCH COMPLETE")
    print("=" * 80)

    print("Epochs:", EPOCHS)

    print(
        "Completed epoch count:",
        len(EPOCHS),
    )

    print(
        "Expected product/enriched outputs:",
        len(EPOCHS) * 4,
    )

    print(
        "Total elapsed hours:",
        round(batch_elapsed / 3600.0, 2),
    )

    print(
        "Batch manifest:",
        MANIFEST_PATH,
    )

    print()
    print(MANIFEST_PATH.read_text(
        encoding="utf-8"
    ))

    print("Sequential execution: YES")
    print("Concurrent Spark jobs: NO")
    print("Resumable output validation: YES")
    print("BATCH PIPELINE: SUCCESS")
    """
).strip() + "\n"


# ============================================================
# TRANSFER BATCH DRIVER TO AWS
# ============================================================

encoded_batch_driver = base64.b64encode(
    batch_driver.encode("utf-8")
).decode("ascii")

prepare_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"mkdir -p {REMOTE_BATCH_ROOT}; "

    f"echo '{encoded_batch_driver}' "
    f"| base64 -d > {REMOTE_BATCH_DRIVER}; "

    f"python3 -m py_compile "
    f"{REMOTE_BATCH_DRIVER}; "

    "echo 'Batch-driver Python syntax: PASS'; "

    f"ls -lh {REMOTE_BATCH_DRIVER}; "

    f"sha256sum {REMOTE_BATCH_DRIVER}; "

    "echo; "
    "echo 'Current system resources:'; "
    "free -h; "
    "df -h / /tmp; "

    "echo; "
    "echo 'Existing Spark jobs:'; "
    "ps -ef "
    "| grep -E "
    "'SparkSubmit|spark-submit' "
    "| grep -v grep "
    "|| true"
)


prepare_result = run_remote(
    prepare_command,
    timeout=240,
    show_output=True,
)


# ============================================================
# RUN BATCH DRIVER
#
# The output is stored remotely with tee.
# Colab receives the complete captured output when the batch
# finishes.
# ============================================================

run_command = (
    "set -o pipefail; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    "export HADOOP_HOME=/usr/local/hadoop; "
    "export PATH=${HADOOP_HOME}/bin:"
    "${HADOOP_HOME}/sbin:${PATH}; "

    f"rm -f {REMOTE_BATCH_LOG}; "

    f"python3 {REMOTE_BATCH_DRIVER} "
    f"2>&1 | tee {REMOTE_BATCH_LOG}; "

    "status=${PIPESTATUS[0]}; "

    "echo BATCH_EXIT_CODE=${status}; "

    "exit ${status}"
)


print()
print("=" * 80)
print("STARTING INTERMEDIATE-EPOCH BATCH")
print("=" * 80)
print("Epochs:", BATCH_EPOCHS)
print("Execution: sequential")
print("Existing valid outputs: validate and skip")
print("Missing or invalid outputs: build and validate")
print("Remote batch log:", REMOTE_BATCH_LOG)
print("Remote manifest:", REMOTE_BATCH_MANIFEST)
print()


batch_result = run_remote(
    run_command,
    timeout=28800,
    show_output=True,
)


# ============================================================
# FINAL NOTEBOOK RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7 NOTEBOOK RESULT")
print("=" * 80)

print(
    "Batch-driver preparation code:",
    prepare_result.returncode,
)

print(
    "Batch execution return code:",
    batch_result.returncode,
)

print(
    "Batch epochs:",
    BATCH_EPOCHS,
)

print(
    "Remote batch log:",
    REMOTE_BATCH_LOG,
)

print(
    "Remote batch manifest:",
    REMOTE_BATCH_MANIFEST,
)

print("CELL 7: SUCCESS")

Batch-driver Python syntax: PASS
-rw-rw-r-- 1 ubuntu ubuntu 35K Nov  4 07:38 /home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py
1c7b87354d647760c54839c224c153d410f3a9901dfe19a044c85b83d39a258a  /home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py

Current system resources:
               total        used        free      shared  buff/cache   available
Mem:           7.6Gi       1.5Gi       5.1Gi        35Mi       1.2Gi       6.1Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        77G   24G   54G  31% /
tmpfs           3.8G   33M  3.8G   1% /tmp

Existing Spark jobs:

REMOTE STDERR:


STARTING INTERMEDIATE-EPOCH BATCH
Epochs: [1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015]
Execution: sequential
Existing valid outputs: validate and skip
Missing or invalid outputs: build and validate
Remote batch log: /home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log
Remote manifest: /home/ubuntu/ghsl_noteboo

RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7A: PATCH AND RESUME INTERMEDIATE-EPOCH BATCH
#
# Fix:
#   Adds the missing `import textwrap` to the remote batch
#   driver created by Cell 7.
#
# This cell:
#   - patches the existing remote driver
#   - verifies the patch
#   - compiles the driver
#   - resumes the full 1980-2015 batch
#
# Do not rerun the original Cell 7 because it would recreate
# the unpatched driver.
# ============================================================

REMOTE_BATCH_ROOT = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch"
)

REMOTE_BATCH_DRIVER = (
    f"{REMOTE_BATCH_ROOT}/"
    "run_intermediate_epochs.py"
)

REMOTE_BATCH_LOG = (
    f"{REMOTE_BATCH_ROOT}/"
    "intermediate_epochs_batch.log"
)

REMOTE_BATCH_MANIFEST = (
    f"{REMOTE_BATCH_ROOT}/"
    "batch_manifest.tsv"
)


# ============================================================
# 1. PATCH THE REMOTE BATCH DRIVER
# ============================================================

patch_command = f'''
set -e

DRIVER="{REMOTE_BATCH_DRIVER}"
BACKUP="${{DRIVER}}.before_textwrap_fix"

if [ ! -f "${{DRIVER}}" ]; then
    echo "ERROR: Batch driver not found:"
    echo "${{DRIVER}}"
    exit 20
fi

cp "${{DRIVER}}" "${{BACKUP}}"

python3 - "${{DRIVER}}" <<'PY'
from pathlib import Path
import sys


driver_path = Path(sys.argv[1])

program = driver_path.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# Verify the expected import section
# ------------------------------------------------------------

expected_import_block = """from pathlib import Path
import hashlib
import subprocess
import sys
import time
"""

patched_import_block = """from pathlib import Path
import hashlib
import subprocess
import sys
import time
import textwrap
"""


if "import textwrap" in program:
    print("textwrap import already present.")

else:
    occurrence_count = program.count(
        expected_import_block
    )

    if occurrence_count != 1:
        raise RuntimeError(
            "Expected exactly one batch-driver import block, "
            f"found {{occurrence_count}}."
        )

    program = program.replace(
        expected_import_block,
        patched_import_block,
        1,
    )

    driver_path.write_text(
        program,
        encoding="utf-8",
    )

    print("Added missing import: textwrap")


# ------------------------------------------------------------
# Verify source and compile
# ------------------------------------------------------------

patched_program = driver_path.read_text(
    encoding="utf-8"
)

if patched_program.count(
    "import textwrap"
) != 1:
    raise RuntimeError(
        "The patched driver must contain exactly one "
        "`import textwrap` statement."
    )

compile(
    patched_program,
    str(driver_path),
    "exec",
)

print("Batch-driver Python syntax: PASS")
print("textwrap import validation: PASS")
PY

echo
echo "============================================================"
echo "PATCHED DRIVER IMPORTS"
echo "============================================================"

sed -n '1,20p' "${{DRIVER}}"

echo
echo "Driver SHA-256:"
sha256sum "${{DRIVER}}"

echo
echo "Backup:"
ls -lh "${{BACKUP}}"

echo
echo "============================================================"
echo "CELL 7A PATCH RESULT"
echo "============================================================"

echo "Missing textwrap import: FIXED"
echo "Batch-driver syntax: PASS"
echo "Spark jobs launched during patch: NO"
echo "HDFS modified during patch: NO"
'''


patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 2. CONFIRM NO OTHER SPARK JOB IS RUNNING
# ============================================================

preflight_command = f'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

echo "============================================================"
echo "BATCH RESUME PRE-FLIGHT"
echo "============================================================"

python3 -m py_compile "{REMOTE_BATCH_DRIVER}"

grep -F 'import textwrap' \
    "{REMOTE_BATCH_DRIVER}" \
    >/dev/null \
    || {{
        echo "ERROR: textwrap import is missing."
        exit 30
    }}

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
        | awk '{{print $2}}' \
        | grep -qx "${{daemon}}" \
        || {{
            echo "ERROR: ${{daemon}} is not running."
            exit 31
        }}
done

SAFE_MODE_STATUS="$(
    hdfs dfsadmin -safemode get 2>&1
)"

echo "${{SAFE_MODE_STATUS}}"

echo "${{SAFE_MODE_STATUS}}" \
    | grep -qi "Safe mode is OFF" \
    || {{
        echo "ERROR: HDFS safe mode is enabled."
        exit 32
    }}

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
    'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${{RUNNING_COUNT}}" -ne 0 ]; then
    echo "ERROR: Another batch or Spark job is running."
    ps -ef \
        | grep -E \
        'run_intermediate_epochs|SparkSubmit|spark-submit' \
        | grep -v grep
    exit 33
fi

echo
echo "Current memory:"
free -h

echo
echo "Current disk:"
df -h / /tmp

echo
echo "Batch resume pre-flight: PASS"
'''


preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 3. RESUME THE COMPLETE BATCH
#
# The batch driver validates existing outputs before deciding
# whether to skip or rebuild each stage.
#
# Population 1980 and SMOD 1980 should be validated and
# skipped. Missing stages will be built sequentially.
# ============================================================

resume_command = f'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

python3 "{REMOTE_BATCH_DRIVER}" \
    2>&1 \
    | tee "{REMOTE_BATCH_LOG}"

status=${{PIPESTATUS[0]}}

echo "BATCH_EXIT_CODE=${{status}}"

exit "${{status}}"
'''


print()
print("=" * 80)
print("RESUMING INTERMEDIATE-EPOCH BATCH")
print("=" * 80)
print(
    "Epochs:",
    [1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015],
)
print("Execution mode: sequential")
print("Existing valid outputs: validate and skip")
print("Remote log:", REMOTE_BATCH_LOG)
print("Remote manifest:", REMOTE_BATCH_MANIFEST)
print()


batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7A NOTEBOOK RESULT")
print("=" * 80)

print(
    "Patch return code:",
    patch_result.returncode,
)

print(
    "Pre-flight return code:",
    preflight_result.returncode,
)

print(
    "Batch return code:",
    batch_result.returncode,
)

print(
    "Remote batch log:",
    REMOTE_BATCH_LOG,
)

print(
    "Remote batch manifest:",
    REMOTE_BATCH_MANIFEST,
)

print("CELL 7A: SUCCESS")

Added missing import: textwrap
Batch-driver Python syntax: PASS
textwrap import validation: PASS

PATCHED DRIVER IMPORTS
from pathlib import Path
import hashlib
import subprocess
import sys
import time
import textwrap


# ========================================================
# CONFIGURATION
# ========================================================

EPOCHS = [
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,

Driver SHA-256:
8ad361682ad61abe8e2027358591418c2c7f2c5bee2ab06ed588c23bd3c58448  /home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py

Backup:
-rw-rw-r-- 1 ubuntu ubuntu 35K Nov  4 07:39 /home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py.before_textwrap_fix

CELL 7A PATCH RESULT
Missing textwrap import: FIXED
Batch-driver syntax: PASS
Spark jobs launched during patch: NO
HDFS modified during patch: NO

REMOTE STDERR:

BATCH RESUME PRE-FLIGHT
Safe mode is OFF

Current memory:
               total        used        free      shared  buff

RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7B: PATCH BUILT-SURFACE GENERATOR AND RESUME BATCH
#
# Fixes:
#   1. Built template has one active epoch=2020 path, not two.
#      The separate grid-reference path remains canonical 1975.
#
#   2. Enrichment-generated programs explicitly initialize
#      SedonaContext before calling ST_AsBinary and
#      ST_PolygonFromEnvelope.
#
# Existing valid outputs are revalidated and skipped.
# Batch execution remains sequential.
#
# Do not rerun original Cell 7 or Cell 7A.
# ============================================================

REMOTE_BATCH_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_BATCH_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_BATCH_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)


# ============================================================
# 1. PATCH THE REMOTE BATCH DRIVER
# ============================================================

patch_command = f'''
set -e

DRIVER="{REMOTE_BATCH_DRIVER}"
BACKUP="${{DRIVER}}.before_built_path_fix"

if [ ! -f "${{DRIVER}}" ]; then
    echo "ERROR: Batch driver is missing:"
    echo "${{DRIVER}}"
    exit 20
fi

cp "${{DRIVER}}" "${{BACKUP}}"

python3 - "${{DRIVER}}" <<'PY'
from pathlib import Path
import sys


driver_path = Path(sys.argv[1])

program = driver_path.read_text(
    encoding="utf-8"
)


# ============================================================
# PATCH 1: BUILT-SURFACE PATH COUNT
#
# Confirmed behavior:
#   - one active 2020 built output path
#   - one canonical 1975 analytical-grid reference
# ============================================================

old_built_block = """        # The validated 2020 built template contains:
        #   1. grid-year 2020 reference path
        #   2. built-surface 2020 output path
        program = replace_exact_count(
            program,
            '    "epoch=2020/"',
            f'    "epoch={{epoch}}/"',
            2,
            "Built grid-reference/output paths",
        )
"""

new_built_block = """        # The validated built template contains:
        #   1. one active built-surface 2020 output path
        #   2. one canonical 1975 analytical-grid reference
        #
        # Only the active output path is changed. The 1975
        # canonical grid reference is preserved.
        program = replace_exact_count(
            program,
            '    "epoch=2020/"',
            f'    "epoch={{epoch}}/"',
            1,
            "Built-surface output path",
        )
"""


if old_built_block in program:
    program = program.replace(
        old_built_block,
        new_built_block,
        1,
    )

elif new_built_block in program:
    print(
        "Built-surface path-count patch "
        "already present."
    )

else:
    raise RuntimeError(
        "Could not locate the expected built-surface "
        "path replacement block."
    )


# ============================================================
# PATCH 2: BUILT-SURFACE STATIC CHECKS
# ============================================================

old_built_requirements = """                f'EPOCH = {{epoch}}',
                f'    "epoch={{epoch}}/"',
                'PRODUCT = "built_surface"',
"""

new_built_requirements = """                f'EPOCH = {{epoch}}',
                f'    "epoch={{epoch}}/"',
                '    "epoch=1975/"',
                'PRODUCT = "built_surface"',
"""


if old_built_requirements in program:
    program = program.replace(
        old_built_requirements,
        new_built_requirements,
        1,
    )

elif new_built_requirements in program:
    print(
        "Built canonical-reference requirement "
        "already present."
    )

else:
    raise RuntimeError(
        "Could not locate built-surface static "
        "requirements."
    )


# ============================================================
# PATCH 3: ENRICHMENT SEDONA INITIALIZATION
#
# The generated enrichment program uses:
#   ST_AsBinary
#   ST_PolygonFromEnvelope
#
# Therefore, it must initialize SedonaContext rather than only
# a plain SparkSession.
# ============================================================

old_enrichment_import = """    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
"""

new_enrichment_import = """    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F
"""


import_count = program.count(
    old_enrichment_import
)

if import_count == 1:
    program = program.replace(
        old_enrichment_import,
        new_enrichment_import,
        1,
    )

elif new_enrichment_import in program:
    print(
        "Enrichment Sedona import already present."
    )

else:
    raise RuntimeError(
        "Could not locate the generated enrichment "
        "program import block."
    )


old_enrichment_session = """    spark = (
        SparkSession.builder
        .appName(
            f"GHSL-Bengaluru-{{EPOCH}}-Enrichment"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")
"""

new_enrichment_session = """    config = (
        SedonaContext.builder()
        .appName(
            f"GHSL-Bengaluru-{{EPOCH}}-Enrichment"
        )
        .getOrCreate()
    )

    spark = SedonaContext.create(config)
    spark.sparkContext.setLogLevel("WARN")
"""


if old_enrichment_session in program:
    program = program.replace(
        old_enrichment_session,
        new_enrichment_session,
        1,
    )

elif new_enrichment_session in program:
    print(
        "Enrichment Sedona context already present."
    )

else:
    raise RuntimeError(
        "Could not locate the generated enrichment "
        "Spark-session block."
    )


# ============================================================
# VERIFY PATCHED DRIVER
# ============================================================

required_fragments = [
    "import textwrap",
    '"Built-surface output path"',
    "f'    \\"epoch={{epoch}}/\\"'",
    "'    \\"epoch=1975/\\"'",
    "from sedona.spark import SedonaContext",
    "spark = SedonaContext.create(config)",
]

missing = [
    fragment
    for fragment in required_fragments
    if fragment not in program
]

if missing:
    raise RuntimeError(
        "Patched driver is missing required content:\\n"
        + "\\n".join(missing)
    )


forbidden_fragments = [
    '"Built grid-reference/output paths",\\n'
    '            2,',
]

present_forbidden = [
    fragment
    for fragment in forbidden_fragments
    if fragment in program
]

if present_forbidden:
    raise RuntimeError(
        "Old built-surface path-count logic remains."
    )


compile(
    program,
    str(driver_path),
    "exec",
)

driver_path.write_text(
    program,
    encoding="utf-8",
)


print(
    "Built active-path count corrected:",
    "2 -> 1",
)

print(
    "Canonical 1975 built-grid reference:",
    "PRESERVED",
)

print(
    "Generated enrichment SedonaContext:",
    "ENABLED",
)

print(
    "Patched batch-driver Python syntax:",
    "PASS",
)
PY


python3 -m py_compile "${{DRIVER}}"

echo
echo "============================================================"
echo "PATCHED BUILT-SURFACE GENERATOR"
echo "============================================================"

grep -n -A 14 -B 5 \
    'Built-surface output path' \
    "${{DRIVER}}"

echo
echo "============================================================"
echo "PATCHED ENRICHMENT SEDONA INITIALIZATION"
echo "============================================================"

grep -n -A 12 -B 5 \
    'SedonaContext.builder' \
    "${{DRIVER}}" \
    | head -n 60

echo
echo "Patched driver SHA-256:"
sha256sum "${{DRIVER}}"

echo
echo "Backup:"
ls -lh "${{BACKUP}}"

echo
echo "============================================================"
echo "CELL 7B PATCH RESULT"
echo "============================================================"

echo "Built active epoch-path count: FIXED"
echo "Canonical 1975 built reference: PRESERVED"
echo "Enrichment Sedona initialization: FIXED"
echo "Driver syntax: PASS"
echo "Spark jobs launched during patch: NO"
echo "HDFS modified during patch: NO"
'''


patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 2. PRE-FLIGHT BEFORE RESUMING
# ============================================================

preflight_command = f'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

DRIVER="{REMOTE_BATCH_DRIVER}"

echo "============================================================"
echo "CELL 7B BATCH RESUME PRE-FLIGHT"
echo "============================================================"

python3 -m py_compile "${{DRIVER}}"

grep -F 'import textwrap' \
    "${{DRIVER}}" >/dev/null

grep -F '"Built-surface output path"' \
    "${{DRIVER}}" >/dev/null

grep -F 'spark = SedonaContext.create(config)' \
    "${{DRIVER}}" >/dev/null

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
        | awk '{{print $2}}' \
        | grep -qx "${{daemon}}" \
        || {{
            echo "ERROR: ${{daemon}} is not running."
            exit 30
        }}
done

SAFE_MODE_STATUS="$(
    hdfs dfsadmin -safemode get 2>&1
)"

echo "${{SAFE_MODE_STATUS}}"

echo "${{SAFE_MODE_STATUS}}" \
    | grep -qi "Safe mode is OFF" \
    || {{
        echo "ERROR: HDFS safe mode is enabled."
        exit 31
    }}

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
    'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${{RUNNING_COUNT}}" -ne 0 ]; then
    echo "ERROR: Another batch or Spark job is running."

    ps -ef \
        | grep -E \
        'run_intermediate_epochs|SparkSubmit|spark-submit' \
        | grep -v grep

    exit 32
fi

echo
echo "Memory:"
free -h

echo
echo "Disk:"
df -h / /tmp

echo
echo "Batch resume pre-flight: PASS"
'''


preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 3. RESUME AUTHORITATIVE BATCH
#
# Expected initial behavior:
#   population 1980 -> validate and skip
#   SMOD 1980       -> validate and skip
#   built 1980      -> build and validate
#   enriched 1980   -> build and validate
# ============================================================

resume_command = f'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

python3 "{REMOTE_BATCH_DRIVER}" \
    2>&1 \
    | tee "{REMOTE_BATCH_LOG}"

status=${{PIPESTATUS[0]}}

echo "BATCH_EXIT_CODE=${{status}}"

exit "${{status}}"
'''


print()
print("=" * 80)
print("RESUMING AUTHORITATIVE INTERMEDIATE-EPOCH BATCH")
print("=" * 80)

print(
    "Epochs:",
    [
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
    ],
)

print("Execution mode: sequential")
print("Existing valid outputs: validate and skip")
print("Remote log:", REMOTE_BATCH_LOG)
print("Remote manifest:", REMOTE_BATCH_MANIFEST)
print()


batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)


# ============================================================
# FINAL NOTEBOOK RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7B NOTEBOOK RESULT")
print("=" * 80)

print(
    "Patch return code:",
    patch_result.returncode,
)

print(
    "Pre-flight return code:",
    preflight_result.returncode,
)

print(
    "Batch return code:",
    batch_result.returncode,
)

print(
    "Remote batch log:",
    REMOTE_BATCH_LOG,
)

print(
    "Remote batch manifest:",
    REMOTE_BATCH_MANIFEST,
)

print("CELL 7B: SUCCESS")

REMOTE STDERR:
Traceback (most recent call last):
  File "<stdin>", line 62, in <module>
RuntimeError: Could not locate the expected built-surface path replacement block.



RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7C: ROBUSTLY PATCH BUILT PATH COUNT AND RESUME BATCH
#
# Fix:
#   The validated built-surface template has:
#     - one active epoch=2020 built output path
#     - one canonical epoch=1975 grid reference
#
#   Therefore:
#     expected active 2020 path count = 1
#
# This patch locates the generator call by its unique
# description instead of matching comments or formatting.
#
# Existing valid outputs are validated and skipped.
# ============================================================

REMOTE_BATCH_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_BATCH_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_BATCH_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)


# ============================================================
# 1. PATCH THE REMOTE DRIVER
# ============================================================

patch_command = f'''
set -e

DRIVER="{REMOTE_BATCH_DRIVER}"
BACKUP="${{DRIVER}}.before_robust_built_fix"

if [ ! -f "${{DRIVER}}" ]; then
    echo "ERROR: Batch driver is missing:"
    echo "${{DRIVER}}"
    exit 20
fi

cp "${{DRIVER}}" "${{BACKUP}}"

python3 - "${{DRIVER}}" <<'PY'
from pathlib import Path
import re
import sys


driver_path = Path(sys.argv[1])

program = driver_path.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# Locate the built-surface replacement call by its unique
# description and inspect it before changing anything.
# ------------------------------------------------------------

call_pattern = re.compile(
    r"""
    program\\s*=\\s*replace_exact_count\\(
        \\s*program\\s*,
        \\s*'\\s{4}"epoch=2020/"'\\s*,
        \\s*f'\\s{4}"epoch=\\{epoch\\}/"'\\s*,
        \\s*(?P<count>[12])\\s*,
        \\s*"Built\\s+grid-reference/output\\s+paths"\\s*,
        \\s*\\)
    """,
    re.VERBOSE,
)


matches = list(
    call_pattern.finditer(program)
)

if len(matches) != 1:
    print(
        "Could not locate exactly one original "
        "built replacement call."
    )

    print(
        "Matches found:",
        len(matches),
    )

    print()
    print("Nearby source lines:")

    lines = program.splitlines()

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        if (
            "Built grid-reference/output paths"
            in line
        ):
            start = max(
                1,
                line_number - 12,
            )

            end = min(
                len(lines),
                line_number + 5,
            )

            for index in range(
                start,
                end + 1,
            ):
                print(
                    f"{index:04d}: "
                    f"{lines[index - 1]}"
                )

    # The driver may already contain the corrected description.
    corrected_marker = (
        '"Built-surface output path"'
    )

    if corrected_marker in program:
        print()
        print(
            "Corrected built-surface marker "
            "already exists."
        )

    else:
        raise RuntimeError(
            "Unable to identify the built-surface "
            "replacement call safely."
        )

else:
    match = matches[0]

    current_count = int(
        match.group("count")
    )

    print(
        "Current built active-path expectation:",
        current_count,
    )

    replacement_call = """
        program = replace_exact_count(
            program,
            '    "epoch=2020/"',
            f'    "epoch={epoch}/"',
            1,
            "Built-surface output path",
        )
    """.strip("\\n")

    program = (
        program[:match.start()]
        + replacement_call
        + program[match.end():]
    )

    print(
        "Built active-path expectation changed:",
        f"{current_count} -> 1",
    )


# ------------------------------------------------------------
# Ensure the generated built program requires the canonical
# 1975 reference.
# ------------------------------------------------------------

built_requirements_anchor = """                f'EPOCH = {epoch}',
                f'    "epoch={epoch}/"',
                'PRODUCT = "built_surface"',
"""

built_requirements_corrected = """                f'EPOCH = {epoch}',
                f'    "epoch={epoch}/"',
                '    "epoch=1975/"',
                'PRODUCT = "built_surface"',
"""


if (
    built_requirements_corrected
    not in program
):
    if (
        built_requirements_anchor
        not in program
    ):
        raise RuntimeError(
            "Could not locate built-surface "
            "static requirements."
        )

    program = program.replace(
        built_requirements_anchor,
        built_requirements_corrected,
        1,
    )

    print(
        "Canonical 1975 built-reference check:",
        "ADDED",
    )

else:
    print(
        "Canonical 1975 built-reference check:",
        "ALREADY PRESENT",
    )


# ------------------------------------------------------------
# Ensure enrichment programs initialize Sedona because the
# generated enrichment code uses ST_AsBinary and
# ST_PolygonFromEnvelope.
# ------------------------------------------------------------

plain_import = """    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
"""

sedona_import = """    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F
"""


if sedona_import not in program:
    if plain_import not in program:
        raise RuntimeError(
            "Could not locate enrichment import block."
        )

    program = program.replace(
        plain_import,
        sedona_import,
        1,
    )

    print(
        "Generated enrichment Sedona import:",
        "ADDED",
    )

else:
    print(
        "Generated enrichment Sedona import:",
        "ALREADY PRESENT",
    )


plain_session = """    spark = (
        SparkSession.builder
        .appName(
            f"GHSL-Bengaluru-{EPOCH}-Enrichment"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")
"""

sedona_session = """    config = (
        SedonaContext.builder()
        .appName(
            f"GHSL-Bengaluru-{EPOCH}-Enrichment"
        )
        .getOrCreate()
    )

    spark = SedonaContext.create(config)
    spark.sparkContext.setLogLevel("WARN")
"""


if sedona_session not in program:
    if plain_session not in program:
        raise RuntimeError(
            "Could not locate enrichment "
            "SparkSession block."
        )

    program = program.replace(
        plain_session,
        sedona_session,
        1,
    )

    print(
        "Generated enrichment SedonaContext:",
        "ADDED",
    )

else:
    print(
        "Generated enrichment SedonaContext:",
        "ALREADY PRESENT",
    )


# ------------------------------------------------------------
# Final source validation
# ------------------------------------------------------------

required_fragments = [
    "import textwrap",
    '"Built-surface output path"',
    "'    \\"epoch=1975/\\"'",
    "from sedona.spark import SedonaContext",
    "spark = SedonaContext.create(config)",
]

missing = [
    fragment
    for fragment in required_fragments
    if fragment not in program
]

if missing:
    raise RuntimeError(
        "Patched driver is missing required "
        "content:\\n"
        + "\\n".join(missing)
    )


if (
    '"Built grid-reference/output paths"'
    in program
):
    raise RuntimeError(
        "Old built path-count marker remains."
    )


compile(
    program,
    str(driver_path),
    "exec",
)

driver_path.write_text(
    program,
    encoding="utf-8",
)


print()
print(
    "Patched batch-driver syntax:",
    "PASS",
)

print(
    "Built output-path count:",
    "1",
)

print(
    "Canonical 1975 built reference:",
    "PRESERVED",
)

print(
    "Enrichment Sedona initialization:",
    "ENABLED",
)
PY


python3 -m py_compile "${{DRIVER}}"

echo
echo "============================================================"
echo "PATCH VERIFICATION"
echo "============================================================"

grep -n -A 8 -B 5 \
    'Built-surface output path' \
    "${{DRIVER}}"

echo
echo "Enrichment Sedona context:"

grep -n -A 10 -B 4 \
    'SedonaContext.builder' \
    "${{DRIVER}}" \
    | tail -n 40

echo
echo "Patched driver SHA-256:"
sha256sum "${{DRIVER}}"

echo
echo "Backup file:"
ls -lh "${{BACKUP}}"

echo
echo "============================================================"
echo "CELL 7C PATCH RESULT"
echo "============================================================"

echo "Built active output-path count: FIXED TO 1"
echo "Canonical 1975 reference: PRESERVED"
echo "Enrichment Sedona initialization: ENABLED"
echo "Driver syntax: PASS"
echo "Spark launched during patch: NO"
echo "HDFS modified during patch: NO"
'''


patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 2. BATCH RESUME PRE-FLIGHT
# ============================================================

preflight_command = f'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

DRIVER="{REMOTE_BATCH_DRIVER}"

echo "============================================================"
echo "CELL 7C BATCH RESUME PRE-FLIGHT"
echo "============================================================"

python3 -m py_compile "${{DRIVER}}"

grep -F '"Built-surface output path"' \
    "${{DRIVER}}" \
    >/dev/null \
    || {{
        echo "ERROR: Corrected built marker missing."
        exit 30
    }}

if grep -Fq \
    '"Built grid-reference/output paths"' \
    "${{DRIVER}}"
then
    echo "ERROR: Old built marker remains."
    exit 31
fi

grep -F \
    'spark = SedonaContext.create(config)' \
    "${{DRIVER}}" \
    >/dev/null \
    || {{
        echo "ERROR: Enrichment SedonaContext missing."
        exit 32
    }}

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
        | awk '{{print $2}}' \
        | grep -qx "${{daemon}}" \
        || {{
            echo "ERROR: ${{daemon}} is not running."
            exit 33
        }}
done

SAFE_MODE_STATUS="$(
    hdfs dfsadmin -safemode get 2>&1
)"

echo "${{SAFE_MODE_STATUS}}"

echo "${{SAFE_MODE_STATUS}}" \
    | grep -qi "Safe mode is OFF" \
    || {{
        echo "ERROR: HDFS safe mode is enabled."
        exit 34
    }}

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
    'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${{RUNNING_COUNT}}" -ne 0 ]; then
    echo "ERROR: Another batch or Spark job is running."

    ps -ef \
        | grep -E \
        'run_intermediate_epochs|SparkSubmit|spark-submit' \
        | grep -v grep

    exit 35
fi

echo
echo "Memory:"
free -h

echo
echo "Disk:"
df -h / /tmp

echo
echo "Batch resume pre-flight: PASS"
'''


preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 3. RESUME BATCH
# ============================================================

resume_command = f'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

python3 "{REMOTE_BATCH_DRIVER}" \
    2>&1 \
    | tee "{REMOTE_BATCH_LOG}"

status=${{PIPESTATUS[0]}}

echo "BATCH_EXIT_CODE=${{status}}"

exit "${{status}}"
'''


print()
print("=" * 80)
print("RESUMING INTERMEDIATE-EPOCH BATCH")
print("=" * 80)

print(
    "Epochs:",
    [
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
    ],
)

print("Execution: sequential")
print("Existing valid outputs: validate and skip")
print("Next missing stage: built surface 1980")
print("Remote log:", REMOTE_BATCH_LOG)
print("Remote manifest:", REMOTE_BATCH_MANIFEST)
print()


batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7C NOTEBOOK RESULT")
print("=" * 80)

print(
    "Patch return code:",
    patch_result.returncode,
)

print(
    "Pre-flight return code:",
    preflight_result.returncode,
)

print(
    "Batch return code:",
    batch_result.returncode,
)

print(
    "Remote batch log:",
    REMOTE_BATCH_LOG,
)

print(
    "Remote batch manifest:",
    REMOTE_BATCH_MANIFEST,
)

print("CELL 7C: SUCCESS")

SyntaxError: unexpected character after line continuation character (2924627718.py, line 78)

In [ ]:
# ============================================================
# CELL 7C: FIX BUILT-SURFACE COUNT AND RESUME BATCH
#
# Fixes the existing remote batch driver:
#   expected built epoch=2020 path count: 2 -> 1
#
# The canonical epoch=1975 reference remains unchanged.
# Existing valid outputs are validated and skipped.
# ============================================================

import base64
import textwrap


REMOTE_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_PATCHER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "patch_built_count.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)


# ============================================================
# 1. CREATE A SMALL REMOTE PATCH PROGRAM
#
# No shell heredoc or nested triple-quoted source is used.
# ============================================================

patch_program = textwrap.dedent(
    """
    from pathlib import Path
    import shutil


    DRIVER = Path(
        "/home/ubuntu/ghsl_notebook05_batch/"
        "run_intermediate_epochs.py"
    )

    BACKUP = Path(
        "/home/ubuntu/ghsl_notebook05_batch/"
        "run_intermediate_epochs.py.before_built_fix"
    )


    if not DRIVER.is_file():
        raise FileNotFoundError(
            f"Batch driver not found: {DRIVER}"
        )


    shutil.copy2(
        DRIVER,
        BACKUP,
    )


    lines = DRIVER.read_text(
        encoding="utf-8"
    ).splitlines()


    original_marker = (
        '"Built grid-reference/output paths",'
    )

    corrected_marker = (
        '"Built-surface output path",'
    )


    original_indexes = [
        index
        for index, line in enumerate(lines)
        if original_marker in line
    ]

    corrected_indexes = [
        index
        for index, line in enumerate(lines)
        if corrected_marker in line
    ]


    if len(original_indexes) == 1:
        marker_index = original_indexes[0]

        count_index = None

        for index in range(
            marker_index - 1,
            max(-1, marker_index - 10),
            -1,
        ):
            if lines[index].strip() in {
                "1,",
                "2,",
            }:
                count_index = index
                break


        if count_index is None:
            raise RuntimeError(
                "Could not locate the built path-count "
                "argument."
            )


        current_count = lines[
            count_index
        ].strip()

        if current_count != "2,":
            raise RuntimeError(
                "Expected built path count 2, found "
                f"{current_count}"
            )


        indentation = lines[count_index][
            :len(lines[count_index])
            - len(lines[count_index].lstrip())
        ]


        lines[count_index] = (
            indentation + "1,"
        )

        lines[marker_index] = (
            lines[marker_index].replace(
                "Built grid-reference/output paths",
                "Built-surface output path",
            )
        )

        print(
            "Built active-path count changed: 2 -> 1"
        )


    elif (
        len(original_indexes) == 0
        and len(corrected_indexes) == 1
    ):
        print(
            "Built path-count fix is already present."
        )


    else:
        print(
            "Original marker count:",
            len(original_indexes),
        )

        print(
            "Corrected marker count:",
            len(corrected_indexes),
        )

        raise RuntimeError(
            "Could not uniquely identify the built "
            "path-count marker."
        )


    program = (
        "\\n".join(lines)
        + "\\n"
    )


    if original_marker in program:
        raise RuntimeError(
            "Original built marker remains after patch."
        )


    if program.count(corrected_marker) != 1:
        raise RuntimeError(
            "Expected exactly one corrected built marker."
        )


    if "import textwrap" not in program:
        raise RuntimeError(
            "The previous textwrap fix is missing."
        )


    if "'    \\"epoch=1975/\\"'" not in program:
        raise RuntimeError(
            "Canonical epoch=1975 marker is missing."
        )


    compile(
        program,
        str(DRIVER),
        "exec",
    )


    DRIVER.write_text(
        program,
        encoding="utf-8",
    )


    print("Canonical 1975 reference: PRESERVED")
    print("textwrap import: PRESENT")
    print("Patched driver syntax: PASS")
    print("Backup:", BACKUP)
    """
).strip() + "\n"


encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")


# ============================================================
# 2. TRANSFER AND APPLY PATCH
# ============================================================

patch_command = (
    "set -e; "

    f"echo '{encoded_patch}' "
    f"| base64 -d > '{REMOTE_PATCHER}'; "

    f"python3 -m py_compile "
    f"'{REMOTE_PATCHER}'; "

    f"python3 '{REMOTE_PATCHER}'; "

    f"python3 -m py_compile "
    f"'{REMOTE_DRIVER}'; "

    "echo; "
    "echo 'Corrected built-surface block:'; "

    f"grep -n -A 8 -B 8 "
    f"'Built-surface output path' "
    f"'{REMOTE_DRIVER}'; "

    "echo; "
    "echo 'Patched driver checksum:'; "

    f"sha256sum '{REMOTE_DRIVER}'"
)


patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 3. PRE-FLIGHT
# ============================================================

preflight_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"

echo "============================================================"
echo "BATCH RESUME PRE-FLIGHT"
echo "============================================================"

python3 -m py_compile "${DRIVER}"

grep -F \
    '"Built-surface output path",' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: Corrected built marker is missing."
        exit 30
    }

if grep -Fq \
    '"Built grid-reference/output paths",' \
    "${DRIVER}"
then
    echo "ERROR: Old built marker remains."
    exit 31
fi

grep -F \
    'import textwrap' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: textwrap import is missing."
        exit 32
    }

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
        | awk '{print $2}' \
        | grep -qx "${daemon}" \
        || {
            echo "ERROR: ${daemon} is not running."
            exit 33
        }
done

SAFE_MODE_STATUS="$(
    hdfs dfsadmin -safemode get 2>&1
)"

echo "${SAFE_MODE_STATUS}"

echo "${SAFE_MODE_STATUS}" \
    | grep -qi "Safe mode is OFF" \
    || {
        echo "ERROR: HDFS safe mode is enabled."
        exit 34
    }

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
    'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${RUNNING_COUNT}" -ne 0 ]; then
    echo "ERROR: Another batch or Spark job is active."

    ps -ef \
        | grep -E \
        'run_intermediate_epochs|SparkSubmit|spark-submit' \
        | grep -v grep

    exit 35
fi

echo
free -h

echo
df -h / /tmp

echo
echo "Batch resume pre-flight: PASS"
'''


preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 4. RESUME BATCH
#
# Population and SMOD 1980 should be validated and skipped.
# Built surface 1980 should be the first new heavy stage.
# ============================================================

resume_command = r'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
LOG="/home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log"

python3 "${DRIVER}" \
    2>&1 \
    | tee "${LOG}"

status=${PIPESTATUS[0]}

echo "BATCH_EXIT_CODE=${status}"

exit "${status}"
'''


print()
print("=" * 80)
print("RESUMING INTERMEDIATE-EPOCH BATCH")
print("=" * 80)

print(
    "Epochs:",
    [
        1980,
        1985,
        1990,
        1995,
        2000,
        2005,
        2010,
        2015,
    ],
)

print("Execution mode: sequential")
print("Existing valid outputs: validate and skip")
print("Expected first new stage: built surface 1980")
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)
print()


batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)


# ============================================================
# FINAL NOTEBOOK RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7C NOTEBOOK RESULT")
print("=" * 80)

print(
    "Patch return code:",
    patch_result.returncode,
)

print(
    "Pre-flight return code:",
    preflight_result.returncode,
)

print(
    "Batch return code:",
    batch_result.returncode,
)

print(
    "Remote log:",
    REMOTE_LOG,
)

print(
    "Remote manifest:",
    REMOTE_MANIFEST,
)

print("CELL 7C: SUCCESS")

Built active-path count changed: 2 -> 1
Canonical 1975 reference: PRESERVED
textwrap import: PRESENT
Patched driver syntax: PASS
Backup: /home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py.before_built_fix

Corrected built-surface block:
613-    # The validated 2020 built template contains:
614-    #   1. grid-year 2020 reference path
615-    #   2. built-surface 2020 output path
616-    program = replace_exact_count(
617-        program,
618-        '    "epoch=2020/"',
619-        f'    "epoch={epoch}/"',
620-        1,
621:        "Built-surface output path",
622-    )
623-
624-    program = replace_exact_count(
625-        program,
626-        "EPOCH = 2020",
627-        f"EPOCH = {epoch}",
628-        1,
629-        "Built active epoch",

Patched driver checksum:
fc258a511fbc58935a097343a1b72297f72b07cf67878bdbdfdd1cd4ee5d8403  /home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py

REMOTE STDERR:

BATCH RESUME PRE-FLIGHT
Safe mode is OFF

               total    

RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7D: FIX ENRICHMENT SEDONA INITIALIZATION AND RESUME
#
# Problem:
#   The generated enrichment program uses:
#       ST_AsBinary(...)
#       ST_PolygonFromEnvelope(...)
#
#   but starts a plain SparkSession. Therefore, Sedona SQL
#   functions are not registered.
#
# Fix:
#   Patch the batch driver's enrichment-program factory to use:
#       SedonaContext.builder()
#       SedonaContext.create(config)
#
#   Also delete the failed generated enrich_1980.py so the
#   patched factory creates a fresh version.
#
# Existing valid outputs will be validated and skipped:
#   population 1980
#   SMOD 1980
#   built surface 1980
#
# First missing stage after resume:
#   enriched 1980
# ============================================================

import base64
import textwrap


REMOTE_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_PATCHER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "patch_enrichment_sedona.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)


# ============================================================
# 1. CREATE REMOTE PATCHER
# ============================================================

patch_program = textwrap.dedent(
    """
    from pathlib import Path
    import shutil


    DRIVER = Path(
        "/home/ubuntu/ghsl_notebook05_batch/"
        "run_intermediate_epochs.py"
    )

    BACKUP = Path(
        "/home/ubuntu/ghsl_notebook05_batch/"
        "run_intermediate_epochs.py.before_enrichment_sedona_fix"
    )


    if not DRIVER.is_file():
        raise FileNotFoundError(
            f"Batch driver not found: {DRIVER}"
        )


    shutil.copy2(
        DRIVER,
        BACKUP,
    )


    program = DRIVER.read_text(
        encoding="utf-8"
    )


    # --------------------------------------------------------
    # Patch the import inside the generated enrichment program
    # only.
    # --------------------------------------------------------

    old_import = '''    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F


    EPOCH = {epoch}
    EXPECTED_ROWS = 3658
'''

    new_import = '''    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    EPOCH = {epoch}
    EXPECTED_ROWS = 3658
'''


    if new_import in program:
        print(
            "Enrichment Sedona import already present."
        )

    elif old_import in program:
        program = program.replace(
            old_import,
            new_import,
            1,
        )

        print(
            "Enrichment Sedona import: ADDED"
        )

    else:
        raise RuntimeError(
            "Could not locate the enrichment import block."
        )


    # --------------------------------------------------------
    # Patch the generated enrichment SparkSession.
    # Braces are literal because this text exists inside the
    # batch driver's program factory.
    # --------------------------------------------------------

    old_session = '''    spark = (
        SparkSession.builder
        .appName(
            f"GHSL-Bengaluru-{EPOCH}-Enrichment"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")
'''

    new_session = '''    config = (
        SedonaContext.builder()
        .appName(
            f"GHSL-Bengaluru-{EPOCH}-Enrichment"
        )
        .getOrCreate()
    )

    spark = SedonaContext.create(config)
    spark.sparkContext.setLogLevel("WARN")
'''


    if new_session in program:
        print(
            "Enrichment SedonaContext already present."
        )

    elif old_session in program:
        program = program.replace(
            old_session,
            new_session,
            1,
        )

        print(
            "Enrichment SedonaContext: ADDED"
        )

    else:
        raise RuntimeError(
            "Could not locate the enrichment "
            "SparkSession block."
        )


    # --------------------------------------------------------
    # Validate the patched factory
    # --------------------------------------------------------

    required_fragments = [
        "import textwrap",
        '"Built-surface output path"',
        "from sedona.spark import SedonaContext",
        "SedonaContext.builder()",
        "spark = SedonaContext.create(config)",
        "ST_AsBinary(",
        "ST_PolygonFromEnvelope(",
    ]


    missing = [
        fragment
        for fragment in required_fragments
        if fragment not in program
    ]


    if missing:
        raise RuntimeError(
            "Patched driver is missing required content:\\n"
            + "\\n".join(missing)
        )


    compile(
        program,
        str(DRIVER),
        "exec",
    )


    DRIVER.write_text(
        program,
        encoding="utf-8",
    )


    print("Batch-driver syntax: PASS")
    print("Built-surface path fix: PRESERVED")
    print("Enrichment Sedona registration: ENABLED")
    print("Backup:", BACKUP)
    """
).strip() + "\n"


encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")


# ============================================================
# 2. APPLY PATCH AND REMOVE FAILED GENERATED ENRICHMENT FILE
# ============================================================

patch_command = (
    "set -e; "

    f"echo '{encoded_patch}' "
    f"| base64 -d > '{REMOTE_PATCHER}'; "

    f"python3 -m py_compile '{REMOTE_PATCHER}'; "
    f"python3 '{REMOTE_PATCHER}'; "

    f"python3 -m py_compile '{REMOTE_DRIVER}'; "

    "rm -f "
    "'/home/ubuntu/ghsl_notebook05_programs/"
    "epoch_1980/enrich_1980.py'; "

    "rm -f "
    "'/home/ubuntu/ghsl_notebook05_programs/"
    "epoch_1980/validate_enriched_1980.py'; "

    "echo; "
    "echo 'Failed generated enrichment files removed.'; "

    "echo; "
    "echo 'Enrichment factory verification:'; "

    f"grep -n -A 16 -B 6 "
    f"'SedonaContext.builder' "
    f"'{REMOTE_DRIVER}' "
    f"| tail -n 50; "

    "echo; "
    "echo 'Patched driver checksum:'; "

    f"sha256sum '{REMOTE_DRIVER}'"
)


patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 3. PRE-FLIGHT
# ============================================================

preflight_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"

echo "============================================================"
echo "ENRICHMENT FIX PRE-FLIGHT"
echo "============================================================"

python3 -m py_compile "${DRIVER}"

grep -F \
    'from sedona.spark import SedonaContext' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: SedonaContext import missing."
        exit 30
    }

grep -F \
    'spark = SedonaContext.create(config)' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: SedonaContext creation missing."
        exit 31
    }

grep -F \
    '"Built-surface output path",' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: Built-surface path fix is missing."
        exit 32
    }

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
        | awk '{print $2}' \
        | grep -qx "${daemon}" \
        || {
            echo "ERROR: ${daemon} is not running."
            exit 33
        }
done

SAFE_MODE_STATUS="$(
    hdfs dfsadmin -safemode get 2>&1
)"

echo "${SAFE_MODE_STATUS}"

echo "${SAFE_MODE_STATUS}" \
    | grep -qi "Safe mode is OFF" \
    || {
        echo "ERROR: HDFS safe mode is enabled."
        exit 34
    }

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
    'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${RUNNING_COUNT}" -ne 0 ]; then
    echo "ERROR: Another batch or Spark job is active."

    ps -ef \
        | grep -E \
        'run_intermediate_epochs|SparkSubmit|spark-submit' \
        | grep -v grep

    exit 35
fi

echo
echo "Existing 1980 outputs:"

for path in \
"/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1980/grid_1km" \
"/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1980/grid_1km" \
"/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=1980/grid_1km"
do
    if hdfs dfs -test -e "${path}"; then
        echo "PASS: ${path}"
    else
        echo "ERROR: Missing ${path}"
        exit 36
    fi
done

echo
free -h

echo
df -h / /tmp

echo
echo "Enrichment fix pre-flight: PASS"
'''


preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 4. RESUME COMPLETE BATCH
#
# This remains attached to Colab. Do not launch another copy.
# ============================================================

resume_command = r'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
LOG="/home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log"

python3 "${DRIVER}" \
    2>&1 \
    | tee "${LOG}"

status=${PIPESTATUS[0]}

echo "BATCH_EXIT_CODE=${status}"

exit "${status}"
'''


print()
print("=" * 80)
print("RESUMING INTERMEDIATE-EPOCH BATCH")
print("=" * 80)

print("1980 population: expected validate and skip")
print("1980 SMOD: expected validate and skip")
print("1980 built surface: expected validate and skip")
print("Next new stage: enriched 1980")
print("Remaining epochs: 1985 through 2015")
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)
print()


batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7D NOTEBOOK RESULT")
print("=" * 80)

print(
    "Patch return code:",
    patch_result.returncode,
)

print(
    "Pre-flight return code:",
    preflight_result.returncode,
)

print(
    "Batch return code:",
    batch_result.returncode,
)

print(
    "Remote log:",
    REMOTE_LOG,
)

print(
    "Remote manifest:",
    REMOTE_MANIFEST,
)

print("CELL 7D: SUCCESS")

REMOTE STDERR:
Sorry: IndentationError: unexpected indent (patch_enrichment_sedona.py, line 2)


RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7D: PATCH ENRICHMENT SEDONA INITIALIZATION
#
# This cell only patches and validates the remote batch driver.
# It does NOT launch Spark or modify HDFS outputs.
# ============================================================

import base64


REMOTE_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_PATCHER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "patch_enrichment_sedona.py"
)


# Important:
# The first line starts at column zero. Do not indent this
# multiline string.
patch_program = r'''from pathlib import Path
import shutil


driver = Path(
    "/home/ubuntu/ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

backup = Path(
    "/home/ubuntu/ghsl_notebook05_batch/"
    "run_intermediate_epochs.py.before_enrichment_fix"
)


if not driver.is_file():
    raise FileNotFoundError(
        f"Batch driver not found: {driver}"
    )


shutil.copy2(
    driver,
    backup,
)


program = driver.read_text(
    encoding="utf-8"
)


# ============================================================
# PATCH ENRICHMENT IMPORT
# ============================================================

old_import = (
    "    from pyspark.sql import SparkSession\n"
    "    from pyspark.sql import functions as F\n"
    "\n"
    "\n"
    "    EPOCH = {epoch}\n"
    "    EXPECTED_ROWS = 3658\n"
)

new_import = (
    "    from sedona.spark import SedonaContext\n"
    "    from pyspark.sql import functions as F\n"
    "\n"
    "\n"
    "    EPOCH = {epoch}\n"
    "    EXPECTED_ROWS = 3658\n"
)


if new_import in program:
    print(
        "Enrichment Sedona import already present."
    )

else:
    import_count = program.count(
        old_import
    )

    if import_count != 1:
        raise RuntimeError(
            "Expected exactly one enrichment import block, "
            f"found {import_count}."
        )

    program = program.replace(
        old_import,
        new_import,
        1,
    )

    print(
        "Enrichment Sedona import: ADDED"
    )


# ============================================================
# PATCH ENRICHMENT SPARK CONTEXT
# ============================================================

old_session = (
    "    spark = (\n"
    "        SparkSession.builder\n"
    "        .appName(\n"
    '            f"GHSL-Bengaluru-{EPOCH}-Enrichment"\n'
    "        )\n"
    "        .getOrCreate()\n"
    "    )\n"
    "\n"
    "    spark.sparkContext.setLogLevel(\"WARN\")\n"
)

new_session = (
    "    config = (\n"
    "        SedonaContext.builder()\n"
    "        .appName(\n"
    '            f"GHSL-Bengaluru-{EPOCH}-Enrichment"\n'
    "        )\n"
    "        .getOrCreate()\n"
    "    )\n"
    "\n"
    "    spark = SedonaContext.create(config)\n"
    "    spark.sparkContext.setLogLevel(\"WARN\")\n"
)


if new_session in program:
    print(
        "Enrichment SedonaContext already present."
    )

else:
    session_count = program.count(
        old_session
    )

    if session_count != 1:
        raise RuntimeError(
            "Expected exactly one enrichment session block, "
            f"found {session_count}."
        )

    program = program.replace(
        old_session,
        new_session,
        1,
    )

    print(
        "Enrichment SedonaContext: ADDED"
    )


# ============================================================
# VALIDATE EXISTING BATCH FIXES
# ============================================================

required_fragments = [
    "import textwrap",
    '"Built-surface output path"',
    "from sedona.spark import SedonaContext",
    "SedonaContext.builder()",
    "spark = SedonaContext.create(config)",
    "ST_AsBinary(",
    "ST_PolygonFromEnvelope(",
]


missing_fragments = [
    fragment
    for fragment in required_fragments
    if fragment not in program
]


if missing_fragments:
    raise RuntimeError(
        "Patched driver is missing required content:\n"
        + "\n".join(missing_fragments)
    )


if (
    '"Built grid-reference/output paths"'
    in program
):
    raise RuntimeError(
        "Old built-surface path marker remains."
    )


# ============================================================
# COMPILE AND SAVE
# ============================================================

compile(
    program,
    str(driver),
    "exec",
)


driver.write_text(
    program,
    encoding="utf-8",
)


# Re-read and compile the saved file.
saved_program = driver.read_text(
    encoding="utf-8"
)

compile(
    saved_program,
    str(driver),
    "exec",
)


print()
print("=" * 80)
print("ENRICHMENT PATCH RESULT")
print("=" * 80)
print("Batch driver:", driver)
print("Backup:", backup)
print("Built-surface path fix: PRESERVED")
print("Enrichment Sedona import: PRESENT")
print("Enrichment SedonaContext: PRESENT")
print("ST_AsBinary usage: PRESENT")
print("ST_PolygonFromEnvelope usage: PRESENT")
print("Batch-driver syntax: PASS")
print("Spark jobs launched: NO")
print("HDFS modified: NO")
print("CELL 7D: SUCCESS")
'''


encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")


patch_command = (
    "set -e; "

    f"echo '{encoded_patch}' "
    f"| base64 -d > '{REMOTE_PATCHER}'; "

    f"python3 -m py_compile "
    f"'{REMOTE_PATCHER}'; "

    f"python3 '{REMOTE_PATCHER}'; "

    f"python3 -m py_compile "
    f"'{REMOTE_DRIVER}'; "

    "echo; "
    "echo 'Remote patcher first lines:'; "

    f"sed -n '1,20p' "
    f"'{REMOTE_PATCHER}'; "

    "echo; "
    "echo 'Enrichment factory verification:'; "

    f"grep -n -A 14 -B 6 "
    f"'SedonaContext.builder' "
    f"'{REMOTE_DRIVER}' "
    f"| tail -n 50; "

    "echo; "
    "echo 'Patched driver SHA-256:'; "

    f"sha256sum '{REMOTE_DRIVER}'"
)


patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7D NOTEBOOK RESULT")
print("=" * 80)
print(
    "Patch return code:",
    patch_result.returncode,
)
print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 7D: SUCCESS")

REMOTE STDERR:
Traceback (most recent call last):
  File "/home/ubuntu/ghsl_notebook05_batch/patch_enrichment_sedona.py", line 67, in <module>
    raise RuntimeError(
    ...<2 lines>...
    )
RuntimeError: Expected exactly one enrichment import block, found 0.



RuntimeError: Remote command failed with return code 1.

In [ ]:
# ============================================================
# CELL 7D-DIAG CORRECTED: INSPECT ENRICHMENT FACTORY
#
# Read-only:
#   - no driver modification
#   - no Spark launch
#   - no HDFS modification
# ============================================================

inspection_command = r'''
set -e

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"

if [ ! -f "${DRIVER}" ]; then
    echo "ERROR: Batch driver not found:"
    echo "${DRIVER}"
    exit 20
fi

echo "============================================================"
echo "ENRICHMENT FACTORY LOCATION"
echo "============================================================"

START_LINE="$(
    grep -n -E \
    '^[[:space:]]*def create_enrichment_program[[:space:]]*\(' \
    "${DRIVER}" \
    | head -n 1 \
    | cut -d: -f1
)"

END_LINE="$(
    grep -n -E \
    '^[[:space:]]*def create_validator[[:space:]]*\(' \
    "${DRIVER}" \
    | head -n 1 \
    | cut -d: -f1
)"

echo "Start line: ${START_LINE:-NOT_FOUND}"
echo "End line: ${END_LINE:-NOT_FOUND}"

if [ -z "${START_LINE}" ]; then
    echo
    echo "Available function definitions:"
    grep -n -E \
    '^[[:space:]]*def[[:space:]]+' \
    "${DRIVER}" \
    || true

    exit 21
fi

if [ -z "${END_LINE}" ]; then
    echo
    echo "create_validator() was not found."
    echo "Printing 450 lines from enrichment start."

    END_LINE="$((START_LINE + 450))"
fi

if [ "${END_LINE}" -le "${START_LINE}" ]; then
    echo "ERROR: Invalid source range."
    exit 22
fi

echo
echo "============================================================"
echo "EXACT create_enrichment_program() SOURCE"
echo "============================================================"

sed -n "${START_LINE},$((END_LINE - 1))p" \
    "${DRIVER}"

echo
echo "============================================================"
echo "RELEVANT MARKERS"
echo "============================================================"

grep -n -E \
'create_enrichment_program|SparkSession|SedonaContext|ST_AsBinary|ST_PolygonFromEnvelope|geometry_wkb' \
"${DRIVER}" \
|| true

echo
echo "============================================================"
echo "DIAGNOSTIC RESULT"
echo "============================================================"

echo "Driver modified: NO"
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "CELL 7D-DIAG: SUCCESS"
'''


result = run_remote(
    inspection_command,
    timeout=120,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7D-DIAG NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 7D-DIAG: SUCCESS")

ENRICHMENT FACTORY LOCATION
Start line: 723
End line: 1171

EXACT create_enrichment_program() SOURCE
def create_enrichment_program(
    epoch,
    epoch_dir,
):
    output_path = (
        epoch_dir
        / f"enrich_{epoch}.py"
    )

    program = f'''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


EPOCH = {epoch}
EXPECTED_ROWS = 3658
EXPECTED_COMPLETE = 3480
EXPECTED_PARTIAL = 178

BASE = (
    "hdfs://localhost:9000"
    "/data/ghsl/curated/pilots/bengaluru"
)

POPULATION_PATH = (
    f"{{BASE}}/product=population/"
    f"epoch={{EPOCH}}/grid_1km"
)

SMOD_PATH = (
    f"{{BASE}}/product=smod/"
    f"epoch={{EPOCH}}/grid_1km"
)

BUILT_PATH = (
    f"{{BASE}}/product=built_surface/"
    f"epoch={{EPOCH}}/grid_1km"
)

OUTPUT_PATH = (
    f"{{BASE}}/enriched_grid_year/"
    f"epoch={{EPOCH}}"
)


spark = (
    SparkSession.builder
    .appName(
        f"GHSL-Bengaluru-{{EPOCH}}-Enrichment"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN

In [ ]:
# ============================================================
# CELL 7E: PATCH EXACT ENRICHMENT FACTORY AND RESUME BATCH
#
# Exact source structure was confirmed by diagnostics:
#
#   from pyspark.sql import SparkSession
#   from pyspark.sql import functions as F
#
#   spark = (
#       SparkSession.builder
#       ...
#   )
#
# Fix:
#   - Replace the generated enrichment import with SedonaContext
#   - Replace SparkSession.builder with SedonaContext.builder
#   - Register Sedona using SedonaContext.create(config)
#   - Delete the failed generated enrichment scripts
#   - Resume the complete intermediate-epoch batch
#
# Existing validated outputs will be skipped.
# ============================================================

import base64


REMOTE_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_PATCHER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "patch_exact_enrichment_factory.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)


# ============================================================
# 1. REMOTE PATCH PROGRAM
#
# The string begins at column zero intentionally.
# ============================================================

patch_program = r'''from pathlib import Path
import shutil


DRIVER = Path(
    "/home/ubuntu/ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

BACKUP = Path(
    "/home/ubuntu/ghsl_notebook05_batch/"
    "run_intermediate_epochs.py.before_exact_enrichment_fix"
)


if not DRIVER.is_file():
    raise FileNotFoundError(
        f"Batch driver not found: {DRIVER}"
    )


shutil.copy2(
    DRIVER,
    BACKUP,
)


program = DRIVER.read_text(
    encoding="utf-8"
)


# ============================================================
# EXACT IMPORT REPLACEMENT
#
# The generated program text starts at column zero inside:
#
#     program = f'''
#
# Therefore, these markers must also start at column zero.
# ============================================================

old_import = (
    "from pyspark.sql import SparkSession\n"
    "from pyspark.sql import functions as F\n"
    "\n"
    "\n"
    "EPOCH = {epoch}\n"
    "EXPECTED_ROWS = 3658\n"
)

new_import = (
    "from sedona.spark import SedonaContext\n"
    "from pyspark.sql import functions as F\n"
    "\n"
    "\n"
    "EPOCH = {epoch}\n"
    "EXPECTED_ROWS = 3658\n"
)


if new_import in program:
    print(
        "Exact enrichment Sedona import "
        "already present."
    )

else:
    import_count = program.count(
        old_import
    )

    if import_count != 1:
        raise RuntimeError(
            "Expected exactly one column-zero enrichment "
            f"import block, found {import_count}."
        )

    program = program.replace(
        old_import,
        new_import,
        1,
    )

    print(
        "Exact enrichment Sedona import: ADDED"
    )


# ============================================================
# EXACT SESSION REPLACEMENT
# ============================================================

old_session = (
    "spark = (\n"
    "    SparkSession.builder\n"
    "    .appName(\n"
    '        f"GHSL-Bengaluru-{EPOCH}-Enrichment"\n'
    "    )\n"
    "    .getOrCreate()\n"
    ")\n"
    "\n"
    'spark.sparkContext.setLogLevel("WARN")\n'
)

new_session = (
    "config = (\n"
    "    SedonaContext.builder()\n"
    "    .appName(\n"
    '        f"GHSL-Bengaluru-{EPOCH}-Enrichment"\n'
    "    )\n"
    "    .getOrCreate()\n"
    ")\n"
    "\n"
    "spark = SedonaContext.create(config)\n"
    'spark.sparkContext.setLogLevel("WARN")\n'
)


if new_session in program:
    print(
        "Exact enrichment SedonaContext "
        "already present."
    )

else:
    session_count = program.count(
        old_session
    )

    if session_count != 1:
        raise RuntimeError(
            "Expected exactly one column-zero enrichment "
            f"session block, found {session_count}."
        )

    program = program.replace(
        old_session,
        new_session,
        1,
    )

    print(
        "Exact enrichment SedonaContext: ADDED"
    )


# ============================================================
# VERIFY ALL PREVIOUS AND CURRENT FIXES
# ============================================================

required_fragments = [
    "import textwrap",
    '"Built-surface output path"',
    "from sedona.spark import SedonaContext",
    "SedonaContext.builder()",
    "spark = SedonaContext.create(config)",
    "ST_AsBinary(",
    "ST_PolygonFromEnvelope(",
]


missing_fragments = [
    fragment
    for fragment in required_fragments
    if fragment not in program
]


if missing_fragments:
    raise RuntimeError(
        "Patched driver is missing required content:\n"
        + "\n".join(missing_fragments)
    )


if (
    '"Built grid-reference/output paths"'
    in program
):
    raise RuntimeError(
        "Old built-surface path-count marker remains."
    )


if old_import in program:
    raise RuntimeError(
        "Old enrichment SparkSession import remains."
    )


if old_session in program:
    raise RuntimeError(
        "Old enrichment SparkSession block remains."
    )


# ============================================================
# COMPILE AND SAVE
# ============================================================

compile(
    program,
    str(DRIVER),
    "exec",
)


DRIVER.write_text(
    program,
    encoding="utf-8",
)


saved_program = DRIVER.read_text(
    encoding="utf-8"
)


compile(
    saved_program,
    str(DRIVER),
    "exec",
)


print()
print("=" * 80)
print("EXACT ENRICHMENT PATCH RESULT")
print("=" * 80)
print("Batch driver:", DRIVER)
print("Backup:", BACKUP)
print("Built-surface path fix: PRESERVED")
print("Column-zero Sedona import: PRESENT")
print("SedonaContext builder: PRESENT")
print("SedonaContext registration: PRESENT")
print("Sedona geometry expressions: PRESERVED")
print("Batch-driver syntax: PASS")
print("Spark jobs launched: NO")
print("HDFS modified: NO")
print("PATCH: SUCCESS")
'''


encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")


# ============================================================
# 2. APPLY PATCH
# ============================================================

patch_command = (
    "set -e; "

    f"echo '{encoded_patch}' "
    f"| base64 -d > '{REMOTE_PATCHER}'; "

    f"python3 -m py_compile "
    f"'{REMOTE_PATCHER}'; "

    f"python3 '{REMOTE_PATCHER}'; "

    f"python3 -m py_compile "
    f"'{REMOTE_DRIVER}'; "

    "echo; "
    "echo 'Patched enrichment factory:'; "

    f"sed -n '723,790p' "
    f"'{REMOTE_DRIVER}'; "

    "echo; "
    "echo 'Patched driver SHA-256:'; "

    f"sha256sum '{REMOTE_DRIVER}'"
)


patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 3. DELETE ONLY FAILED GENERATED ENRICHMENT ARTIFACTS
#
# No valid HDFS enriched 1980 output exists yet.
# Product outputs remain untouched.
# ============================================================

cleanup_command = r'''
set -e

EPOCH_DIR="/home/ubuntu/ghsl_notebook05_programs/epoch_1980"

rm -f "${EPOCH_DIR}/enrich_1980.py"
rm -f "${EPOCH_DIR}/validate_enriched_1980.py"

echo "Removed failed generated files:"
echo "${EPOCH_DIR}/enrich_1980.py"
echo "${EPOCH_DIR}/validate_enriched_1980.py"

echo
echo "Existing 1980 HDFS products:"

for path in \
"/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1980/grid_1km" \
"/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1980/grid_1km" \
"/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=1980/grid_1km"
do
    if hdfs dfs -test -e "${path}"; then
        echo "PASS: ${path}"
    else
        echo "ERROR: Missing ${path}"
        exit 20
    fi
done

echo
echo "Valid product outputs preserved: PASS"
'''


cleanup_result = run_remote(
    cleanup_command,
    timeout=120,
    show_output=True,
)


# ============================================================
# 4. PRE-FLIGHT
# ============================================================

preflight_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"

echo "============================================================"
echo "BATCH RESUME PRE-FLIGHT"
echo "============================================================"

python3 -m py_compile "${DRIVER}"

grep -F \
    'from sedona.spark import SedonaContext' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: SedonaContext import is missing."
        exit 30
    }

grep -F \
    'spark = SedonaContext.create(config)' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: SedonaContext registration is missing."
        exit 31
    }

grep -F \
    '"Built-surface output path",' \
    "${DRIVER}" \
    >/dev/null \
    || {
        echo "ERROR: Built-surface count fix is missing."
        exit 32
    }

for daemon in NameNode DataNode SecondaryNameNode; do
    jps \
        | awk '{print $2}' \
        | grep -qx "${daemon}" \
        || {
            echo "ERROR: ${daemon} is not running."
            exit 33
        }
done

SAFE_MODE_STATUS="$(
    hdfs dfsadmin -safemode get 2>&1
)"

echo "${SAFE_MODE_STATUS}"

echo "${SAFE_MODE_STATUS}" \
    | grep -qi "Safe mode is OFF" \
    || {
        echo "ERROR: HDFS safe mode is enabled."
        exit 34
    }

RUNNING_COUNT="$(
    ps -ef \
    | grep -E \
    'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${RUNNING_COUNT}" -ne 0 ]; then
    echo "ERROR: Another batch or Spark job is active."

    ps -ef \
        | grep -E \
        'run_intermediate_epochs|SparkSubmit|spark-submit' \
        | grep -v grep

    exit 35
fi

echo
echo "Memory:"
free -h

echo
echo "Disk:"
df -h / /tmp

echo
echo "Batch resume pre-flight: PASS"
'''


preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 5. RESUME COMPLETE BATCH
#
# Expected:
#   population 1980    validate and skip
#   SMOD 1980          validate and skip
#   built surface 1980 validate and skip
#   enrichment 1980    regenerate and run
# ============================================================

resume_command = r'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
LOG="/home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log"

python3 "${DRIVER}" \
    2>&1 \
    | tee "${LOG}"

status=${PIPESTATUS[0]}

echo "BATCH_EXIT_CODE=${status}"

exit "${status}"
'''


print()
print("=" * 80)
print("RESUMING INTERMEDIATE-EPOCH BATCH")
print("=" * 80)
print("1980 population: validate and skip")
print("1980 SMOD: validate and skip")
print("1980 built surface: validate and skip")
print("Next new stage: enriched 1980")
print("Then continue: 1985 through 2015")
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)
print()


batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7E NOTEBOOK RESULT")
print("=" * 80)

print(
    "Patch return code:",
    patch_result.returncode,
)

print(
    "Cleanup return code:",
    cleanup_result.returncode,
)

print(
    "Pre-flight return code:",
    preflight_result.returncode,
)

print(
    "Batch return code:",
    batch_result.returncode,
)

print(
    "Remote log:",
    REMOTE_LOG,
)

print(
    "Remote manifest:",
    REMOTE_MANIFEST,
)

print("CELL 7E: SUCCESS")

SyntaxError: unterminated string literal (detected at line 449) (2863816244.py, line 449)

In [ ]:
# ============================================================
# CELL 7F: PATCH ONLY create_enrichment_program() AND RESUME
#
# Root cause:
#   The batch driver contains two SparkSession imports:
#     1. create_enrichment_program()
#     2. create_validator()
#
# Fix:
#   Patch only the source range between:
#     def create_enrichment_program(...)
#     def create_validator(...)
#
# Existing population, SMOD, and built-surface outputs remain
# untouched. The batch will validate and skip them.
# ============================================================

import base64


REMOTE_DRIVER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "run_intermediate_epochs.py"
)

REMOTE_PATCHER = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "patch_enrichment_function_only.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "intermediate_epochs_batch.log"
)

REMOTE_MANIFEST = (
    "/home/ubuntu/"
    "ghsl_notebook05_batch/"
    "batch_manifest.tsv"
)


# ============================================================
# 1. BUILD REMOTE PATCHER
# ============================================================

patch_lines = [
    "from pathlib import Path",
    "import shutil",
    "",
    'driver = Path("/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py")',
    'backup = Path("/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py.before_function_scoped_enrichment_fix")',
    "",
    "if not driver.is_file():",
    '    raise FileNotFoundError(f"Batch driver not found: {driver}")',
    "",
    "shutil.copy2(driver, backup)",
    "",
    'program = driver.read_text(encoding="utf-8")',
    "",
    "# Locate the exact enrichment-factory source range.",
    'start_marker = "def create_enrichment_program("',
    'end_marker = "def create_validator("',
    "",
    "start_count = program.count(start_marker)",
    "end_count = program.count(end_marker)",
    "",
    "if start_count != 1:",
    '    raise RuntimeError(f"Expected one enrichment factory, found {start_count}.")',
    "",
    "if end_count != 1:",
    '    raise RuntimeError(f"Expected one validator factory, found {end_count}.")',
    "",
    "start_index = program.index(start_marker)",
    "end_index = program.index(end_marker)",
    "",
    "if end_index <= start_index:",
    '    raise RuntimeError("Invalid enrichment-factory source range.")',
    "",
    "prefix = program[:start_index]",
    "enrichment_section = program[start_index:end_index]",
    "suffix = program[end_index:]",
    "",
    "# Patch only the import inside create_enrichment_program().",
    'old_import = "from pyspark.sql import SparkSession\\nfrom pyspark.sql import functions as F\\n"',
    'new_import = "from sedona.spark import SedonaContext\\nfrom pyspark.sql import functions as F\\n"',
    "",
    "old_import_count = enrichment_section.count(old_import)",
    "new_import_count = enrichment_section.count(new_import)",
    "",
    "if old_import_count == 1:",
    "    enrichment_section = enrichment_section.replace(",
    "        old_import,",
    "        new_import,",
    "        1,",
    "    )",
    '    print("Enrichment Sedona import: ADDED")',
    "elif old_import_count == 0 and new_import_count == 1:",
    '    print("Enrichment Sedona import: ALREADY PRESENT")',
    "else:",
    "    raise RuntimeError(",
    '        "Unexpected enrichment import counts: "',
    '        f"old={old_import_count}, new={new_import_count}."',
    "    )",
    "",
    "# Patch only the session inside create_enrichment_program().",
    'old_session = (',
    '    "spark = (\\n"',
    '    "    SparkSession.builder\\n"',
    '    "    .appName(\\n"',
    '    "        f\\"GHSL-Bengaluru-{EPOCH}-Enrichment\\"\\n"',
    '    "    )\\n"',
    '    "    .getOrCreate()\\n"',
    '    ")\\n"',
    '    "\\n"',
    '    "spark.sparkContext.setLogLevel(\\"WARN\\")\\n"',
    ')',
    "",
    'new_session = (',
    '    "config = (\\n"',
    '    "    SedonaContext.builder()\\n"',
    '    "    .appName(\\n"',
    '    "        f\\"GHSL-Bengaluru-{EPOCH}-Enrichment\\"\\n"',
    '    "    )\\n"',
    '    "    .getOrCreate()\\n"',
    '    ")\\n"',
    '    "\\n"',
    '    "spark = SedonaContext.create(config)\\n"',
    '    "spark.sparkContext.setLogLevel(\\"WARN\\")\\n"',
    ')',
    "",
    "old_session_count = enrichment_section.count(old_session)",
    "new_session_count = enrichment_section.count(new_session)",
    "",
    "if old_session_count == 1:",
    "    enrichment_section = enrichment_section.replace(",
    "        old_session,",
    "        new_session,",
    "        1,",
    "    )",
    '    print("Enrichment SedonaContext: ADDED")',
    "elif old_session_count == 0 and new_session_count == 1:",
    '    print("Enrichment SedonaContext: ALREADY PRESENT")',
    "else:",
    "    raise RuntimeError(",
    '        "Unexpected enrichment session counts: "',
    '        f"old={old_session_count}, new={new_session_count}."',
    "    )",
    "",
    "# Validate only the enrichment section.",
    "required_enrichment = [",
    '    "from sedona.spark import SedonaContext",',
    '    "SedonaContext.builder()",',
    '    "spark = SedonaContext.create(config)",',
    '    "ST_AsBinary(",',
    '    "ST_PolygonFromEnvelope(",',
    "]",
    "",
    "missing = [",
    "    item",
    "    for item in required_enrichment",
    "    if item not in enrichment_section",
    "]",
    "",
    "if missing:",
    '    raise RuntimeError("Patched enrichment section is missing:\\n" + "\\n".join(missing))',
    "",
    'if "from pyspark.sql import SparkSession" in enrichment_section:',
    '    raise RuntimeError("Old SparkSession import remains in enrichment section.")',
    "",
    'if "SparkSession.builder" in enrichment_section:',
    '    raise RuntimeError("Old SparkSession builder remains in enrichment section.")',
    "",
    "# Reassemble the full batch driver.",
    "patched_program = prefix + enrichment_section + suffix",
    "",
    "# Preserve previous batch-driver fixes.",
    'required_driver_markers = [',
    '    "import textwrap",',
    '    "\\"Built-surface output path\\"",',
    ']',
    "",
    "missing_driver_markers = [",
    "    item",
    "    for item in required_driver_markers",
    "    if item not in patched_program",
    "]",
    "",
    "if missing_driver_markers:",
    '    raise RuntimeError("Previous driver fixes are missing:\\n" + "\\n".join(missing_driver_markers))',
    "",
    'if "\\"Built grid-reference/output paths\\"" in patched_program:',
    '    raise RuntimeError("Old built-surface path marker remains.")',
    "",
    "# The validator factory may retain plain SparkSession.",
    "validator_section = patched_program[end_index:]",
    "",
    'if "from pyspark.sql import SparkSession" not in validator_section:',
    '    raise RuntimeError("Validator SparkSession import was unexpectedly modified.")',
    "",
    'compile(patched_program, str(driver), "exec")',
    'driver.write_text(patched_program, encoding="utf-8")',
    "",
    'saved = driver.read_text(encoding="utf-8")',
    'compile(saved, str(driver), "exec")',
    "",
    'print("Function-scoped enrichment patch: PASS")',
    'print("Validator factory preserved: PASS")',
    'print("Built-surface path fix preserved: PASS")',
    'print("Batch-driver syntax: PASS")',
    'print("Backup:", backup)',
]

patch_program = "\n".join(patch_lines) + "\n"

encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")


# ============================================================
# 2. APPLY PATCH
# ============================================================

patch_command = (
    "set -e; "
    f"echo '{encoded_patch}' | base64 -d > '{REMOTE_PATCHER}'; "
    f"python3 -m py_compile '{REMOTE_PATCHER}'; "
    f"python3 '{REMOTE_PATCHER}'; "
    f"python3 -m py_compile '{REMOTE_DRIVER}'; "
    "echo; "
    "echo 'Patched enrichment function header:'; "
    f"sed -n '723,790p' '{REMOTE_DRIVER}'; "
    "echo; "
    "echo 'Validator import remains:'; "
    f"sed -n '1238,1265p' '{REMOTE_DRIVER}'; "
    "echo; "
    "echo 'Driver checksum:'; "
    f"sha256sum '{REMOTE_DRIVER}'"
)

patch_result = run_remote(
    patch_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 3. REMOVE FAILED ENRICHMENT SCRIPTS ONLY
# ============================================================

cleanup_command = r'''
set -e

EPOCH_DIR="/home/ubuntu/ghsl_notebook05_programs/epoch_1980"

rm -f "${EPOCH_DIR}/enrich_1980.py"
rm -f "${EPOCH_DIR}/validate_enriched_1980.py"

echo "Failed enrichment scripts removed."

echo
echo "Validated 1980 HDFS products:"

for path in \
"/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1980/grid_1km" \
"/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1980/grid_1km" \
"/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=1980/grid_1km"
do
    if hdfs dfs -test -e "${path}"; then
        echo "PASS: ${path}"
    else
        echo "ERROR: Missing ${path}"
        exit 20
    fi
done
'''

cleanup_result = run_remote(
    cleanup_command,
    timeout=120,
    show_output=True,
)


# ============================================================
# 4. PRE-FLIGHT
# ============================================================

preflight_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"

python3 -m py_compile "${DRIVER}"

START_LINE="$(
    grep -n '^def create_enrichment_program' "${DRIVER}" \
    | head -n 1 \
    | cut -d: -f1
)"

END_LINE="$(
    grep -n '^def create_validator' "${DRIVER}" \
    | head -n 1 \
    | cut -d: -f1
)"

sed -n "${START_LINE},$((END_LINE - 1))p" "${DRIVER}" \
| grep -F 'from sedona.spark import SedonaContext' \
>/dev/null

sed -n "${START_LINE},$((END_LINE - 1))p" "${DRIVER}" \
| grep -F 'spark = SedonaContext.create(config)' \
>/dev/null

grep -F '"Built-surface output path"' "${DRIVER}" \
>/dev/null

for daemon in NameNode DataNode SecondaryNameNode; do
    jps | awk '{print $2}' | grep -qx "${daemon}" || {
        echo "ERROR: ${daemon} is not running."
        exit 30
    }
done

SAFE_MODE_STATUS="$(hdfs dfsadmin -safemode get 2>&1)"
echo "${SAFE_MODE_STATUS}"

echo "${SAFE_MODE_STATUS}" \
| grep -qi "Safe mode is OFF" || {
    echo "ERROR: HDFS safe mode enabled."
    exit 31
}

RUNNING_COUNT="$(
    ps -ef \
    | grep -E 'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep \
    | wc -l \
    | tr -d ' '
)"

if [ "${RUNNING_COUNT}" -ne 0 ]; then
    echo "ERROR: Another batch or Spark process is active."
    ps -ef \
    | grep -E 'run_intermediate_epochs|SparkSubmit|spark-submit' \
    | grep -v grep
    exit 32
fi

echo
free -h

echo
df -h / /tmp

echo
echo "Batch resume pre-flight: PASS"
'''

preflight_result = run_remote(
    preflight_command,
    timeout=180,
    show_output=True,
)


# ============================================================
# 5. RESUME BATCH
# ============================================================

resume_command = r'''
set -o pipefail

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

DRIVER="/home/ubuntu/ghsl_notebook05_batch/run_intermediate_epochs.py"
LOG="/home/ubuntu/ghsl_notebook05_batch/intermediate_epochs_batch.log"

python3 "${DRIVER}" 2>&1 | tee "${LOG}"

status=${PIPESTATUS[0]}

echo "BATCH_EXIT_CODE=${status}"

exit "${status}"
'''

print()
print("=" * 80)
print("RESUMING INTERMEDIATE-EPOCH BATCH")
print("=" * 80)
print("1980 population: validate and skip")
print("1980 SMOD: validate and skip")
print("1980 built surface: validate and skip")
print("Next new stage: enriched 1980")
print("Then continue: 1985 through 2015")
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)
print()

batch_result = run_remote(
    resume_command,
    timeout=28800,
    show_output=True,
)


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 80)
print("CELL 7F NOTEBOOK RESULT")
print("=" * 80)
print("Patch return code:", patch_result.returncode)
print("Cleanup return code:", cleanup_result.returncode)
print("Pre-flight return code:", preflight_result.returncode)
print("Batch return code:", batch_result.returncode)
print("Remote log:", REMOTE_LOG)
print("Remote manifest:", REMOTE_MANIFEST)
print("CELL 7F: SUCCESS")

Enrichment Sedona import: ADDED

REMOTE STDERR:
Traceback (most recent call last):
  File "/home/ubuntu/ghsl_notebook05_batch/patch_enrichment_function_only.py", line 98, in <module>
    raise RuntimeError(
    ...<2 lines>...
    )
RuntimeError: Unexpected enrichment session counts: old=0, new=0.



RuntimeError: Remote command failed with return code 1.